# Brooklyn ambulance siting with busy units and centroid uncertainty

Choose five new standby posts from the existing 20 candidates. For each possible centroid position, the fastest post responds with probability $1-p$. If it is busy, the second-fastest distinct post responds and is assumed free. The 66 listed firehouses are modeled as staffed ambulance posts. The default $p=0.25$ is an assumption.

Each ZIP has one uncertain centroid and retains its full call weight. The uncertainty region is the intersection of the ZIP polygon and a **500 m Euclidean ball** around the nominal centroid. The notebook approximates the region with a **250 m grid inside the ball**, adding four diagonal boundary points. Keep the same five sites across all points. For each ZIP, take its highest expected drive time over those points, then minimize the call-weighted average across ZIPs. The result protects against the evaluated grid, and smaller grid spacing tests the sensitivity to that approximation.

Run from the project root with a Julia kernel. Use the original Julia packages plus JSON3 (`import Pkg; Pkg.add("JSON3")`). The notebook reads the same two nominal routing matrices and `data/modzcta.geojson`. Default grid routes are included for the current inputs. Changing the geometry, sites, radius, or grid spacing can trigger a new Valhalla routing run. Set the radius and busy probability to zero to recover the original model.

In [12]:
using CSV, DataFrames, JuMP, Gurobi, Printf
using Downloads, JSON3, SHA

## 1. Configuration

In [13]:
DATA_DIR   = "data"
OUTPUT_DIR = "results"

DEMAND_FILE        = joinpath(DATA_DIR, "demand.csv")                     # area_id, lat, lon, calls
CANDIDATE_FILE     = joinpath(DATA_DIR, "candidate_sites.csv")            # site_id, lat, lon, source (20 pins)
TRAVEL_TIME_FILE   = joinpath(DATA_DIR, "travel_times_routed.csv")        # site_id, area_id, ..., minutes
EXISTING_FILE      = joinpath(DATA_DIR, "FDNY_Firehouse_Listing_20261115.csv")  # all NYC firehouses
BOROUGH            = "Brooklyn"
EXISTING_TIME_FILE = joinpath(DATA_DIR, "travel_times_existing_routed.csv")               # site_id, area_id, ..., minutes

P          = 5               # number of new standby sites to open
WEIGHT_COL = :calls          # w_i: call count per ZIP area
TIME_COL   = :minutes        # time column in both travel-time files

TIME_LIMIT_SEC = 300
ZIP_FILE = joinpath(DATA_DIR, "modzcta.geojson")
RADIUS_M = 500.0
GRID_SPACING_M = 250.0
BUSY_PROB = 0.25
MATRIX_URL = "https://valhalla1.openstreetmap.de/sources_to_targets"
MAX_SNAP_M = 250
@assert RADIUS_M >= 0 && GRID_SPACING_M > 0 && 0 <= BUSY_PROB <= 1

## 2. Load ZIP demand and sites

In [14]:
demand = CSV.read(DEMAND_FILE, DataFrame)
demand.area_id = string.(demand.area_id)
dropmissing!(demand, [:lat, :lon, WEIGHT_COL])
filter!(row -> row[WEIGHT_COL] > 0, demand)

I_areas = demand.area_id
w = Float64.(demand[!, WEIGHT_COL])
println("Demand areas |I| = $(length(I_areas)),  total $(WEIGHT_COL) = $(sum(w))")
first(demand, 5)

Demand areas |I| = 37,  total calls = 202491.0


Row,area_id,lat,lon,calls
,String,Float64,Float64,Int64
1,11201,40.6946,-73.9903,11460
2,11203,40.6487,-73.9339,7205
3,11204,40.621,-73.9859,2877
4,11205,40.6965,-73.9664,3461
5,11206,40.7024,-73.9419,8664


In [15]:
candidates = CSV.read(CANDIDATE_FILE, DataFrame)
candidates.site_id = string.(candidates.site_id)
J_sites = candidates.site_id

# Same construction as TravelTimes.ipynb, so site_ids line up with EXISTING_TIME_FILE.
firehouses = CSV.read(EXISTING_FILE, DataFrame; normalizenames=true)
firehouses = dropmissing(filter(r -> r.Borough == BOROUGH, firehouses), [:Latitude, :Longitude])
existing = DataFrame(
    site_id = [@sprintf("FH_%03d", k) for k in 1:nrow(firehouses)],
    name    = firehouses.FacilityName,
    address = firehouses.FacilityAddress,
    lat     = Float64.(firehouses.Latitude),
    lon     = Float64.(firehouses.Longitude),
)
K_sites = existing.site_id

isempty(intersect(J_sites, K_sites)) || error("Candidate and existing site_ids overlap")
println("Candidate sites |J| = $(length(J_sites)),  existing firehouses |K| = $(length(K_sites))")
first(candidates, 5)

Candidate sites |J| = 20,  existing firehouses |K| = 66


Row,site_id,lat,lon,source
,String,Float64,Float64,String15
1,1,40.6382,-73.9125,alternate
2,2,40.6611,-73.8861,alternate
3,3,40.6547,-73.9392,original
4,4,40.6217,-73.9494,original
5,5,40.5901,-73.941,alternate


## 3. Load nominal drive times and route centroid offsets

In [16]:
function load_travel_times(path, sites, I_areas; col=TIME_COL)
    tt = CSV.read(path, DataFrame)
    tt.site_id = string.(tt.site_id)
    tt.area_id = string.(tt.area_id)
    jidx = Dict(s => k for (k, s) in enumerate(sites))
    iidx = Dict(a => k for (k, a) in enumerate(I_areas))
    t = fill(NaN, length(sites), length(I_areas))
    for r in eachrow(tt)
        haskey(jidx, r.site_id) && haskey(iidx, r.area_id) || continue
        t[jidx[r.site_id], iidx[r.area_id]] = r[col]
    end
    n_missing = count(isnan, t)
    n_missing == 0 || error("$(n_missing) (site, area) pairs missing from $(path): check that the site file and travel-time file use the same site_ids")
    return t
end

t       = load_travel_times(TRAVEL_TIME_FILE, J_sites, I_areas)
t_exist = load_travel_times(EXISTING_TIME_FILE, K_sites, I_areas)


nominal_times = vcat(t_exist, t)
site_ids = vcat(K_sites, J_sites)
n_existing = length(K_sites)
sites = vcat(select(existing, :site_id, :lat, :lon),
             select(candidates, :site_id, :lat, :lon))

Row,site_id,lat,lon
,String,Float64,Float64
1,FH_001,40.6955,-73.9828
2,FH_002,40.6871,-73.9828
3,FH_003,40.7,-73.9923
4,FH_004,40.6931,-73.9971
5,FH_005,40.6721,-73.9997
6,FH_006,40.6804,-74.0064
7,FH_007,40.6518,-74.0053
8,FH_008,40.6746,-73.9767
9,FH_009,40.6718,-73.9872


In [17]:

function bundled_offset_times(sites, offsets)
    # Use bundled routes only if the actual site and target coordinates match.
    bundled = JSON3.read(raw"""{"sites":[{"id":"FH_001","lat":40.69545,"lon":-73.982843},{"id":"FH_002","lat":40.68707,"lon":-73.982768},{"id":"FH_003","lat":40.700029,"lon":-73.992273},{"id":"FH_004","lat":40.693072,"lon":-73.99708},{"id":"FH_005","lat":40.672118,"lon":-73.999666},{"id":"FH_006","lat":40.680386,"lon":-74.006438},{"id":"FH_007","lat":40.651814,"lon":-74.005327},{"id":"FH_008","lat":40.674649,"lon":-73.97667},{"id":"FH_009","lat":40.671782,"lon":-73.987239},{"id":"FH_010","lat":40.66495,"lon":-73.981452},{"id":"FH_011","lat":40.681192,"lon":-73.973851},{"id":"FH_012","lat":40.673473,"lon":-73.961641},{"id":"FH_013","lat":40.670443,"lon":-73.933495},{"id":"FH_014","lat":40.674781,"lon":-73.921728},{"id":"FH_015","lat":40.682712,"lon":-73.911396},{"id":"FH_016","lat":40.692759,"lon":-73.937993},{"id":"FH_017","lat":40.689591,"lon":-73.924433},{"id":"FH_018","lat":40.68441,"lon":-73.935771},{"id":"FH_019","lat":40.698066,"lon":-73.947507},{"id":"FH_020","lat":40.685175,"lon":-73.951098},{"id":"FH_021","lat":40.695608,"lon":-73.956614},{"id":"FH_022","lat":40.692799,"lon":-73.973006},{"id":"FH_023","lat":40.701379,"lon":-73.962289},{"id":"FH_024","lat":40.713175,"lon":-73.961113},{"id":"FH_025","lat":40.706236,"lon":-73.95062},{"id":"FH_026","lat":40.730747,"lon":-73.951144},{"id":"FH_027","lat":40.718619,"lon":-73.949024},{"id":"FH_028","lat":40.715301,"lon":-73.928159},{"id":"FH_029","lat":40.705312,"lon":-73.931932},{"id":"FH_030","lat":40.698312,"lon":-73.92659},{"id":"FH_031","lat":40.70363,"lon":-73.91602},{"id":"FH_032","lat":40.696397,"lon":-73.914659},{"id":"FH_033","lat":40.688597,"lon":-73.90814},{"id":"FH_034","lat":40.67057,"lon":-73.907891},{"id":"FH_035","lat":40.660966,"lon":-73.917952},{"id":"FH_036","lat":40.675471,"lon":-73.892695},{"id":"FH_037","lat":40.665123,"lon":-73.895447},{"id":"FH_038","lat":40.677748,"lon":-73.872797},{"id":"FH_039","lat":40.669446,"lon":-73.865383},{"id":"FH_040","lat":40.650355,"lon":-73.928888},{"id":"FH_041","lat":40.660257,"lon":-73.953293},{"id":"FH_042","lat":40.648746,"lon":-73.949987},{"id":"FH_043","lat":40.640024,"lon":-73.966669},{"id":"FH_044","lat":40.636444,"lon":-73.95064},{"id":"FH_045","lat":40.651175,"lon":-73.975574},{"id":"FH_046","lat":40.640182,"lon":-73.990752},{"id":"FH_047","lat":40.62802,"lon":-73.997775},{"id":"FH_048","lat":40.638157,"lon":-74.025161},{"id":"FH_049","lat":40.617389,"lon":-74.02952},{"id":"FH_050","lat":40.619955,"lon":-74.012537},{"id":"FH_051","lat":40.606037,"lon":-74.003495},{"id":"FH_052","lat":40.627757,"lon":-73.97558},{"id":"FH_053","lat":40.612127,"lon":-73.979153},{"id":"FH_054","lat":40.609365,"lon":-73.959131},{"id":"FH_055","lat":40.598957,"lon":-73.988518},{"id":"FH_056","lat":40.57776,"lon":-73.993011},{"id":"FH_057","lat":40.598496,"lon":-73.962008},{"id":"FH_058","lat":40.576735,"lon":-73.97616},{"id":"FH_059","lat":40.646611,"lon":-73.903569},{"id":"FH_060","lat":40.620133,"lon":-73.915859},{"id":"FH_061","lat":40.615256,"lon":-73.928169},{"id":"FH_062","lat":40.584275,"lon":-73.959413},{"id":"FH_063","lat":40.602267,"lon":-73.935188},{"id":"FH_064","lat":40.705184,"lon":-73.971631},{"id":"FH_065","lat":40.645405,"lon":-74.01318},{"id":"FH_066","lat":40.671045,"lon":-73.918868},{"id":"1","lat":40.638178,"lon":-73.91247},{"id":"2","lat":40.661051,"lon":-73.886086},{"id":"3","lat":40.654714,"lon":-73.939198},{"id":"4","lat":40.621677,"lon":-73.949438},{"id":"5","lat":40.590072,"lon":-73.940958},{"id":"6","lat":40.686776,"lon":-73.962669},{"id":"7","lat":40.707172,"lon":-73.943111},{"id":"8","lat":40.63863,"lon":-74.007143},{"id":"9","lat":40.606271,"lon":-73.97752},{"id":"10","lat":40.667563,"lon":-73.945086},{"id":"11","lat":40.688006,"lon":-73.989392},{"id":"12","lat":40.667465,"lon":-73.975446},{"id":"13","lat":40.662464,"lon":-73.94903},{"id":"14","lat":40.627646,"lon":-73.983141},{"id":"15","lat":40.609788,"lon":-73.905646},{"id":"16","lat":40.737272,"lon":-73.955127},{"id":"17","lat":40.612217,"lon":-73.961702},{"id":"18","lat":40.575287,"lon":-74.001835},{"id":"19","lat":40.631925,"lon":-74.033008},{"id":"20","lat":40.621345,"lon":-73.999983}],"targets":[{"id":"11201_G1","lat":40.69464362095849,"lon":-73.99624233943771},{"id":"11201_G2","lat":40.6923953184269,"lon":-73.99327700511597},{"id":"11201_G3","lat":40.69464362095849,"lon":-73.99327700511597},{"id":"11201_G4","lat":40.69689192349008,"lon":-73.99327700511597},{"id":"11201_G5","lat":40.69014701589531,"lon":-73.99031167079423},{"id":"11201_G6","lat":40.6923953184269,"lon":-73.99031167079423},{"id":"11201_G7","lat":40.69689192349008,"lon":-73.99031167079423},{"id":"11201_G8","lat":40.69914022602166,"lon":-73.99031167079423},{"id":"11201_G9","lat":40.6923953184269,"lon":-73.98734633647248},{"id":"11201_G10","lat":40.69464362095849,"lon":-73.98734633647248},{"id":"11201_G11","lat":40.69689192349008,"lon":-73.98734633647248},{"id":"11201_G12","lat":40.69464362095849,"lon":-73.98438100215074},{"id":"11201_G13","lat":40.691464041025995,"lon":-73.994505286809},{"id":"11201_G14","lat":40.69782320089098,"lon":-73.994505286809},{"id":"11201_G15","lat":40.691464041025995,"lon":-73.98611805477945},{"id":"11201_G16","lat":40.69782320089098,"lon":-73.98611805477945},{"id":"11203_G1","lat":40.64869669712154,"lon":-73.93982413853112},{"id":"11203_G2","lat":40.646448394589946,"lon":-73.93686084684299},{"id":"11203_G3","lat":40.64869669712154,"lon":-73.93686084684299},{"id":"11203_G4","lat":40.65094499965313,"lon":-73.93686084684299},{"id":"11203_G5","lat":40.64420009205836,"lon":-73.93389755515484},{"id":"11203_G6","lat":40.646448394589946,"lon":-73.93389755515484},{"id":"11203_G7","lat":40.65094499965313,"lon":-73.93389755515484},{"id":"11203_G8","lat":40.65319330218471,"lon":-73.93389755515484},{"id":"11203_G9","lat":40.646448394589946,"lon":-73.9309342634667},{"id":"11203_G10","lat":40.64869669712154,"lon":-73.9309342634667},{"id":"11203_G11","lat":40.65094499965313,"lon":-73.9309342634667},{"id":"11203_G12","lat":40.64869669712154,"lon":-73.92797097177856},{"id":"11203_G13","lat":40.645517117189044,"lon":-73.93808828244948},{"id":"11203_G14","lat":40.65187627705403,"lon":-73.93808828244948},{"id":"11203_G15","lat":40.645517117189044,"lon":-73.9297068278602},{"id":"11203_G16","lat":40.65187627705403,"lon":-73.9297068278602},{"id":"11204_G1","lat":40.62101169311929,"lon":-73.99185660347449},{"id":"11204_G2","lat":40.6187633905877,"lon":-73.98889454028394},{"id":"11204_G3","lat":40.62101169311929,"lon":-73.98889454028394},{"id":"11204_G4","lat":40.62325999565088,"lon":-73.98889454028394},{"id":"11204_G5","lat":40.61651508805612,"lon":-73.9859324770934},{"id":"11204_G6","lat":40.6187633905877,"lon":-73.9859324770934},{"id":"11204_G7","lat":40.62325999565088,"lon":-73.9859324770934},{"id":"11204_G8","lat":40.62550829818247,"lon":-73.9859324770934},{"id":"11204_G9","lat":40.6187633905877,"lon":-73.98297041390286},{"id":"11204_G10","lat":40.62101169311929,"lon":-73.98297041390286},{"id":"11204_G11","lat":40.62325999565088,"lon":-73.98297041390286},{"id":"11204_G12","lat":40.62101169311929,"lon":-73.98000835071231},{"id":"11204_G13","lat":40.6178321131868,"lon":-73.99012146703008},{"id":"11204_G14","lat":40.624191273051785,"lon":-73.99012146703008},{"id":"11204_G15","lat":40.6178321131868,"lon":-73.98174348715672},{"id":"11204_G16","lat":40.624191273051785,"lon":-73.98174348715672},{"id":"11205_G1","lat":40.69651653634531,"lon":-73.97230002246329},{"id":"11205_G2","lat":40.694268233813716,"lon":-73.96933460477831},{"id":"11205_G3","lat":40.69651653634531,"lon":-73.96933460477831},{"id":"11205_G5","lat":40.69201993128213,"lon":-73.96636918709335},{"id":"11205_G6","lat":40.694268233813716,"lon":-73.96636918709335},{"id":"11205_G9","lat":40.694268233813716,"lon":-73.96340376940839},{"id":"11205_G10","lat":40.69651653634531,"lon":-73.96340376940839},{"id":"11205_G12","lat":40.69651653634531,"lon":-73.96043835172341},{"id":"11205_G13","lat":40.693336956412814,"lon":-73.97056292100153},{"id":"11205_G15","lat":40.693336956412814,"lon":-73.96217545318517},{"id":"11206_G1","lat":40.70241350046831,"lon":-73.94778498991045},{"id":"11206_G2","lat":40.70016519793672,"lon":-73.94481930970106},{"id":"11206_G3","lat":40.70241350046831,"lon":-73.94481930970106},{"id":"11206_G4","lat":40.7046618029999,"lon":-73.94481930970106},{"id":"11206_G5","lat":40.697916895405136,"lon":-73.94185362949166},{"id":"11206_G6","lat":40.70016519793672,"lon":-73.94185362949166},{"id":"11206_G7","lat":40.7046618029999,"lon":-73.94185362949166},{"id":"11206_G8","lat":40.706910105531485,"lon":-73.94185362949166},{"id":"11206_G9","lat":40.70016519793672,"lon":-73.93888794928226},{"id":"11206_G10","lat":40.70241350046831,"lon":-73.93888794928226},{"id":"11206_G11","lat":40.7046618029999,"lon":-73.93888794928226},{"id":"11206_G12","lat":40.70241350046831,"lon":-73.93592226907288},{"id":"11206_G13","lat":40.69923392053582,"lon":-73.94604773466544},{"id":"11206_G14","lat":40.7055930804008,"lon":-73.94604773466544},{"id":"11206_G15","lat":40.69923392053582,"lon":-73.93765952431788},{"id":"11206_G16","lat":40.7055930804008,"lon":-73.93765952431788},{"id":"11207_G1","lat":40.671648072694744,"lon":-73.89903225509882},{"id":"11207_G2","lat":40.66939977016315,"lon":-73.89606794366591},{"id":"11207_G3","lat":40.671648072694744,"lon":-73.89606794366591},{"id":"11207_G4","lat":40.673896375226334,"lon":-73.89606794366591},{"id":"11207_G5","lat":40.66715146763157,"lon":-73.89310363223302},{"id":"11207_G6","lat":40.66939977016315,"lon":-73.89310363223302},{"id":"11207_G7","lat":40.673896375226334,"lon":-73.89310363223302},{"id":"11207_G8","lat":40.67614467775792,"lon":-73.89310363223302},{"id":"11207_G9","lat":40.66939977016315,"lon":-73.89013932080013},{"id":"11207_G10","lat":40.671648072694744,"lon":-73.89013932080013},{"id":"11207_G11","lat":40.673896375226334,"lon":-73.89013932080013},{"id":"11207_G12","lat":40.671648072694744,"lon":-73.88717500936723},{"id":"11207_G13","lat":40.66846849276225,"lon":-73.89729580166451},{"id":"11207_G14","lat":40.674827652627236,"lon":-73.89729580166451},{"id":"11207_G15","lat":40.66846849276225,"lon":-73.88891146280153},{"id":"11207_G16","lat":40.674827652627236,"lon":-73.88891146280153},{"id":"11209_G1","lat":40.62086274030151,"lon":-74.03751135598588},{"id":"11209_G2","lat":40.61861443776992,"lon":-74.03454929940037},{"id":"11209_G3","lat":40.62086274030151,"lon":-74.03454929940037},{"id":"11209_G4","lat":40.6231110428331,"lon":-74.03454929940037},{"id":"11209_G5","lat":40.61636613523834,"lon":-74.03158724281485},{"id":"11209_G6","lat":40.61861443776992,"lon":-74.03158724281485},{"id":"11209_G7","lat":40.6231110428331,"lon":-74.03158724281485},{"id":"11209_G8","lat":40.625359345364686,"lon":-74.03158724281485},{"id":"11209_G9","lat":40.61861443776992,"lon":-74.02862518622933},{"id":"11209_G10","lat":40.62086274030151,"lon":-74.02862518622933},{"id":"11209_G11","lat":40.6231110428331,"lon":-74.02862518622933},{"id":"11209_G12","lat":40.62086274030151,"lon":-74.02566312964382},{"id":"11209_G13","lat":40.61768316036902,"lon":-74.03577622341061},{"id":"11209_G14","lat":40.624042320234004,"lon":-74.03577622341061},{"id":"11209_G15","lat":40.61768316036902,"lon":-74.0273982622191},{"id":"11209_G16","lat":40.624042320234004,"lon":-74.0273982622191},{"id":"11208_G1","lat":40.667474208909226,"lon":-73.87595442226271},{"id":"11208_G2","lat":40.665225906377636,"lon":-73.87299029636492},{"id":"11208_G3","lat":40.667474208909226,"lon":-73.87299029636492},{"id":"11208_G4","lat":40.66972251144082,"lon":-73.87299029636492},{"id":"11208_G5","lat":40.66297760384605,"lon":-73.87002617046714},{"id":"11208_G6","lat":40.665225906377636,"lon":-73.87002617046714},{"id":"11208_G7","lat":40.66972251144082,"lon":-73.87002617046714},{"id":"11208_G8","lat":40.6719708139724,"lon":-73.87002617046714},{"id":"11208_G9","lat":40.665225906377636,"lon":-73.86706204456937},{"id":"11208_G10","lat":40.667474208909226,"lon":-73.86706204456937},{"id":"11208_G11","lat":40.66972251144082,"lon":-73.86706204456937},{"id":"11208_G12","lat":40.667474208909226,"lon":-73.86409791867158},{"id":"11208_G13","lat":40.664294628976734,"lon":-73.87421807751237},{"id":"11208_G14","lat":40.67065378884172,"lon":-73.87421807751237},{"id":"11208_G15","lat":40.664294628976734,"lon":-73.86583426342192},{"id":"11208_G16","lat":40.67065378884172,"lon":-73.86583426342192},{"id":"11210_G1","lat":40.625764064313785,"lon":-73.95032032601979},{"id":"11210_G2","lat":40.623515761782194,"lon":-73.94735805206857},{"id":"11210_G3","lat":40.625764064313785,"lon":-73.94735805206857},{"id":"11210_G4","lat":40.628012366845375,"lon":-73.94735805206857},{"id":"11210_G5","lat":40.62126745925061,"lon":-73.94439577811734},{"id":"11210_G6","lat":40.623515761782194,"lon":-73.94439577811734},{"id":"11210_G7","lat":40.628012366845375,"lon":-73.94439577811734},{"id":"11210_G8","lat":40.63026066937696,"lon":-73.94439577811734},{"id":"11210_G9","lat":40.623515761782194,"lon":-73.94143350416611},{"id":"11210_G10","lat":40.625764064313785,"lon":-73.94143350416611},{"id":"11210_G11","lat":40.628012366845375,"lon":-73.94143350416611},{"id":"11210_G12","lat":40.625764064313785,"lon":-73.93847123021489},{"id":"11210_G13","lat":40.62258448438129,"lon":-73.94858506611463},{"id":"11210_G14","lat":40.62894364424628,"lon":-73.94858506611463},{"id":"11210_G16","lat":40.62894364424628,"lon":-73.94020649012005},{"id":"11211_G1","lat":40.71146977898183,"lon":-73.963297773078},{"id":"11211_G2","lat":40.70922147645024,"lon":-73.9603316895443},{"id":"11211_G3","lat":40.71146977898183,"lon":-73.9603316895443},{"id":"11211_G4","lat":40.71371808151342,"lon":-73.9603316895443},{"id":"11211_G5","lat":40.706973173918655,"lon":-73.95736560601061},{"id":"11211_G6","lat":40.70922147645024,"lon":-73.95736560601061},{"id":"11211_G7","lat":40.71371808151342,"lon":-73.95736560601061},{"id":"11211_G8","lat":40.715966384045004,"lon":-73.95736560601061},{"id":"11211_G9","lat":40.70922147645024,"lon":-73.95439952247692},{"id":"11211_G10","lat":40.71146977898183,"lon":-73.95439952247692},{"id":"11211_G11","lat":40.71371808151342,"lon":-73.95439952247692},{"id":"11211_G12","lat":40.71146977898183,"lon":-73.95143343894323},{"id":"11211_G13","lat":40.70829019904934,"lon":-73.9615602815711},{"id":"11211_G14","lat":40.71464935891432,"lon":-73.9615602815711},{"id":"11211_G15","lat":40.70829019904934,"lon":-73.95317093045013},{"id":"11211_G16","lat":40.71464935891432,"lon":-73.95317093045013},{"id":"11212_G1","lat":40.66459271042337,"lon":-73.92048319973613},{"id":"11212_G2","lat":40.66234440789178,"lon":-73.91751920190298},{"id":"11212_G3","lat":40.66459271042337,"lon":-73.91751920190298},{"id":"11212_G4","lat":40.66684101295496,"lon":-73.91751920190298},{"id":"11212_G5","lat":40.660096105360196,"lon":-73.91455520406981},{"id":"11212_G6","lat":40.66234440789178,"lon":-73.91455520406981},{"id":"11212_G7","lat":40.66684101295496,"lon":-73.91455520406981},{"id":"11212_G8","lat":40.669089315486545,"lon":-73.91455520406981},{"id":"11212_G9","lat":40.66234440789178,"lon":-73.91159120623665},{"id":"11212_G10","lat":40.66459271042337,"lon":-73.91159120623665},{"id":"11212_G11","lat":40.66684101295496,"lon":-73.91159120623665},{"id":"11212_G12","lat":40.66459271042337,"lon":-73.9086272084035},{"id":"11212_G13","lat":40.66141313049088,"lon":-73.9187469300043},{"id":"11212_G14","lat":40.66777229035586,"lon":-73.9187469300043},{"id":"11212_G15","lat":40.66141313049088,"lon":-73.91036347813532},{"id":"11212_G16","lat":40.66777229035586,"lon":-73.91036347813532},{"id":"11213_G1","lat":40.67145235690273,"lon":-73.94236290219088},{"id":"11213_G2","lat":40.66920405437114,"lon":-73.93939859945874},{"id":"11213_G3","lat":40.67145235690273,"lon":-73.93939859945874},{"id":"11213_G4","lat":40.673700659434324,"lon":-73.93939859945874},{"id":"11213_G5","lat":40.66695575183956,"lon":-73.9364342967266},{"id":"11213_G6","lat":40.66920405437114,"lon":-73.9364342967266},{"id":"11213_G7","lat":40.673700659434324,"lon":-73.9364342967266},{"id":"11213_G8","lat":40.67594896196591,"lon":-73.9364342967266},{"id":"11213_G9","lat":40.66920405437114,"lon":-73.93346999399446},{"id":"11213_G10","lat":40.67145235690273,"lon":-73.93346999399446},{"id":"11213_G11","lat":40.673700659434324,"lon":-73.93346999399446},{"id":"11213_G12","lat":40.67145235690273,"lon":-73.93050569126233},{"id":"11213_G13","lat":40.66827277697024,"lon":-73.94062645385337},{"id":"11213_G14","lat":40.674631936835226,"lon":-73.94062645385337},{"id":"11213_G15","lat":40.66827277697024,"lon":-73.93224213959984},{"id":"11213_G16","lat":40.674631936835226,"lon":-73.93224213959984},{"id":"11218_G1","lat":40.644382184885494,"lon":-73.98322921339718},{"id":"11218_G2","lat":40.6421338823539,"lon":-73.98026611327425},{"id":"11218_G3","lat":40.644382184885494,"lon":-73.98026611327425},{"id":"11218_G4","lat":40.646630487417085,"lon":-73.98026611327425},{"id":"11218_G5","lat":40.63988557982232,"lon":-73.97730301315131},{"id":"11218_G6","lat":40.6421338823539,"lon":-73.97730301315131},{"id":"11218_G7","lat":40.646630487417085,"lon":-73.97730301315131},{"id":"11218_G8","lat":40.64887878994867,"lon":-73.97730301315131},{"id":"11218_G9","lat":40.6421338823539,"lon":-73.97433991302837},{"id":"11218_G10","lat":40.644382184885494,"lon":-73.97433991302837},{"id":"11218_G11","lat":40.646630487417085,"lon":-73.97433991302837},{"id":"11218_G12","lat":40.644382184885494,"lon":-73.97137681290545},{"id":"11218_G13","lat":40.641202604953,"lon":-73.98149346953184},{"id":"11218_G15","lat":40.641202604953,"lon":-73.97311255677079},{"id":"11218_G16","lat":40.647561764817986,"lon":-73.97311255677079},{"id":"11214_G1","lat":40.598420087565906,"lon":-74.00217623392291},{"id":"11214_G2","lat":40.596171785034315,"lon":-73.99921517194835},{"id":"11214_G3","lat":40.598420087565906,"lon":-73.99921517194835},{"id":"11214_G4","lat":40.600668390097496,"lon":-73.99921517194835},{"id":"11214_G5","lat":40.59392348250273,"lon":-73.99625410997379},{"id":"11214_G6","lat":40.596171785034315,"lon":-73.99625410997379},{"id":"11214_G7","lat":40.600668390097496,"lon":-73.99625410997379},{"id":"11214_G8","lat":40.60291669262908,"lon":-73.99625410997379},{"id":"11214_G9","lat":40.596171785034315,"lon":-73.99329304799923},{"id":"11214_G10","lat":40.598420087565906,"lon":-73.99329304799923},{"id":"11214_G11","lat":40.600668390097496,"lon":-73.99329304799923},{"id":"11214_G12","lat":40.598420087565906,"lon":-73.99033198602467},{"id":"11214_G13","lat":40.59524050763341,"lon":-74.00044168397724},{"id":"11214_G14","lat":40.6015996674984,"lon":-74.00044168397724},{"id":"11214_G15","lat":40.59524050763341,"lon":-73.99206653597034},{"id":"11214_G16","lat":40.6015996674984,"lon":-73.99206653597034},{"id":"11215_G1","lat":40.66205720270209,"lon":-73.9917601062575},{"id":"11215_G2","lat":40.659808900170496,"lon":-73.9887962210965},{"id":"11215_G3","lat":40.66205720270209,"lon":-73.9887962210965},{"id":"11215_G4","lat":40.66430550523368,"lon":-73.9887962210965},{"id":"11215_G5","lat":40.65756059763891,"lon":-73.98583233593547},{"id":"11215_G6","lat":40.659808900170496,"lon":-73.98583233593547},{"id":"11215_G7","lat":40.66430550523368,"lon":-73.98583233593547},{"id":"11215_G8","lat":40.66655380776526,"lon":-73.98583233593547},{"id":"11215_G9","lat":40.659808900170496,"lon":-73.98286845077445},{"id":"11215_G10","lat":40.66205720270209,"lon":-73.98286845077445},{"id":"11215_G11","lat":40.66430550523368,"lon":-73.98286845077445},{"id":"11215_G12","lat":40.66205720270209,"lon":-73.97990456561344},{"id":"11215_G13","lat":40.658877622769594,"lon":-73.9900239025275},{"id":"11215_G14","lat":40.66523678263458,"lon":-73.9900239025275},{"id":"11215_G15","lat":40.658877622769594,"lon":-73.98164076934344},{"id":"11215_G16","lat":40.66523678263458,"lon":-73.98164076934344},{"id":"11216_G1","lat":40.68141562786529,"lon":-73.95265900947831},{"id":"11216_G2","lat":40.6791673253337,"lon":-73.94969426370923},{"id":"11216_G3","lat":40.68141562786529,"lon":-73.94969426370923},{"id":"11216_G4","lat":40.68366393039688,"lon":-73.94969426370923},{"id":"11216_G5","lat":40.67691902280212,"lon":-73.94672951794014},{"id":"11216_G6","lat":40.6791673253337,"lon":-73.94672951794014},{"id":"11216_G7","lat":40.68366393039688,"lon":-73.94672951794014},{"id":"11216_G8","lat":40.685912232928466,"lon":-73.94672951794014},{"id":"11216_G9","lat":40.6791673253337,"lon":-73.94376477217105},{"id":"11216_G10","lat":40.68141562786529,"lon":-73.94376477217105},{"id":"11216_G12","lat":40.68141562786529,"lon":-73.94080002640197},{"id":"11216_G13","lat":40.6782360479328,"lon":-73.95092230161578},{"id":"11216_G14","lat":40.684595207797784,"lon":-73.95092230161578},{"id":"11216_G15","lat":40.6782360479328,"lon":-73.94253673426451},{"id":"11217_G1","lat":40.68141525200155,"lon":-73.98598897148663},{"id":"11217_G2","lat":40.67916694946996,"lon":-73.98302422573427},{"id":"11217_G3","lat":40.68141525200155,"lon":-73.98302422573427},{"id":"11217_G4","lat":40.68366355453314,"lon":-73.98302422573427},{"id":"11217_G5","lat":40.676918646938375,"lon":-73.9800594799819},{"id":"11217_G6","lat":40.67916694946996,"lon":-73.9800594799819},{"id":"11217_G7","lat":40.68366355453314,"lon":-73.9800594799819},{"id":"11217_G8","lat":40.685911857064724,"lon":-73.9800594799819},{"id":"11217_G9","lat":40.67916694946996,"lon":-73.97709473422954},{"id":"11217_G10","lat":40.68141525200155,"lon":-73.97709473422954},{"id":"11217_G11","lat":40.68366355453314,"lon":-73.97709473422954},{"id":"11217_G12","lat":40.68141525200155,"lon":-73.97412998847717},{"id":"11217_G14","lat":40.68459483193404,"lon":-73.98425226363389},{"id":"11217_G15","lat":40.67823567206906,"lon":-73.97586669632992},{"id":"11217_G16","lat":40.68459483193404,"lon":-73.97586669632992},{"id":"11219_G1","lat":40.63343263409161,"lon":-74.00129265190157},{"id":"11219_G2","lat":40.63118433156002,"lon":-73.99833003775429},{"id":"11219_G3","lat":40.63343263409161,"lon":-73.99833003775429},{"id":"11219_G4","lat":40.6356809366232,"lon":-73.99833003775429},{"id":"11219_G5","lat":40.62893602902844,"lon":-73.99536742360702},{"id":"11219_G6","lat":40.63118433156002,"lon":-73.99536742360702},{"id":"11219_G7","lat":40.6356809366232,"lon":-73.99536742360702},{"id":"11219_G8","lat":40.637929239154786,"lon":-73.99536742360702},{"id":"11219_G9","lat":40.63118433156002,"lon":-73.99240480945976},{"id":"11219_G10","lat":40.63343263409161,"lon":-73.99240480945976},{"id":"11219_G11","lat":40.6356809366232,"lon":-73.99240480945976},{"id":"11219_G12","lat":40.63343263409161,"lon":-73.98944219531248},{"id":"11219_G13","lat":40.63025305415912,"lon":-73.99955719271418},{"id":"11219_G14","lat":40.636612214024105,"lon":-73.99955719271418},{"id":"11219_G15","lat":40.63025305415912,"lon":-73.99117765449986},{"id":"11219_G16","lat":40.636612214024105,"lon":-73.99117765449986},{"id":"11220_G1","lat":40.64077359924393,"lon":-74.02407957309202},{"id":"11220_G2","lat":40.63852529671234,"lon":-74.02111663315908},{"id":"11220_G3","lat":40.64077359924393,"lon":-74.02111663315908},{"id":"11220_G4","lat":40.64302190177552,"lon":-74.02111663315908},{"id":"11220_G5","lat":40.636276994180754,"lon":-74.01815369322615},{"id":"11220_G6","lat":40.63852529671234,"lon":-74.01815369322615},{"id":"11220_G7","lat":40.64302190177552,"lon":-74.01815369322615},{"id":"11220_G8","lat":40.6452702043071,"lon":-74.01815369322615},{"id":"11220_G9","lat":40.63852529671234,"lon":-74.01519075329323},{"id":"11220_G10","lat":40.64077359924393,"lon":-74.01519075329323},{"id":"11220_G11","lat":40.64302190177552,"lon":-74.01519075329323},{"id":"11220_G12","lat":40.64077359924393,"lon":-74.01222781336028},{"id":"11220_G13","lat":40.637594019311436,"lon":-74.0223439230638},{"id":"11220_G14","lat":40.64395317917642,"lon":-74.0223439230638},{"id":"11220_G15","lat":40.637594019311436,"lon":-74.0139634633885},{"id":"11220_G16","lat":40.64395317917642,"lon":-74.0139634633885},{"id":"11221_G1","lat":40.692511973278734,"lon":-73.93149328176085},{"id":"11221_G2","lat":40.69026367074714,"lon":-73.92852804230891},{"id":"11221_G3","lat":40.692511973278734,"lon":-73.92852804230891},{"id":"11221_G4","lat":40.694760275810324,"lon":-73.92852804230891},{"id":"11221_G5","lat":40.68801536821556,"lon":-73.92556280285697},{"id":"11221_G6","lat":40.69026367074714,"lon":-73.92556280285697},{"id":"11221_G7","lat":40.694760275810324,"lon":-73.92556280285697},{"id":"11221_G8","lat":40.69700857834191,"lon":-73.92556280285697},{"id":"11221_G9","lat":40.69026367074714,"lon":-73.92259756340503},{"id":"11221_G10","lat":40.692511973278734,"lon":-73.92259756340503},{"id":"11221_G11","lat":40.694760275810324,"lon":-73.92259756340503},{"id":"11221_G12","lat":40.692511973278734,"lon":-73.9196323239531},{"id":"11221_G13","lat":40.68933239334624,"lon":-73.92975628470559},{"id":"11221_G14","lat":40.695691553211226,"lon":-73.92975628470559},{"id":"11221_G15","lat":40.68933239334624,"lon":-73.92136932100836},{"id":"11221_G16","lat":40.695691553211226,"lon":-73.92136932100836},{"id":"11222_G1","lat":40.72848608108555,"lon":-73.95370701739124},{"id":"11222_G2","lat":40.72623777855396,"lon":-73.95074017553371},{"id":"11222_G3","lat":40.72848608108555,"lon":-73.95074017553371},{"id":"11222_G4","lat":40.73073438361714,"lon":-73.95074017553371},{"id":"11222_G5","lat":40.723989476022375,"lon":-73.94777333367618},{"id":"11222_G6","lat":40.72623777855396,"lon":-73.94777333367618},{"id":"11222_G7","lat":40.73073438361714,"lon":-73.94777333367618},{"id":"11222_G8","lat":40.73298268614872,"lon":-73.94777333367618},{"id":"11222_G9","lat":40.72623777855396,"lon":-73.94480649181865},{"id":"11222_G10","lat":40.72848608108555,"lon":-73.94480649181865},{"id":"11222_G11","lat":40.73073438361714,"lon":-73.94480649181865},{"id":"11222_G12","lat":40.72848608108555,"lon":-73.94183964996112},{"id":"11222_G13","lat":40.72530650115306,"lon":-73.95196908166852},{"id":"11222_G14","lat":40.73166566101804,"lon":-73.95196908166852},{"id":"11222_G15","lat":40.72530650115306,"lon":-73.94357758568384},{"id":"11222_G16","lat":40.73166566101804,"lon":-73.94357758568384},{"id":"11223_G1","lat":40.59656687173119,"lon":-73.97845288820362},{"id":"11223_G2","lat":40.5943185691996,"lon":-73.97549190830951},{"id":"11223_G3","lat":40.59656687173119,"lon":-73.97549190830951},{"id":"11223_G4","lat":40.59881517426278,"lon":-73.97549190830951},{"id":"11223_G5","lat":40.59207026666802,"lon":-73.97253092841541},{"id":"11223_G6","lat":40.5943185691996,"lon":-73.97253092841541},{"id":"11223_G7","lat":40.59881517426278,"lon":-73.97253092841541},{"id":"11223_G8","lat":40.601063476794366,"lon":-73.97253092841541},{"id":"11223_G9","lat":40.5943185691996,"lon":-73.96956994852131},{"id":"11223_G10","lat":40.59656687173119,"lon":-73.96956994852131},{"id":"11223_G11","lat":40.59881517426278,"lon":-73.96956994852131},{"id":"11223_G12","lat":40.59656687173119,"lon":-73.9666089686272},{"id":"11223_G13","lat":40.5933872917987,"lon":-73.97671838633957},{"id":"11223_G14","lat":40.599746451663684,"lon":-73.97671838633957},{"id":"11223_G15","lat":40.5933872917987,"lon":-73.96834347049125},{"id":"11223_G16","lat":40.599746451663684,"lon":-73.96834347049125},{"id":"11224_G1","lat":40.57696095397775,"lon":-73.99703607357016},{"id":"11224_G2","lat":40.57471265144616,"lon":-73.99407596156998},{"id":"11224_G3","lat":40.57696095397775,"lon":-73.99407596156998},{"id":"11224_G4","lat":40.579209256509344,"lon":-73.99407596156998},{"id":"11224_G5","lat":40.57246434891458,"lon":-73.99111584956981},{"id":"11224_G6","lat":40.57471265144616,"lon":-73.99111584956981},{"id":"11224_G7","lat":40.579209256509344,"lon":-73.99111584956981},{"id":"11224_G9","lat":40.57471265144616,"lon":-73.98815573756964},{"id":"11224_G10","lat":40.57696095397775,"lon":-73.98815573756964},{"id":"11224_G12","lat":40.57696095397775,"lon":-73.98519562556946},{"id":"11224_G13","lat":40.57378137404526,"lon":-73.9953020801066},{"id":"11224_G15","lat":40.57378137404526,"lon":-73.98692961903302},{"id":"11225_G1","lat":40.66326615175426,"lon":-73.95653238116603},{"id":"11225_G2","lat":40.66101784922267,"lon":-73.95356844228388},{"id":"11225_G3","lat":40.66326615175426,"lon":-73.95356844228388},{"id":"11225_G4","lat":40.66551445428585,"lon":-73.95356844228388},{"id":"11225_G5","lat":40.65876954669108,"lon":-73.95060450340173},{"id":"11225_G6","lat":40.66101784922267,"lon":-73.95060450340173},{"id":"11225_G7","lat":40.66551445428585,"lon":-73.95060450340173},{"id":"11225_G8","lat":40.66776275681743,"lon":-73.95060450340173},{"id":"11225_G9","lat":40.66101784922267,"lon":-73.94764056451959},{"id":"11225_G10","lat":40.66326615175426,"lon":-73.94764056451959},{"id":"11225_G11","lat":40.66551445428585,"lon":-73.94764056451959},{"id":"11225_G12","lat":40.66326615175426,"lon":-73.94467662563744},{"id":"11225_G13","lat":40.660086571821765,"lon":-73.95479614596691},{"id":"11225_G14","lat":40.66644573168675,"lon":-73.95479614596691},{"id":"11226_G1","lat":40.64590786004898,"lon":-73.9626187820861},{"id":"11226_G2","lat":40.64365955751739,"lon":-73.95965561422763},{"id":"11226_G3","lat":40.64590786004898,"lon":-73.95965561422763},{"id":"11226_G4","lat":40.64815616258057,"lon":-73.95965561422763},{"id":"11226_G5","lat":40.641411254985805,"lon":-73.95669244636916},{"id":"11226_G6","lat":40.64365955751739,"lon":-73.95669244636916},{"id":"11226_G7","lat":40.64815616258057,"lon":-73.95669244636916},{"id":"11226_G8","lat":40.65040446511215,"lon":-73.95669244636916},{"id":"11226_G9","lat":40.64365955751739,"lon":-73.95372927851069},{"id":"11226_G10","lat":40.64590786004898,"lon":-73.95372927851069},{"id":"11226_G11","lat":40.64815616258057,"lon":-73.95372927851069},{"id":"11226_G12","lat":40.64590786004898,"lon":-73.95076611065222},{"id":"11226_G13","lat":40.642728280116486,"lon":-73.96088299854219},{"id":"11226_G14","lat":40.64908743998147,"lon":-73.96088299854219},{"id":"11226_G15","lat":40.642728280116486,"lon":-73.95250189419613},{"id":"11226_G16","lat":40.64908743998147,"lon":-73.95250189419613},{"id":"11228_G1","lat":40.61649663405339,"lon":-74.01893739405284},{"id":"11228_G2","lat":40.6142483315218,"lon":-74.01597553105188},{"id":"11228_G3","lat":40.61649663405339,"lon":-74.01597553105188},{"id":"11228_G4","lat":40.61874493658498,"lon":-74.01597553105188},{"id":"11228_G5","lat":40.61200002899022,"lon":-74.01301366805092},{"id":"11228_G6","lat":40.6142483315218,"lon":-74.01301366805092},{"id":"11228_G7","lat":40.61874493658498,"lon":-74.01301366805092},{"id":"11228_G8","lat":40.620993239116565,"lon":-74.01301366805092},{"id":"11228_G9","lat":40.6142483315218,"lon":-74.01005180504995},{"id":"11228_G10","lat":40.61649663405339,"lon":-74.01005180504995},{"id":"11228_G11","lat":40.61874493658498,"lon":-74.01005180504995},{"id":"11228_G12","lat":40.61649663405339,"lon":-74.00708994204899},{"id":"11228_G13","lat":40.6133170541209,"lon":-74.01720237487676},{"id":"11228_G14","lat":40.61967621398588,"lon":-74.01720237487676},{"id":"11228_G15","lat":40.6133170541209,"lon":-74.00882496122507},{"id":"11228_G16","lat":40.61967621398588,"lon":-74.00882496122507},{"id":"11230_G1","lat":40.62235511455446,"lon":-73.97157406831984},{"id":"11230_G2","lat":40.62010681202287,"lon":-73.96861194555564},{"id":"11230_G3","lat":40.62235511455446,"lon":-73.96861194555564},{"id":"11230_G4","lat":40.62460341708605,"lon":-73.96861194555564},{"id":"11230_G5","lat":40.617858509491285,"lon":-73.96564982279143},{"id":"11230_G6","lat":40.62010681202287,"lon":-73.96564982279143},{"id":"11230_G7","lat":40.62460341708605,"lon":-73.96564982279143},{"id":"11230_G8","lat":40.62685171961763,"lon":-73.96564982279143},{"id":"11230_G9","lat":40.62010681202287,"lon":-73.96268770002723},{"id":"11230_G10","lat":40.62235511455446,"lon":-73.96268770002723},{"id":"11230_G11","lat":40.62460341708605,"lon":-73.96268770002723},{"id":"11230_G12","lat":40.62235511455446,"lon":-73.95972557726303},{"id":"11230_G13","lat":40.61917553462197,"lon":-73.96983889697799},{"id":"11230_G14","lat":40.62553469448695,"lon":-73.96983889697799},{"id":"11230_G15","lat":40.61917553462197,"lon":-73.96146074860488},{"id":"11230_G16","lat":40.62553469448695,"lon":-73.96146074860488},{"id":"11229_G1","lat":40.601879888416626,"lon":-73.95247327877665},{"id":"11229_G2","lat":40.599631585885035,"lon":-73.94951206354419},{"id":"11229_G3","lat":40.601879888416626,"lon":-73.94951206354419},{"id":"11229_G4","lat":40.60412819094822,"lon":-73.94951206354419},{"id":"11229_G5","lat":40.59738328335345,"lon":-73.94655084831174},{"id":"11229_G6","lat":40.599631585885035,"lon":-73.94655084831174},{"id":"11229_G7","lat":40.60412819094822,"lon":-73.94655084831174},{"id":"11229_G8","lat":40.6063764934798,"lon":-73.94655084831174},{"id":"11229_G9","lat":40.599631585885035,"lon":-73.9435896330793},{"id":"11229_G10","lat":40.601879888416626,"lon":-73.9435896330793},{"id":"11229_G11","lat":40.60412819094822,"lon":-73.9435896330793},{"id":"11229_G12","lat":40.601879888416626,"lon":-73.94062841784684},{"id":"11229_G13","lat":40.59870030848413,"lon":-73.95073863905458},{"id":"11229_G14","lat":40.60505946834912,"lon":-73.95073863905458},{"id":"11229_G15","lat":40.59870030848413,"lon":-73.94236305756891},{"id":"11229_G16","lat":40.60505946834912,"lon":-73.94236305756891},{"id":"11231_G1","lat":40.67615783487769,"lon":-74.01197241361005},{"id":"11231_G2","lat":40.6739095323461,"lon":-74.00900790166702},{"id":"11231_G3","lat":40.67615783487769,"lon":-74.00900790166702},{"id":"11231_G4","lat":40.67840613740928,"lon":-74.00900790166702},{"id":"11231_G5","lat":40.67166122981452,"lon":-74.00604338972398},{"id":"11231_G6","lat":40.6739095323461,"lon":-74.00604338972398},{"id":"11231_G7","lat":40.67840613740928,"lon":-74.00604338972398},{"id":"11231_G8","lat":40.68065443994087,"lon":-74.00604338972398},{"id":"11231_G9","lat":40.6739095323461,"lon":-74.00307887778095},{"id":"11231_G10","lat":40.67615783487769,"lon":-74.00307887778095},{"id":"11231_G11","lat":40.67840613740928,"lon":-74.00307887778095},{"id":"11231_G12","lat":40.67615783487769,"lon":-74.00011436583792},{"id":"11231_G13","lat":40.6729782549452,"lon":-74.01023584271964},{"id":"11231_G14","lat":40.679337414810185,"lon":-74.01023584271964},{"id":"11231_G15","lat":40.6729782549452,"lon":-74.00185093672833},{"id":"11231_G16","lat":40.679337414810185,"lon":-74.00185093672833},{"id":"11232_G1","lat":40.657191408710766,"lon":-74.01221329140078},{"id":"11232_G2","lat":40.654943106179175,"lon":-74.00924962242422},{"id":"11232_G3","lat":40.657191408710766,"lon":-74.00924962242422},{"id":"11232_G4","lat":40.65943971124236,"lon":-74.00924962242422},{"id":"11232_G5","lat":40.65269480364759,"lon":-74.00628595344767},{"id":"11232_G6","lat":40.654943106179175,"lon":-74.00628595344767},{"id":"11232_G7","lat":40.65943971124236,"lon":-74.00628595344767},{"id":"11232_G8","lat":40.66168801377394,"lon":-74.00628595344767},{"id":"11232_G9","lat":40.654943106179175,"lon":-74.00332228447112},{"id":"11232_G10","lat":40.657191408710766,"lon":-74.00332228447112},{"id":"11232_G11","lat":40.65943971124236,"lon":-74.00332228447112},{"id":"11232_G12","lat":40.657191408710766,"lon":-74.00035861549456},{"id":"11232_G13","lat":40.65401182877827,"lon":-74.01047721430871},{"id":"11232_G15","lat":40.65401182877827,"lon":-74.00209469258664},{"id":"11232_G16","lat":40.66037098864326,"lon":-74.00209469258664},{"id":"11233_G1","lat":40.67685532584006,"lon":-73.92279661905727},{"id":"11233_G2","lat":40.67460702330847,"lon":-73.91983207609877},{"id":"11233_G3","lat":40.67685532584006,"lon":-73.91983207609877},{"id":"11233_G4","lat":40.67910362837165,"lon":-73.91983207609877},{"id":"11233_G5","lat":40.672358720776884,"lon":-73.91686753314028},{"id":"11233_G6","lat":40.67460702330847,"lon":-73.91686753314028},{"id":"11233_G7","lat":40.67910362837165,"lon":-73.91686753314028},{"id":"11233_G8","lat":40.68135193090323,"lon":-73.91686753314028},{"id":"11233_G9","lat":40.67460702330847,"lon":-73.91390299018178},{"id":"11233_G10","lat":40.67685532584006,"lon":-73.91390299018178},{"id":"11233_G11","lat":40.67910362837165,"lon":-73.91390299018178},{"id":"11233_G12","lat":40.67685532584006,"lon":-73.91093844722329},{"id":"11233_G13","lat":40.673675745907566,"lon":-73.92106002999842},{"id":"11233_G14","lat":40.68003490577255,"lon":-73.92106002999842},{"id":"11233_G15","lat":40.673675745907566,"lon":-73.91267503628214},{"id":"11233_G16","lat":40.68003490577255,"lon":-73.91267503628214},{"id":"11234_G1","lat":40.583915025296164,"lon":-73.92817357300973},{"id":"11234_G3","lat":40.583915025296164,"lon":-73.92521315327201},{"id":"11234_G10","lat":40.583915025296164,"lon":-73.9192923137966},{"id":"11234_G12","lat":40.583915025296164,"lon":-73.91633189405889},{"id":"11234_G16","lat":40.58709460522866,"lon":-73.91806606779092},{"id":"11235_G1","lat":40.584355409816474,"lon":-73.95481800306837},{"id":"11235_G2","lat":40.582107107284884,"lon":-73.95185756383877},{"id":"11235_G3","lat":40.584355409816474,"lon":-73.95185756383877},{"id":"11235_G4","lat":40.586603712348065,"lon":-73.95185756383877},{"id":"11235_G5","lat":40.5798588047533,"lon":-73.94889712460916},{"id":"11235_G6","lat":40.582107107284884,"lon":-73.94889712460916},{"id":"11235_G7","lat":40.586603712348065,"lon":-73.94889712460916},{"id":"11235_G8","lat":40.58885201487965,"lon":-73.94889712460916},{"id":"11235_G10","lat":40.584355409816474,"lon":-73.94593668537955},{"id":"11235_G11","lat":40.586603712348065,"lon":-73.94593668537955},{"id":"11235_G12","lat":40.584355409816474,"lon":-73.94297624614995},{"id":"11235_G13","lat":40.58117582988398,"lon":-73.95308381791826},{"id":"11235_G14","lat":40.58753498974897,"lon":-73.95308381791826},{"id":"11235_G15","lat":40.58117582988398,"lon":-73.94471043130007},{"id":"11235_G16","lat":40.58753498974897,"lon":-73.94471043130007},{"id":"11236_G1","lat":40.63997966964458,"lon":-73.90614420513207},{"id":"11236_G2","lat":40.63773136711299,"lon":-73.90318130043883},{"id":"11236_G3","lat":40.63997966964458,"lon":-73.90318130043883},{"id":"11236_G4","lat":40.64222797217617,"lon":-73.90318130043883},{"id":"11236_G5","lat":40.63548306458141,"lon":-73.90021839574561},{"id":"11236_G6","lat":40.63773136711299,"lon":-73.90021839574561},{"id":"11236_G7","lat":40.64222797217617,"lon":-73.90021839574561},{"id":"11236_G8","lat":40.644476274707756,"lon":-73.90021839574561},{"id":"11236_G9","lat":40.63773136711299,"lon":-73.89725549105239},{"id":"11236_G10","lat":40.63997966964458,"lon":-73.89725549105239},{"id":"11236_G11","lat":40.64222797217617,"lon":-73.89725549105239},{"id":"11236_G12","lat":40.63997966964458,"lon":-73.89429258635916},{"id":"11236_G13","lat":40.63680008971209,"lon":-73.90440857574679},{"id":"11236_G14","lat":40.643159249577074,"lon":-73.90440857574679},{"id":"11236_G15","lat":40.63680008971209,"lon":-73.89602821574444},{"id":"11236_G16","lat":40.643159249577074,"lon":-73.89602821574444},{"id":"11237_G1","lat":40.70324379301033,"lon":-73.92703795690203},{"id":"11237_G2","lat":40.70099549047874,"lon":-73.92407223972295},{"id":"11237_G3","lat":40.70324379301033,"lon":-73.92407223972295},{"id":"11237_G4","lat":40.70549209554192,"lon":-73.92407223972295},{"id":"11237_G5","lat":40.698747187947156,"lon":-73.92110652254385},{"id":"11237_G6","lat":40.70099549047874,"lon":-73.92110652254385},{"id":"11237_G7","lat":40.70549209554192,"lon":-73.92110652254385},{"id":"11237_G8","lat":40.707740398073504,"lon":-73.92110652254385},{"id":"11237_G9","lat":40.70099549047874,"lon":-73.91814080536476},{"id":"11237_G10","lat":40.70324379301033,"lon":-73.91814080536476},{"id":"11237_G11","lat":40.70549209554192,"lon":-73.91814080536476},{"id":"11237_G12","lat":40.70324379301033,"lon":-73.91517508818568},{"id":"11237_G14","lat":40.70642337294282,"lon":-73.92530068000069},{"id":"11237_G15","lat":40.70006421307784,"lon":-73.91691236508701},{"id":"11237_G16","lat":40.70642337294282,"lon":-73.91691236508701},{"id":"11238_G1","lat":40.67659973831762,"lon":-73.96986043204168},{"id":"11238_G2","lat":40.67435143578603,"lon":-73.96689590044858},{"id":"11238_G3","lat":40.67659973831762,"lon":-73.96689590044858},{"id":"11238_G4","lat":40.67884804084921,"lon":-73.96689590044858},{"id":"11238_G5","lat":40.672103133254446,"lon":-73.96393136885547},{"id":"11238_G6","lat":40.67435143578603,"lon":-73.96393136885547},{"id":"11238_G7","lat":40.67884804084921,"lon":-73.96393136885547},{"id":"11238_G8","lat":40.681096343380794,"lon":-73.96393136885547},{"id":"11238_G9","lat":40.67435143578603,"lon":-73.96096683726236},{"id":"11238_G10","lat":40.67659973831762,"lon":-73.96096683726236},{"id":"11238_G11","lat":40.67884804084921,"lon":-73.96096683726236},{"id":"11238_G12","lat":40.67659973831762,"lon":-73.95800230566925},{"id":"11238_G13","lat":40.67342015838513,"lon":-73.96812384964053},{"id":"11238_G14","lat":40.67977931825011,"lon":-73.96812384964053},{"id":"11238_G15","lat":40.67342015838513,"lon":-73.9597388880704},{"id":"11238_G16","lat":40.67977931825011,"lon":-73.9597388880704},{"id":"11239_G2","lat":40.64462288621848,"lon":-73.88510055942476},{"id":"11239_G3","lat":40.64687118875007,"lon":-73.88510055942476},{"id":"11239_G4","lat":40.64911949128166,"lon":-73.88510055942476},{"id":"11239_G6","lat":40.64462288621848,"lon":-73.88213734879463},{"id":"11239_G7","lat":40.64911949128166,"lon":-73.88213734879463},{"id":"11239_G8","lat":40.65136779381324,"lon":-73.88213734879463},{"id":"11239_G9","lat":40.64462288621848,"lon":-73.8791741381645},{"id":"11239_G10","lat":40.64687118875007,"lon":-73.8791741381645},{"id":"11239_G11","lat":40.64911949128166,"lon":-73.8791741381645},{"id":"11239_G14","lat":40.65005076868256,"lon":-73.88632796145593},{"id":"11239_G15","lat":40.643691608817576,"lon":-73.87794673613332}],"routes":[[1.921,1.746,1.394,1.187,1.4,1.254,null,null,1.44,1.39,0.64,1.099,1.959,1.252,1.662,0.948,8.175,8.514,8.23,8.256,8.877,8.812,8.241,8.093,8.997,8.749,8.5,8.863,8.405,7.807,9.126,8.49,10.023,10.315,10.057,9.741,10.4,10.39,9.623,9.543,10.154,10.027,9.71,10.01,10.653,9.749,10.23,9.547,1.354,1.402,1.315,1.784,1.64,2.011,6.367,2.004,1.318,2.092,3.618,3.472,3.612,3.949,3.78,3.842,4.127,4.381,4.061,4.025,4.394,4.262,3.453,4.179,3.995,4.978,8.82,9.167,8.918,8.591,9.826,9.579,8.867,8.716,9.56,9.442,9.248,9.674,9.202,8.609,9.757,9.506,10.986,10.854,10.757,10.468,11.008,10.715,10.14,10.014,10.623,10.312,10.113,10.369,11.017,10.559,10.837,10.299,11.305,11.575,11.71,11.211,11.972,11.818,11.559,11.429,12.046,11.96,12.067,12.207,11.456,11.045,12.252,11.702,9.062,9.837,9.548,8.89,10.09,9.533,9.294,8.811,9.842,9.55,9.263,9.527,9.704,8.87,9.225,3.717,3.549,4.071,4.25,3.168,3.591,4.123,4.597,3.478,3.679,4.298,3.945,3.624,4.125,3.643,4.263,7.922,8.183,8.081,8.065,8.537,8.474,8.145,7.889,8.736,8.586,8.359,8.838,8.089,7.697,8.897,8.351,5.452,5.895,5.703,5.531,6.404,6.142,5.892,5.409,6.796,6.132,5.931,6.788,5.896,5.227,6.524,5.957,7.196,7.286,6.994,6.73,7.608,7.795,6.922,6.662,7.468,7.218,7.397,7.204,7.486,7.659,6.855,13.009,15.753,13.045,12.578,13.678,13.009,12.589,12.226,13.04,12.657,12.542,12.604,13.311,12.544,13.051,12.324,4.857,5.228,4.997,4.561,5.707,5.706,4.761,4.323,5.192,5.047,4.514,4.94,5.552,4.51,5.199,4.83,3.775,4.079,4.06,4.051,4.574,4.453,4.3,4.641,4.517,4.592,4.903,4.018,3.966,5.261,2.322,2.74,2.413,2.239,2.887,2.706,2.202,1.853,2.665,2.326,2.365,2.282,2.012,2.659,1.842,8.994,9.26,8.948,8.606,9.292,8.996,8.522,8.315,9.179,9.158,8.569,8.801,9.435,8.603,9.286,8.451,8.217,9.174,8.339,8.089,8.815,8.219,7.759,7.444,8.183,7.938,7.624,7.904,8.416,7.965,8.37,8.021,5.016,5.357,5.115,4.916,5.778,5.472,5.227,5.272,5.571,5.489,5.533,5.97,5.292,4.749,5.667,5.693,5.753,5.966,5.986,6.19,5.464,5.653,6.319,6.732,5.805,6.052,6.655,6.003,5.399,6.502,5.806,6.64,12.669,12.728,12.493,12.261,13.396,12.789,12.138,11.863,13.141,12.477,12.229,12.902,12.928,12.445,13.044,12.278,15.902,15.998,15.757,15.582,16.028,15.904,15.42,15.549,15.279,15.059,16.177,15.616,5.138,5.637,5.487,5.155,6.094,5.867,5.408,5.182,6.094,5.91,5.707,6.084,5.947,5.347,6.688,6.843,6.689,6.618,7.131,6.976,6.526,6.277,7.172,7.148,6.776,7.231,7.402,6.23,7.765,7.024,11.134,11.431,11.359,10.975,11.644,11.406,11.055,11.05,11.29,11.131,10.971,10.995,11.415,11.08,11.482,10.979,9.692,10.08,9.833,9.645,10.277,10.01,9.4,9.122,9.832,10.042,9.233,9.509,10.054,9.695,9.871,9.302,11.615,11.981,11.738,11.446,12.153,11.906,11.491,11.244,12.146,11.937,12.0,11.917,11.992,11.558,12.375,11.507,4.584,4.796,4.3,4.204,4.192,4.153,3.949,3.887,3.938,3.73,3.992,3.624,4.651,4.347,3.752,3.268,6.407,6.188,6.148,6.224,6.214,6.039,5.779,5.748,6.108,5.769,5.838,5.845,6.524,6.3,5.275,6.397,6.918,7.075,6.633,7.384,7.15,6.731,6.696,7.655,7.214,6.979,7.382,6.906,6.503,7.675,7.222,14.346,14.597,15.1,15.342,14.273,14.227,14.407,14.164,13.608,14.903,14.653,13.261,13.088,13.737,13.5,13.716,14.391,13.448,15.022,13.282,10.555,10.891,10.969,10.811,11.271,11.603,11.181,10.973,11.446,11.368,11.325,11.761,11.062,10.803,11.584,11.839,5.153,5.656,5.462,5.722,5.435,5.572,5.756,5.949,5.857,6.069,6.092,6.298,5.277,6.002,6.241,3.05,3.562,3.291,3.212,3.998,3.634,3.096,2.979,4.042,3.494,3.303,3.762,3.775,2.792,3.961,3.265,11.999,11.864,11.622,12.215,12.186,11.666,12.399,12.602,12.106,11.526,12.398],[2.356,1.482,1.786,2.134,1.09,1.359,null,null,1.207,1.457,1.728,1.483,1.874,2.199,1.075,2.036,6.82,7.159,6.875,6.901,7.522,7.457,6.886,6.738,7.642,7.394,7.145,7.633,7.05,6.452,7.838,7.203,8.668,8.96,8.702,8.386,9.045,9.035,8.268,8.188,8.799,8.672,8.355,8.655,9.298,8.394,8.875,8.192,2.44,2.068,2.315,1.953,2.603,2.478,7.407,2.931,1.82,2.559,4.658,4.512,4.652,4.989,4.704,4.882,5.167,5.421,5.053,5.065,5.434,5.302,4.38,5.219,4.87,6.018,7.754,8.156,7.974,7.738,8.973,8.726,8.014,7.863,8.707,8.589,8.395,8.821,8.136,7.543,8.904,8.653,9.402,9.27,9.173,8.884,9.424,9.154,8.556,8.43,9.062,8.751,8.552,8.846,9.433,8.975,9.314,8.776,10.452,10.722,10.857,10.358,11.119,10.965,10.706,10.576,11.193,11.107,11.214,11.354,10.603,10.192,11.399,10.849,7.707,8.482,8.193,7.535,8.735,8.178,7.939,7.456,8.487,8.195,7.908,8.172,8.349,7.515,7.87,4.757,4.589,5.111,5.29,4.208,4.631,5.163,5.637,4.518,4.719,5.338,4.985,4.664,5.165,4.683,5.361,6.692,6.953,6.851,6.835,7.307,7.244,6.938,6.682,7.506,7.379,7.152,7.687,6.859,6.467,7.667,7.203,4.245,4.665,4.496,4.423,5.174,4.912,4.685,4.301,5.589,4.925,4.823,5.581,4.666,4.069,5.294,4.849,5.841,5.931,5.639,5.375,6.253,6.44,5.567,5.307,6.113,5.863,6.042,5.849,6.131,6.304,5.5,11.654,14.398,11.69,11.223,12.323,11.654,11.234,10.871,11.685,11.302,11.187,11.249,11.956,11.189,11.696,10.969,3.443,3.873,3.541,3.147,4.352,4.351,3.406,2.909,3.837,3.692,3.159,3.585,4.197,3.086,3.844,3.475,2.859,3.163,3.144,3.135,3.466,3.387,3.384,3.758,3.601,3.676,3.987,2.952,3.082,4.153,0.732,1.326,0.823,0.649,1.473,1.292,0.612,0.263,1.251,0.912,0.951,1.049,0.422,1.304,0.882,7.58,7.846,7.534,7.192,7.878,7.582,7.108,6.901,7.765,7.803,7.155,7.446,8.021,7.189,7.872,7.096,6.633,7.735,6.883,6.505,7.376,6.763,6.303,5.86,6.769,6.482,6.168,6.49,6.855,6.381,6.956,6.565,4.464,4.983,4.71,5.159,5.481,5.228,5.127,5.321,5.327,5.372,5.416,5.846,4.782,4.997,5.423,5.576,6.851,7.064,7.084,7.288,6.562,6.751,7.417,7.83,6.903,7.15,7.753,7.101,6.497,7.6,6.904,7.738,11.314,11.373,11.138,10.906,12.041,11.434,10.783,10.508,11.786,11.122,10.874,11.547,11.573,11.09,11.689,10.923,14.547,14.643,14.402,14.227,14.673,14.549,14.065,14.194,13.924,13.704,14.822,14.261,3.783,4.282,4.132,3.925,4.739,4.512,4.178,3.952,4.739,4.618,4.477,4.854,4.592,4.117,5.333,5.488,5.334,5.263,5.776,5.621,5.171,4.922,5.817,5.793,5.421,5.876,6.047,4.875,6.41,5.669,9.611,9.908,9.836,9.458,10.23,9.992,9.538,9.533,9.876,9.717,9.557,9.581,9.892,9.563,10.068,9.462,8.337,8.725,8.478,8.29,8.922,8.655,8.045,7.767,8.477,8.687,7.878,8.154,8.699,8.34,8.516,7.947,10.26,10.626,10.383,10.091,10.798,10.551,10.136,9.889,10.791,10.582,10.645,10.562,10.637,10.203,11.02,10.152,3.968,4.101,3.605,3.607,3.455,3.416,3.35,3.29,3.201,2.993,3.393,2.887,3.914,3.75,3.015,2.664,4.823,4.604,4.564,4.64,4.758,4.455,4.195,4.164,4.524,4.313,4.254,4.431,4.94,4.716,3.691,5.331,5.852,6.009,5.717,6.276,6.084,5.815,6.093,6.547,6.148,6.063,6.401,5.798,5.587,6.567,6.509,12.991,13.242,13.745,13.987,12.918,12.872,13.052,12.809,12.253,13.548,13.298,11.906,11.733,12.382,12.145,12.361,13.036,12.093,13.667,11.927,9.325,9.661,9.739,9.581,10.041,10.373,9.951,9.743,10.216,10.138,10.095,10.531,9.832,9.573,10.354,10.609,6.011,6.036,6.151,6.735,5.71,5.903,6.387,6.983,6.132,6.4,6.472,6.573,6.317,6.277,6.621,1.695,2.207,1.936,2.055,2.791,2.427,1.988,2.063,2.835,2.336,2.195,2.604,2.42,1.635,2.754,2.193,10.933,10.798,10.769,11.149,11.245,10.813,11.546,11.749,11.253,10.604,11.545],[1.505,1.768,1.43,0.771,1.777,1.66,null,null,1.671,1.421,1.217,1.864,1.543,0.836,2.039,1.159,9.155,9.494,9.21,9.236,9.857,9.792,9.221,9.073,9.977,9.729,9.48,9.968,9.385,8.787,10.173,9.538,10.582,10.874,10.616,10.3,10.959,10.949,10.182,10.102,10.713,10.586,10.269,10.569,11.212,10.308,10.789,10.106,2.325,2.816,2.571,3.234,3.054,3.445,7.448,3.418,2.787,3.526,4.699,4.553,4.693,5.03,5.187,4.923,5.208,5.462,5.142,5.106,5.475,5.343,4.729,5.26,5.281,6.059,10.018,10.42,10.204,9.877,11.112,10.865,10.153,10.002,10.846,10.728,10.534,10.96,10.4,9.807,11.043,10.792,11.066,10.934,10.837,10.548,11.088,10.818,10.22,10.094,10.726,10.415,10.216,10.51,11.097,10.639,10.978,10.44,12.591,12.861,12.996,12.497,13.258,13.104,12.845,12.715,13.332,13.246,13.353,13.493,12.742,12.331,13.538,12.988,10.042,10.817,10.528,9.87,11.07,10.513,10.274,9.791,10.822,10.53,10.243,10.507,10.684,9.85,10.205,4.798,4.63,5.152,5.331,4.249,4.672,5.204,5.678,4.559,4.76,5.379,5.026,4.705,5.206,4.724,5.402,9.027,9.288,9.186,9.17,9.642,9.579,9.273,9.017,9.841,9.714,9.487,10.022,9.194,8.802,10.002,9.538,6.58,7.0,6.831,6.729,7.509,7.247,7.02,6.607,7.924,7.26,7.129,7.916,7.001,6.404,7.629,7.155,7.755,7.845,7.553,7.289,8.167,8.354,7.46,7.199,7.948,7.698,8.062,8.184,8.045,8.139,7.569,13.362,16.312,13.524,13.071,14.237,13.568,13.082,12.719,13.599,13.216,13.101,13.163,13.87,12.897,13.61,12.883,5.158,5.716,5.29,5.188,6.312,6.469,5.62,5.244,5.885,5.904,5.494,5.799,5.971,5.046,6.029,5.81,4.973,5.277,5.258,5.249,5.772,5.651,5.498,5.872,5.715,5.79,6.101,5.216,5.196,6.459,3.134,3.661,3.228,3.086,3.808,3.627,3.049,2.747,3.586,3.247,3.286,3.384,2.856,3.639,3.04,9.285,9.551,9.239,8.897,9.583,9.287,8.813,8.606,9.47,9.563,8.86,9.36,9.726,8.894,9.577,8.965,8.297,9.399,8.561,8.169,9.04,8.441,7.981,7.524,8.471,8.16,7.846,8.192,8.519,8.045,8.658,8.243,6.302,6.643,6.401,6.202,7.064,6.758,6.513,6.558,6.857,6.775,6.819,7.256,6.578,6.035,6.953,6.979,6.892,7.105,7.125,7.329,6.603,6.792,7.458,7.871,6.944,7.191,7.794,7.142,6.538,7.641,6.945,7.779,13.228,13.287,13.052,12.82,13.955,13.348,12.697,12.422,13.7,13.036,12.788,13.382,13.487,13.004,13.603,12.806,16.461,16.557,16.316,16.141,16.587,16.463,15.979,16.108,15.838,15.618,16.736,16.175,6.118,6.617,6.467,6.26,7.074,6.847,6.513,6.287,7.074,6.953,6.812,7.189,6.927,6.452,7.668,7.823,7.669,7.598,8.111,7.956,7.506,7.257,8.152,8.128,7.756,8.211,8.382,7.21,8.745,8.004,11.275,11.572,11.5,11.122,11.935,11.697,11.202,11.197,11.581,11.422,11.262,11.286,11.556,11.227,11.773,11.126,10.172,10.56,10.313,10.133,11.014,10.904,10.362,10.084,10.812,11.022,10.213,10.489,10.534,10.175,10.851,10.282,12.595,12.961,12.718,12.426,13.133,12.886,12.471,12.224,13.126,12.917,12.98,12.897,12.972,12.538,13.355,12.487,4.168,4.38,3.884,3.788,4.272,4.157,3.533,3.471,4.018,3.81,3.576,3.704,4.347,3.931,3.832,2.852,6.487,6.268,6.228,6.304,6.436,6.119,5.859,5.828,6.188,5.991,5.918,6.136,6.604,6.38,5.355,7.595,8.116,8.273,7.831,8.582,8.348,7.929,7.982,8.853,8.412,8.177,8.58,8.104,7.701,8.873,8.508,15.326,15.577,16.08,16.322,15.253,15.207,15.387,15.144,14.588,15.883,15.633,14.241,14.068,14.717,14.48,14.696,15.371,14.428,16.002,14.262,11.66,11.996,12.074,11.916,12.376,12.708,12.286,12.078,12.551,12.473,12.43,12.866,12.167,11.908,12.689,12.944,6.234,6.928,6.543,6.803,6.721,6.858,6.837,7.03,7.102,7.219,7.177,7.542,6.358,7.247,7.328,4.03,4.542,4.271,4.39,5.126,4.762,4.294,4.177,5.17,4.671,4.501,4.939,4.755,3.97,5.089,4.463,13.197,13.062,12.908,13.413,13.472,12.952,13.685,13.888,13.392,12.812,13.684],[0.607,0.872,0.653,0.936,1.154,1.294,null,null,1.326,1.322,1.279,1.79,0.647,1.001,1.416,1.587,8.557,8.896,8.612,8.638,9.259,9.194,8.623,8.475,9.379,9.131,8.882,9.37,8.787,8.189,9.575,8.94,9.702,9.994,9.736,9.42,10.079,10.069,9.302,9.222,9.833,9.706,9.389,9.689,10.332,9.428,9.909,9.226,2.791,2.839,2.752,3.221,3.077,3.448,7.804,3.441,2.755,3.529,5.055,4.909,5.049,5.386,5.217,5.279,5.564,5.818,5.498,5.462,5.831,5.699,4.89,5.616,5.432,6.415,9.485,9.887,9.705,9.469,10.704,10.457,9.745,9.594,10.438,10.32,10.126,10.552,9.867,9.274,10.635,10.384,10.264,10.132,10.035,9.746,10.286,10.016,9.418,9.292,9.924,9.613,9.414,9.708,10.295,9.837,10.176,9.638,12.183,12.453,12.588,12.089,12.85,12.696,12.437,12.307,12.924,12.838,12.945,13.085,12.334,11.923,13.13,12.58,9.444,10.219,9.93,9.272,10.472,9.915,9.676,9.193,10.224,9.932,9.645,9.909,10.086,9.252,9.607,5.154,4.986,5.508,5.687,4.605,5.028,5.56,6.034,4.915,5.116,5.735,5.382,5.061,5.562,5.08,5.7,8.429,8.69,8.588,8.572,9.044,8.981,8.675,8.419,9.243,9.116,8.889,9.424,8.596,8.204,9.404,8.94,5.982,6.402,6.233,6.16,6.911,6.649,6.422,6.038,7.326,6.662,6.56,7.318,6.403,5.806,7.031,6.586,6.875,6.965,6.673,6.409,7.287,7.474,6.58,6.319,7.068,6.818,7.182,7.355,7.165,7.259,6.689,12.56,15.432,12.722,12.257,13.357,12.688,12.268,11.905,12.719,12.336,12.221,12.283,12.99,12.095,12.73,12.003,4.356,4.914,4.41,4.386,5.51,5.667,4.818,4.45,5.083,5.102,4.874,4.997,5.169,4.244,5.227,5.19,4.581,4.885,4.866,4.857,5.203,5.118,5.106,5.48,5.323,5.398,5.709,4.683,4.804,5.89,2.364,2.973,2.458,2.336,3.21,2.939,2.299,2.0,2.988,2.649,2.688,2.786,2.106,3.041,2.604,8.483,8.749,8.437,8.095,8.781,8.485,8.011,7.804,8.668,8.761,8.058,8.48,8.924,8.092,8.775,8.13,7.495,8.597,7.759,7.348,8.238,7.639,7.179,6.722,7.669,7.358,7.044,7.39,7.717,7.224,7.856,7.441,6.186,6.705,6.432,6.353,7.203,6.909,6.664,6.709,7.008,6.926,6.97,7.407,6.504,6.186,7.104,7.13,7.19,7.403,7.423,7.627,6.901,7.09,7.756,8.169,7.242,7.489,8.092,7.44,6.836,7.939,7.243,8.077,12.348,12.407,12.172,11.94,13.075,12.468,11.817,11.542,12.82,12.156,11.908,12.502,12.607,12.124,12.723,11.926,15.581,15.677,15.436,15.261,15.707,15.583,15.099,15.228,14.958,14.738,15.856,15.295,5.52,6.019,5.869,5.662,6.476,6.249,5.915,5.689,6.476,6.355,6.214,6.591,6.329,5.854,7.07,7.225,7.071,7.0,7.513,7.358,6.908,6.659,7.554,7.53,7.158,7.613,7.784,6.612,8.147,7.406,10.473,10.77,10.679,10.295,11.133,10.895,10.375,10.37,10.779,10.62,10.46,10.484,10.754,10.4,10.971,10.299,9.292,9.68,9.433,9.253,10.134,10.024,9.482,9.204,10.109,10.381,9.615,9.891,9.654,9.295,10.253,9.684,11.997,12.363,12.12,11.828,12.535,12.288,11.873,11.626,12.528,12.319,12.382,12.299,12.374,11.94,12.757,11.889,2.927,3.139,2.643,2.547,3.139,2.916,2.292,2.23,2.835,2.627,2.335,2.681,3.106,2.69,2.909,1.956,5.685,5.466,5.426,5.502,5.634,5.317,5.057,5.026,5.386,5.189,5.116,5.334,5.781,5.578,4.553,7.062,7.583,7.74,7.439,8.013,7.815,7.537,7.815,8.284,7.879,7.785,8.132,7.535,7.309,8.304,8.231,14.728,14.979,15.482,15.724,14.655,14.609,14.754,14.546,13.99,15.25,15.0,13.643,13.47,14.119,13.882,14.098,14.738,13.83,15.369,13.664,11.062,11.398,11.476,11.318,11.778,12.11,11.688,11.48,11.953,11.875,11.832,12.268,11.569,11.31,12.091,12.346,6.59,7.093,6.899,7.159,6.872,7.009,7.193,7.386,7.294,7.506,7.529,7.735,6.714,7.439,7.678,3.432,3.944,3.673,3.792,4.528,4.164,3.725,3.785,4.572,4.073,3.932,4.341,4.157,3.372,4.491,3.924,12.664,12.529,12.5,12.88,12.976,12.544,13.277,13.48,12.984,12.335,13.276],[3.142,2.442,2.746,3.232,2.39,2.661,null,null,2.511,2.761,3.032,3.289,2.66,3.297,2.527,3.34,7.399,7.649,7.454,7.48,7.72,7.947,7.465,7.386,8.221,7.973,7.724,8.242,7.54,7.1,8.167,7.782,6.869,7.161,6.903,6.587,7.246,7.236,6.469,6.389,7.0,6.873,6.556,6.856,7.499,6.595,7.076,6.393,4.544,4.347,4.505,4.53,4.83,4.757,9.557,5.194,4.064,4.838,6.808,6.662,6.802,7.139,6.97,7.032,7.317,7.681,7.251,7.215,7.694,7.562,6.643,7.369,7.162,8.168,9.473,9.924,9.745,9.944,10.718,10.456,10.236,10.15,10.502,10.384,10.617,10.616,9.885,9.83,10.654,10.582,7.353,7.221,7.124,6.835,7.375,7.105,6.507,6.381,7.013,6.702,6.503,6.797,7.384,6.926,7.265,6.727,12.202,12.472,12.609,12.147,12.869,12.717,12.495,12.365,12.945,12.859,13.12,13.106,12.353,11.944,13.151,12.784,7.842,8.617,8.328,7.67,8.87,8.313,8.074,7.591,8.622,8.33,8.043,8.307,8.484,7.65,8.005,6.907,6.739,7.261,7.44,6.358,6.781,7.313,7.787,6.668,6.869,7.488,7.135,6.814,7.315,6.833,7.453,8.041,8.302,8.2,8.184,8.656,8.593,8.38,8.128,8.855,8.79,8.614,9.036,8.208,7.816,9.016,8.675,5.853,6.014,6.367,6.687,6.523,6.261,6.556,6.565,7.096,6.796,7.087,7.36,6.015,6.139,6.643,7.113,4.042,4.132,3.84,3.576,4.454,4.641,3.747,3.486,4.235,3.985,4.349,4.522,4.332,4.426,3.856,9.649,12.599,9.811,9.358,10.524,9.855,9.369,9.006,9.886,9.503,9.388,9.45,10.157,9.184,9.897,9.17,1.445,2.003,1.577,1.475,2.599,2.756,1.907,1.539,2.172,2.191,1.963,2.086,2.258,1.333,2.316,2.279,5.479,5.744,5.764,5.755,5.73,5.858,6.004,6.378,6.151,6.296,6.607,5.423,5.702,6.417,2.308,2.396,2.331,2.637,2.447,2.812,2.758,2.883,2.895,3.004,3.295,3.313,2.633,2.971,3.502,5.572,5.838,5.526,5.184,5.87,5.574,5.1,4.893,5.757,5.85,5.147,5.647,6.013,5.181,5.864,5.252,4.584,5.686,4.848,4.456,5.327,4.728,4.268,3.811,4.758,4.447,4.133,4.479,4.806,4.332,4.945,4.53,7.084,7.603,7.33,7.779,8.101,7.848,7.747,7.941,7.947,7.992,8.036,8.466,7.402,7.612,8.043,8.196,8.943,9.156,9.176,9.38,8.654,8.843,9.509,9.922,8.995,9.242,9.845,9.193,8.589,9.692,8.996,9.83,9.515,9.574,9.339,9.107,10.242,9.635,8.984,8.709,9.987,9.323,9.075,9.669,9.774,9.291,9.89,9.093,12.748,12.844,12.603,12.428,12.874,12.75,12.266,12.395,12.125,11.905,13.023,12.462,5.132,5.631,5.481,5.274,6.088,5.861,5.527,5.301,6.088,5.967,5.826,6.203,5.941,5.466,5.331,5.446,5.577,5.573,5.911,5.756,5.744,5.495,6.015,6.046,5.994,6.449,6.005,5.237,6.545,6.242,7.562,7.859,7.787,7.409,8.222,7.984,7.489,7.484,7.868,7.709,7.549,7.573,7.843,7.514,8.06,7.413,6.459,6.847,6.6,6.42,7.301,7.191,6.649,6.371,7.276,7.548,6.895,7.307,6.821,6.462,7.533,6.964,10.01,10.472,10.246,9.954,10.893,10.646,10.231,9.984,10.921,10.712,10.78,10.697,10.483,10.051,11.155,10.287,1.224,1.356,1.075,1.462,0.692,0.653,1.194,1.832,0.456,0.72,1.788,0.614,1.151,1.619,0.252,1.296,2.774,2.555,2.515,2.591,2.723,2.406,2.146,2.115,2.475,2.278,2.205,2.423,2.891,2.667,1.642,7.802,8.118,8.48,8.337,8.399,8.522,8.435,8.713,8.553,8.619,8.683,8.872,7.921,8.207,8.573,9.129,13.126,13.377,13.88,14.122,13.053,11.822,11.921,12.006,11.287,12.417,12.167,11.641,11.513,11.966,11.86,12.287,11.905,11.187,12.536,12.022,10.425,10.761,10.839,10.681,11.141,11.473,11.051,10.853,11.316,11.238,11.205,11.641,10.932,10.673,11.454,11.719,8.343,8.519,8.548,8.912,8.298,8.435,8.784,9.139,8.72,8.932,8.955,9.161,8.467,8.865,9.104,3.566,4.078,3.807,4.319,4.392,4.259,4.252,4.683,4.443,4.406,4.459,4.674,4.291,3.873,4.362,4.664,12.287,12.152,12.324,12.503,12.599,12.368,13.101,13.304,12.808,11.958,13.1],[2.952,2.692,2.996,3.281,3.002,3.273,null,null,3.256,3.506,3.622,4.034,2.47,3.346,3.272,3.93,8.551,8.801,8.606,8.632,8.872,9.099,8.617,8.538,9.373,9.125,8.876,9.394,8.692,8.252,9.319,8.934,8.021,8.313,8.055,7.739,8.398,8.388,7.621,7.541,8.152,8.025,7.708,8.008,8.651,7.747,8.228,7.545,5.134,5.092,5.095,5.275,5.42,5.502,10.147,5.784,4.809,5.583,7.398,7.252,7.392,7.729,7.56,7.622,7.907,8.161,7.841,7.805,8.174,8.042,7.233,7.959,7.775,8.758,10.286,10.737,10.558,10.757,11.531,11.269,11.049,10.963,11.315,11.197,11.43,11.429,10.698,10.643,11.467,11.395,8.505,8.373,8.276,7.987,8.527,8.257,7.659,7.533,8.165,7.854,7.655,7.949,8.536,8.078,8.417,7.879,13.015,13.285,13.422,12.96,13.682,13.53,13.308,13.178,13.758,13.672,13.933,13.919,13.166,12.757,13.964,13.597,8.994,9.769,9.48,8.822,10.022,9.465,9.226,8.743,9.774,9.482,9.195,9.459,9.636,8.802,9.157,7.497,7.329,7.851,8.03,6.948,7.371,7.903,8.377,7.258,7.459,8.078,7.725,7.404,7.905,7.423,8.043,8.854,9.115,9.013,8.997,9.469,9.406,9.193,8.941,9.668,9.603,9.427,9.849,9.021,8.629,9.829,9.488,6.666,6.827,7.18,7.5,7.336,7.074,7.369,7.378,7.909,7.609,7.9,8.173,6.828,6.952,7.456,7.926,5.194,5.284,4.992,4.728,5.606,5.793,4.899,4.638,5.387,5.137,5.501,5.674,5.484,5.578,5.008,10.801,13.751,10.963,10.51,11.676,11.007,10.521,10.158,11.038,10.655,10.54,10.602,11.309,10.336,11.049,10.322,2.597,3.155,2.729,2.627,3.751,3.908,3.059,2.691,3.324,3.343,3.115,3.238,3.41,2.485,3.468,3.431,6.28,6.556,6.565,6.556,6.543,6.67,6.805,7.179,6.963,7.097,7.408,6.235,6.503,7.23,3.053,3.209,3.144,3.449,3.26,3.625,3.57,3.695,3.708,3.817,4.108,4.126,3.378,3.784,4.303,6.724,6.99,6.678,6.336,7.022,6.726,6.252,6.045,6.909,7.002,6.299,6.799,7.165,6.333,7.016,6.404,5.736,6.838,6.0,5.608,6.479,5.88,5.42,4.963,5.91,5.599,5.285,5.631,5.958,5.484,6.097,5.682,7.885,8.404,8.131,8.524,8.902,8.649,8.548,8.742,8.748,8.793,8.837,9.267,8.203,8.357,8.844,8.997,9.533,9.746,9.766,9.97,9.244,9.433,10.099,10.512,9.585,9.832,10.435,9.783,9.179,10.282,9.586,10.42,10.667,10.726,10.491,10.259,11.394,10.787,10.136,9.861,11.139,10.475,10.227,10.821,10.926,10.443,11.042,10.245,13.9,13.996,13.755,13.58,14.026,13.902,13.418,13.547,13.277,13.057,14.175,13.614,5.945,6.444,6.294,6.087,6.901,6.674,6.34,6.114,6.901,6.78,6.639,7.016,6.754,6.279,6.483,6.598,6.729,6.725,7.063,6.908,6.896,6.647,7.167,7.198,7.146,7.601,7.157,6.389,7.697,7.394,8.714,9.011,8.939,8.561,9.374,9.136,8.641,8.636,9.02,8.861,8.701,8.725,8.995,8.666,9.212,8.565,7.611,7.999,7.752,7.572,8.453,8.343,7.801,7.523,8.428,8.7,8.047,8.459,7.973,7.614,8.685,8.116,11.162,11.624,11.398,11.106,12.045,11.798,11.383,11.136,12.073,11.864,11.932,11.849,11.635,11.203,12.307,11.439,0.69,1.163,0.668,0.31,1.167,0.944,0.32,0.807,1.076,0.868,1.109,0.922,1.134,0.453,1.15,1.604,3.926,3.707,3.667,3.743,3.875,3.558,3.298,3.267,3.627,3.43,3.357,3.575,4.043,3.819,2.794,8.614,8.931,9.292,9.138,9.212,9.335,9.236,9.514,9.366,9.431,9.484,9.684,8.734,9.008,9.386,9.93,14.278,14.529,15.032,15.274,14.205,12.974,13.073,13.158,12.439,13.569,13.319,12.793,12.665,13.118,13.012,13.439,13.057,12.339,13.688,13.174,11.487,11.823,11.901,11.743,12.203,12.535,12.113,11.905,12.378,12.3,12.257,12.693,11.994,11.735,12.516,12.771,8.933,9.264,9.242,9.502,9.043,9.18,9.529,9.729,9.465,9.677,9.7,9.906,9.057,9.61,9.849,4.379,4.891,4.62,5.132,5.205,5.072,5.065,5.484,5.256,5.219,5.272,5.487,5.104,4.686,5.175,5.476,13.154,13.019,13.182,13.37,13.466,13.226,13.959,14.162,13.666,12.825,13.958],[5.937,5.237,5.541,6.027,5.185,5.456,null,null,5.306,5.556,5.827,5.849,5.455,6.092,5.322,6.135,6.593,6.779,6.807,6.833,6.85,7.077,6.818,6.773,7.431,7.326,7.077,7.595,6.67,6.487,7.297,7.135,4.134,4.537,4.228,3.972,4.896,4.977,4.176,4.152,4.804,4.841,4.319,4.977,4.875,3.826,5.014,4.156,6.787,6.414,6.661,6.299,6.949,6.824,11.753,7.277,6.166,6.905,9.004,8.858,8.998,9.335,9.05,9.228,9.513,9.767,9.399,9.411,9.78,9.648,8.726,9.565,9.216,10.364,10.656,11.012,10.928,11.127,11.144,11.127,11.419,11.333,11.31,11.472,11.8,11.704,10.693,11.013,11.325,11.765,5.137,4.989,4.907,4.618,5.143,4.628,4.275,4.149,4.38,4.225,4.026,4.116,5.152,4.709,4.584,4.046,12.268,12.229,12.541,12.686,12.603,12.553,12.779,12.904,12.781,12.726,13.449,12.973,12.11,12.444,12.987,13.122,6.498,7.381,7.092,6.8,7.802,7.301,7.099,6.721,7.752,7.46,7.173,7.437,7.136,6.667,7.135,9.103,8.935,9.457,9.636,8.554,8.977,9.509,9.983,8.864,9.065,9.684,9.331,9.01,9.511,9.029,9.707,8.712,8.809,8.871,8.988,8.729,9.1,9.184,9.311,9.18,9.461,9.42,9.543,8.59,8.999,9.341,9.622,7.036,7.197,7.554,7.876,7.706,7.444,7.743,7.754,8.279,7.983,8.276,8.543,7.198,7.328,7.826,8.302,2.835,2.663,2.912,2.648,2.986,3.173,2.941,3.448,3.342,3.262,3.421,3.765,2.551,3.319,3.186,6.83,10.237,6.992,6.539,7.947,7.215,6.55,6.187,7.271,6.905,6.63,6.988,7.394,6.365,7.434,6.491,1.784,2.181,2.385,2.18,2.686,3.001,2.768,2.574,2.771,2.896,2.998,3.105,1.965,2.221,2.822,3.314,6.796,6.974,7.081,7.122,6.919,7.088,7.371,7.774,7.381,7.613,7.924,6.653,7.098,7.606,4.195,3.75,4.077,4.383,3.632,4.168,4.504,4.629,4.084,4.193,4.484,4.502,4.523,4.158,5.192,2.753,3.019,2.707,2.365,3.051,2.755,2.281,2.074,2.938,3.031,2.328,3.136,3.194,2.362,3.045,2.433,2.43,2.921,2.252,2.309,2.562,2.132,1.672,1.664,1.93,1.851,1.537,1.651,2.329,2.185,2.117,1.791,8.809,9.102,9.055,9.505,9.497,9.347,9.472,9.667,9.446,9.5,9.598,9.909,8.857,9.343,9.486,9.758,11.197,11.41,11.43,11.634,10.908,11.097,11.763,12.176,11.249,11.496,12.099,11.447,10.843,11.946,11.25,12.084,7.46,7.728,7.427,7.208,8.396,7.789,7.138,6.863,8.141,7.477,7.229,7.976,7.736,7.28,8.044,7.278,10.386,10.482,10.241,10.066,10.512,10.388,9.904,10.033,9.763,9.543,10.661,10.1,5.915,5.839,6.089,6.355,5.98,6.364,6.608,6.484,6.296,6.574,6.848,7.004,5.832,6.547,4.65,4.576,4.707,4.891,5.041,4.886,5.038,4.848,5.145,5.176,5.288,5.743,5.135,4.59,5.675,5.536,4.881,5.178,5.106,4.722,5.403,5.165,4.802,4.797,5.049,4.89,4.73,4.754,5.162,4.827,5.241,4.726,4.839,5.375,5.128,4.948,5.828,5.718,5.195,4.955,5.86,6.132,5.479,5.891,5.322,4.988,6.117,5.548,8.594,9.056,8.83,8.538,9.477,9.23,8.815,8.568,9.505,9.296,9.436,9.531,9.067,8.635,9.809,9.113,4.279,4.411,4.127,4.338,3.747,3.708,3.989,4.627,3.511,3.515,4.583,3.409,4.206,4.481,3.307,4.091,2.191,0.753,0.842,2.008,0.127,0.506,1.165,1.532,0.486,0.794,1.484,1.022,0.752,0.678,1.402,9.032,9.305,9.71,9.595,9.586,9.709,9.693,10.03,9.736,9.849,9.941,10.102,9.108,9.465,9.756,10.387,11.96,12.211,12.714,12.956,11.887,10.3,10.399,10.449,9.794,10.895,10.645,10.148,10.02,10.473,10.367,10.794,10.383,9.694,11.014,10.571,9.741,10.106,10.218,10.068,10.486,10.818,10.438,10.24,10.703,10.625,10.592,11.028,10.226,10.06,10.841,11.106,10.357,10.382,10.497,11.081,10.055,10.249,10.733,11.329,10.477,10.746,10.818,10.918,10.663,10.622,10.967,4.753,5.265,4.994,5.508,5.575,5.442,5.441,6.079,5.626,5.595,5.648,5.863,5.478,5.062,5.545,5.902,11.674,11.539,11.711,11.89,11.986,11.755,12.488,12.691,12.195,11.345,12.487],[3.794,3.085,3.221,3.554,2.669,2.779,null,null,2.627,2.877,3.013,2.671,3.519,3.619,2.495,3.321,5.412,5.751,5.467,5.493,6.114,6.049,5.478,5.33,6.234,5.986,5.737,6.225,5.642,5.044,6.43,5.795,7.091,7.383,7.125,6.809,7.468,7.458,6.691,6.611,7.222,7.095,6.778,7.078,7.721,6.817,7.298,6.615,3.16,2.787,3.034,2.672,3.322,3.197,8.126,3.65,2.539,3.278,5.377,5.231,5.371,5.708,5.423,5.601,5.886,6.14,5.772,5.784,6.153,6.021,5.099,5.938,5.589,6.737,6.716,7.167,6.988,7.187,7.961,7.699,7.479,7.393,7.745,7.627,7.86,7.859,7.128,7.073,7.897,7.825,9.033,8.748,8.771,8.514,8.902,8.359,8.171,8.045,8.056,8.113,7.922,7.792,8.911,8.605,8.26,7.722,9.445,9.715,9.852,9.39,10.112,9.96,9.738,9.608,10.188,10.102,10.363,10.349,9.596,9.187,10.394,10.027,6.299,7.074,6.785,6.127,7.327,6.77,6.531,6.048,7.079,6.787,6.5,6.764,6.941,6.107,6.462,5.476,5.308,5.83,6.009,4.927,5.35,5.882,6.356,5.237,5.438,6.057,5.704,5.383,5.884,5.402,6.08,5.284,5.545,5.443,5.427,5.899,5.836,5.623,5.371,6.098,6.033,5.857,6.279,5.451,5.059,6.259,5.918,3.096,3.257,3.618,4.094,3.766,3.504,3.807,3.972,4.339,4.047,4.494,4.603,3.258,3.476,3.886,4.52,4.264,4.354,4.062,3.798,4.676,4.863,3.975,3.714,4.463,4.213,4.524,4.331,4.554,4.654,3.982,10.077,12.821,10.113,9.646,10.746,10.077,9.657,9.294,10.108,9.725,9.61,9.672,10.379,9.612,10.119,9.392,2.28,2.179,2.806,1.984,2.775,2.776,1.712,1.746,2.143,1.998,1.465,1.891,2.674,2.319,2.287,1.781,3.014,3.192,3.299,3.34,3.137,3.306,3.589,3.992,3.599,3.831,4.142,2.871,3.316,3.824,1.381,0.864,1.263,1.567,0.44,1.282,1.688,1.805,0.89,0.999,1.169,0.894,1.707,0.493,1.584,6.148,6.395,6.083,5.76,6.427,6.131,5.657,5.469,6.314,6.226,5.638,5.869,6.57,5.76,6.421,5.519,6.283,6.597,6.148,6.155,6.238,6.028,5.568,5.51,5.606,5.747,5.433,5.327,6.225,6.031,5.793,5.467,5.182,5.442,5.428,5.878,5.715,5.72,5.845,6.04,5.717,5.873,5.971,6.127,5.075,5.716,5.704,6.131,7.57,7.783,7.803,8.007,7.281,7.47,8.136,8.549,7.622,7.869,8.472,7.82,7.216,8.319,7.623,8.457,9.737,9.796,9.561,9.329,10.464,9.857,9.206,8.931,10.209,9.545,9.297,9.897,9.996,9.513,10.112,9.321,12.97,13.066,12.825,12.65,13.096,12.972,12.488,12.617,12.347,12.127,13.245,12.684,2.375,2.874,2.724,2.517,3.331,3.104,2.77,2.544,3.331,3.21,3.069,3.446,3.184,2.709,3.925,4.08,3.926,3.855,4.368,4.213,3.763,3.514,4.409,4.385,4.013,4.468,4.639,3.467,5.002,4.261,8.492,8.728,8.549,8.165,8.779,8.541,8.245,8.24,8.425,8.266,8.106,8.13,8.744,8.27,8.578,8.169,6.687,7.075,6.828,6.648,7.404,7.137,6.527,6.249,7.069,7.279,6.47,6.746,7.049,6.69,7.108,6.539,8.852,9.218,8.975,8.683,9.39,9.143,8.728,8.481,9.383,9.174,9.237,9.154,9.229,8.795,9.612,8.744,3.932,4.065,3.569,3.78,3.419,3.38,3.431,3.786,3.165,2.957,3.742,2.851,3.878,3.923,2.979,2.714,4.473,4.254,4.214,4.29,4.023,4.105,3.845,3.814,3.951,3.578,3.904,3.268,4.59,3.879,3.341,5.25,5.369,5.928,5.813,5.65,5.773,5.911,6.248,5.796,6.044,6.159,6.193,5.172,5.683,5.816,6.605,11.583,11.834,12.337,12.579,11.51,11.464,11.644,11.401,10.845,12.14,11.89,10.498,10.325,10.974,10.737,10.953,11.628,10.685,12.259,10.519,7.917,8.253,8.331,8.173,8.633,8.965,8.543,8.335,8.808,8.73,8.687,9.123,8.424,8.165,8.946,9.201,6.73,6.755,6.87,7.454,6.428,6.622,7.106,7.702,6.85,7.119,7.191,7.291,7.036,6.995,7.34,0.817,1.329,1.058,1.726,1.635,1.502,1.639,2.297,1.686,1.743,1.866,2.011,1.542,1.21,1.605,2.12,9.584,9.449,9.612,9.8,9.896,9.656,10.389,10.592,10.096,9.255,10.388],[3.963,3.104,3.408,3.894,2.854,3.125,null,null,2.975,3.225,3.467,3.125,3.481,3.959,2.893,3.775,6.743,7.082,6.798,6.824,7.445,7.38,6.809,6.661,7.565,7.317,7.068,7.556,6.973,6.375,7.761,7.126,6.838,7.13,6.872,6.556,7.215,7.205,6.438,6.358,6.969,6.842,6.525,6.825,7.468,6.564,7.045,6.362,4.082,3.71,3.957,3.595,4.245,4.12,9.049,4.573,3.462,4.201,6.3,6.154,6.294,6.631,6.346,6.524,6.809,7.063,6.695,6.707,7.076,6.944,6.022,6.861,6.512,7.66,8.047,8.498,8.319,8.518,9.292,9.03,8.81,8.724,9.076,8.958,9.191,9.19,8.459,8.404,9.228,9.156,7.773,7.625,7.543,7.254,7.779,7.264,6.911,6.785,7.172,6.861,6.662,6.956,7.788,7.345,7.424,6.886,10.776,11.046,11.183,10.721,11.443,11.291,11.069,10.939,11.519,11.433,11.694,11.68,10.927,10.518,11.725,11.358,7.63,8.405,8.116,7.458,8.658,8.101,7.862,7.379,8.41,8.118,7.831,8.095,8.272,7.438,7.793,6.399,6.231,6.753,6.932,5.85,6.273,6.805,7.279,6.16,6.361,6.98,6.627,6.306,6.807,6.325,7.003,6.615,6.876,6.774,6.758,7.23,7.167,6.954,6.702,7.429,7.364,7.188,7.61,6.782,6.39,7.59,7.249,4.427,4.588,4.945,5.267,5.097,4.835,5.134,5.145,5.67,5.374,5.667,5.934,4.589,4.719,5.217,5.693,4.011,4.101,3.809,3.545,4.423,4.61,3.716,3.455,4.204,3.954,4.318,4.248,4.301,4.395,3.825,9.687,12.568,9.849,9.393,10.493,9.824,9.404,9.041,9.855,9.472,9.357,9.419,10.126,9.222,9.866,9.139,1.473,2.027,1.546,1.199,2.623,2.665,1.631,0.961,2.036,1.915,1.385,1.81,2.289,1.059,2.18,1.701,4.148,4.326,4.433,4.474,4.31,4.44,4.723,5.131,4.733,4.965,5.276,4.005,4.455,4.997,1.471,0.975,1.353,1.659,1.023,1.393,1.78,1.905,1.475,1.584,1.875,1.893,1.799,1.549,2.524,5.61,5.876,5.564,5.222,5.908,5.612,5.138,4.931,5.795,5.888,5.185,5.616,6.051,5.219,5.902,5.266,5.05,5.789,4.888,4.922,5.43,4.768,4.308,4.277,4.798,4.487,4.173,4.519,4.965,4.798,4.985,4.57,6.106,6.493,6.352,6.801,6.854,6.738,6.769,6.963,6.837,6.891,6.989,7.266,6.214,6.639,6.843,7.149,8.493,8.706,8.726,8.93,8.204,8.393,9.059,9.472,8.545,8.792,9.395,8.743,8.139,9.242,8.546,9.38,9.484,9.543,9.308,9.076,10.211,9.604,8.953,8.678,9.956,9.292,9.044,9.638,9.743,9.26,9.859,9.062,12.717,12.813,12.572,12.397,12.843,12.719,12.235,12.364,12.094,11.874,12.992,12.431,3.706,4.205,4.055,3.848,4.662,4.435,4.101,3.875,4.662,4.541,4.4,4.777,4.515,4.04,5.055,5.241,5.257,5.186,5.699,5.544,5.094,4.845,5.74,5.716,5.344,5.799,5.8,4.798,6.333,5.592,7.721,8.018,7.946,7.59,8.26,8.022,7.67,7.665,7.906,7.747,7.587,7.611,8.002,7.695,8.098,7.594,6.428,6.816,6.569,6.389,7.27,7.054,6.444,6.166,7.071,7.343,6.69,7.102,6.79,6.431,7.328,6.759,9.805,10.267,10.041,9.749,10.688,10.441,10.026,9.779,10.714,10.505,10.568,10.485,10.278,9.846,10.943,10.075,2.702,2.835,2.339,2.55,2.189,2.15,2.201,2.839,1.935,1.727,2.795,1.621,2.648,2.693,1.749,1.958,3.24,3.021,2.981,3.057,2.763,2.872,2.612,2.581,2.941,2.318,2.671,2.461,3.357,3.072,2.108,6.384,6.696,7.062,6.947,6.977,7.1,7.045,7.382,7.127,7.201,7.293,7.454,6.499,6.817,7.147,7.739,12.914,13.165,13.668,13.91,12.841,11.646,11.745,11.872,11.101,12.241,11.991,11.455,11.327,11.78,11.674,12.101,11.729,11.001,12.36,11.817,9.248,9.584,9.662,9.504,9.964,10.296,9.874,9.666,10.139,10.061,10.018,10.454,9.755,9.496,10.277,10.532,7.653,7.678,7.793,8.377,7.352,7.545,8.029,8.625,7.774,8.042,8.114,8.215,7.959,7.919,8.263,2.144,2.656,2.385,2.899,2.966,2.833,2.832,3.436,3.017,2.986,3.039,3.254,2.869,2.453,2.936,3.246,10.915,10.78,10.943,11.131,11.227,10.987,11.72,11.923,11.427,10.586,11.719],[4.866,4.166,4.47,4.819,3.934,4.044,null,null,3.892,4.142,4.278,3.936,4.384,4.884,3.76,4.586,6.086,6.407,6.141,6.167,6.478,6.705,6.152,6.073,6.908,6.66,6.411,6.929,6.298,5.787,6.925,6.469,5.971,6.263,6.005,5.689,6.348,6.338,5.571,5.491,6.102,5.975,5.658,5.958,6.601,5.697,6.178,5.495,4.33,3.957,4.204,3.842,4.492,4.367,9.296,4.82,3.709,4.448,6.547,6.401,6.541,6.878,6.593,6.771,7.056,7.31,6.942,6.954,7.323,7.191,6.269,7.108,6.759,7.907,7.886,8.337,8.158,8.357,9.131,8.869,8.649,8.563,8.915,8.797,9.03,9.029,8.298,8.243,9.067,8.995,8.156,7.868,7.891,7.64,8.022,7.479,7.3,7.174,7.176,7.233,7.051,6.912,8.031,7.731,7.38,6.842,10.615,10.885,11.022,10.56,11.282,11.13,10.908,10.778,11.358,11.272,11.533,11.519,10.766,10.357,11.564,11.197,6.6,7.375,7.086,6.428,7.628,7.071,6.832,6.349,7.38,7.088,6.801,7.065,7.242,6.408,6.763,6.646,6.478,7.0,7.179,6.097,6.52,7.052,7.526,6.407,6.608,7.227,6.874,6.553,7.054,6.572,7.25,6.454,6.715,6.613,6.597,7.069,7.006,6.793,6.541,7.268,7.203,7.027,7.449,6.621,6.229,7.429,7.088,4.266,4.427,4.788,5.264,4.936,4.674,4.977,5.142,5.509,5.217,5.664,5.773,4.428,4.646,5.056,5.69,3.144,3.234,2.942,2.678,3.556,3.743,2.855,2.594,3.343,3.093,3.404,3.211,3.434,3.534,2.862,8.957,11.701,8.993,8.526,9.626,8.957,8.537,8.174,8.988,8.605,8.49,8.552,9.259,8.492,8.999,8.272,1.4,1.059,1.947,1.104,1.655,1.656,0.592,0.866,1.023,0.878,0.345,0.771,1.617,1.46,1.167,0.031,4.184,4.362,4.469,4.51,4.307,4.476,4.759,5.162,4.769,5.001,5.312,4.041,4.486,4.994,2.664,2.166,2.546,2.851,1.764,2.584,2.972,3.07,2.216,2.296,2.434,2.159,2.991,1.842,2.849,5.028,5.275,4.963,4.64,5.307,5.011,4.537,4.349,5.194,5.106,4.518,4.749,5.45,4.64,5.301,4.399,5.451,5.717,5.289,5.323,5.358,5.169,4.709,4.678,4.726,4.888,4.574,4.447,5.355,5.199,4.913,4.587,6.352,6.612,6.598,7.048,6.885,6.89,7.015,7.21,6.887,7.043,7.141,7.297,6.245,6.886,6.874,7.301,8.74,8.953,8.973,9.177,8.451,8.64,9.306,9.719,8.792,9.039,9.642,8.99,8.386,9.489,8.793,9.627,8.617,8.676,8.441,8.209,9.344,8.737,8.086,7.811,9.089,8.425,8.177,8.777,8.876,8.393,8.992,8.201,11.85,11.946,11.705,11.53,11.976,11.852,11.368,11.497,11.227,11.007,12.125,11.564,3.545,4.044,3.894,3.687,4.501,4.274,3.94,3.714,4.501,4.38,4.239,4.616,4.354,3.879,4.018,4.204,4.335,4.26,4.669,4.514,4.431,4.182,4.773,4.804,4.681,5.136,4.763,3.924,5.303,4.929,7.372,7.608,7.429,7.045,7.659,7.421,7.125,7.12,7.305,7.146,6.986,7.01,7.624,7.15,7.458,7.049,5.567,5.955,5.708,5.528,6.284,6.017,5.407,5.129,6.034,6.306,5.653,6.065,5.929,5.57,6.291,5.722,8.768,9.23,9.004,8.712,9.651,9.404,8.989,8.742,9.679,9.47,9.538,9.455,9.241,8.809,9.913,9.045,3.365,3.497,3.056,3.267,2.833,2.794,2.918,3.556,2.597,2.444,3.512,2.338,3.292,3.41,2.393,3.02,3.641,3.422,3.382,3.458,3.164,3.273,3.013,2.982,3.071,2.719,3.072,2.388,3.758,2.999,2.509,6.42,6.539,7.098,6.983,6.82,6.943,7.081,7.418,6.966,7.214,7.329,7.363,6.342,6.853,6.986,7.775,11.884,12.135,12.638,12.88,11.811,10.609,10.708,10.835,10.064,11.204,10.954,10.418,10.29,10.743,10.637,11.064,10.692,9.964,11.323,10.78,9.087,9.423,9.501,9.343,9.803,10.135,9.713,9.505,9.978,9.9,9.857,10.293,9.594,9.335,10.116,10.371,7.9,7.925,8.04,8.624,7.598,7.792,8.276,8.872,8.02,8.289,8.361,8.461,8.206,8.165,8.51,1.987,2.499,2.228,2.896,2.805,2.672,2.809,3.467,2.856,2.913,3.036,3.181,2.712,2.38,2.775,3.29,10.754,10.619,10.782,10.97,11.066,10.826,11.559,11.762,11.266,10.425,11.558],[3.591,2.857,3.018,3.351,2.465,2.576,null,null,2.424,2.674,2.808,2.466,3.249,3.416,2.292,3.116,6.034,6.389,6.089,6.115,6.9,6.687,6.1,5.936,6.856,6.608,6.359,6.677,6.28,5.652,6.94,6.304,8.068,8.36,8.102,7.786,8.445,8.435,7.668,7.588,8.199,8.072,7.755,8.055,8.698,7.794,8.275,7.592,2.411,2.038,2.285,1.923,2.573,2.448,7.377,2.901,1.79,2.529,4.628,4.482,4.622,4.959,4.674,4.852,5.137,5.391,5.023,5.035,5.404,5.272,4.35,5.189,4.84,5.988,6.924,7.326,7.144,6.908,8.143,7.896,7.184,7.033,7.877,7.759,7.565,7.991,7.306,6.713,8.074,7.823,10.29,10.009,10.032,9.771,10.163,9.62,9.428,9.302,9.317,9.374,9.179,9.053,10.172,9.862,9.521,8.983,9.622,9.892,10.027,9.528,10.289,10.135,9.876,9.746,10.363,10.277,10.384,10.524,9.773,9.362,10.569,10.019,7.107,7.882,7.593,6.935,8.135,7.578,7.339,6.856,7.887,7.595,7.308,7.572,7.749,6.915,7.27,4.727,4.559,5.081,5.26,4.178,4.601,5.133,5.607,4.488,4.689,5.308,4.955,4.634,5.135,4.653,5.331,5.736,5.997,5.895,5.879,6.351,6.288,5.959,5.703,6.55,6.4,6.173,6.711,5.903,5.511,6.711,6.224,3.266,3.709,3.517,3.345,4.218,3.956,3.706,3.223,4.61,3.946,3.745,4.602,3.71,3.041,4.338,3.771,5.241,5.331,5.039,4.775,5.653,5.84,4.967,4.707,5.513,5.263,5.442,5.249,5.531,5.704,4.9,11.054,13.798,11.09,10.623,11.723,11.054,10.634,10.271,11.085,10.702,10.587,10.649,11.356,10.589,11.096,10.369,3.541,3.641,4.063,3.245,3.752,3.751,3.174,3.007,3.442,3.191,2.927,3.353,3.965,3.576,3.244,2.994,2.265,2.443,2.55,2.591,2.388,2.557,2.84,3.243,2.85,3.082,3.393,2.122,2.567,3.075,2.007,1.806,1.731,1.565,1.571,1.772,1.686,1.426,1.349,1.01,1.177,0.695,1.705,1.072,1.205,7.125,7.372,7.06,6.737,7.404,7.108,6.634,6.446,7.291,7.203,6.615,6.846,7.547,6.737,7.398,6.496,7.54,7.858,7.405,7.412,7.499,7.285,6.825,6.767,6.867,7.004,6.69,6.588,7.482,7.288,7.054,6.728,4.433,4.693,4.679,5.129,4.966,4.971,5.096,5.291,4.968,5.124,5.222,5.378,4.326,4.967,4.955,5.382,6.821,7.034,7.054,7.258,6.532,6.721,7.387,7.8,6.873,7.12,7.723,7.071,6.467,7.57,6.874,7.708,10.714,10.773,10.538,10.306,11.441,10.834,10.183,9.908,11.186,10.522,10.274,10.947,10.973,10.49,11.089,10.323,13.947,14.043,13.802,13.627,14.073,13.949,13.465,13.594,13.324,13.104,14.222,13.661,2.998,3.61,3.347,2.969,3.96,3.686,3.222,2.996,3.96,3.724,3.521,3.898,3.92,3.161,4.733,4.888,4.734,4.663,5.176,5.021,4.571,4.322,5.217,5.193,4.821,5.276,5.447,4.275,5.81,5.069,9.469,9.705,9.526,9.142,9.756,9.518,9.222,9.217,9.402,9.243,9.083,9.107,9.721,9.247,9.555,9.146,7.737,8.125,7.878,7.69,8.322,8.055,7.445,7.167,7.877,8.087,7.278,7.554,8.099,7.74,7.916,7.347,9.66,10.026,9.783,9.491,10.198,9.951,9.536,9.289,10.191,9.982,10.045,9.962,10.037,9.603,10.42,9.552,5.189,5.322,4.826,4.892,4.676,4.637,4.635,4.575,4.422,4.214,4.678,4.108,5.135,5.035,4.236,3.949,5.73,5.511,5.471,5.547,5.28,5.362,5.102,5.071,5.212,4.835,5.161,4.529,5.847,5.14,4.598,4.501,4.917,5.179,5.064,5.198,5.254,5.162,5.499,5.469,5.318,5.41,5.571,4.72,4.934,5.489,5.856,12.391,12.642,13.145,13.387,12.318,12.272,12.452,12.209,11.653,12.948,12.698,11.306,11.133,11.782,11.545,11.761,12.436,11.493,13.067,11.327,8.369,8.705,8.783,8.625,9.085,9.417,8.995,8.787,9.26,9.182,9.139,9.575,8.876,8.617,9.398,9.653,5.981,6.006,6.121,6.705,5.679,5.873,6.357,6.953,6.101,6.37,6.442,6.542,6.287,6.246,6.591,1.463,1.363,1.096,0.977,1.812,1.448,0.91,1.548,1.856,1.308,1.117,1.576,1.576,0.557,1.775,1.371,10.036,9.901,9.939,10.252,10.348,9.983,10.716,10.919,10.423,9.707,10.715],[4.739,4.03,4.166,4.499,3.614,3.724,null,null,3.572,3.822,3.958,3.616,4.464,4.564,3.44,4.266,4.516,4.871,4.571,4.597,5.382,5.169,4.582,4.418,5.338,5.09,4.841,5.149,4.762,4.134,5.412,4.776,7.674,7.908,7.708,7.392,7.953,7.943,7.216,7.136,7.707,7.58,7.303,7.563,8.246,7.4,7.783,7.14,3.56,2.788,3.035,2.392,3.048,2.691,7.285,2.803,2.781,2.614,4.536,4.39,4.53,4.867,4.558,4.758,5.045,5.299,4.907,4.941,5.312,5.178,4.234,5.097,4.724,5.896,5.44,5.891,5.712,5.802,6.732,6.47,6.094,6.008,6.469,6.351,6.475,6.583,5.852,5.688,6.666,6.549,10.734,10.417,10.447,10.218,10.571,10.033,9.896,9.833,9.732,9.811,9.637,9.306,10.58,10.309,9.576,9.488,8.214,8.484,8.619,8.12,8.881,8.727,8.468,8.338,8.955,8.869,9.093,9.116,8.365,7.954,9.161,8.757,5.91,6.685,6.396,5.738,6.938,6.381,6.142,5.659,6.69,6.398,6.111,6.375,6.552,5.718,6.073,4.635,4.467,4.989,5.168,4.086,4.509,5.041,5.515,4.396,4.597,5.216,4.863,4.542,5.043,4.561,5.239,4.15,4.411,4.309,4.29,4.765,4.685,4.347,4.095,4.964,4.808,4.581,5.126,4.317,3.922,5.125,4.642,1.654,2.121,2.239,2.72,2.635,2.368,2.428,2.598,2.998,2.668,3.12,3.161,2.127,2.1,2.752,3.146,4.904,4.937,4.702,4.438,5.089,5.168,4.288,4.027,4.684,4.434,4.481,4.282,5.137,4.808,3.95,10.562,13.182,10.501,10.131,11.107,10.438,10.018,9.779,10.469,10.086,9.971,10.033,10.74,10.097,10.48,9.753,4.068,3.77,4.594,3.772,3.729,3.728,3.303,3.534,3.419,3.168,3.056,3.379,4.328,4.107,3.221,2.971,1.64,1.818,1.925,1.966,1.763,1.932,2.215,2.993,2.225,2.457,2.768,1.497,1.989,2.45,2.788,2.338,2.512,2.531,2.099,2.553,2.652,2.73,2.025,1.976,2.114,1.839,2.671,1.628,2.509,6.792,7.035,6.723,6.404,7.067,6.771,6.297,6.113,6.954,6.809,6.278,6.452,7.21,6.404,7.061,6.159,8.071,8.385,7.936,7.943,8.026,7.816,7.356,7.298,7.394,7.535,7.221,7.115,8.013,7.819,7.581,7.255,4.315,4.446,4.561,5.013,4.553,4.768,4.978,5.175,4.721,5.006,5.104,5.127,4.079,4.851,4.697,5.264,6.729,6.942,6.962,7.166,6.44,6.629,7.295,7.708,6.781,7.028,7.631,6.979,6.375,7.478,6.782,7.616,10.098,10.157,9.922,9.69,10.825,10.218,9.567,9.292,10.57,9.906,9.658,10.007,10.357,9.874,10.33,9.431,13.331,13.427,13.186,13.011,13.457,13.333,12.849,12.978,12.708,12.488,13.606,13.045,1.632,2.244,1.981,1.603,2.442,2.168,1.704,1.413,2.442,2.206,2.003,2.38,2.554,1.795,3.638,3.793,3.566,3.51,4.008,3.853,3.403,3.099,3.889,3.865,3.493,3.94,4.352,3.18,4.529,3.733,9.136,9.368,9.193,8.809,9.419,9.181,8.889,8.884,9.065,8.906,8.746,8.77,9.384,8.914,9.161,8.813,6.797,7.112,6.865,6.595,7.227,6.96,6.35,6.072,6.782,6.992,6.183,6.459,7.159,6.793,6.821,6.252,8.565,8.879,8.586,8.294,9.001,8.754,8.339,8.092,8.994,8.785,8.848,8.765,8.942,8.443,9.223,8.355,5.72,5.853,5.357,5.568,5.207,5.168,5.219,5.508,4.953,4.745,5.53,4.639,5.666,5.711,4.767,4.502,6.261,6.042,6.002,6.078,5.811,5.893,5.633,5.602,5.739,5.366,5.692,5.056,6.378,5.603,5.129,3.876,3.99,4.554,4.439,4.271,4.394,4.537,4.874,4.411,4.659,4.785,4.808,3.793,4.309,4.431,5.231,11.194,11.445,11.948,12.19,11.121,11.177,11.357,11.114,10.558,11.853,11.603,10.211,10.038,10.687,10.348,10.564,11.341,10.398,11.972,10.13,6.841,7.177,7.255,7.097,7.557,7.889,7.467,7.201,7.732,7.654,7.553,7.989,7.348,7.089,7.87,8.067,5.865,5.89,6.005,6.589,5.561,5.757,6.241,6.837,5.983,6.252,6.326,6.424,6.193,6.128,6.475,1.549,0.84,1.277,1.124,0.849,0.485,0.907,1.364,0.244,1.011,1.214,0.636,1.053,1.291,0.163,1.303,8.45,8.315,8.478,8.666,8.762,8.522,9.255,9.458,8.962,8.121,9.254],[7.211,6.502,6.638,6.971,6.086,6.196,null,null,6.044,6.294,6.43,6.088,6.936,7.036,5.912,6.738,2.979,3.334,3.034,3.06,3.47,3.234,2.694,2.362,2.97,2.721,2.472,2.737,3.225,2.597,3.0,2.364,8.995,9.08,9.029,8.713,9.042,9.032,8.388,8.308,8.796,8.669,8.475,8.652,9.418,8.721,8.872,8.312,6.04,5.268,5.515,4.872,5.528,5.024,8.269,5.13,5.261,4.947,4.709,4.367,4.403,4.623,4.101,4.048,4.599,5.04,3.959,4.302,4.612,4.339,4.781,4.853,3.776,4.974,3.161,3.612,3.433,3.569,4.41,4.148,3.861,3.775,4.19,4.072,4.242,4.304,3.573,3.455,4.346,4.27,12.079,11.762,11.792,11.563,11.916,11.378,11.241,11.221,11.077,11.156,10.982,10.651,11.925,11.654,10.921,10.833,5.894,6.164,6.301,5.839,6.561,6.409,6.187,6.057,6.637,6.551,6.812,6.798,6.045,5.636,6.843,6.476,6.508,6.811,6.63,5.974,6.748,6.34,6.043,5.67,6.252,6.176,5.852,5.756,6.882,5.954,5.524,6.381,5.922,6.59,6.599,5.655,5.866,6.472,6.736,5.622,5.905,6.2,5.661,6.308,6.734,5.433,6.223,1.733,1.994,1.892,1.872,2.348,2.285,2.068,1.816,2.547,2.482,2.302,2.728,1.9,1.504,2.708,2.363,1.053,0.643,0.943,1.518,0.884,0.383,0.867,1.746,0.58,0.742,1.289,0.93,1.21,1.512,0.335,1.315,6.249,6.258,6.047,5.783,6.41,6.489,5.633,5.372,6.01,5.779,5.802,5.603,6.458,6.129,5.271,11.651,14.271,11.59,11.22,12.196,11.527,11.107,10.868,11.558,11.175,11.06,11.122,11.829,11.186,11.569,10.842,6.448,6.148,6.975,6.152,5.97,5.969,5.681,5.914,5.66,5.409,5.401,5.62,6.706,6.488,5.462,5.212,3.107,2.84,2.977,3.306,2.14,2.346,2.828,3.189,2.339,2.702,2.19,2.89,3.637,1.89,5.264,4.747,4.992,5.003,4.508,5.033,5.124,5.21,4.505,4.448,4.586,4.311,5.143,4.108,4.989,8.137,8.38,8.068,7.749,8.412,8.116,7.642,7.458,8.299,8.13,7.623,7.773,8.555,7.749,8.406,7.504,10.399,10.117,10.208,10.321,9.758,10.088,9.68,9.679,9.374,9.859,9.57,9.42,10.23,10.197,9.347,9.56,3.357,2.949,3.069,3.521,3.048,3.268,3.484,3.683,3.221,3.511,3.609,3.624,2.584,3.36,3.192,3.769,7.446,7.508,7.528,7.704,6.854,7.145,7.833,8.146,7.195,7.442,8.045,7.334,7.092,8.154,7.196,7.971,10.947,10.909,10.674,10.443,11.556,10.949,10.298,10.023,11.103,10.439,10.191,10.526,11.109,10.656,10.849,9.95,14.277,14.359,14.132,13.957,14.361,14.237,13.795,13.882,13.654,13.421,14.538,13.949,2.638,2.786,2.61,2.506,2.594,2.32,2.094,1.807,2.105,2.116,1.742,1.846,2.776,2.303,4.959,5.069,4.75,4.699,5.192,4.827,4.592,4.034,4.764,4.614,4.428,4.14,5.571,4.501,5.091,3.93,10.481,10.713,10.538,10.154,10.764,10.526,10.234,10.229,10.41,10.251,10.091,10.115,10.729,10.259,10.482,10.158,7.886,8.201,7.954,7.684,8.316,8.049,7.439,7.161,7.871,8.081,7.272,7.548,8.248,7.882,7.91,7.341,8.886,8.967,8.674,8.382,9.089,8.842,8.374,8.127,8.748,8.539,8.603,8.497,9.089,8.531,8.978,8.11,8.101,8.234,7.738,7.949,7.588,7.549,7.6,7.955,7.334,7.126,7.911,7.02,8.047,8.092,7.148,6.883,8.642,8.423,8.383,8.459,8.192,8.274,8.014,7.983,7.981,7.747,8.073,7.436,8.759,7.844,7.51,1.78,1.794,2.457,2.353,2.074,2.198,2.451,2.813,2.178,2.426,2.699,2.575,1.597,2.223,2.198,3.081,10.717,10.968,11.471,11.713,10.642,11.496,11.676,11.433,10.88,12.172,11.922,10.532,10.272,11.006,10.436,10.652,11.66,10.72,12.291,10.218,4.429,4.765,4.843,4.685,5.145,5.477,5.055,4.784,5.32,5.242,5.136,5.572,4.936,4.677,5.458,5.65,4.473,4.399,4.613,5.197,4.067,4.265,4.849,5.445,4.489,4.758,4.835,4.93,4.924,4.634,4.984,4.021,3.15,3.757,3.604,2.715,2.965,3.387,3.7,2.756,3.491,3.585,2.997,3.092,3.771,2.721,3.639,6.033,5.898,6.061,6.249,6.345,6.105,6.838,7.041,6.545,5.704,6.837],[7.649,6.94,7.076,7.409,6.524,6.634,null,null,6.482,6.732,6.868,6.526,7.373,7.474,6.35,7.176,4.193,4.335,3.968,3.994,4.273,4.037,3.505,3.179,3.773,3.524,3.278,3.328,4.439,3.7,3.802,3.046,10.209,10.294,10.243,9.927,10.256,10.246,9.602,9.522,10.01,9.883,9.689,9.866,10.632,9.935,10.086,9.526,6.38,5.608,5.855,5.212,5.868,5.28,7.983,5.361,5.601,5.203,4.423,4.13,4.117,4.337,3.815,3.762,4.313,4.584,3.673,3.846,4.156,3.883,4.625,4.567,3.49,4.403,2.517,2.967,2.788,2.562,3.803,3.548,2.844,2.693,3.537,3.419,3.225,3.651,2.931,2.373,3.734,3.483,13.293,12.976,13.006,12.777,13.13,12.592,12.455,12.435,12.291,12.37,12.196,11.865,13.139,12.868,12.135,12.047,5.282,5.552,5.687,5.188,5.949,5.795,5.536,5.406,6.023,5.937,6.044,6.184,5.433,5.022,6.229,5.679,7.234,7.359,7.286,6.743,7.296,6.995,6.701,6.439,6.8,6.724,6.437,6.304,7.43,6.723,6.127,6.095,5.636,6.304,6.313,5.369,5.58,6.186,6.45,5.336,5.619,5.914,5.375,6.022,6.448,5.147,5.937,1.533,1.794,1.633,1.566,2.148,1.958,1.62,1.338,2.198,2.035,1.808,2.346,1.7,1.198,2.401,1.859,2.3,2.238,1.99,1.774,2.446,1.979,1.52,2.001,1.708,1.484,1.238,1.204,2.805,2.166,1.931,1.264,7.463,7.472,7.261,6.997,7.624,7.703,6.847,6.586,7.224,6.993,7.016,6.817,7.672,7.343,6.485,12.865,14.979,12.804,12.434,13.115,12.724,12.321,12.082,12.599,12.372,12.257,12.163,13.043,12.4,12.589,12.056,7.388,7.26,7.91,7.092,7.219,7.218,6.793,6.854,6.909,6.658,6.546,6.869,7.812,7.423,6.711,6.461,3.363,3.094,3.231,3.56,2.396,2.6,3.082,3.357,2.593,2.87,2.358,3.146,3.893,2.144,5.854,5.653,5.578,5.412,5.418,5.619,5.533,5.55,5.196,4.857,5.024,4.72,5.552,4.919,5.329,9.351,9.594,9.282,8.963,9.626,9.33,8.856,8.672,9.513,9.344,8.837,8.987,9.769,8.963,9.62,8.718,11.387,11.331,11.252,11.259,10.972,11.132,10.672,10.614,10.588,10.851,10.537,10.435,11.329,11.135,10.561,10.575,3.273,2.262,2.834,2.765,1.853,2.073,2.564,2.92,2.026,2.385,2.553,2.429,2.385,2.842,1.997,2.713,7.16,7.222,7.242,7.418,6.568,6.859,7.537,7.85,6.899,7.146,7.661,6.878,6.806,7.868,6.754,7.515,11.495,11.457,11.222,10.991,12.104,11.497,10.846,10.571,11.651,10.987,10.739,11.074,11.657,11.204,11.397,10.498,14.825,14.907,14.68,14.505,14.909,14.785,14.343,14.43,14.202,13.969,15.086,14.497,3.919,4.0,3.885,3.902,3.808,3.534,3.561,3.402,3.319,3.391,3.209,3.231,3.99,3.77,6.173,6.283,5.964,5.913,6.406,6.041,5.806,5.248,5.978,5.828,5.642,5.354,6.785,5.715,6.305,5.144,11.695,11.927,11.752,11.368,11.978,11.74,11.448,11.443,11.624,11.465,11.305,11.329,11.943,11.473,11.696,11.372,9.1,9.139,8.963,8.693,9.035,8.767,8.446,8.375,8.518,8.728,8.209,8.237,9.151,9.013,8.557,8.12,9.434,9.515,9.222,8.93,9.637,9.39,8.922,8.675,9.296,9.087,9.151,9.045,9.637,9.079,9.526,8.658,9.036,9.169,8.673,8.739,8.523,8.484,8.482,8.422,8.269,8.061,8.525,7.955,8.982,8.882,8.083,7.796,9.577,9.358,9.318,9.394,9.127,9.209,8.949,8.918,9.059,8.682,9.008,8.376,9.694,8.987,8.445,0.823,0.193,0.971,0.927,0.787,0.712,1.025,1.392,1.058,1.108,1.273,1.361,0.309,0.802,1.078,1.719,11.264,11.515,12.018,12.26,11.189,12.044,12.224,11.981,11.428,12.72,12.47,11.08,10.82,11.554,10.984,11.2,12.208,11.268,12.839,10.766,4.675,5.011,4.828,4.67,5.391,5.551,4.958,4.587,5.278,5.059,4.939,5.375,5.182,4.635,5.416,5.453,3.902,3.695,3.943,4.514,3.144,3.498,4.166,4.762,3.566,3.835,4.127,4.007,4.353,3.576,4.266,5.039,4.33,4.566,3.944,4.19,3.975,3.727,3.956,3.734,3.795,3.841,3.253,4.543,4.111,4.058,3.895,5.693,5.558,5.587,5.909,6.005,5.631,6.364,6.567,6.071,5.364,6.363],[8.495,7.801,7.937,7.761,7.385,7.495,null,null,7.172,7.301,7.214,6.874,8.235,7.826,7.211,7.522,5.768,5.884,5.541,5.567,5.709,5.586,5.08,4.754,5.284,5.097,4.853,4.867,5.993,5.275,5.269,4.621,11.779,11.721,11.813,11.428,11.683,11.673,11.058,10.951,11.482,11.355,11.145,11.322,12.059,11.51,11.513,10.955,6.203,5.628,5.661,5.418,5.547,5.066,7.502,4.88,5.643,4.989,3.942,3.649,3.636,3.856,3.334,3.281,3.832,4.103,3.192,3.365,3.675,3.402,4.144,4.086,3.009,3.916,2.333,2.5,2.251,1.924,3.159,2.912,2.2,2.049,2.893,2.775,2.581,3.007,2.715,2.119,3.09,2.839,14.868,14.551,14.581,14.352,14.705,14.167,14.03,14.01,13.866,13.945,13.771,13.44,14.714,14.443,13.71,13.622,4.638,4.908,5.043,4.544,5.305,5.151,4.892,4.762,5.379,5.293,5.4,5.54,4.789,4.378,5.585,5.035,8.623,8.748,8.675,8.132,8.685,8.384,8.09,7.828,8.189,8.113,7.826,7.693,8.819,8.112,7.516,5.614,5.155,5.823,5.832,4.888,5.099,5.705,5.969,4.855,5.138,5.433,4.894,5.541,5.967,4.666,5.456,2.842,2.893,2.65,2.538,3.049,2.768,2.317,1.923,2.647,2.386,2.1,2.385,3.082,2.283,2.752,1.898,3.801,3.739,3.491,3.273,3.984,3.483,3.022,3.153,3.219,2.987,2.737,2.863,4.306,3.665,3.435,2.763,9.038,9.047,8.836,8.572,9.199,9.278,8.422,8.161,8.799,8.568,8.591,8.392,9.247,8.918,8.06,14.292,16.368,14.231,13.861,14.504,14.113,13.748,13.509,13.988,13.761,13.646,13.552,14.47,13.827,13.978,13.483,8.841,8.713,9.363,8.545,8.672,8.671,8.246,8.307,8.362,8.111,7.999,8.322,9.265,8.876,8.164,7.914,3.717,3.665,3.522,3.705,3.895,3.561,3.329,3.339,3.164,3.073,2.656,4.313,4.24,3.105,7.23,7.106,7.031,6.865,6.871,7.072,6.986,6.549,6.649,6.31,6.477,6.018,6.92,6.372,6.084,10.926,11.169,10.857,10.538,11.201,10.905,10.431,10.247,11.088,10.919,10.412,10.562,11.344,10.538,11.195,10.293,12.84,12.906,12.705,12.712,12.547,12.585,12.125,12.067,12.163,12.304,11.99,11.888,12.782,12.588,12.136,12.028,2.792,1.84,2.353,2.284,1.537,1.592,2.083,2.433,1.355,1.709,2.066,1.714,2.062,2.361,1.177,2.226,6.679,6.741,6.761,6.937,6.087,6.378,7.056,7.369,6.418,6.665,7.18,6.397,6.325,7.387,6.273,7.034,12.884,12.846,12.611,12.38,13.493,12.886,12.235,11.96,13.04,12.376,12.128,12.463,13.046,12.593,12.786,11.887,16.214,16.296,16.069,15.894,16.298,16.174,15.732,15.819,15.591,15.358,16.475,15.886,5.494,5.575,5.46,5.477,5.383,5.109,5.136,4.902,4.894,4.966,4.784,4.806,5.565,5.345,7.748,7.858,7.539,7.488,7.882,7.616,7.381,6.823,7.553,7.403,7.217,6.929,8.261,7.29,7.727,6.719,13.27,13.502,13.327,12.943,13.553,13.315,13.023,13.018,13.199,13.04,12.88,12.904,13.518,13.048,13.271,12.947,10.535,10.528,10.352,10.082,10.424,10.156,9.835,9.802,9.907,10.117,9.598,9.626,10.54,10.402,9.946,9.509,10.823,10.904,10.611,10.319,11.026,10.779,10.311,10.064,10.685,10.476,10.54,10.434,11.026,10.468,10.915,10.047,10.466,10.599,10.103,10.105,9.953,9.914,9.848,9.788,9.699,9.491,9.891,9.385,10.412,10.248,9.513,9.162,11.03,10.811,10.771,10.847,10.58,10.662,10.402,10.371,10.512,10.135,10.461,9.829,11.147,10.44,9.898,2.025,2.065,1.559,1.232,1.761,1.512,1.067,0.757,1.485,1.193,1.241,0.878,2.066,1.422,1.83,0.62,12.653,12.904,13.407,13.649,12.578,13.433,13.613,13.37,12.817,14.109,13.859,12.469,12.209,12.943,12.373,12.589,13.597,12.657,14.228,12.155,5.38,5.716,5.359,5.042,6.037,6.082,5.189,4.818,5.648,5.29,5.17,5.606,5.887,5.006,5.786,5.684,3.415,3.208,3.456,4.012,2.657,3.01,3.664,4.247,3.073,3.333,3.625,3.35,3.866,2.896,3.739,6.492,5.783,6.019,5.397,5.662,5.445,5.18,4.951,5.204,5.248,4.992,4.722,5.996,5.484,5.528,4.89,5.441,5.328,4.955,5.657,5.519,4.999,5.732,5.935,5.439,4.859,5.731],[5.785,5.076,5.212,5.322,4.66,4.77,null,null,4.447,4.697,4.775,4.387,5.51,5.387,4.486,5.083,5.335,5.69,5.39,5.416,6.334,5.988,5.401,5.237,6.009,5.76,5.511,5.776,5.581,4.953,6.039,5.403,11.047,11.258,11.081,10.765,11.233,11.223,10.566,10.486,10.987,10.86,10.653,10.843,11.596,10.773,11.063,10.49,3.879,3.141,3.312,2.693,3.198,2.579,5.409,2.537,3.156,2.502,1.77,1.39,1.654,1.895,1.476,1.593,2.065,2.572,1.504,1.847,2.157,1.884,1.804,2.125,1.321,2.632,5.282,5.551,5.302,4.975,6.21,5.963,5.251,5.1,5.944,5.826,5.632,6.058,5.664,5.07,6.141,5.89,13.476,13.344,13.247,12.958,13.498,13.006,12.63,12.504,12.703,12.76,12.565,12.439,13.507,13.049,12.907,12.369,7.689,7.959,8.094,7.595,8.356,8.202,7.943,7.813,8.43,8.344,8.451,8.591,7.84,7.429,8.636,8.086,8.815,9.228,8.939,8.281,9.359,8.696,8.399,7.997,8.926,8.636,8.208,8.435,9.299,8.261,7.88,3.521,3.062,3.73,3.739,2.795,3.006,3.612,3.876,2.762,3.045,3.34,2.801,3.448,3.874,2.573,3.363,4.772,5.033,4.931,4.899,5.387,5.195,4.857,4.574,5.434,5.271,5.023,5.308,4.939,4.54,5.637,4.821,2.986,2.924,2.676,2.458,3.442,3.171,2.865,2.336,3.618,2.929,2.679,3.583,3.491,2.85,3.377,2.705,8.277,8.31,8.075,7.811,8.462,8.541,7.661,7.4,8.057,7.807,7.854,7.655,8.51,8.181,7.323,13.842,16.462,13.781,13.411,14.387,13.718,13.298,13.059,13.749,13.366,13.251,13.313,14.02,13.377,13.76,13.033,6.927,6.98,7.449,6.631,6.939,6.938,6.513,6.393,6.629,6.378,6.266,6.589,7.351,6.962,6.431,6.181,2.512,2.569,2.317,2.153,2.955,2.61,2.006,1.531,2.195,1.868,1.691,3.25,2.688,2.281,4.806,5.192,4.897,4.723,4.957,5.158,4.686,4.337,4.735,4.396,4.563,4.081,4.496,4.458,3.859,10.165,10.408,10.096,9.777,10.44,10.144,9.67,9.486,10.327,10.182,9.651,9.825,10.583,9.777,10.434,9.532,10.707,11.244,10.791,10.579,10.885,10.671,10.211,9.934,10.253,10.39,10.076,9.974,10.868,10.455,10.44,10.114,1.066,1.678,1.312,1.6,2.099,1.793,1.659,1.762,1.892,1.921,1.965,2.402,1.487,1.438,1.988,2.125,4.708,4.781,4.801,5.005,4.251,4.447,5.134,5.543,4.592,4.839,5.442,4.79,4.354,5.454,4.593,5.427,13.378,13.437,13.202,12.97,14.105,13.498,12.847,12.572,13.817,13.153,12.905,13.24,13.637,13.154,13.563,12.664,16.611,16.707,16.466,16.291,16.737,16.613,16.129,16.258,15.988,15.768,16.886,16.325,4.698,5.061,4.81,4.543,4.869,4.595,4.131,3.84,4.401,4.34,3.901,4.005,5.051,4.34,7.011,7.166,6.881,6.83,7.323,7.018,6.723,6.165,6.955,6.907,6.559,6.436,7.702,6.553,7.387,6.226,12.509,12.741,12.566,12.182,12.792,12.554,12.262,12.257,12.438,12.279,12.119,12.143,12.757,12.287,12.534,12.186,10.077,10.392,10.145,9.875,10.507,10.24,9.63,9.352,10.062,10.272,9.463,9.739,10.439,10.073,10.101,9.532,11.6,11.681,11.388,11.096,11.803,11.556,11.088,10.841,11.462,11.253,11.317,11.234,11.803,11.245,11.692,10.824,7.843,7.976,7.48,7.482,7.33,7.291,7.225,7.165,7.076,6.868,7.268,6.762,7.789,7.625,6.89,6.539,8.897,8.678,8.638,8.714,8.666,8.529,8.269,8.238,8.598,8.221,8.328,7.915,9.014,8.526,7.765,3.129,3.65,3.807,3.316,4.023,3.774,3.329,3.017,4.294,3.833,3.503,3.801,3.651,3.029,4.314,3.543,13.663,13.914,14.417,14.659,13.59,14.21,14.39,14.147,13.594,14.886,14.636,13.246,12.986,13.72,13.15,13.366,14.374,13.434,15.005,12.932,7.468,7.804,7.882,7.724,8.184,8.516,8.094,7.741,8.359,8.213,8.093,8.529,7.975,7.716,8.497,8.607,2.461,2.477,2.601,3.185,2.207,2.344,2.837,3.433,2.629,2.841,2.913,3.07,2.899,2.774,3.062,4.759,4.213,4.283,3.827,4.405,4.041,3.631,3.368,4.005,3.748,3.492,3.523,4.426,3.744,4.329,3.39,8.461,8.326,8.006,8.677,8.57,8.05,8.783,8.986,8.49,7.91,8.782],[7.153,6.459,6.595,6.419,6.043,6.153,null,null,5.83,5.959,5.872,5.532,6.893,6.484,5.869,6.18,5.775,5.917,5.55,5.576,5.855,5.619,5.087,4.761,5.355,5.106,4.86,4.91,6.021,5.282,5.384,4.628,11.609,11.803,11.643,11.327,11.778,11.768,11.111,11.031,11.532,11.405,11.198,11.388,12.141,11.335,11.608,11.035,4.861,4.286,4.319,4.076,4.205,3.724,6.16,3.538,4.301,3.647,2.6,2.307,2.294,2.514,1.992,1.939,2.49,2.761,1.85,2.023,2.333,2.06,2.802,2.744,1.667,2.58,3.559,3.828,3.579,3.252,4.487,4.24,3.528,3.377,4.221,4.103,3.909,4.335,3.941,3.347,4.418,4.167,14.417,14.136,14.159,13.898,14.29,13.747,13.555,13.429,13.444,13.501,13.306,13.18,14.299,13.989,13.511,13.11,5.966,6.236,6.371,5.872,6.633,6.479,6.22,6.09,6.707,6.621,6.728,6.868,6.117,5.706,6.913,6.363,8.816,8.941,8.868,8.325,8.878,8.577,8.283,8.021,8.382,8.306,8.019,7.886,9.012,8.305,7.709,4.272,3.813,4.481,4.49,3.546,3.757,4.363,4.627,3.513,3.796,4.091,3.552,4.199,4.625,3.324,4.114,3.115,3.376,3.215,3.148,3.73,3.472,3.134,2.851,3.711,3.548,3.3,3.585,3.282,2.78,3.914,3.098,3.527,3.465,3.217,2.999,3.767,3.266,2.805,2.877,3.002,2.77,2.52,2.647,4.032,3.391,3.218,2.546,8.839,8.872,8.637,8.373,9.024,9.103,8.223,7.962,8.619,8.369,8.416,8.217,9.072,8.743,7.885,14.387,16.561,14.326,13.956,14.697,14.263,13.843,13.604,14.181,13.911,13.796,13.745,14.565,13.922,14.171,13.578,7.668,7.547,8.19,7.372,7.506,7.505,7.08,7.134,7.196,6.945,6.833,7.156,8.092,7.703,6.998,6.748,3.057,3.114,2.862,2.7,3.5,3.155,2.553,2.078,2.74,2.413,2.232,3.795,3.235,2.822,6.034,5.933,5.858,5.692,5.698,5.899,5.813,5.353,5.476,5.137,5.304,4.822,5.724,5.199,4.845,10.727,10.97,10.658,10.339,11.002,10.706,10.232,10.048,10.889,10.744,10.213,10.387,11.145,10.339,10.996,10.094,11.667,11.985,11.532,11.539,11.626,11.412,10.952,10.894,10.994,11.131,10.817,10.715,11.609,11.415,11.181,10.855,1.45,0.498,1.011,0.942,0.276,0.25,0.741,1.097,0.203,0.562,0.73,0.688,0.702,1.019,0.265,0.89,5.337,5.399,5.419,5.595,4.745,5.036,5.714,6.027,5.076,5.323,5.838,5.055,4.983,6.045,4.931,5.692,13.077,13.039,12.804,12.573,13.686,13.079,12.428,12.153,13.233,12.569,12.321,12.656,13.239,12.786,12.979,12.08,16.407,16.489,16.262,16.087,16.491,16.367,15.925,16.012,15.784,15.551,16.668,16.079,5.243,5.582,5.355,5.088,5.39,5.116,4.676,4.385,4.901,4.885,4.446,4.55,5.572,4.885,7.573,7.728,7.426,7.375,7.868,7.563,7.268,6.71,7.5,7.41,7.104,6.936,8.247,7.115,7.887,6.726,13.071,13.303,13.128,12.744,13.354,13.116,12.824,12.819,13.0,12.841,12.681,12.705,13.319,12.849,13.096,12.748,10.622,10.721,10.545,10.275,10.617,10.349,10.028,9.897,10.1,10.31,9.791,9.819,10.733,10.595,10.139,9.702,11.016,11.097,10.804,10.512,11.219,10.972,10.504,10.257,10.878,10.669,10.733,10.627,11.219,10.661,11.108,10.24,9.226,9.359,8.863,8.865,8.713,8.674,8.608,8.548,8.459,8.251,8.651,8.145,9.172,9.008,8.273,7.922,9.857,9.638,9.598,9.674,9.407,9.489,9.229,9.198,9.339,8.962,9.288,8.656,9.974,9.267,8.725,1.809,1.89,2.045,1.554,2.3,2.051,1.606,1.294,2.571,2.11,1.78,2.078,1.891,1.206,2.591,1.82,12.846,13.097,13.6,13.842,12.771,13.626,13.806,13.563,13.01,14.302,14.052,12.662,12.402,13.136,12.566,12.782,13.79,12.85,14.421,12.348,6.257,6.593,6.41,6.242,6.973,7.133,6.389,6.018,6.848,6.49,6.37,6.806,6.764,6.206,6.986,6.884,2.079,1.872,2.12,2.691,1.321,1.675,2.343,2.939,1.743,2.012,2.304,2.184,2.53,1.753,2.443,5.326,4.78,4.85,4.394,4.967,4.603,4.198,3.935,4.567,4.31,4.054,4.085,4.993,4.311,4.891,3.952,6.738,6.603,6.283,6.954,6.847,6.327,7.06,7.263,6.767,6.187,7.059],[6.261,5.552,5.688,5.837,5.136,5.246,null,null,4.971,5.221,5.29,4.948,5.986,5.902,4.962,5.598,4.361,4.716,4.416,4.442,5.36,5.014,4.427,4.263,5.035,4.786,4.537,4.802,4.607,3.979,5.065,4.429,10.093,10.287,10.127,9.811,10.262,10.252,9.595,9.515,10.016,9.889,9.682,9.872,10.625,9.819,10.092,9.519,4.777,4.004,4.251,3.608,4.198,3.579,6.418,3.541,3.998,3.502,2.779,2.399,2.663,2.904,2.485,2.597,3.074,3.581,2.508,2.851,3.161,2.888,2.813,3.134,2.325,3.636,4.572,4.973,4.791,4.524,5.759,5.512,4.8,4.649,5.493,5.375,5.181,5.607,4.954,4.36,5.69,5.439,13.153,12.836,12.866,12.637,12.99,12.452,12.315,12.295,12.151,12.23,12.056,11.725,12.999,12.728,11.995,11.907,7.238,7.508,7.643,7.144,7.905,7.751,7.492,7.362,7.979,7.893,8.0,8.14,7.389,6.978,8.185,7.635,7.844,8.257,7.968,7.31,8.385,7.722,7.425,7.026,7.952,7.662,7.234,7.461,8.328,7.29,6.906,4.53,4.071,4.739,4.748,3.804,4.015,4.621,4.885,3.771,4.054,4.349,3.81,4.457,4.883,3.582,4.372,3.798,4.059,3.957,3.934,4.413,4.326,3.988,3.706,4.566,4.403,4.176,4.596,3.965,3.566,4.769,4.109,2.012,1.95,1.702,1.484,2.468,2.197,1.891,1.362,2.644,1.955,1.705,2.609,2.517,1.876,2.403,1.731,7.323,7.356,7.121,6.857,7.508,7.587,6.707,6.446,7.103,6.853,6.9,6.701,7.556,7.227,6.369,12.871,15.491,12.81,12.44,13.416,12.747,12.327,12.088,12.778,12.395,12.28,12.342,13.049,12.406,12.789,12.062,6.646,6.518,7.168,6.35,6.477,6.476,6.051,6.112,6.167,5.916,5.804,6.127,7.07,6.681,5.969,5.719,1.541,1.598,1.346,1.513,1.984,1.639,1.036,1.221,1.224,0.897,0.717,2.279,2.048,1.307,5.035,4.911,4.836,4.67,4.676,4.877,4.791,4.354,4.454,4.115,4.282,3.823,4.725,4.177,3.833,9.211,9.454,9.142,8.823,9.486,9.19,8.716,8.532,9.373,9.228,8.697,8.871,9.629,8.823,9.48,8.578,10.645,10.963,10.51,10.517,10.604,10.39,9.93,9.872,9.972,10.109,9.795,9.693,10.587,10.393,10.159,9.833,1.906,2.034,2.152,2.604,2.134,2.354,2.569,2.766,2.307,2.596,2.694,2.71,1.667,2.442,2.278,2.854,5.717,5.79,5.81,6.014,5.26,5.456,6.143,6.552,5.601,5.848,6.451,5.799,5.363,6.463,5.602,6.436,12.407,12.466,12.231,11.999,13.134,12.527,11.876,11.601,12.843,12.179,11.931,12.266,12.666,12.183,12.589,11.69,15.64,15.736,15.495,15.32,15.766,15.642,15.158,15.287,15.017,14.797,15.915,15.354,3.727,4.09,3.839,3.572,3.898,3.624,3.16,2.869,3.43,3.369,2.93,3.034,4.08,3.369,6.057,6.212,5.91,5.859,6.352,6.047,5.752,5.194,5.984,5.936,5.588,5.465,6.731,5.599,6.416,5.255,11.555,11.787,11.612,11.228,11.838,11.6,11.308,11.303,11.484,11.325,11.165,11.189,11.803,11.333,11.58,11.232,9.106,9.421,9.174,8.904,9.536,9.269,8.659,8.381,9.091,9.301,8.492,8.768,9.468,9.102,9.13,8.561,10.626,10.707,10.414,10.122,10.829,10.582,10.114,9.867,10.488,10.279,10.343,10.26,10.829,10.271,10.718,9.85,8.271,8.404,7.908,7.91,7.758,7.719,7.653,7.593,7.504,7.296,7.696,7.19,8.217,8.053,7.318,6.967,8.835,8.616,8.576,8.652,8.385,8.467,8.207,8.176,8.317,7.94,8.266,7.634,8.952,8.245,7.703,2.155,2.676,2.833,2.342,3.155,2.908,2.44,2.501,3.426,2.967,2.688,3.091,2.677,2.212,3.446,3.134,12.689,12.94,13.443,13.685,12.616,13.236,13.416,13.173,12.62,13.912,13.662,12.272,12.012,12.746,12.176,12.392,13.4,12.46,14.031,11.958,6.494,6.83,6.908,6.75,7.21,7.542,7.12,6.849,7.385,7.307,7.201,7.637,7.001,6.742,7.523,7.715,3.465,3.481,3.605,4.189,3.152,3.348,3.841,4.437,3.574,3.843,3.917,4.015,3.903,3.719,4.066,4.297,3.588,3.824,3.202,3.509,3.292,2.985,2.756,3.051,3.053,2.797,2.569,3.801,3.289,3.375,2.695,7.751,7.616,7.555,7.967,8.063,7.599,8.332,8.535,8.039,7.422,8.331],[4.756,4.581,4.229,4.022,4.235,4.089,null,null,3.828,3.914,3.475,3.487,4.794,4.087,4.277,3.783,6.065,6.42,6.12,6.146,7.064,6.718,6.131,5.967,6.887,6.639,6.39,6.679,6.311,5.683,6.942,6.306,10.968,11.187,11.002,10.686,11.162,11.152,10.495,10.415,10.916,10.789,10.582,10.772,11.525,10.694,10.992,10.419,2.489,2.071,1.922,2.094,1.808,1.679,4.369,1.141,2.256,1.602,0.969,0.589,0.853,1.094,0.675,0.877,1.264,1.771,1.096,1.06,1.531,1.297,0.722,1.324,0.89,2.045,5.856,6.062,5.813,5.486,6.721,6.474,5.762,5.611,6.455,6.337,6.143,6.569,6.238,5.644,6.652,6.401,13.38,13.248,13.151,12.862,13.402,12.927,12.534,12.408,12.624,12.681,12.486,12.36,13.411,12.953,12.828,12.29,8.2,8.47,8.605,8.106,8.867,8.713,8.454,8.324,8.941,8.855,8.962,9.102,8.351,7.94,9.147,8.597,8.744,9.157,8.868,8.21,9.41,8.757,8.518,8.035,9.066,8.774,8.487,8.751,9.228,8.19,8.351,2.392,1.934,2.602,2.611,1.529,1.952,2.484,2.836,1.722,2.005,2.3,1.761,2.265,2.746,1.533,2.323,5.512,5.773,5.585,5.473,6.049,5.769,5.431,5.148,6.008,5.845,5.597,5.882,5.679,5.177,6.211,5.395,3.303,3.654,3.406,3.188,4.172,3.901,3.595,3.066,4.521,3.832,3.582,4.486,3.779,3.028,4.28,3.608,8.198,8.231,7.996,7.732,8.383,8.462,7.582,7.321,7.978,7.728,7.775,7.576,8.431,8.102,7.244,13.771,16.391,13.71,13.34,14.316,13.647,13.227,12.988,13.678,13.295,13.18,13.242,13.949,13.306,13.689,12.962,6.848,6.901,7.37,6.552,6.86,6.859,6.434,6.314,6.55,6.299,6.187,6.51,7.272,6.883,6.352,6.102,2.441,2.498,2.246,2.082,2.993,2.648,2.044,1.569,2.233,1.906,2.374,3.179,2.617,3.011,4.71,5.113,4.801,4.627,4.878,5.079,4.59,4.241,4.656,4.317,4.484,4.002,4.4,4.379,3.78,10.086,10.329,10.017,9.698,10.361,10.065,9.591,9.407,10.248,10.103,9.572,9.746,10.504,9.698,10.355,9.453,10.611,11.165,10.712,10.483,10.806,10.592,10.132,9.838,10.174,10.311,9.997,9.895,10.789,10.359,10.361,10.035,1.911,2.252,2.01,1.811,2.673,2.367,2.122,2.167,2.466,2.384,2.428,2.865,2.187,1.644,2.562,2.588,3.813,3.98,4.0,4.204,3.45,3.646,4.333,4.742,3.791,4.038,4.641,3.989,3.459,4.562,3.792,4.626,13.307,13.366,13.131,12.899,14.034,13.427,12.776,12.501,13.779,13.115,12.867,13.216,13.566,13.083,13.539,12.64,16.54,16.636,16.395,16.22,16.666,16.542,16.058,16.187,15.917,15.697,16.815,16.254,4.627,4.99,4.739,4.472,4.798,4.524,4.06,3.769,4.439,4.378,3.939,4.043,4.98,4.269,6.932,7.087,6.81,6.759,7.252,6.947,6.652,6.094,6.884,6.836,6.488,6.365,7.631,6.474,7.316,6.155,12.43,12.662,12.487,12.103,12.713,12.475,12.183,12.178,12.359,12.2,12.04,12.064,12.678,12.208,12.455,12.107,10.006,10.321,10.074,9.804,10.436,10.169,9.559,9.281,9.991,10.201,9.392,9.668,10.368,10.002,10.03,9.461,11.604,11.685,11.392,11.1,11.807,11.56,11.092,10.845,11.466,11.257,11.32,11.237,11.807,11.249,11.695,10.827,7.419,7.631,7.135,7.039,7.027,6.988,6.784,6.722,6.773,6.565,6.827,6.459,7.486,7.182,6.587,6.103,8.801,8.582,8.542,8.618,8.587,8.433,8.173,8.142,8.502,8.142,8.232,7.836,8.918,8.447,7.669,3.893,4.287,4.395,3.951,4.597,4.348,3.903,3.591,4.868,4.407,4.077,4.375,4.288,3.603,4.888,4.117,13.666,13.917,14.42,14.662,13.593,14.214,14.394,14.151,13.598,14.89,14.64,13.25,12.99,13.724,13.154,13.37,14.378,13.438,15.009,12.936,8.371,8.707,8.785,8.539,9.087,9.419,8.686,8.315,9.145,8.787,8.667,9.103,8.878,8.503,9.283,9.181,2.188,2.551,2.497,2.757,2.33,2.467,2.791,2.984,2.752,2.964,2.987,3.193,2.312,2.897,3.136,4.68,4.134,4.204,3.748,4.326,3.962,3.552,3.289,3.926,3.669,3.413,3.444,4.347,3.665,4.25,3.311,9.003,8.89,8.517,9.219,9.081,8.561,9.294,9.497,9.001,8.421,9.293],[4.998,4.289,4.425,4.574,3.873,3.983,null,null,3.708,3.958,4.027,3.685,4.723,4.639,3.699,4.335,5.825,6.18,5.88,5.906,6.691,6.478,5.891,5.727,6.647,6.399,6.15,6.454,6.071,5.443,6.717,6.081,9.156,9.39,9.19,8.874,9.435,9.425,8.698,8.618,9.189,9.062,8.785,9.045,9.728,8.882,9.265,8.622,3.284,2.546,2.717,2.097,2.603,1.984,6.108,1.946,2.561,1.907,3.002,2.856,2.996,3.333,3.022,3.222,3.511,3.836,3.371,3.405,3.778,3.642,2.698,3.563,3.188,4.362,6.235,6.636,6.454,6.156,7.391,7.144,6.432,6.281,7.125,7.007,6.813,7.239,6.617,6.023,7.322,7.071,12.123,11.842,11.865,11.604,11.996,11.453,11.261,11.135,11.15,11.207,11.012,10.788,12.005,11.695,11.058,10.816,8.87,9.14,9.275,8.776,9.537,9.383,9.124,8.994,9.611,9.525,9.632,9.772,9.021,8.61,9.817,9.267,7.391,8.11,7.821,7.163,8.363,7.858,7.623,7.14,8.171,7.879,7.592,7.856,8.033,7.143,7.554,3.458,3.29,3.812,3.991,2.909,3.332,3.864,4.338,3.219,3.42,4.039,3.518,3.365,3.866,3.29,4.062,5.45,5.711,5.609,5.586,6.065,5.978,5.64,5.36,6.22,6.057,5.83,6.253,5.617,5.218,6.423,5.766,2.959,3.426,3.181,2.963,3.943,3.673,3.37,2.841,4.296,3.607,3.357,4.261,3.435,2.684,4.055,3.383,6.386,6.419,6.184,5.92,6.571,6.65,5.77,5.509,6.166,5.916,5.963,5.764,6.619,6.29,5.432,12.044,14.664,11.983,11.613,12.589,11.92,11.5,11.261,11.951,11.568,11.453,11.515,12.222,11.579,11.962,11.235,5.374,5.253,5.896,5.078,5.212,5.211,4.786,4.84,4.902,4.651,4.539,4.862,5.798,5.409,4.704,4.454,1.221,1.451,1.199,1.035,2.009,1.871,1.284,1.457,1.889,1.794,2.262,1.465,0.986,2.694,3.74,3.639,3.564,3.398,3.404,3.605,3.519,3.059,3.182,2.843,3.01,2.528,3.43,2.905,2.57,8.274,8.517,8.205,7.886,8.549,8.253,7.779,7.595,8.436,8.291,7.76,7.934,8.692,7.886,8.543,7.641,9.373,9.691,9.238,9.245,9.332,9.118,8.658,8.6,8.7,8.837,8.523,8.421,9.315,9.121,8.887,8.561,2.779,2.91,3.025,3.477,3.18,3.232,3.442,3.639,3.185,3.47,3.568,3.592,2.543,3.315,3.169,3.728,5.552,5.765,5.785,5.989,5.263,5.452,6.118,6.531,5.604,5.851,6.454,5.802,5.198,6.301,5.605,6.439,11.58,11.639,11.404,11.172,12.307,11.7,11.049,10.774,12.052,11.388,11.14,11.489,11.839,11.356,11.812,10.913,14.813,14.909,14.668,14.493,14.939,14.815,14.331,14.46,14.19,13.97,15.088,14.527,3.113,3.725,3.462,3.084,3.751,3.477,3.013,2.722,3.751,3.515,3.312,3.689,3.933,3.222,5.12,5.275,5.048,4.992,5.49,5.335,4.885,4.58,5.37,5.346,4.974,5.318,5.834,4.662,6.01,5.108,10.618,10.85,10.675,10.291,10.901,10.663,10.371,10.366,10.547,10.388,10.228,10.252,10.866,10.396,10.643,10.295,8.279,8.594,8.347,8.077,8.709,8.442,7.832,7.554,8.264,8.474,7.665,7.941,8.641,8.275,8.303,7.734,10.047,10.36,10.067,9.775,10.482,10.235,9.82,9.573,10.419,10.21,10.273,10.19,10.424,9.924,10.648,9.78,6.976,7.109,6.613,6.615,6.463,6.424,6.358,6.298,6.209,6.001,6.401,5.895,6.922,6.758,6.023,5.672,7.563,7.344,7.304,7.38,7.113,7.195,6.935,6.904,7.045,6.668,6.994,6.362,7.68,6.973,6.431,3.812,4.333,4.49,4.005,4.81,4.565,4.103,4.198,5.081,4.629,4.351,4.754,4.332,3.875,5.101,4.724,12.619,12.87,13.373,13.615,12.546,12.659,12.839,12.596,12.04,13.335,13.085,11.693,11.52,12.169,11.829,12.045,12.823,11.88,13.454,11.611,8.146,8.482,8.56,8.402,8.862,9.194,8.772,8.501,9.037,8.959,8.853,9.289,8.653,8.394,9.175,9.367,4.329,4.354,4.469,5.053,4.025,4.221,4.705,5.301,4.447,4.716,4.79,4.888,4.657,4.592,4.939,3.032,2.392,2.556,2.006,2.514,2.15,1.789,1.56,2.114,1.857,1.601,1.632,2.605,2.017,2.438,1.499,9.414,9.279,9.187,9.63,9.726,9.231,9.964,10.167,9.671,9.085,9.963],[4.109,3.934,3.582,3.375,3.588,3.442,null,null,3.181,3.267,2.828,2.84,4.147,3.44,3.63,3.136,6.847,7.202,6.902,6.928,7.713,7.5,6.913,6.749,7.669,7.421,7.172,7.476,7.093,6.465,7.739,7.103,10.321,10.555,10.355,10.039,10.6,10.59,9.863,9.783,10.354,10.227,9.95,10.21,10.893,10.047,10.43,9.787,1.842,1.424,1.275,1.447,1.161,1.032,4.653,0.494,1.609,0.955,1.559,1.413,1.553,1.89,1.582,1.779,2.068,2.381,1.998,1.962,2.335,2.199,1.255,2.12,1.797,2.919,6.763,6.969,6.72,6.393,7.628,7.381,6.669,6.518,7.362,7.244,7.05,7.476,7.145,6.551,7.559,7.308,12.733,12.601,12.504,12.215,12.755,12.28,11.887,11.761,11.977,12.034,11.839,11.713,12.764,12.306,12.181,11.643,9.107,9.377,9.512,9.013,9.774,9.62,9.361,9.231,9.848,9.762,9.869,10.009,9.258,8.847,10.054,9.504,8.556,9.132,8.843,8.185,9.385,8.88,8.66,8.177,9.208,8.916,8.629,8.893,9.198,8.165,8.591,2.003,1.835,2.357,2.536,1.454,1.877,2.409,2.883,1.764,1.965,2.584,2.063,1.91,2.411,1.835,2.607,6.419,6.68,6.492,6.38,6.956,6.676,6.338,6.055,6.915,6.752,6.504,6.789,6.586,6.084,7.118,6.302,3.981,4.448,4.203,3.985,4.965,4.695,4.392,3.863,5.318,4.629,4.379,5.283,4.457,3.706,5.077,4.405,7.551,7.584,7.349,7.085,7.736,7.815,6.935,6.674,7.331,7.081,7.128,6.929,7.784,7.455,6.597,13.209,15.829,13.148,12.778,13.754,13.085,12.665,12.426,13.116,12.733,12.618,12.68,13.387,12.744,13.127,12.4,6.201,6.254,6.723,5.905,6.213,6.212,5.787,5.667,5.903,5.652,5.54,5.863,6.625,6.236,5.705,5.455,2.386,2.473,2.221,2.057,3.031,2.893,2.306,2.469,2.911,2.806,3.274,2.63,2.151,3.716,4.063,4.466,4.154,3.98,4.231,4.432,3.943,3.594,4.009,3.67,3.837,3.355,3.753,3.732,3.133,9.439,9.682,9.37,9.051,9.714,9.418,8.944,8.76,9.601,9.456,8.925,9.099,9.857,9.051,9.708,8.806,9.964,10.518,10.065,9.836,10.159,9.945,9.485,9.191,9.527,9.664,9.35,9.248,10.142,9.712,9.714,9.388,2.818,3.159,2.917,2.718,3.58,3.274,3.029,3.074,3.373,3.291,3.335,3.772,3.094,2.551,3.469,3.495,4.097,4.31,4.33,4.534,3.808,3.997,4.663,5.076,4.149,4.396,4.999,4.347,3.743,4.846,4.15,4.984,12.745,12.804,12.569,12.337,13.472,12.865,12.214,11.939,13.217,12.553,12.305,12.654,13.004,12.521,12.977,12.078,15.978,16.074,15.833,15.658,16.104,15.98,15.496,15.625,15.355,15.135,16.253,15.692,4.278,4.89,4.627,4.249,4.773,4.499,4.035,3.744,4.773,4.537,4.334,4.711,4.955,4.244,6.285,6.44,6.213,6.157,6.655,6.5,6.05,5.745,6.535,6.511,6.139,6.34,6.999,5.827,7.175,6.13,11.783,12.015,11.84,11.456,12.066,11.828,11.536,11.531,11.712,11.553,11.393,11.417,12.031,11.561,11.808,11.46,9.444,9.759,9.512,9.242,9.874,9.607,8.997,8.719,9.429,9.639,8.83,9.106,9.806,9.44,9.468,8.899,11.212,11.525,11.232,10.94,11.647,11.4,10.985,10.738,11.441,11.232,11.295,11.212,11.589,11.089,11.67,10.802,6.772,6.984,6.488,6.392,6.38,6.341,6.137,6.075,6.126,5.918,6.18,5.812,6.839,6.535,5.94,5.456,8.154,7.935,7.895,7.971,7.94,7.786,7.526,7.495,7.855,7.495,7.585,7.189,8.271,7.8,7.022,4.8,5.194,5.302,4.858,5.504,5.255,4.81,4.498,5.775,5.314,4.984,5.282,5.195,4.51,5.795,5.024,13.641,13.892,14.395,14.637,13.568,13.824,14.004,13.761,13.205,14.5,14.25,12.858,12.685,13.334,12.994,13.21,13.988,13.045,14.619,12.776,9.168,9.504,9.582,9.424,9.884,10.216,9.593,9.222,10.052,9.694,9.574,10.01,9.675,9.41,10.19,10.088,3.09,3.458,3.399,3.659,3.237,3.374,3.693,3.886,3.659,3.871,3.894,4.1,3.214,3.804,4.043,4.033,3.487,3.557,3.101,3.679,3.315,2.905,2.642,3.279,3.022,2.766,2.797,3.7,3.018,3.603,2.664,9.91,9.797,9.424,10.126,9.988,9.468,10.201,10.404,9.908,9.328,10.2],[2.814,2.332,2.287,2.08,1.916,2.026,null,null,1.426,1.512,1.533,1.085,2.766,2.145,1.875,1.841,7.194,7.549,7.249,7.275,8.06,7.847,7.26,7.096,8.016,7.768,7.519,7.837,7.44,6.812,8.1,7.464,9.277,9.569,9.311,8.995,9.654,9.644,8.877,8.797,9.408,9.281,8.964,9.264,9.907,9.003,9.484,8.801,0.824,0.523,0.77,0.706,1.058,0.933,5.862,1.386,0.24,1.014,3.113,2.967,3.107,3.444,3.162,3.337,3.622,3.876,3.521,3.52,3.889,3.757,2.835,3.674,3.338,4.473,7.794,8.196,7.957,7.63,8.865,8.618,7.906,7.755,8.599,8.481,8.287,8.713,8.176,7.583,8.796,8.545,10.978,10.846,10.749,10.46,11.0,10.575,10.132,10.006,10.272,10.298,10.099,10.008,11.009,10.551,10.476,9.938,10.344,10.614,10.749,10.25,11.011,10.857,10.598,10.468,11.085,10.999,11.106,11.246,10.495,10.084,11.291,10.741,8.294,9.069,8.78,8.122,9.322,8.765,8.526,8.043,9.074,8.782,8.495,8.759,8.936,8.102,8.457,3.212,3.044,3.566,3.745,2.663,3.086,3.618,4.092,2.973,3.174,3.793,3.44,3.119,3.62,3.138,3.816,6.896,7.157,7.055,7.039,7.511,7.448,7.119,6.863,7.71,7.56,7.333,7.812,7.063,6.671,7.871,7.325,4.426,4.869,4.677,4.505,5.378,5.116,4.866,4.383,5.77,5.106,4.905,5.762,4.87,4.201,5.498,4.931,6.45,6.54,6.248,5.984,6.862,7.049,6.176,5.916,6.722,6.472,6.651,6.458,6.74,6.913,6.109,12.263,15.007,12.299,11.832,12.932,12.263,11.843,11.48,12.294,11.911,11.796,11.858,12.565,11.798,12.305,11.578,4.496,4.596,4.983,4.2,4.961,4.96,4.129,3.962,4.56,4.4,3.882,4.308,4.92,4.496,4.453,4.198,2.749,3.053,3.034,3.025,3.548,3.427,3.274,3.563,3.491,3.566,3.877,2.992,2.888,4.235,2.308,2.726,2.399,2.225,2.526,2.692,2.188,1.839,2.304,1.965,2.132,1.65,1.998,2.027,1.428,8.334,8.581,8.269,7.946,8.613,8.317,7.843,7.655,8.5,8.412,7.824,8.055,8.756,7.946,8.607,7.705,8.209,8.813,8.325,8.081,8.454,8.205,7.745,7.436,7.822,7.924,7.61,7.543,8.402,7.957,8.009,7.683,4.055,4.396,4.154,3.955,4.817,4.511,4.266,4.311,4.61,4.528,4.572,5.009,4.331,3.788,4.706,4.732,5.306,5.519,5.539,5.743,5.017,5.206,5.872,6.285,5.358,5.605,6.208,5.556,4.952,6.055,5.359,6.193,11.923,11.982,11.747,11.515,12.65,12.043,11.392,11.117,12.395,11.731,11.483,12.156,12.182,11.699,12.298,11.532,15.156,15.252,15.011,14.836,15.282,15.158,14.674,14.803,14.533,14.313,15.431,14.87,4.158,4.77,4.507,4.129,5.12,4.846,4.382,4.156,5.12,4.884,4.681,5.058,5.08,4.321,5.942,6.097,5.921,5.865,6.363,6.208,5.758,5.509,6.404,6.38,6.008,6.463,6.656,5.484,6.997,6.256,10.678,10.914,10.735,10.351,10.965,10.727,10.431,10.426,10.611,10.452,10.292,10.316,10.93,10.456,10.764,10.355,8.946,9.334,9.087,8.899,9.531,9.264,8.654,8.376,9.086,9.296,8.487,8.763,9.308,8.949,9.125,8.556,10.869,11.235,10.97,10.678,11.385,11.138,10.723,10.476,11.378,11.169,11.232,11.149,11.246,10.812,11.607,10.739,5.19,5.365,4.869,4.81,4.719,4.68,4.553,4.493,4.465,4.257,4.596,4.151,5.178,4.953,4.279,3.867,6.399,6.18,6.14,6.216,6.2,6.031,5.771,5.74,6.1,5.755,5.83,5.484,6.516,6.095,5.267,5.371,5.892,6.049,5.607,6.358,6.124,5.705,5.735,6.629,6.188,5.953,6.356,5.88,5.477,6.649,6.261,13.578,13.829,14.332,14.574,13.505,13.481,13.661,13.418,12.862,14.157,13.907,12.515,12.342,12.991,12.732,12.948,13.645,12.702,14.276,12.514,9.529,9.865,9.943,9.785,10.245,10.577,10.155,9.947,10.42,10.342,10.299,10.735,10.036,9.777,10.558,10.813,4.584,4.695,4.724,5.217,4.474,4.611,4.96,5.444,4.896,5.108,5.131,5.337,4.772,5.041,5.28,2.418,2.572,2.305,2.186,2.972,2.608,2.07,1.953,3.016,2.468,2.277,2.736,2.785,1.766,2.935,2.239,10.973,10.838,10.661,11.189,11.225,10.705,11.438,11.641,11.145,10.565,11.437],[4.011,3.836,3.484,3.277,3.49,3.344,null,null,3.127,3.213,2.73,2.786,4.049,3.342,3.576,2.986,7.553,7.908,7.608,7.634,8.419,8.206,7.619,7.455,8.375,8.127,7.878,8.185,7.799,7.171,8.448,7.812,10.497,10.789,10.531,10.215,10.874,10.864,10.097,10.017,10.628,10.501,10.184,10.484,11.127,10.223,10.704,10.021,1.454,1.326,1.177,1.393,1.063,1.342,4.274,0.802,1.555,1.265,1.892,1.778,1.886,2.255,2.412,2.148,2.433,2.499,2.367,2.331,2.7,2.568,1.954,2.463,2.506,3.284,7.472,7.678,7.429,7.102,8.337,8.09,7.378,7.227,8.071,7.953,7.759,8.185,7.854,7.26,8.268,8.017,12.679,12.547,12.45,12.161,12.701,12.246,11.833,11.707,11.943,11.999,11.8,11.679,12.71,12.252,12.147,11.609,9.816,10.086,10.221,9.722,10.483,10.329,10.07,9.94,10.557,10.471,10.578,10.718,9.967,9.556,10.763,10.213,9.09,9.838,9.549,8.891,10.091,9.561,9.322,8.839,9.87,9.578,9.291,9.555,9.732,8.871,9.253,1.527,1.359,1.881,2.082,1.201,1.624,2.03,2.504,1.511,1.712,2.205,1.978,1.434,1.935,1.676,2.354,7.128,7.389,7.201,7.089,7.665,7.385,7.047,6.764,7.624,7.461,7.213,7.498,7.295,6.793,7.827,7.011,4.689,5.156,4.911,4.694,5.672,5.403,5.1,4.572,6.027,5.338,5.088,5.992,5.164,4.414,5.786,5.114,7.67,7.76,7.468,7.204,8.082,8.269,7.396,7.136,7.843,7.593,7.64,7.441,7.96,7.967,7.109,13.483,16.227,13.519,13.052,14.152,13.483,13.063,12.7,13.514,13.131,13.016,13.078,13.785,13.018,13.525,12.798,6.167,6.222,6.684,5.871,6.181,6.18,5.755,5.633,5.871,5.62,5.508,5.831,6.591,6.197,5.673,5.423,2.92,3.224,3.07,2.906,3.737,3.599,3.155,3.32,3.662,3.657,4.048,3.164,2.685,4.424,4.009,4.427,4.1,3.926,4.197,4.393,3.889,3.54,3.975,3.636,3.803,3.321,3.699,3.698,3.099,9.554,9.801,9.489,9.166,9.833,9.537,9.063,8.875,9.72,9.632,9.044,9.275,9.976,9.166,9.827,8.925,9.91,10.484,10.026,9.782,10.125,9.906,9.446,9.137,9.493,9.625,9.311,9.214,10.103,9.658,9.68,9.354,3.527,3.868,3.626,3.427,4.289,3.983,3.738,3.783,4.082,4.0,4.044,4.481,3.803,3.26,4.178,4.204,3.816,4.047,4.051,4.257,3.555,3.734,4.386,4.799,3.896,4.143,4.746,4.094,3.48,4.565,3.897,4.731,13.143,13.202,12.967,12.735,13.87,13.263,12.612,12.337,13.615,12.951,12.703,13.166,13.402,12.919,13.489,12.59,16.376,16.472,16.231,16.056,16.502,16.378,15.894,16.023,15.753,15.533,16.651,16.09,4.812,5.424,5.161,4.783,5.479,5.205,4.741,4.45,5.479,5.243,5.04,5.417,5.661,4.95,6.797,6.952,6.725,6.669,7.167,7.012,6.562,6.279,7.069,7.045,6.673,7.046,7.511,6.339,7.709,6.836,11.898,12.134,11.955,11.571,12.185,11.947,11.651,11.646,11.831,11.672,11.512,11.536,12.15,11.676,11.984,11.575,9.956,10.271,10.024,9.754,10.386,10.119,9.509,9.231,9.941,10.151,9.342,9.618,10.318,9.952,9.98,9.411,11.724,12.059,11.766,11.474,12.181,11.934,11.519,11.272,12.147,11.938,12.001,11.918,12.101,11.623,12.376,11.508,6.674,6.886,6.39,6.294,6.282,6.243,6.039,5.977,6.028,5.82,6.082,5.714,6.741,6.437,5.842,5.358,8.1,7.881,7.841,7.917,7.901,7.732,7.472,7.441,7.801,7.456,7.531,7.155,8.217,7.766,6.968,5.509,5.903,6.011,5.567,6.213,5.964,5.519,5.207,6.484,6.023,5.693,5.991,5.904,5.219,6.504,5.733,14.347,14.598,15.101,15.343,14.274,14.336,14.516,14.273,13.717,15.012,14.762,13.37,13.197,13.846,13.528,13.744,14.5,13.557,15.131,13.31,9.877,10.213,10.291,10.133,10.593,10.925,10.302,9.931,10.761,10.403,10.283,10.719,10.384,10.119,10.899,10.797,3.459,4.153,3.768,4.028,3.946,4.083,4.062,4.255,4.327,4.444,4.402,4.767,3.583,4.472,4.553,4.001,3.456,3.525,3.07,3.776,3.412,2.874,2.623,3.813,3.272,3.017,3.331,3.669,2.986,3.739,2.915,10.619,10.506,10.133,10.835,10.697,10.177,10.91,11.113,10.617,10.037,10.909],[5.809,5.634,5.282,5.075,5.288,5.142,null,null,4.925,5.011,4.528,4.584,5.847,5.14,5.374,4.784,8.677,9.032,8.732,8.758,9.676,9.33,8.743,8.579,9.242,8.993,8.744,9.009,8.923,8.295,9.272,8.636,12.295,12.587,12.329,12.013,12.672,12.662,11.895,11.815,12.426,12.299,11.982,12.282,12.925,12.021,12.502,11.819,3.252,3.124,2.975,3.191,2.861,3.127,2.994,2.587,3.353,3.05,2.305,2.621,2.297,2.685,3.026,2.726,2.6,2.465,2.945,2.909,2.863,3.146,2.797,2.433,3.084,3.447,8.05,8.256,8.007,7.68,8.915,8.668,7.956,7.805,8.649,8.531,8.337,8.763,8.432,7.838,8.846,8.595,14.477,14.345,14.248,13.959,14.499,14.044,13.631,13.505,13.741,13.797,13.598,13.477,14.508,14.05,13.945,13.407,10.394,10.664,10.799,10.3,11.061,10.907,10.648,10.518,11.135,11.049,11.156,11.296,10.545,10.134,11.341,10.791,10.734,11.15,10.861,10.203,11.403,10.898,10.678,10.195,11.226,10.934,10.647,10.911,11.221,10.183,10.609,0.903,0.902,0.546,0.409,1.353,1.018,0.814,0.994,1.379,1.103,0.925,1.402,0.965,0.262,1.721,1.191,7.706,7.967,7.779,7.667,8.243,7.963,7.625,7.342,8.202,8.039,7.791,8.076,7.873,7.371,8.405,7.589,5.915,6.266,6.018,5.8,6.784,6.513,6.207,5.678,6.851,6.283,6.033,6.7,6.391,5.64,6.731,6.059,9.468,9.558,9.266,9.002,9.88,9.993,9.113,8.852,9.509,9.259,9.306,9.107,9.758,9.633,8.775,15.281,18.007,15.317,14.85,15.932,15.263,14.843,14.498,15.294,14.911,14.796,14.858,15.565,14.816,15.305,14.578,7.965,8.02,8.482,7.669,7.979,7.978,7.553,7.431,7.669,7.418,7.306,7.629,8.389,7.995,7.471,7.221,4.434,4.491,4.239,4.075,5.049,4.911,4.324,4.181,4.845,4.518,4.986,4.808,4.329,5.623,5.807,6.225,5.898,5.724,5.995,6.191,5.687,5.338,5.773,5.434,5.601,5.119,5.497,5.496,4.897,11.352,11.599,11.287,10.964,11.631,11.335,10.861,10.673,11.518,11.43,10.842,11.073,11.774,10.964,11.625,10.723,11.708,12.282,11.824,11.58,11.923,11.704,11.244,10.935,11.291,11.423,11.109,11.012,11.901,11.456,11.478,11.152,4.105,4.446,4.204,4.005,4.867,4.561,4.316,4.361,4.66,4.578,4.622,5.059,4.381,3.838,4.756,4.782,2.161,2.374,2.394,2.598,2.513,2.061,2.727,3.14,2.248,2.495,3.098,2.703,1.807,2.91,2.249,3.112,14.923,14.982,14.747,14.515,15.65,15.043,14.392,14.117,15.395,14.731,14.483,14.832,15.182,14.699,15.155,14.256,18.156,18.252,18.011,17.836,18.282,18.158,17.674,17.803,17.533,17.313,18.431,17.87,6.456,6.983,6.732,6.427,6.791,6.517,6.053,5.762,6.791,6.555,6.352,6.655,6.973,6.262,8.463,8.618,8.391,8.335,8.833,8.678,8.228,7.923,8.713,8.689,8.317,8.358,9.177,8.005,9.309,8.148,13.696,13.932,13.753,13.369,13.983,13.745,13.449,13.444,13.629,13.47,13.31,13.334,13.948,13.474,13.782,13.373,11.622,11.937,11.69,11.42,12.052,11.785,11.175,10.897,11.607,11.817,11.008,11.276,11.984,11.618,11.646,11.077,13.39,13.678,13.385,13.093,13.8,13.553,13.085,12.838,13.459,13.25,13.313,13.23,13.767,13.242,13.688,12.82,8.472,8.684,8.188,8.092,8.08,8.041,7.837,7.775,7.826,7.618,7.88,7.512,8.539,8.235,7.64,7.156,9.898,9.679,9.639,9.715,9.699,9.53,9.27,9.239,9.599,9.254,9.329,8.953,10.015,9.564,8.766,6.087,6.481,6.589,6.145,6.791,6.542,6.097,5.785,7.062,6.601,6.271,6.569,6.482,5.797,7.082,6.311,15.659,15.91,16.413,16.655,15.586,16.002,16.182,15.939,15.383,16.678,16.428,15.036,14.863,15.512,15.147,15.363,16.166,15.223,16.797,14.929,10.701,11.037,11.001,10.733,11.417,11.724,10.88,10.509,11.339,10.981,10.861,11.297,11.208,10.697,11.477,11.375,4.037,4.731,4.346,4.324,4.524,4.661,4.358,4.289,4.794,4.753,4.555,4.92,3.998,4.939,4.587,5.799,5.254,5.323,4.868,5.574,5.21,4.672,4.421,5.457,5.07,4.815,4.975,5.467,4.784,5.537,4.713,11.197,11.084,10.711,11.413,11.275,10.755,11.488,11.691,11.195,10.615,11.487],[5.095,4.92,4.568,4.361,4.574,4.428,null,null,4.211,4.297,3.814,3.87,5.133,4.426,4.66,4.07,7.036,7.391,7.091,7.117,8.035,7.689,7.102,6.938,7.601,7.352,7.103,7.368,7.282,6.654,7.631,6.995,11.478,11.712,11.512,11.196,11.757,11.747,11.02,10.94,11.511,11.384,11.107,11.367,12.05,11.204,11.587,10.944,2.538,2.41,2.261,2.477,2.147,2.159,3.21,1.619,2.639,2.082,0.664,0.98,0.656,1.044,1.385,1.085,1.055,0.92,1.304,1.268,1.318,1.505,1.156,0.884,1.443,1.902,6.409,6.615,6.366,6.039,7.274,7.027,6.315,6.164,7.008,6.89,6.696,7.122,6.791,6.197,7.205,6.954,13.763,13.631,13.534,13.245,13.785,13.33,12.917,12.791,13.027,13.083,12.884,12.763,13.794,13.336,13.231,12.693,8.753,9.023,9.158,8.659,9.42,9.266,9.007,8.877,9.494,9.408,9.515,9.655,8.904,8.493,9.7,9.15,9.443,9.856,9.567,8.909,10.109,9.604,9.384,8.901,9.932,9.64,9.353,9.617,9.927,8.889,9.315,1.409,0.95,1.618,1.627,0.683,0.807,1.462,1.677,0.563,0.846,1.141,0.602,1.336,1.762,0.374,1.164,6.065,6.326,6.138,6.026,6.602,6.322,5.984,5.701,6.561,6.398,6.15,6.435,6.232,5.73,6.764,5.948,4.274,4.625,4.377,4.159,5.143,4.872,4.566,4.037,5.21,4.642,4.392,5.059,4.75,3.999,5.09,4.418,8.708,8.741,8.506,8.242,8.893,8.972,8.092,7.831,8.488,8.238,8.285,8.086,8.941,8.612,7.754,14.366,16.986,14.305,13.935,14.911,14.242,13.822,13.583,14.273,13.89,13.775,13.837,14.544,13.901,14.284,13.557,7.251,7.306,7.768,6.955,7.265,7.264,6.839,6.717,6.955,6.704,6.592,6.915,7.675,7.281,6.757,6.507,3.14,3.197,2.945,2.781,3.755,3.617,3.015,2.54,3.204,2.877,3.345,3.787,3.308,3.982,5.093,5.511,5.184,5.01,5.281,5.477,4.973,4.624,5.059,4.72,4.887,4.405,4.783,4.782,4.183,10.596,10.839,10.527,10.208,10.871,10.575,10.101,9.917,10.758,10.613,10.082,10.256,11.014,10.208,10.865,9.963,10.994,11.568,11.11,10.866,11.209,10.99,10.53,10.221,10.577,10.709,10.395,10.298,11.187,10.742,10.764,10.438,2.464,2.805,2.563,2.364,3.226,2.92,2.675,2.72,3.019,2.937,2.981,3.418,2.74,2.197,3.115,3.141,2.654,2.867,2.887,3.091,2.365,2.554,3.22,3.633,2.706,2.953,3.556,2.904,2.3,3.403,2.707,3.541,13.902,13.961,13.726,13.494,14.629,14.022,13.371,13.096,14.374,13.71,13.462,13.811,14.161,13.678,14.134,13.235,17.135,17.231,16.99,16.815,17.261,17.137,16.653,16.782,16.512,16.292,17.41,16.849,5.326,5.689,5.438,5.171,5.497,5.223,4.759,4.468,5.41,5.261,4.91,5.014,5.679,4.968,7.442,7.597,7.37,7.314,7.812,7.646,7.207,6.793,7.583,7.535,7.187,7.064,8.156,6.984,8.015,6.854,12.94,13.172,12.997,12.613,13.223,12.985,12.693,12.688,12.869,12.71,12.55,12.574,13.188,12.718,12.965,12.617,10.601,10.916,10.669,10.399,11.031,10.764,10.154,9.876,10.586,10.796,9.987,10.263,10.963,10.597,10.625,10.056,12.303,12.384,12.091,11.799,12.506,12.259,11.791,11.544,12.165,11.956,12.019,11.936,12.506,11.948,12.394,11.526,7.758,7.97,7.474,7.378,7.366,7.327,7.123,7.061,7.112,6.904,7.166,6.798,7.825,7.521,6.926,6.442,9.184,8.965,8.925,9.001,8.985,8.816,8.556,8.525,8.885,8.54,8.615,8.239,9.301,8.85,8.052,4.446,4.84,4.948,4.504,5.15,4.901,4.456,4.144,5.421,4.96,4.63,4.928,4.841,4.156,5.441,4.67,14.365,14.616,15.119,15.361,14.292,14.913,15.093,14.85,14.297,15.589,15.339,13.949,13.689,14.423,13.853,14.069,15.077,14.137,15.708,13.635,9.06,9.396,9.36,9.092,9.776,10.083,9.239,8.868,9.698,9.34,9.22,9.656,9.567,9.056,9.836,9.734,2.396,3.09,2.705,2.826,2.883,3.02,2.86,2.791,3.264,3.255,3.057,3.422,2.5,3.409,3.089,5.085,4.54,4.609,4.154,4.836,4.472,3.958,3.707,4.436,4.179,3.923,3.954,4.753,4.07,4.76,3.821,9.556,9.443,9.07,9.772,9.634,9.114,9.847,10.05,9.554,8.974,9.846],[7.837,7.662,7.31,7.103,7.316,7.17,null,null,6.953,7.039,6.556,6.612,7.875,7.168,7.402,6.812,9.659,10.014,9.714,9.74,10.658,10.312,9.725,9.561,10.205,9.956,9.707,9.972,9.905,9.277,10.235,9.599,14.323,14.615,14.357,14.041,14.7,14.69,13.923,13.843,14.454,14.327,14.01,14.31,14.953,14.049,14.53,13.847,5.28,5.152,5.003,5.219,4.889,5.155,4.444,4.615,5.381,5.078,3.47,3.699,3.432,3.453,3.996,3.696,3.181,2.948,3.915,3.72,3.271,3.705,4.181,3.205,4.054,3.855,8.886,8.974,8.725,8.398,9.633,9.386,8.674,8.523,9.367,9.249,9.055,9.481,9.249,8.799,9.564,9.313,16.505,16.373,16.276,15.987,16.527,16.072,15.659,15.533,15.769,15.825,15.626,15.505,16.536,16.078,15.973,15.435,11.112,11.382,11.517,11.018,11.779,11.625,11.366,11.236,11.853,11.767,11.874,12.014,11.263,10.852,12.059,11.509,12.543,12.956,12.667,12.009,13.209,12.704,12.484,12.001,13.032,12.74,12.453,12.717,13.027,11.989,12.204,3.086,3.129,2.773,2.495,3.268,2.933,2.696,2.148,3.276,3.018,2.375,2.547,3.192,2.616,3.195,2.194,8.667,8.928,8.74,8.628,9.204,8.924,8.586,8.303,9.163,9.0,8.752,9.037,8.834,8.332,9.366,8.55,7.299,7.248,7.0,6.782,7.766,7.495,7.189,6.66,7.814,7.253,7.003,7.661,7.775,7.024,7.701,7.029,11.496,11.586,11.294,11.03,11.908,12.021,11.141,10.88,11.537,11.287,11.334,11.135,11.786,11.661,10.803,17.309,20.035,17.345,16.878,17.96,17.291,16.871,16.526,17.322,16.939,16.824,16.886,17.593,16.844,17.333,16.606,9.993,10.048,10.51,9.697,10.007,10.006,9.581,9.459,9.697,9.446,9.334,9.657,10.417,10.023,9.499,9.249,6.24,6.297,6.045,5.881,6.855,6.644,6.04,5.565,6.229,5.902,6.015,6.836,6.357,6.605,7.835,8.253,7.926,7.752,8.023,8.219,7.715,7.366,7.801,7.462,7.629,7.147,7.525,7.524,6.925,13.38,13.627,13.315,12.992,13.659,13.363,12.889,12.701,13.546,13.458,12.87,13.101,13.802,12.992,13.653,12.751,13.736,14.31,13.852,13.608,13.951,13.732,13.272,12.963,13.319,13.451,13.137,13.04,13.929,13.484,13.506,13.18,5.075,5.407,5.165,4.723,5.828,5.522,5.034,5.079,5.621,5.296,5.34,5.777,5.342,4.556,5.717,5.5,0.502,0.695,0.309,0.073,1.493,1.166,0.644,0.677,1.199,1.067,0.742,1.278,0.844,0.563,1.2,0.756,16.951,17.01,16.775,16.543,17.678,17.071,16.42,16.145,17.423,16.759,16.511,16.86,17.21,16.727,17.183,16.284,20.184,20.28,20.039,19.864,20.31,20.186,19.702,19.831,19.561,19.341,20.459,19.898,8.426,8.789,8.538,8.271,8.597,8.323,7.859,7.568,8.435,8.361,7.935,8.039,8.779,8.068,10.491,10.646,10.419,10.363,10.861,10.706,10.256,9.893,10.683,10.635,10.287,10.164,11.205,10.033,11.115,9.954,15.724,15.96,15.781,15.397,16.011,15.773,15.477,15.472,15.657,15.498,15.338,15.362,15.976,15.502,15.81,15.401,13.65,13.965,13.718,13.448,14.08,13.813,13.203,12.925,13.635,13.845,13.036,13.312,14.012,13.646,13.674,13.105,15.403,15.484,15.191,14.899,15.606,15.359,14.891,14.644,15.265,15.056,15.119,15.036,15.606,15.048,15.494,14.626,10.5,10.712,10.216,10.12,10.108,10.069,9.865,9.803,9.854,9.646,9.908,9.54,10.567,10.263,9.668,9.184,11.926,11.707,11.667,11.743,11.727,11.558,11.298,11.267,11.627,11.282,11.357,10.981,12.043,11.592,10.794,7.048,7.442,7.55,7.106,7.752,7.503,7.058,6.746,8.023,7.562,7.232,7.53,7.443,6.758,8.043,7.272,17.465,17.716,18.219,18.461,17.392,18.013,18.193,17.95,17.397,18.689,18.439,17.049,16.789,17.523,16.953,17.169,18.177,17.237,18.808,16.735,11.664,12.0,11.962,11.694,12.38,12.685,11.841,11.47,12.3,11.942,11.822,12.258,12.171,11.658,12.438,12.336,4.347,5.002,4.656,4.59,4.947,5.002,4.624,4.555,5.06,5.019,4.821,5.186,4.264,5.205,4.853,7.827,7.282,7.351,6.896,7.602,7.238,6.7,6.449,7.485,7.098,6.843,7.003,7.495,6.812,7.565,6.741,11.915,11.802,11.429,12.131,11.993,11.473,12.206,12.409,11.913,11.333,12.205],[6.301,6.126,5.774,5.567,5.78,5.634,null,null,5.417,5.503,5.02,5.076,6.339,5.632,5.866,5.276,8.288,8.643,8.343,8.369,9.287,8.941,8.354,8.19,8.841,8.592,8.343,8.608,8.534,7.906,8.871,8.235,12.787,13.079,12.821,12.505,13.164,13.154,12.387,12.307,12.918,12.791,12.474,12.774,13.417,12.513,12.994,12.311,3.744,3.616,3.467,3.683,3.353,3.619,2.908,3.079,3.845,3.542,1.934,2.275,1.896,2.284,2.625,2.325,2.012,1.877,2.544,2.508,2.2,2.634,2.645,1.845,2.683,2.784,7.649,7.855,7.606,7.279,8.514,8.267,7.555,7.404,8.248,8.13,7.936,8.362,8.031,7.437,8.445,8.194,14.969,14.837,14.74,14.451,14.991,14.536,14.123,13.997,14.233,14.289,14.09,13.969,15.0,14.542,14.437,13.899,9.993,10.263,10.398,9.899,10.66,10.506,10.247,10.117,10.734,10.648,10.755,10.895,10.144,9.733,10.94,10.39,11.007,11.42,11.131,10.473,11.673,11.168,10.948,10.465,11.496,11.204,10.917,11.181,11.491,10.453,10.811,2.096,1.721,1.643,1.365,1.732,1.397,1.16,1.115,1.74,1.482,0.839,1.011,2.062,1.5,1.659,0.658,7.305,7.566,7.378,7.266,7.842,7.562,7.224,6.941,7.801,7.638,7.39,7.675,7.472,6.97,8.004,7.188,5.763,5.877,5.629,5.411,6.395,6.124,5.818,5.289,6.45,5.882,5.632,6.299,6.239,5.488,6.33,5.658,9.96,10.05,9.758,9.494,10.372,10.485,9.605,9.344,10.001,9.751,9.798,9.599,10.25,10.125,9.267,15.773,18.499,15.809,15.342,16.424,15.755,15.335,14.99,15.786,15.403,15.288,15.35,16.057,15.308,15.797,15.07,8.457,8.512,8.974,8.161,8.471,8.47,8.045,7.923,8.161,7.91,7.798,8.121,8.881,8.487,7.963,7.713,4.704,4.761,4.509,4.345,5.319,5.108,4.504,4.029,4.693,4.366,4.644,5.3,4.821,5.234,6.299,6.717,6.39,6.216,6.487,6.683,6.179,5.83,6.265,5.926,6.093,5.611,5.989,5.988,5.389,11.844,12.091,11.779,11.456,12.123,11.827,11.353,11.165,12.01,11.922,11.334,11.565,12.266,11.456,12.117,11.215,12.2,12.774,12.316,12.072,12.415,12.196,11.736,11.427,11.783,11.915,11.601,11.504,12.393,11.948,11.97,11.644,3.704,4.045,3.803,3.604,4.466,4.16,3.915,3.96,4.259,4.177,4.221,4.658,3.98,3.437,4.355,4.381,1.566,1.629,1.649,1.853,1.122,1.295,1.982,2.395,1.463,1.71,2.313,1.684,1.212,2.302,1.464,2.321,15.415,15.474,15.239,15.007,16.142,15.535,14.884,14.609,15.887,15.223,14.975,15.324,15.674,15.191,15.647,14.748,18.648,18.744,18.503,18.328,18.774,18.65,18.166,18.295,18.025,17.805,18.923,18.362,6.89,7.253,7.002,6.735,7.061,6.787,6.323,6.032,6.899,6.825,6.399,6.503,7.243,6.532,8.955,9.11,8.883,8.827,9.325,9.17,8.72,8.357,9.147,9.099,8.751,8.628,9.669,8.497,9.579,8.418,14.188,14.424,14.245,13.861,14.475,14.237,13.941,13.936,14.121,13.962,13.802,13.826,14.44,13.966,14.274,13.865,12.114,12.429,12.182,11.912,12.544,12.277,11.667,11.389,12.099,12.309,11.5,11.776,12.476,12.11,12.138,11.569,13.867,13.948,13.655,13.363,14.07,13.823,13.355,13.108,13.729,13.52,13.583,13.5,14.07,13.512,13.958,13.09,8.964,9.176,8.68,8.584,8.572,8.533,8.329,8.267,8.318,8.11,8.372,8.004,9.031,8.727,8.132,7.648,10.39,10.171,10.131,10.207,10.191,10.022,9.762,9.731,10.091,9.746,9.821,9.445,10.507,10.056,9.258,5.686,6.08,6.188,5.744,6.39,6.141,5.696,5.384,6.661,6.2,5.87,6.168,6.081,5.396,6.681,5.91,15.929,16.18,16.683,16.925,15.856,16.477,16.657,16.414,15.861,17.153,16.903,15.513,15.253,15.987,15.417,15.633,16.641,15.701,17.272,15.199,10.3,10.636,10.6,10.332,11.016,11.323,10.479,10.108,10.938,10.58,10.46,10.896,10.807,10.296,11.076,10.974,3.276,3.931,3.585,3.519,3.876,3.931,3.553,3.484,3.989,3.948,3.75,4.115,3.193,4.134,3.782,6.291,5.746,5.815,5.36,6.066,5.702,5.164,4.913,5.949,5.562,5.307,5.467,5.959,5.276,6.029,5.205,10.796,10.683,10.31,11.012,10.874,10.354,11.087,11.29,10.794,10.214,11.086],[7.551,7.376,7.024,6.817,7.03,6.884,null,null,6.667,6.753,6.27,6.326,7.589,6.882,7.116,6.526,8.575,8.93,8.63,8.656,9.044,8.808,8.268,7.936,8.544,8.295,8.046,8.311,8.821,8.193,8.574,7.938,14.017,14.251,14.051,13.735,14.296,14.286,13.559,13.479,14.05,13.923,13.646,13.906,14.589,13.743,14.126,13.483,4.994,4.866,4.717,4.933,4.603,4.698,4.359,4.158,5.095,4.621,2.761,2.802,2.664,2.556,2.912,2.612,2.284,1.864,2.795,2.335,1.996,2.098,3.38,2.308,2.864,1.895,6.925,7.013,6.764,6.437,7.672,7.425,6.713,6.562,7.406,7.288,7.094,7.52,7.288,6.871,7.603,7.352,16.219,16.087,15.99,15.701,16.241,15.786,15.373,15.247,15.483,15.539,15.34,15.219,16.25,15.792,15.687,15.149,8.538,8.857,8.78,8.281,9.078,8.888,8.546,8.386,9.116,9.009,9.003,9.256,8.738,8.115,9.322,8.638,11.908,12.321,12.032,11.374,12.322,11.936,11.639,11.23,11.826,11.75,11.448,11.33,12.392,11.354,11.12,3.374,2.971,3.094,2.816,2.982,2.647,2.611,2.826,2.7,2.459,2.29,2.092,3.34,2.951,2.619,2.313,6.849,7.11,6.916,6.804,7.38,7.1,6.762,6.479,7.339,7.176,6.926,7.211,7.016,6.514,7.542,6.724,6.226,6.164,5.916,5.698,6.662,6.161,5.7,5.576,6.153,5.665,5.415,6.0,6.731,6.09,6.113,5.441,11.21,11.28,11.008,10.744,11.432,11.511,10.631,10.37,11.027,10.777,10.824,10.625,11.48,11.151,10.293,16.905,19.525,16.844,16.474,17.45,16.781,16.361,16.122,16.812,16.429,16.314,16.376,17.083,16.44,16.823,16.096,9.707,9.762,10.224,9.411,9.721,9.72,9.295,9.173,9.411,9.16,9.048,9.371,10.131,9.737,9.213,8.963,5.605,5.662,5.41,5.246,6.188,5.843,5.239,4.764,5.428,5.101,4.931,6.326,5.781,5.521,7.549,7.967,7.64,7.466,7.737,7.933,7.429,7.08,7.515,7.176,7.343,6.861,7.239,7.238,6.639,13.094,13.341,13.029,12.706,13.373,13.077,12.603,12.415,13.26,13.152,12.584,12.795,13.516,12.706,13.367,12.465,13.45,14.024,13.566,13.322,13.665,13.446,12.986,12.677,13.033,13.165,12.851,12.754,13.643,13.198,13.22,12.894,3.439,3.622,3.504,2.926,4.01,3.704,3.196,3.119,3.803,3.447,3.385,3.465,3.681,2.851,3.899,3.545,3.287,3.265,3.285,3.318,2.584,2.798,3.095,3.408,2.457,2.704,3.219,2.436,2.933,3.57,2.312,3.073,16.441,16.483,16.248,16.017,17.13,16.523,15.872,15.597,16.677,16.013,15.765,16.1,16.683,16.217,16.423,15.524,19.674,19.77,19.529,19.354,19.8,19.676,19.192,19.321,19.051,18.831,19.949,19.388,7.791,8.154,7.903,7.636,7.962,7.688,7.224,6.933,7.634,7.573,7.134,7.238,8.144,7.433,9.981,10.136,9.909,9.853,10.351,10.111,9.746,9.258,10.048,10.0,9.652,9.529,10.695,9.523,10.48,9.319,15.438,15.674,15.495,15.111,15.725,15.487,15.191,15.186,15.371,15.212,15.052,15.076,15.69,15.216,15.504,15.115,13.14,13.455,13.208,12.938,13.57,13.303,12.693,12.415,13.125,13.335,12.526,12.802,13.502,13.136,13.164,12.595,14.46,14.541,14.248,13.956,14.663,14.416,13.948,13.701,14.322,14.113,14.177,14.071,14.663,14.105,14.552,13.684,10.214,10.426,9.93,9.834,9.822,9.783,9.579,9.517,9.568,9.36,9.622,9.254,10.281,9.977,9.382,8.898,11.64,11.421,11.381,11.457,11.441,11.272,11.012,10.981,11.341,10.996,11.071,10.695,11.757,11.306,10.508,5.263,5.624,5.726,5.288,5.928,5.679,5.234,4.922,6.199,5.738,5.408,5.704,5.625,4.94,6.219,5.446,16.291,16.542,17.045,17.287,16.216,17.07,17.25,17.007,16.454,17.746,17.496,16.106,15.846,16.58,16.01,16.226,17.234,16.294,17.865,15.792,9.991,10.327,10.144,9.868,10.707,10.867,10.015,9.644,10.474,10.116,9.996,10.432,10.498,9.832,10.612,10.51,2.053,2.047,2.23,1.635,2.519,2.324,1.669,1.6,2.105,2.064,1.866,2.231,1.309,2.25,1.898,7.541,6.996,7.065,6.61,7.316,6.952,6.414,6.163,6.975,6.718,6.462,6.493,7.209,6.526,7.279,6.36,9.954,9.841,9.468,10.17,10.032,9.512,10.245,10.448,9.952,9.372,10.244],[6.521,6.346,5.994,5.787,6.0,5.854,null,null,5.637,5.723,5.24,5.296,6.559,5.852,6.086,5.496,7.134,7.489,7.189,7.215,7.649,7.413,6.873,6.541,7.149,6.9,6.651,6.916,7.38,6.752,7.179,6.543,12.841,13.054,12.875,12.559,13.029,13.019,12.362,12.282,12.783,12.656,12.449,12.639,13.392,12.567,12.859,12.286,3.964,3.836,3.687,3.903,3.573,3.555,4.888,3.017,4.065,3.478,1.71,1.41,1.469,1.505,1.471,1.171,1.391,1.203,1.317,0.857,0.988,0.62,1.988,1.648,1.446,0.881,5.641,5.729,5.48,5.153,6.388,6.141,5.429,5.278,6.122,6.004,5.81,6.236,6.004,5.587,6.319,6.068,15.189,15.057,14.96,14.671,15.211,14.756,14.343,14.217,14.453,14.509,14.31,14.189,15.22,14.762,14.657,14.119,7.867,8.137,8.219,7.72,8.517,8.327,7.985,7.825,8.555,8.448,8.442,8.695,8.018,7.554,8.761,8.077,10.611,10.99,10.735,10.077,10.927,10.495,10.198,9.793,10.431,10.355,10.007,9.935,11.061,10.057,9.679,3.382,2.923,3.554,3.345,2.656,2.733,3.14,3.355,2.489,2.772,2.819,2.492,3.309,3.48,2.3,2.842,5.565,5.826,5.632,5.52,6.096,5.816,5.478,5.195,6.055,5.892,5.642,5.927,5.732,5.23,6.258,5.44,4.785,4.723,4.475,4.257,5.241,4.743,4.282,4.135,4.758,4.247,3.997,4.605,5.29,4.649,4.695,4.023,10.071,10.104,9.869,9.605,10.256,10.335,9.455,9.194,9.851,9.601,9.648,9.449,10.304,9.975,9.117,15.638,18.258,15.577,15.207,16.183,15.514,15.094,14.855,15.545,15.162,15.047,15.109,15.816,15.173,15.556,14.829,8.677,8.732,9.194,8.381,8.691,8.69,8.265,8.143,8.381,8.13,8.018,8.341,9.101,8.707,8.183,7.933,4.308,4.365,4.113,3.949,4.751,4.406,3.802,3.327,3.991,3.664,3.49,5.046,4.484,4.08,6.519,6.937,6.61,6.436,6.707,6.903,6.399,6.05,6.485,6.146,6.313,5.831,6.209,6.208,5.609,11.959,12.202,11.89,11.571,12.234,11.938,11.464,11.28,12.121,11.976,11.445,11.619,12.377,11.571,12.228,11.326,12.42,12.994,12.536,12.292,12.635,12.416,11.956,11.647,12.003,12.135,11.821,11.724,12.613,12.168,12.19,11.864,2.021,2.338,2.109,1.642,2.726,2.42,1.912,1.835,2.519,2.163,2.101,2.181,2.286,1.5,2.615,2.261,3.816,3.794,3.814,3.849,3.115,3.329,3.626,3.939,2.988,3.235,3.75,2.967,3.462,4.101,2.843,3.604,15.126,15.088,14.853,14.622,15.735,15.128,14.477,14.202,15.282,14.618,14.37,14.705,15.288,14.835,15.028,14.129,18.407,18.503,18.262,18.087,18.533,18.409,17.925,18.054,17.784,17.564,18.682,18.121,6.494,6.857,6.606,6.339,6.665,6.391,5.927,5.636,6.197,6.136,5.697,5.801,6.847,6.136,8.805,8.96,8.677,8.626,9.119,8.814,8.519,7.961,8.751,8.703,8.355,8.232,9.498,8.347,9.183,8.022,14.303,14.535,14.36,13.976,14.586,14.348,14.056,14.051,14.232,14.073,13.913,13.937,14.551,14.081,14.328,13.98,11.873,12.188,11.941,11.671,12.303,12.036,11.426,11.148,11.858,12.068,11.259,11.535,12.235,11.869,11.897,11.328,13.065,13.146,12.853,12.561,13.268,13.021,12.553,12.306,12.927,12.718,12.782,12.676,13.268,12.71,13.157,12.289,9.184,9.396,8.9,8.804,8.792,8.753,8.549,8.487,8.538,8.33,8.592,8.224,9.251,8.947,8.352,7.868,10.61,10.391,10.351,10.427,10.411,10.242,9.982,9.951,10.311,9.966,10.041,9.665,10.727,10.276,9.478,3.979,4.34,4.442,4.004,4.644,4.395,3.95,3.638,4.915,4.454,4.124,4.42,4.341,3.656,4.935,4.162,14.896,15.147,15.65,15.892,14.821,15.675,15.855,15.612,15.059,16.351,16.101,14.711,14.451,15.185,14.615,14.831,15.839,14.899,16.47,14.397,8.608,8.944,8.86,8.584,9.324,9.583,8.731,8.36,9.19,8.832,8.712,9.148,9.115,8.548,9.328,9.226,0.635,1.329,0.944,1.128,1.235,1.29,1.162,1.355,1.427,1.544,1.502,1.867,0.683,1.572,1.653,6.511,5.966,6.035,5.58,6.199,5.835,5.384,5.133,5.799,5.542,5.286,5.317,6.179,5.496,6.123,5.184,8.67,8.557,8.184,8.886,8.748,8.228,8.961,9.164,8.668,8.088,8.96],[6.619,6.245,6.092,5.885,5.829,5.939,null,null,5.339,5.425,5.338,4.998,6.657,5.95,5.765,5.646,6.567,6.723,6.354,6.38,6.661,6.425,5.885,5.553,6.161,5.912,5.663,5.928,6.813,6.074,6.191,5.555,12.326,12.539,12.36,12.044,12.514,12.504,11.847,11.767,12.268,12.141,11.934,12.124,12.877,12.052,12.344,11.771,4.327,3.752,3.785,3.605,3.671,3.19,5.626,3.004,3.767,3.113,2.066,1.773,1.76,1.98,1.458,1.405,1.95,1.975,1.316,1.237,1.547,1.12,2.268,2.21,1.133,1.564,4.798,4.886,4.637,4.31,5.545,5.298,4.586,4.435,5.279,5.161,4.967,5.393,5.161,4.722,5.476,5.225,14.755,14.623,14.526,14.237,14.777,14.285,13.909,13.783,13.982,14.039,13.844,13.718,14.786,14.328,14.186,13.648,7.024,7.294,7.429,6.93,7.691,7.537,7.278,7.148,7.765,7.679,7.786,7.926,7.175,6.764,7.971,7.421,9.877,10.002,9.929,9.386,9.939,9.638,9.344,9.082,9.443,9.367,9.08,8.947,10.073,9.366,8.77,3.738,3.279,3.947,3.956,3.012,3.223,3.829,4.093,2.979,3.262,3.557,3.018,3.665,4.091,2.79,3.58,4.59,4.851,4.663,4.551,5.127,4.847,4.509,4.226,5.086,4.923,4.675,4.96,4.757,4.255,5.289,4.473,4.27,4.208,3.96,3.742,4.535,4.034,3.573,3.62,3.77,3.538,3.288,3.617,4.775,4.134,3.986,3.314,9.556,9.589,9.354,9.09,9.741,9.82,8.94,8.679,9.336,9.086,9.133,8.934,9.789,9.46,8.602,15.123,17.622,15.062,14.692,15.668,14.999,14.579,14.34,15.03,14.647,14.532,14.594,15.301,14.658,15.041,14.314,8.206,8.259,8.728,7.91,8.218,8.217,7.792,7.672,7.908,7.657,7.545,7.868,8.63,8.241,7.71,7.46,3.793,3.85,3.598,3.434,4.236,3.891,3.287,2.812,3.476,3.149,2.975,4.531,3.969,3.565,6.085,6.471,6.176,6.002,6.236,6.437,5.965,5.616,6.014,5.675,5.842,5.36,5.775,5.737,5.138,11.444,11.687,11.375,11.056,11.719,11.423,10.949,10.765,11.606,11.461,10.93,11.104,11.862,11.056,11.713,10.811,11.986,12.523,12.07,11.858,12.164,11.95,11.49,11.213,11.532,11.669,11.355,11.253,12.147,11.734,11.719,11.393,1.352,1.33,1.121,0.635,1.751,1.445,0.946,0.991,1.544,1.208,1.252,1.358,1.298,0.468,1.64,1.412,4.772,4.75,4.77,4.946,4.096,4.387,4.928,5.241,4.29,4.537,5.052,4.269,4.418,5.396,4.145,4.906,14.138,14.1,13.865,13.634,14.747,14.14,13.489,13.214,14.294,13.63,13.382,13.717,14.3,13.847,14.04,13.141,17.468,17.55,17.323,17.148,17.552,17.428,16.986,17.073,16.845,16.612,17.729,17.14,5.979,6.342,6.091,5.824,6.15,5.876,5.412,5.121,5.682,5.621,5.182,5.286,6.332,5.621,8.29,8.445,8.162,8.111,8.604,8.299,8.004,7.446,8.236,8.188,7.84,7.717,8.983,7.832,8.668,7.507,13.788,14.02,13.845,13.461,14.071,13.833,13.541,13.536,13.717,13.558,13.398,13.422,14.036,13.566,13.813,13.465,11.358,11.673,11.426,11.156,11.678,11.41,10.911,10.633,11.161,11.371,10.744,10.88,11.72,11.354,11.2,10.763,12.077,12.158,11.865,11.573,12.28,12.033,11.565,11.318,11.939,11.73,11.794,11.688,12.28,11.722,12.169,11.301,9.103,9.255,8.759,8.723,8.609,8.57,8.466,8.406,8.355,8.147,8.509,8.041,9.068,8.866,8.169,7.78,10.176,9.957,9.917,9.993,9.945,9.808,9.548,9.517,9.877,9.5,9.607,9.194,10.293,9.805,9.044,2.971,3.365,3.473,3.029,3.675,3.426,2.981,2.669,3.946,3.485,3.155,3.453,3.366,2.681,3.966,3.195,13.908,14.159,14.662,14.904,13.833,14.687,14.867,14.624,14.071,15.363,15.113,13.723,13.463,14.197,13.627,13.843,14.851,13.911,15.482,13.409,7.62,7.956,7.885,7.617,8.336,8.608,7.764,7.393,8.223,7.865,7.745,8.181,8.127,7.581,8.361,8.259,1.063,0.92,1.168,1.743,0.699,0.836,1.395,1.991,1.121,1.333,1.356,1.562,1.514,1.266,1.505,6.038,5.492,5.562,5.106,5.684,5.32,4.91,4.647,5.284,5.027,4.771,4.802,5.705,5.023,5.608,4.669,7.827,7.714,7.341,8.043,7.905,7.385,8.118,8.321,7.825,7.245,8.117],[7.928,7.554,7.401,7.194,7.138,7.248,null,null,6.648,6.734,6.647,6.307,7.966,7.259,6.996,6.955,7.712,7.868,7.499,7.525,7.806,7.57,7.03,6.698,7.306,7.057,6.808,7.022,7.958,7.219,7.336,6.7,13.515,13.711,13.549,13.233,13.686,13.676,13.019,12.939,13.44,13.313,13.106,13.296,14.049,13.241,13.516,12.943,5.473,5.061,5.094,4.914,4.98,4.499,6.187,4.313,5.076,4.422,3.284,2.919,2.978,3.104,2.767,2.68,2.923,2.502,2.625,2.366,2.577,2.129,3.497,2.947,2.442,2.312,5.188,5.276,5.027,4.7,5.935,5.688,4.976,4.825,5.448,5.33,5.133,5.56,5.551,5.134,5.645,5.391,15.986,15.854,15.757,15.468,16.008,15.516,15.14,15.014,15.213,15.27,15.075,14.949,16.017,15.559,15.417,14.879,6.527,6.846,6.769,6.27,7.067,6.877,6.535,6.375,7.105,6.998,6.992,7.245,6.727,6.104,7.311,6.627,10.928,11.053,10.98,10.437,10.99,10.689,10.395,10.133,10.494,10.418,10.131,9.998,11.124,10.417,9.821,4.729,4.27,4.853,4.644,4.003,4.032,4.439,4.654,3.788,4.071,4.118,3.791,4.656,4.779,3.599,4.141,5.227,5.488,5.29,5.178,5.754,5.474,5.136,4.853,5.713,5.55,5.3,5.585,5.394,4.892,5.916,5.098,5.438,5.376,5.128,4.91,5.68,5.179,4.718,4.788,4.915,4.683,4.433,4.561,5.943,5.302,5.131,4.459,10.745,10.778,10.543,10.279,10.93,11.009,10.129,9.868,10.525,10.275,10.322,10.123,10.978,10.649,9.791,16.295,18.673,16.234,15.864,16.809,16.171,15.751,15.512,16.202,15.819,15.704,15.766,16.473,15.83,16.213,15.486,9.437,9.451,9.959,9.141,9.41,9.409,8.984,8.903,9.1,8.849,8.737,9.06,9.861,9.472,8.902,8.652,4.965,5.022,4.77,4.606,5.408,5.063,4.459,3.984,4.648,4.321,4.143,5.703,5.141,4.733,7.316,7.702,7.407,7.233,7.467,7.668,7.196,6.847,7.245,6.906,7.073,6.591,7.006,6.968,6.369,12.633,12.876,12.564,12.245,12.908,12.612,12.138,11.954,12.795,12.65,12.119,12.293,13.051,12.245,12.902,12.0,13.217,13.754,13.301,13.089,13.395,13.181,12.721,12.444,12.763,12.9,12.586,12.484,13.378,12.965,12.95,12.624,2.577,2.148,2.324,1.69,2.388,2.082,1.728,1.66,2.181,1.825,1.763,1.839,2.444,1.767,2.277,1.923,5.115,5.093,5.113,5.148,4.414,4.628,4.925,5.238,4.287,4.534,5.049,4.266,4.761,5.4,4.142,4.903,15.189,15.151,14.916,14.685,15.798,15.191,14.54,14.265,15.345,14.681,14.433,14.768,15.351,14.898,15.091,14.192,18.519,18.601,18.374,18.199,18.603,18.479,18.037,18.124,17.896,17.663,18.78,18.191,7.151,7.514,7.263,6.996,7.322,7.048,6.584,6.293,6.838,6.793,6.354,6.458,7.504,6.793,9.479,9.634,9.334,9.283,9.776,9.471,9.176,8.618,9.408,9.347,9.012,8.873,10.155,9.021,9.824,8.663,14.977,15.209,15.034,14.65,15.26,15.022,14.73,14.725,14.906,14.747,14.587,14.611,15.225,14.755,15.002,14.654,12.53,12.833,12.598,12.328,12.729,12.461,12.083,11.805,12.212,12.422,11.903,11.931,12.845,12.526,12.251,11.814,13.128,13.209,12.916,12.624,13.331,13.084,12.616,12.369,12.99,12.781,12.845,12.739,13.331,12.773,13.22,12.352,10.353,10.486,9.99,9.992,9.84,9.801,9.735,9.675,9.586,9.378,9.778,9.272,10.299,10.135,9.4,9.049,11.407,11.188,11.148,11.224,11.176,11.039,10.779,10.748,11.108,10.731,10.838,10.425,11.524,11.036,10.275,3.789,4.002,4.1,3.666,4.302,4.053,3.608,3.296,4.573,4.112,3.782,4.078,4.003,3.318,4.593,3.82,14.958,15.209,15.712,15.954,14.883,15.738,15.918,15.675,15.122,16.414,16.164,14.774,14.514,15.248,14.678,14.894,15.902,14.962,16.533,14.46,8.369,8.705,8.522,8.242,9.085,9.245,8.389,8.018,8.848,8.49,8.37,8.806,8.876,8.206,8.986,8.884,1.728,1.062,1.31,0.986,0.989,0.863,0.647,0.938,0.644,0.322,0.295,0.17,1.114,0.627,0.432,7.23,6.684,6.754,6.298,6.873,6.509,6.102,5.839,6.473,6.216,5.96,5.991,6.897,6.215,6.797,5.858,8.217,8.104,7.731,8.433,8.295,7.744,8.508,8.711,8.088,7.635,8.507],[7.72,7.346,7.193,6.986,6.93,7.04,null,null,6.44,6.526,6.439,6.099,7.758,7.051,6.788,6.747,6.957,7.099,6.732,6.758,7.037,6.801,6.269,5.943,6.537,6.288,6.042,6.092,7.203,6.464,6.566,5.81,12.791,12.985,12.825,12.509,12.96,12.95,12.293,12.213,12.714,12.587,12.38,12.57,13.323,12.517,12.79,12.217,5.428,4.853,4.886,4.706,4.772,4.291,6.468,4.105,4.868,4.214,3.167,2.874,2.861,3.081,2.559,2.506,2.986,2.783,2.417,2.338,2.583,2.125,3.369,3.228,2.234,2.461,4.146,4.234,3.985,3.658,4.893,4.646,3.934,3.783,4.627,4.509,4.315,4.741,4.509,4.092,4.824,4.573,15.599,15.318,15.341,15.08,15.472,14.929,14.737,14.611,14.626,14.683,14.488,14.362,15.481,15.171,14.693,14.292,6.364,6.642,6.606,6.107,6.904,6.714,6.372,6.212,6.942,6.835,6.829,7.082,6.523,5.941,7.148,6.464,9.998,10.123,10.05,9.507,10.06,9.759,9.465,9.203,9.564,9.488,9.201,9.068,10.194,9.487,8.891,4.839,4.38,5.048,4.925,4.113,4.313,4.72,4.935,4.069,4.352,4.399,4.072,4.766,5.06,3.88,4.422,4.297,4.38,4.137,4.025,4.601,4.321,3.983,3.7,4.56,4.397,4.147,4.432,4.464,3.77,4.763,3.945,4.709,4.647,4.399,4.181,4.949,4.448,3.987,4.059,4.184,3.952,3.702,3.829,5.214,4.573,4.4,3.728,10.021,10.054,9.819,9.555,10.206,10.285,9.405,9.144,9.801,9.551,9.598,9.399,10.254,9.925,9.067,15.569,17.743,15.508,15.138,15.879,15.445,15.025,14.786,15.363,15.093,14.978,14.927,15.747,15.104,15.353,14.76,8.85,8.729,9.372,8.554,8.688,8.687,8.262,8.316,8.378,8.127,8.015,8.338,9.274,8.885,8.18,7.93,4.239,4.296,4.044,3.882,4.682,4.337,3.735,3.26,3.922,3.595,3.414,4.977,4.417,4.004,7.108,7.115,7.04,6.874,6.88,7.081,6.988,6.535,6.658,6.319,6.486,6.004,6.798,6.381,5.981,11.909,12.152,11.84,11.521,12.184,11.888,11.414,11.23,12.071,11.926,11.395,11.569,12.327,11.521,12.178,11.276,12.849,13.167,12.714,12.721,12.808,12.594,12.134,12.076,12.176,12.313,11.999,11.897,12.791,12.597,12.363,12.037,2.369,1.634,2.116,1.482,1.458,1.386,1.457,1.452,1.177,1.115,1.204,0.686,1.884,1.559,1.125,1.364,5.396,5.374,5.394,5.429,4.695,4.909,5.206,5.519,4.568,4.815,5.33,4.547,5.042,5.681,4.423,5.184,14.259,14.221,13.986,13.755,14.868,14.261,13.61,13.335,14.415,13.751,13.503,13.838,14.421,13.968,14.161,13.262,17.589,17.671,17.444,17.269,17.673,17.549,17.107,17.194,16.966,16.733,17.85,17.261,6.425,6.764,6.537,6.27,6.572,6.298,5.858,5.567,6.083,6.067,5.628,5.732,6.754,6.067,8.755,8.91,8.608,8.557,9.05,8.745,8.45,7.892,8.682,8.592,8.286,8.118,9.429,8.297,9.069,7.908,14.253,14.485,14.31,13.926,14.536,14.298,14.006,14.001,14.182,14.023,13.863,13.887,14.501,14.031,14.278,13.93,11.804,11.903,11.727,11.457,11.799,11.531,11.21,11.079,11.282,11.492,10.973,11.001,11.915,11.777,11.321,10.884,12.198,12.279,11.986,11.694,12.401,12.154,11.686,11.439,12.06,11.851,11.915,11.809,12.401,11.843,12.29,11.422,10.145,10.278,9.782,9.784,9.632,9.593,9.527,9.467,9.378,9.17,9.57,9.064,10.091,9.927,9.192,8.841,11.039,10.82,10.78,10.856,10.589,10.671,10.411,10.38,10.521,10.144,10.47,9.838,11.156,10.449,9.907,2.991,3.072,2.947,2.62,3.149,2.9,2.455,2.143,3.42,2.959,2.629,2.925,3.073,2.388,3.44,2.667,14.028,14.279,14.782,15.024,13.953,14.808,14.988,14.745,14.192,15.484,15.234,13.844,13.584,14.318,13.748,13.964,14.972,14.032,15.603,13.53,7.427,7.763,7.406,7.089,8.084,8.129,7.236,6.865,7.695,7.337,7.217,7.653,7.934,7.053,7.833,7.731,1.554,1.01,1.258,1.813,0.718,0.811,1.465,2.061,0.874,1.134,1.426,1.151,1.717,0.697,1.565,6.508,5.962,6.032,5.576,6.149,5.785,5.38,5.117,5.749,5.492,5.236,5.267,6.175,5.493,6.073,5.134,7.175,7.062,6.689,7.391,7.253,6.733,7.466,7.669,7.173,6.593,7.465],[8.842,8.314,8.315,8.108,7.898,8.008,null,null,7.562,7.648,7.561,7.221,8.748,8.173,7.724,7.869,6.365,6.481,6.138,6.164,6.306,6.183,5.677,5.351,5.881,5.694,5.45,5.464,6.59,5.872,5.866,5.218,12.349,12.32,12.381,11.996,12.282,12.272,11.626,11.546,12.036,11.909,11.713,11.892,12.658,12.107,12.139,11.55,6.55,5.975,6.008,5.828,5.894,5.413,7.849,5.227,5.99,5.336,4.289,3.996,3.983,4.203,3.681,3.628,4.127,4.152,3.539,3.46,3.724,3.266,4.491,4.433,3.356,3.71,2.695,2.783,2.534,2.207,3.442,3.195,2.483,2.332,3.176,3.058,2.864,3.29,3.058,2.641,3.373,3.122,15.465,15.148,15.178,14.949,15.302,14.764,14.627,14.607,14.463,14.542,14.368,14.037,15.311,15.04,14.307,14.219,4.921,5.191,5.326,4.827,5.588,5.434,5.175,5.045,5.662,5.576,5.683,5.823,5.072,4.661,5.868,5.318,9.22,9.345,9.272,8.729,9.282,8.981,8.687,8.425,8.786,8.71,8.423,8.29,9.416,8.709,8.113,5.961,5.502,6.17,6.179,5.235,5.446,6.052,6.316,5.202,5.485,5.78,5.241,5.888,6.314,5.013,5.803,3.439,3.49,3.247,3.135,3.646,3.365,2.914,2.52,3.244,2.983,2.697,2.982,3.679,2.88,3.349,2.495,4.398,4.336,4.088,3.87,4.581,4.08,3.619,3.75,3.816,3.584,3.334,3.46,4.903,4.262,4.032,3.36,9.635,9.644,9.433,9.169,9.796,9.875,9.019,8.758,9.396,9.165,9.188,8.989,9.844,9.515,8.657,14.918,16.965,14.829,14.487,15.101,14.71,14.346,14.135,14.585,14.358,14.243,14.149,15.068,14.453,14.575,14.081,9.438,9.31,9.96,9.142,9.269,9.268,8.843,8.904,8.959,8.708,8.596,8.919,9.862,9.473,8.761,8.511,4.309,4.259,4.114,4.297,4.492,4.158,3.921,3.92,3.758,3.665,3.25,4.91,4.832,3.702,7.827,7.703,7.628,7.462,7.468,7.669,7.583,7.146,7.246,6.907,7.074,6.615,7.517,6.969,6.678,11.523,11.766,11.454,11.135,11.798,11.502,11.028,10.844,11.685,11.516,11.009,11.159,11.941,11.135,11.792,10.89,13.437,13.503,13.302,13.309,13.144,13.182,12.722,12.664,12.76,12.901,12.587,12.485,13.379,13.185,12.733,12.625,3.305,2.367,2.87,2.418,2.118,2.119,2.217,2.304,1.91,1.848,1.937,1.543,2.617,2.495,1.758,2.097,6.823,6.801,6.821,6.856,6.122,6.336,6.633,6.946,5.995,6.242,6.757,5.974,6.469,7.108,5.85,6.611,13.481,13.443,13.208,12.977,14.09,13.483,12.832,12.557,13.637,12.973,12.725,13.06,13.643,13.19,13.383,12.484,16.811,16.893,16.666,16.491,16.895,16.771,16.329,16.416,16.188,15.955,17.072,16.483,6.091,6.172,6.057,6.074,5.98,5.706,5.733,5.499,5.491,5.563,5.381,5.403,6.162,5.942,8.345,8.455,8.136,8.085,8.479,8.213,7.978,7.42,8.15,8.0,7.814,7.526,8.858,7.887,8.324,7.316,13.867,14.099,13.924,13.54,14.15,13.912,13.62,13.615,13.796,13.637,13.477,13.501,14.115,13.645,13.868,13.544,11.124,11.125,10.949,10.679,11.021,10.753,10.432,10.399,10.504,10.714,10.195,10.223,11.137,10.999,10.543,10.106,11.42,11.501,11.208,10.916,11.623,11.376,10.908,10.661,11.282,11.073,11.137,11.031,11.623,11.065,11.512,10.644,11.063,11.196,10.7,10.702,10.55,10.511,10.445,10.385,10.296,10.088,10.488,9.982,11.009,10.845,10.11,9.759,11.627,11.408,11.368,11.444,11.177,11.259,10.999,10.968,11.109,10.732,11.058,10.426,11.744,11.037,10.495,2.622,2.662,2.156,1.829,2.358,2.109,1.664,1.354,2.082,1.79,1.838,1.475,2.663,2.019,2.427,1.217,13.25,13.501,14.004,14.246,13.175,14.03,14.21,13.967,13.414,14.706,14.456,13.066,12.806,13.54,12.97,13.186,14.194,13.254,14.825,12.752,5.977,6.313,5.956,5.639,6.634,6.679,5.786,5.415,6.245,5.887,5.767,6.203,6.484,5.603,6.383,6.281,2.695,2.437,2.685,3.241,1.886,2.239,2.893,3.473,2.302,2.562,2.854,2.579,3.144,2.125,2.965,7.089,6.38,6.616,5.994,6.259,6.042,5.777,5.548,5.801,5.845,5.589,5.319,6.593,6.081,6.125,5.487,5.724,5.611,5.238,5.94,5.802,5.282,6.015,6.218,5.722,5.142,6.014],[9.221,8.512,8.648,8.981,8.096,8.206,null,null,8.054,8.304,8.44,8.098,8.945,9.046,7.922,8.748,4.593,4.707,4.364,4.39,4.532,4.409,3.905,3.579,4.107,3.92,3.678,3.69,4.816,4.1,4.092,3.446,10.573,10.544,10.607,10.222,10.506,10.496,9.852,9.772,10.26,10.133,9.939,10.116,10.882,10.335,10.336,9.776,7.932,7.18,7.39,6.784,7.276,6.795,9.231,6.609,7.173,6.718,5.671,5.378,5.365,5.585,5.063,5.01,5.561,5.832,4.921,5.094,5.404,5.131,5.873,5.815,4.738,5.645,0.827,1.278,1.099,1.316,2.119,1.857,1.729,1.833,1.856,1.738,1.988,1.97,1.239,1.515,2.053,1.936,13.693,13.376,13.406,13.177,13.53,12.992,12.855,12.835,12.691,12.77,12.596,12.265,13.539,13.268,12.535,12.447,3.601,3.871,4.006,3.507,4.268,4.114,3.855,3.725,4.342,4.256,4.48,4.503,3.752,3.341,4.548,4.144,7.446,7.571,7.498,6.955,7.508,7.207,6.913,6.651,7.012,6.936,6.649,6.516,7.642,6.935,6.339,7.343,6.884,7.552,7.561,6.617,6.828,7.434,7.698,6.584,6.867,7.162,6.623,7.27,7.696,6.395,7.185,1.665,1.704,1.416,1.304,1.79,1.509,1.014,0.62,1.318,0.993,0.707,0.992,1.905,1.096,1.359,0.505,3.152,2.883,3.032,3.346,2.846,2.622,2.697,3.573,2.395,2.526,2.81,2.244,3.45,3.603,2.574,2.836,7.863,7.872,7.661,7.397,8.024,8.103,7.247,6.986,7.624,7.393,7.416,7.217,8.072,7.743,6.885,13.115,15.191,13.054,12.684,13.327,12.936,12.571,12.332,12.811,12.584,12.469,12.375,13.293,12.65,12.801,12.306,8.671,8.373,9.197,8.375,8.202,8.193,7.906,8.137,7.907,7.656,7.648,7.867,8.931,8.71,7.709,7.459,4.935,4.661,4.798,5.127,3.968,4.171,4.649,4.922,4.16,4.435,3.923,4.718,5.465,3.715,7.391,6.941,7.115,6.984,6.702,7.156,7.105,7.122,6.628,6.429,6.596,6.292,7.124,6.231,6.901,9.751,9.994,9.682,9.363,10.026,9.73,9.256,9.072,9.913,9.744,9.237,9.387,10.169,9.363,10.02,9.118,12.013,11.731,11.822,11.935,11.372,11.702,11.294,11.313,10.988,11.473,11.184,11.034,11.844,11.811,10.961,11.174,4.521,3.569,4.082,4.013,3.266,3.321,3.812,4.162,3.084,3.438,3.795,3.443,3.791,4.09,2.906,3.955,8.408,8.47,8.49,8.666,7.816,8.107,8.785,9.098,8.147,8.394,8.909,8.126,8.054,9.116,8.002,8.763,11.707,11.669,11.434,11.203,12.316,11.709,11.058,10.783,11.863,11.199,10.951,11.286,11.869,11.416,11.609,10.71,15.037,15.119,14.892,14.717,15.121,14.997,14.555,14.642,14.414,14.181,15.298,14.709,4.319,4.4,4.285,4.302,4.208,3.934,3.961,4.044,3.719,3.791,3.609,3.631,4.39,4.17,6.573,6.683,6.364,6.313,6.705,6.441,6.206,5.648,6.378,6.228,6.042,5.754,7.084,6.115,6.55,5.544,12.095,12.327,12.152,11.768,12.378,12.14,11.848,11.843,12.024,11.865,11.705,11.729,12.343,11.873,12.096,11.772,9.35,9.351,9.175,8.905,9.247,8.979,8.658,8.625,8.73,8.94,8.421,8.449,9.363,9.225,8.769,8.332,9.646,9.727,9.434,9.142,9.849,9.602,9.134,8.887,9.508,9.299,9.363,9.257,9.849,9.291,9.738,8.87,10.323,10.456,9.96,10.171,9.81,9.771,9.822,9.994,9.556,9.348,10.097,9.242,10.269,10.314,9.37,9.105,10.864,10.444,10.497,10.681,9.984,10.295,10.236,10.205,9.828,9.969,10.295,9.659,10.609,9.691,9.732,2.391,1.882,1.793,2.067,1.057,1.551,1.845,2.351,1.058,1.306,2.093,1.093,1.685,2.367,1.078,1.599,11.476,11.727,12.23,12.472,11.401,12.256,12.436,12.193,11.64,12.932,12.682,11.292,11.032,11.766,11.196,11.412,12.42,11.48,13.051,10.978,3.987,4.323,3.966,3.649,4.644,4.689,3.796,3.425,4.255,3.897,3.777,4.213,4.494,3.613,4.393,4.291,5.144,4.937,5.185,5.741,4.386,4.739,5.393,5.976,4.802,5.062,5.354,5.079,5.595,4.625,5.468,6.152,5.432,5.88,5.516,4.997,5.088,5.299,5.528,4.847,5.367,5.413,4.825,5.374,5.683,5.003,5.467,3.999,3.864,3.893,4.215,4.311,3.937,4.67,4.873,4.377,3.67,4.669],[8.978,8.269,8.405,8.738,7.853,7.963,null,null,7.811,8.061,8.197,7.855,8.703,8.803,7.679,8.505,3.005,3.083,2.74,2.766,2.908,2.785,2.316,2.063,2.483,2.296,2.089,2.066,3.192,2.585,2.468,1.857,8.949,8.92,8.983,8.598,8.882,8.872,8.228,8.148,8.636,8.509,8.315,8.492,9.258,8.719,8.712,8.152,7.842,7.07,7.317,6.674,7.33,6.825,9.532,6.91,7.063,6.748,5.972,5.679,5.666,5.886,5.364,5.311,5.862,6.133,5.222,5.395,5.705,5.432,6.174,6.116,5.039,5.952,2.648,2.811,2.92,3.134,3.025,2.763,3.49,3.404,3.108,3.271,3.804,3.503,2.492,3.084,2.961,3.752,12.37,12.053,12.083,11.854,12.207,11.669,11.532,11.512,11.368,11.447,11.273,10.942,12.216,11.945,11.212,11.124,4.469,4.698,4.916,4.454,5.072,5.022,4.802,4.672,5.25,5.166,5.427,5.413,4.579,4.251,5.456,5.091,5.822,5.947,5.874,5.331,5.884,5.583,5.289,5.027,5.388,5.312,5.025,4.892,6.018,5.311,4.715,7.644,7.185,7.853,7.862,6.918,7.129,7.735,7.999,6.885,7.168,7.463,6.924,7.571,7.997,6.696,7.486,0.521,0.34,0.516,0.794,0.467,0.631,0.989,1.303,0.614,1.103,1.224,1.073,0.089,0.991,0.775,1.424,2.855,2.403,2.745,3.319,2.154,2.142,2.669,3.547,1.917,2.24,2.783,1.958,2.87,3.314,2.094,2.809,6.54,6.395,6.338,6.074,6.547,6.626,5.896,5.663,6.147,6.07,5.939,5.74,6.595,6.266,5.429,11.491,13.567,11.43,11.06,11.703,11.312,10.947,10.708,11.187,10.96,10.845,10.751,11.669,11.026,11.177,10.682,7.636,7.231,8.326,7.647,6.879,6.87,7.134,7.681,6.747,6.789,7.083,7.0,7.789,7.983,6.549,6.979,4.908,4.64,4.777,5.106,3.941,4.146,4.628,4.903,4.139,4.416,3.904,4.691,5.438,3.69,7.054,6.549,6.778,6.77,6.31,6.819,6.891,6.989,6.307,6.215,6.353,6.078,6.91,5.91,6.768,8.428,8.596,8.359,8.04,8.628,8.332,7.933,7.749,8.493,8.267,7.914,7.91,8.771,8.04,8.6,7.698,10.69,10.408,10.499,10.612,10.049,10.379,9.971,9.99,9.665,10.15,9.861,9.711,10.521,10.488,9.638,9.851,4.822,3.811,4.383,4.314,3.402,3.622,4.113,4.469,3.575,3.934,4.102,3.978,3.931,4.391,3.52,4.262,8.709,8.771,8.791,8.967,8.117,8.408,9.086,9.399,8.448,8.695,9.21,8.427,8.355,9.417,8.303,9.064,10.083,10.045,9.81,9.579,10.692,10.085,9.434,9.159,10.239,9.575,9.327,9.662,10.245,9.792,9.985,9.086,13.413,13.495,13.268,13.093,13.497,13.373,12.931,13.018,12.79,12.557,13.674,13.085,3.627,3.708,3.593,3.61,3.235,3.242,3.269,3.372,3.027,3.099,2.917,2.939,3.698,3.478,5.049,5.097,4.785,4.749,5.081,4.823,4.642,4.144,4.76,4.61,4.478,4.145,5.46,4.624,4.926,3.959,10.772,10.929,10.829,10.445,10.98,10.742,10.525,10.52,10.626,10.467,10.307,10.331,10.945,10.55,10.619,10.449,7.726,7.727,7.551,7.281,7.623,7.355,7.034,7.001,7.106,7.316,6.797,6.825,7.739,7.601,7.145,6.708,8.022,8.103,7.81,7.518,8.225,7.978,7.51,7.263,7.884,7.675,7.739,7.633,8.225,7.667,8.114,7.246,9.868,10.001,9.505,9.716,9.355,9.316,9.367,9.722,9.101,8.893,9.678,8.787,9.814,9.859,8.915,8.65,9.822,9.121,9.174,9.639,8.661,8.972,9.194,9.163,8.505,8.857,9.253,8.522,9.286,8.368,8.691,2.372,1.857,2.154,2.428,1.771,1.895,2.526,2.913,1.832,2.08,2.774,2.229,1.66,2.351,1.852,2.735,9.852,10.103,10.606,10.848,9.777,10.632,10.812,10.569,10.016,11.308,11.058,9.668,9.408,10.142,9.572,9.788,10.796,9.856,11.427,9.354,3.169,3.505,3.195,3.037,3.873,3.918,3.277,2.906,3.645,3.378,3.258,3.694,3.676,3.002,3.783,3.772,5.451,5.244,5.492,6.063,4.693,5.047,5.715,6.311,5.115,5.384,5.676,5.556,5.902,5.125,5.815,5.788,4.952,5.559,5.406,4.517,4.767,5.189,5.501,4.558,5.293,5.386,4.798,4.894,5.573,4.523,5.44,4.152,4.017,4.18,4.368,4.464,4.224,4.957,5.16,4.664,3.823,4.956],[9.887,9.153,9.314,9.563,8.761,8.872,null,null,8.697,8.947,9.016,8.674,9.545,9.628,8.588,9.324,6.305,6.421,6.078,6.104,6.246,6.123,5.617,5.291,5.821,5.634,5.39,5.404,6.53,5.812,5.806,5.158,12.229,12.2,12.263,11.878,12.162,12.152,11.508,11.428,11.916,11.789,11.595,11.772,12.538,11.999,11.992,11.432,8.091,7.516,7.549,7.306,7.435,6.954,9.39,6.768,7.531,6.877,5.83,5.537,5.524,5.744,5.222,5.169,5.72,5.788,5.08,5.05,5.36,5.087,6.032,5.974,4.897,5.585,1.108,1.196,0.947,0.62,1.501,1.556,0.543,0.08,0.924,0.806,0.612,1.038,1.471,1.054,1.121,0.87,15.405,15.088,15.118,14.889,15.242,14.704,14.567,14.547,14.403,14.482,14.308,13.977,15.251,14.98,14.247,14.159,2.669,2.939,3.074,2.575,3.336,3.182,2.923,2.793,3.41,3.324,3.431,3.571,2.82,2.409,3.616,3.066,8.981,9.106,9.033,8.498,9.043,8.742,8.448,8.194,8.547,8.471,8.184,8.051,9.177,8.478,7.882,7.502,7.043,7.711,7.72,6.776,6.987,7.593,7.857,6.743,7.026,7.321,6.782,7.429,7.855,6.554,7.344,3.379,3.43,3.187,3.075,3.586,3.305,2.847,2.453,3.141,2.817,2.537,2.679,3.619,2.82,3.18,2.338,4.62,4.422,4.381,4.445,4.558,4.161,4.046,4.326,3.902,3.875,3.909,3.593,4.989,4.837,4.113,3.935,9.575,9.584,9.373,9.109,9.736,9.815,8.959,8.698,9.336,9.105,9.128,8.929,9.784,9.455,8.597,14.771,16.726,14.59,14.34,14.862,14.471,14.107,13.988,14.346,14.119,14.004,13.91,14.829,14.306,14.336,13.842,10.014,9.722,10.536,9.718,9.681,9.68,9.255,9.48,9.371,9.12,9.008,9.331,10.28,10.049,9.173,8.923,5.472,5.14,5.277,5.593,5.067,4.734,5.128,5.227,4.639,4.914,4.402,5.486,6.012,4.278,8.403,8.279,8.204,8.038,8.044,8.245,8.159,7.722,7.822,7.483,7.65,7.191,8.093,7.545,7.501,11.463,11.706,11.394,11.075,11.738,11.442,10.968,10.784,11.625,11.456,10.949,11.099,11.881,11.075,11.732,10.83,13.725,13.443,13.534,13.647,13.084,13.414,13.006,13.025,12.7,13.185,12.896,12.746,13.556,13.523,12.673,12.886,4.68,3.728,4.241,3.955,3.425,3.48,3.754,4.102,3.243,3.378,3.735,3.383,3.95,4.032,3.065,3.895,8.567,8.563,8.583,8.759,7.909,8.2,8.741,9.054,8.103,8.35,8.865,8.082,8.213,9.209,7.958,8.719,13.242,13.204,12.969,12.738,13.851,13.244,12.593,12.318,13.398,12.734,12.486,12.821,13.404,12.951,13.144,12.245,16.572,16.654,16.427,16.252,16.656,16.532,16.09,16.177,15.949,15.716,16.833,16.244,6.031,6.112,5.997,6.014,5.92,5.646,5.673,5.583,5.431,5.503,5.321,5.343,6.102,5.882,8.285,8.395,8.076,8.025,8.39,8.153,7.918,7.36,8.09,7.94,7.754,7.466,8.798,7.827,8.206,7.256,13.807,14.039,13.864,13.48,14.09,13.852,13.56,13.555,13.736,13.577,13.417,13.441,14.055,13.585,13.808,13.484,10.893,10.886,10.71,10.44,10.782,10.514,10.193,10.281,10.265,10.475,9.956,9.984,10.898,10.76,10.304,9.867,11.181,11.262,10.969,10.677,11.384,11.137,10.669,10.422,11.043,10.834,10.898,10.792,11.384,10.826,11.273,10.405,11.639,11.772,11.276,11.278,11.126,11.087,11.021,10.961,10.872,10.664,11.064,10.558,11.585,11.421,10.686,10.335,12.203,11.984,11.944,12.02,11.696,11.835,11.575,11.544,11.54,11.308,11.634,11.002,12.32,11.403,11.071,2.954,2.984,2.356,2.546,2.406,2.309,2.318,2.645,2.132,1.869,2.566,1.652,2.985,2.846,2.427,2.007,12.893,13.144,13.647,13.889,12.818,13.791,13.971,13.728,13.175,14.467,14.217,12.827,12.567,13.301,12.731,12.947,13.955,13.015,14.586,12.513,5.318,5.65,5.297,4.979,5.968,6.013,5.122,4.745,5.58,5.22,4.909,5.178,5.821,5.002,5.716,5.098,5.032,4.774,5.022,5.578,4.223,4.576,5.23,5.466,4.639,4.899,5.032,4.749,5.481,4.462,4.958,7.501,6.792,7.192,6.57,6.505,6.437,6.353,6.124,6.196,6.421,6.165,5.895,6.882,6.657,6.487,6.063,4.109,3.996,3.623,4.325,4.067,3.22,4.385,4.588,3.564,3.527,4.21],[10.597,9.863,10.024,10.277,9.471,9.582,null,null,9.411,9.661,9.73,9.388,10.255,10.342,9.298,10.038,5.238,5.245,4.973,4.999,4.908,4.947,4.556,4.387,4.645,4.529,4.329,4.299,5.354,4.909,4.576,4.097,10.85,10.821,10.884,10.499,10.783,10.773,10.129,10.049,10.537,10.41,10.216,10.393,11.159,10.62,10.613,10.053,8.805,8.23,8.263,8.02,8.149,7.668,10.104,7.482,8.245,7.591,6.544,6.251,6.238,6.458,5.936,5.883,6.434,6.651,5.794,5.913,6.223,5.95,6.746,6.688,5.611,6.448,0.929,0.881,0.935,1.052,0.854,0.678,1.468,1.572,1.023,1.187,1.721,1.419,0.72,1.443,0.876,1.669,14.597,14.28,14.31,14.081,14.434,13.896,13.759,13.739,13.595,13.674,13.5,13.169,14.443,14.172,13.439,13.351,2.424,2.684,2.831,2.369,3.081,2.939,2.717,2.587,3.167,3.081,3.342,3.328,2.565,2.166,3.373,3.006,7.602,7.727,7.654,7.119,7.664,7.363,7.069,6.815,7.168,7.092,6.805,6.672,7.798,7.099,6.503,8.216,7.757,8.425,8.434,7.49,7.701,8.307,8.571,7.457,7.74,8.035,7.496,8.143,8.569,7.268,8.058,2.436,2.436,2.153,2.383,2.522,2.241,2.093,2.268,2.05,1.727,1.783,1.588,2.676,2.744,2.089,1.725,4.8,4.459,4.683,4.997,4.21,4.198,4.348,5.036,3.973,4.177,4.461,3.895,4.926,5.254,4.15,4.487,8.767,8.635,8.565,8.301,8.787,8.866,8.151,7.89,8.387,8.297,8.179,7.98,8.835,8.506,7.789,13.392,15.347,13.211,12.961,13.483,13.092,12.728,12.609,12.967,12.74,12.625,12.531,13.45,12.927,12.957,12.463,9.863,9.458,10.553,9.874,9.106,9.097,9.361,9.737,8.974,9.016,9.224,9.227,10.016,10.21,8.776,9.035,6.186,5.854,5.991,6.307,5.619,5.444,5.842,5.941,5.353,5.628,5.116,6.196,6.726,4.988,9.042,8.592,8.766,8.635,8.353,8.807,8.756,8.432,8.279,8.08,8.247,7.901,8.775,7.882,8.211,10.655,10.836,10.586,10.267,10.868,10.572,10.16,9.976,10.733,10.507,10.141,10.15,11.011,10.267,10.84,9.938,12.917,12.635,12.726,12.839,12.276,12.606,12.198,12.217,11.892,12.377,12.088,11.938,12.748,12.715,11.865,12.078,5.394,4.442,4.955,4.818,4.139,4.194,4.617,4.965,3.957,4.241,4.598,4.246,4.664,4.895,3.779,4.758,9.281,9.343,9.363,9.539,8.689,8.98,9.604,9.917,8.966,9.213,9.728,8.945,8.927,9.989,8.821,9.582,11.863,11.825,11.59,11.359,12.472,11.865,11.214,10.939,12.019,11.355,11.107,11.442,12.025,11.572,11.765,10.866,15.193,15.275,15.048,14.873,15.277,15.153,14.711,14.798,14.57,14.337,15.454,14.865,5.683,5.764,5.649,5.666,5.462,5.298,5.325,5.428,5.083,5.155,4.973,4.995,5.754,5.534,7.282,7.259,6.947,6.982,7.011,6.985,6.875,6.384,6.922,6.772,6.647,6.307,7.46,6.864,6.827,6.192,12.999,13.169,13.056,12.672,13.22,12.982,12.752,12.747,12.866,12.707,12.547,12.571,13.185,12.777,12.859,12.676,9.514,9.507,9.331,9.061,9.403,9.135,8.814,8.902,8.886,9.096,8.577,8.605,9.519,9.381,8.925,8.488,9.802,9.883,9.59,9.298,10.005,9.758,9.29,9.043,9.664,9.455,9.519,9.413,10.005,9.447,9.894,9.026,11.924,12.057,11.561,11.772,11.411,11.372,11.423,11.645,11.157,10.949,11.734,10.843,11.87,11.915,10.971,10.706,12.049,11.348,11.401,11.866,10.888,11.199,11.421,11.39,10.732,11.084,11.48,10.749,11.513,10.595,10.918,3.664,3.533,3.066,3.26,2.708,3.019,3.032,3.359,2.588,2.579,3.28,2.362,3.336,3.56,2.729,2.721,11.514,11.765,12.268,12.51,11.439,12.412,12.592,12.349,11.796,13.088,12.838,11.448,11.188,11.922,11.352,11.568,12.576,11.636,13.207,11.134,3.997,4.333,3.976,3.659,4.654,4.699,3.806,3.435,4.265,3.907,3.612,3.881,4.504,3.623,4.403,3.801,5.895,5.637,5.885,6.441,5.086,5.439,6.093,6.474,5.502,5.762,5.895,5.612,6.344,5.325,5.966,7.803,7.008,7.531,7.167,6.573,6.739,6.95,6.834,6.498,7.018,6.875,6.476,6.95,7.334,6.579,6.773,2.927,2.814,2.441,3.143,3.005,2.485,3.218,3.421,2.925,2.345,3.217],[11.657,10.923,11.084,11.333,10.531,10.642,null,null,10.467,10.717,10.786,10.444,11.315,11.398,10.358,11.094,7.63,7.637,7.365,7.391,7.3,7.339,6.948,6.769,7.037,6.921,6.721,6.691,7.746,7.29,6.968,6.489,13.242,13.213,13.276,12.891,13.175,13.165,12.521,12.441,12.929,12.802,12.608,12.785,13.551,13.012,13.005,12.445,9.861,9.286,9.319,9.076,9.205,8.724,11.16,8.538,9.301,8.647,7.6,7.307,7.294,7.514,6.992,6.939,7.49,7.558,6.85,6.82,7.13,6.857,7.802,7.744,6.667,7.355,2.583,2.669,2.419,2.098,2.831,2.908,1.902,1.85,2.274,2.159,1.658,1.772,2.941,2.532,2.471,1.606,16.883,16.566,16.596,16.367,16.72,16.182,16.045,16.025,15.881,15.96,15.786,15.455,16.729,16.458,15.725,15.637,1.633,1.923,1.53,1.029,1.748,1.522,1.061,0.901,1.65,1.524,1.518,1.771,1.804,0.998,1.908,1.153,9.994,10.119,10.046,9.511,10.056,9.755,9.461,9.207,9.56,9.484,9.197,9.064,10.19,9.491,8.895,9.272,8.813,9.481,9.49,8.546,8.757,9.363,9.627,8.513,8.796,9.091,8.552,9.199,9.625,8.324,9.114,4.857,4.908,4.665,4.553,5.064,4.783,4.318,3.928,4.612,4.288,4.008,4.15,5.097,4.298,4.651,3.813,6.12,5.922,5.881,6.195,6.036,5.661,5.546,6.096,5.402,5.375,5.659,5.093,6.489,6.452,5.613,5.685,11.053,11.027,10.851,10.587,11.179,11.258,10.437,10.176,10.779,10.583,10.571,10.372,11.227,10.898,10.075,15.784,17.739,15.603,15.353,15.875,15.484,15.12,15.001,15.359,15.132,15.017,14.923,15.842,15.319,15.349,14.855,11.52,11.222,12.046,11.224,11.181,11.18,10.755,10.986,10.871,10.62,10.508,10.831,11.78,11.559,10.673,10.423,7.242,6.91,7.047,7.363,6.817,6.504,6.898,6.997,6.409,6.684,6.172,7.256,7.782,6.048,10.173,9.79,9.964,9.808,9.551,10.005,9.929,9.492,9.477,9.253,9.42,8.961,9.863,9.08,9.271,12.941,13.184,12.872,12.553,13.216,12.92,12.446,12.262,13.103,12.899,12.427,12.542,13.359,12.553,13.21,12.308,15.203,14.921,15.012,15.125,14.562,14.892,14.484,14.503,14.178,14.663,14.374,14.224,15.034,15.001,14.151,14.364,6.45,5.498,6.011,5.725,5.195,5.25,5.524,5.872,5.013,5.148,5.505,5.153,5.72,5.802,4.835,5.665,10.337,10.333,10.353,10.529,9.679,9.97,10.389,10.702,9.751,9.998,10.513,9.73,9.983,10.864,9.62,10.367,14.255,14.217,13.982,13.751,14.864,14.257,13.606,13.331,14.411,13.747,13.499,13.834,14.417,13.964,14.157,13.258,17.585,17.667,17.44,17.265,17.669,17.545,17.103,17.19,16.962,16.729,17.846,17.257,7.509,7.59,7.475,7.492,7.398,7.124,7.151,7.083,6.909,6.981,6.799,6.821,7.58,7.36,9.674,9.651,9.376,9.381,9.403,9.414,9.267,8.776,9.337,9.164,9.039,8.699,9.852,9.256,9.219,8.584,15.285,15.517,15.342,14.958,15.568,15.33,15.038,15.033,15.214,15.055,14.895,14.919,15.533,15.063,15.251,14.962,11.906,11.899,11.723,11.453,11.795,11.527,11.206,11.294,11.278,11.488,10.969,10.997,11.911,11.773,11.317,10.88,12.194,12.275,11.982,11.69,12.397,12.15,11.682,11.435,12.056,11.847,11.911,11.756,12.397,11.839,12.286,11.39,13.172,13.305,12.809,13.02,12.659,12.62,12.671,12.731,12.405,12.197,12.834,12.091,13.118,13.163,12.219,11.954,13.713,13.494,13.454,13.53,13.174,13.345,13.085,13.054,13.018,12.818,13.144,12.508,13.799,12.881,12.581,4.724,4.731,4.126,4.316,3.906,4.079,4.088,4.415,3.679,3.639,4.336,3.422,4.534,4.616,3.927,3.777,13.826,14.077,14.58,14.822,13.751,14.804,14.984,14.741,14.188,15.48,15.23,13.84,13.58,14.314,13.744,13.96,14.968,14.028,15.599,13.526,6.002,6.334,5.981,5.663,6.652,6.697,5.806,5.429,6.264,5.904,5.593,5.862,6.505,5.811,6.4,5.782,6.76,6.212,6.46,6.5,5.924,6.013,6.16,6.235,6.076,5.829,5.809,5.526,6.628,5.899,5.727,9.001,8.292,8.729,8.34,8.005,7.937,8.123,7.894,7.696,8.191,7.935,7.665,8.382,8.427,7.987,7.833,4.683,4.57,4.197,4.899,4.638,3.791,4.956,5.054,4.135,4.101,4.781],[13.068,12.334,12.495,12.744,11.942,12.053,null,null,11.878,12.128,12.197,11.855,12.726,12.809,11.769,12.505,7.119,7.126,6.854,6.88,6.789,6.828,6.438,6.269,6.526,6.41,6.211,6.18,7.235,6.791,6.469,5.979,12.731,12.702,12.765,12.38,12.664,12.654,12.01,11.93,12.418,12.291,12.097,12.142,13.04,12.501,12.362,11.934,11.272,10.697,10.73,10.487,10.616,10.135,12.571,9.949,10.712,10.058,9.011,8.718,8.705,8.925,8.403,8.35,8.901,8.969,8.261,8.231,8.541,8.268,9.213,9.155,8.078,8.766,3.43,3.382,3.266,3.383,2.648,2.893,3.313,3.261,2.554,2.566,2.791,2.49,3.221,3.941,2.464,2.739,16.769,16.452,16.482,16.253,16.606,16.068,15.931,15.911,15.767,15.846,15.672,15.341,16.615,16.344,15.611,15.523,1.112,1.507,0.838,0.888,1.332,1.106,0.497,0.667,1.234,1.108,0.226,1.355,1.388,0.996,1.492,0.139,9.333,9.458,9.385,8.85,9.395,9.094,8.8,8.546,8.899,8.823,8.536,8.403,9.529,8.83,8.234,10.683,10.224,10.892,10.901,9.957,10.168,10.774,11.038,9.924,10.207,10.502,9.963,10.61,11.036,9.735,10.525,5.148,5.148,4.865,4.888,4.98,4.953,4.598,4.773,4.762,4.439,4.288,4.3,5.25,5.249,4.801,4.23,7.305,6.978,7.187,7.501,6.779,6.717,6.852,7.507,6.49,6.681,6.965,6.399,7.495,7.758,6.669,6.991,10.794,10.517,10.592,10.435,10.669,10.748,10.212,10.358,10.269,10.244,10.061,9.862,10.717,10.388,9.745,15.273,17.078,14.942,14.71,15.214,14.823,14.459,14.358,14.698,14.471,14.356,14.262,15.181,14.808,14.688,14.194,12.331,11.926,13.021,12.342,11.574,11.565,11.829,12.256,11.442,11.484,11.743,11.695,12.484,12.678,11.244,11.554,8.653,8.321,8.458,8.774,8.123,7.915,8.309,8.408,7.82,8.095,7.583,8.667,9.193,7.459,11.546,11.096,11.27,11.139,10.857,11.311,11.26,10.903,10.783,10.584,10.751,10.372,11.274,10.386,10.682,12.827,12.718,12.709,12.439,12.75,12.454,12.283,12.148,12.615,12.389,12.264,12.032,12.893,12.439,12.722,11.82,15.089,14.807,14.898,15.011,14.448,14.778,14.37,14.389,14.064,14.549,14.26,14.11,14.92,14.887,14.037,14.25,7.861,6.909,7.422,7.136,6.606,6.661,6.935,7.283,6.424,6.559,6.916,6.564,7.131,7.213,6.246,7.076,11.748,11.744,11.764,11.94,11.09,11.381,11.829,12.142,11.186,11.438,11.947,11.165,11.394,12.304,11.041,11.801,13.594,13.556,13.321,13.09,14.203,13.596,12.945,12.67,13.75,13.086,12.838,13.173,13.756,13.303,13.496,12.597,16.924,17.006,16.779,16.604,17.008,16.884,16.442,16.529,16.301,16.068,17.185,16.596,8.252,8.333,8.218,8.235,8.052,7.867,7.894,7.997,7.652,7.724,7.542,7.564,8.323,8.103,9.163,9.14,8.828,8.863,8.892,8.866,8.756,8.266,8.803,8.653,8.528,8.188,9.341,8.746,8.708,8.073,15.171,15.051,15.228,14.844,15.102,14.864,14.686,14.682,14.748,14.589,14.429,14.453,15.067,14.949,14.741,14.611,11.245,11.238,11.062,10.792,11.134,10.866,10.545,10.737,10.617,10.827,10.308,10.336,11.25,11.112,10.656,10.219,11.533,11.614,11.321,11.029,11.736,11.489,11.021,10.774,11.395,11.186,11.25,11.103,11.736,11.178,11.625,10.737,14.443,14.576,14.08,14.291,13.93,13.891,13.942,14.142,13.676,13.468,14.245,13.362,14.389,14.434,13.49,13.225,14.418,13.52,13.573,14.235,13.06,13.371,13.63,13.759,12.904,13.256,13.849,13.118,13.685,12.767,13.287,6.135,6.037,5.537,5.727,5.212,5.49,5.499,5.826,5.088,5.05,5.747,4.833,5.84,6.027,5.233,5.188,13.173,13.424,13.927,14.169,13.098,14.143,14.323,14.08,13.527,14.819,14.569,13.179,12.919,13.653,13.083,13.299,14.307,13.367,14.938,12.865,5.349,5.681,5.328,5.01,5.999,6.044,5.153,4.776,5.611,5.251,4.94,5.209,5.852,5.158,5.747,5.129,8.195,7.647,7.895,7.935,7.359,7.448,7.595,7.67,7.511,7.264,7.244,6.961,8.063,7.334,7.162,10.307,9.527,10.035,9.671,9.092,9.243,9.454,9.305,9.002,9.522,9.346,8.98,9.469,9.838,9.098,9.244,4.03,3.917,3.544,4.246,3.985,3.138,4.303,4.506,3.482,3.448,4.128],[9.387,8.678,8.814,9.147,8.262,8.372,null,null,8.22,8.47,8.606,8.264,9.112,9.212,8.088,8.914,1.06,1.162,0.795,0.821,1.1,0.864,0.694,0.721,0.6,0.351,0.467,0.225,1.271,1.327,0.629,0.252,7.182,7.153,7.216,6.831,7.115,7.105,6.461,6.381,6.869,6.742,6.548,6.725,7.491,6.952,6.945,6.385,8.288,7.516,7.763,7.12,7.776,7.33,10.456,7.436,7.509,7.253,6.896,6.603,6.59,6.81,6.288,6.235,6.786,7.057,6.146,6.319,6.629,6.356,7.087,7.04,5.963,6.975,4.299,4.469,4.571,4.788,4.245,4.455,5.141,5.055,4.767,4.9,5.398,5.096,4.15,4.735,4.587,5.346,11.189,10.872,10.902,10.673,11.026,10.488,10.351,10.331,10.187,10.266,10.092,9.761,11.035,10.764,10.031,9.943,5.365,5.326,5.638,5.787,5.7,5.65,5.876,6.005,5.878,5.823,6.546,6.07,5.207,5.545,6.084,6.219,4.315,4.44,4.367,3.824,4.377,4.076,3.782,3.52,3.881,3.805,3.518,3.385,4.511,3.804,3.208,8.568,8.109,8.777,8.786,7.842,8.053,8.659,8.923,7.809,8.092,8.387,7.848,8.495,8.921,7.62,8.41,2.116,2.052,2.199,2.445,1.936,2.343,2.641,2.954,2.387,2.789,2.877,2.786,1.797,2.642,2.548,3.079,3.359,2.907,3.249,3.824,2.447,2.682,3.173,4.052,2.81,2.743,3.29,2.851,3.166,3.818,2.987,3.316,5.05,4.773,4.848,4.855,4.925,4.96,4.632,4.778,4.39,4.365,4.317,4.079,4.973,4.508,4.165,9.724,12.06,9.663,9.293,10.196,9.6,9.18,8.941,9.631,9.248,9.133,9.195,9.902,9.259,9.642,8.915,6.751,6.346,7.441,6.762,5.994,5.985,6.249,6.998,5.862,5.904,6.198,6.115,6.904,7.098,5.664,6.324,5.413,5.146,5.283,5.612,4.446,4.652,5.134,5.495,4.645,5.008,4.496,5.196,5.943,4.196,7.463,6.978,7.187,7.179,6.739,7.228,7.3,7.398,6.753,6.624,6.762,6.487,7.319,6.356,7.177,7.247,6.974,6.965,6.859,7.006,6.71,6.539,6.568,6.871,6.645,6.52,6.288,7.149,6.859,6.978,6.076,9.509,9.227,9.318,9.431,8.868,9.198,8.79,8.809,8.484,8.969,8.68,8.53,9.34,9.307,8.457,8.67,5.647,4.95,5.07,5.522,4.874,5.094,5.485,5.684,5.047,5.406,5.574,5.45,4.585,5.361,5.018,5.734,9.633,9.695,9.715,9.891,9.041,9.332,10.01,10.323,9.372,9.619,10.134,9.351,9.279,10.341,9.227,9.988,8.576,8.538,8.303,8.072,9.185,8.578,7.927,7.652,8.732,8.068,7.82,8.155,8.738,8.285,8.478,7.579,11.906,11.988,11.761,11.586,11.99,11.866,11.424,11.511,11.283,11.05,12.167,11.578,3.728,3.32,3.57,3.711,2.816,3.186,3.562,3.668,2.805,3.083,3.13,3.04,3.31,3.771,3.104,3.174,2.855,2.804,3.265,2.902,2.697,2.522,2.839,2.689,2.533,2.224,3.644,3.002,3.159,2.014,9.591,9.307,9.648,9.264,9.358,9.12,8.942,8.938,9.004,8.845,8.685,8.709,9.323,9.369,8.997,8.867,5.959,6.22,6.027,5.757,6.116,5.848,5.512,5.234,5.599,5.809,5.29,5.318,6.232,5.955,5.638,5.201,6.515,6.596,6.303,6.011,6.718,6.471,6.003,5.756,6.377,6.168,6.232,6.126,6.718,6.16,6.607,5.739,9.137,9.269,8.985,9.196,8.605,8.566,8.847,9.485,8.369,8.373,9.441,8.267,9.064,9.339,8.165,8.949,8.838,7.94,7.993,8.655,7.48,7.791,8.05,8.179,7.324,7.676,8.269,7.538,8.105,7.187,7.707,3.705,3.329,3.805,3.948,3.422,3.546,4.046,4.413,3.483,3.731,4.294,3.88,3.132,3.823,3.503,4.386,8.346,8.597,9.1,9.342,8.271,9.125,9.305,9.062,8.509,9.801,9.551,8.161,7.901,8.635,8.065,8.281,9.289,8.349,9.92,7.847,2.687,3.023,3.135,2.987,3.403,3.735,3.486,3.288,3.75,3.672,3.64,4.076,3.194,3.108,3.887,4.154,6.474,6.4,6.614,7.198,6.068,6.266,6.85,7.446,6.49,6.759,6.836,6.931,6.925,6.597,6.985,6.197,5.398,6.005,5.852,4.963,5.213,5.635,6.006,5.062,5.739,5.891,5.303,5.34,6.019,5.027,5.945,4.771,4.636,4.808,4.987,5.083,4.852,5.585,5.788,5.292,4.442,5.584],[6.275,5.566,5.702,6.035,5.15,5.26,null,null,5.108,5.358,5.494,5.152,6.0,6.1,4.976,5.802,2.71,3.065,2.765,2.791,3.483,3.363,2.776,2.623,3.532,3.284,3.035,3.553,2.956,2.338,3.728,3.093,6.632,6.765,6.666,6.35,6.74,6.73,6.073,5.993,6.494,6.367,6.16,6.35,7.103,6.358,6.57,5.997,5.311,4.539,4.786,4.143,4.799,4.359,8.627,4.465,4.532,4.282,5.5,5.354,5.494,5.831,5.52,5.72,6.009,6.355,5.859,5.903,6.276,6.14,5.196,6.061,5.676,6.86,4.987,5.438,5.259,5.476,5.991,5.729,5.829,5.743,5.797,5.898,6.148,6.13,5.18,5.423,5.927,6.096,9.709,9.392,9.422,9.193,9.546,9.008,8.871,8.851,8.707,8.786,8.612,8.281,9.555,9.284,8.551,8.463,7.475,7.721,7.882,7.42,8.095,7.99,7.768,7.638,8.218,8.132,8.393,8.379,7.602,7.217,8.424,8.057,4.483,4.902,4.613,3.955,5.155,4.65,4.43,3.947,4.978,4.686,4.399,4.663,4.973,3.935,4.361,5.977,5.809,6.331,6.51,5.428,5.851,6.383,6.857,5.738,5.939,6.558,6.037,5.884,6.385,5.809,6.581,3.314,3.563,3.473,3.475,3.91,3.854,3.671,3.642,4.116,4.063,3.907,4.297,3.469,3.33,4.277,4.109,2.095,2.108,2.654,3.205,2.041,2.276,2.767,3.083,3.111,3.007,3.524,3.324,1.901,2.554,2.612,3.55,3.879,3.895,3.677,3.413,4.047,4.126,3.263,3.002,3.647,3.409,3.439,3.24,4.095,3.766,2.908,9.349,11.969,9.288,8.918,9.894,9.225,8.805,8.566,9.256,8.873,8.758,8.82,9.527,8.884,9.267,8.54,4.975,4.57,5.665,4.986,4.218,4.209,4.473,4.978,4.086,4.128,4.422,4.339,5.128,5.322,3.888,4.276,2.442,2.555,2.727,2.893,2.349,2.555,3.105,3.941,2.848,3.126,3.295,2.225,2.972,2.935,4.351,3.866,4.075,4.067,3.627,4.116,4.188,4.286,3.645,3.512,3.65,3.375,4.207,3.248,4.065,5.767,6.01,5.698,5.379,6.042,5.746,5.272,5.088,5.929,5.767,5.253,5.41,6.185,5.379,6.036,5.134,8.029,7.747,7.838,7.951,7.388,7.718,7.31,7.329,7.004,7.489,7.2,7.05,7.86,7.827,6.977,7.19,5.257,5.385,5.503,5.955,5.475,5.695,5.92,6.117,5.648,5.947,6.045,6.051,5.018,5.793,5.619,6.205,8.071,8.284,8.304,8.508,7.782,7.971,8.637,9.05,8.123,8.37,8.973,8.321,7.717,8.82,8.124,8.958,8.885,8.944,8.709,8.477,9.612,9.005,8.354,8.079,9.357,8.693,8.445,8.794,9.144,8.661,9.117,8.218,12.118,12.214,11.973,11.798,12.244,12.12,11.636,11.765,11.495,11.275,12.393,11.832,0.619,0.086,0.336,0.602,0.543,0.611,0.855,1.361,0.543,0.821,1.095,1.251,0.725,0.794,2.596,2.707,2.388,2.337,2.83,2.525,2.23,1.672,2.462,2.438,2.066,2.11,3.209,2.138,3.061,1.9,8.111,8.343,8.168,7.784,8.394,8.156,7.864,7.859,8.04,7.881,7.721,7.745,8.359,7.889,8.119,7.788,5.584,5.899,5.652,5.382,6.014,5.747,5.137,4.859,5.569,5.779,4.97,5.246,5.946,5.58,5.608,5.039,7.349,7.43,7.137,6.845,7.552,7.305,6.837,6.59,7.211,7.002,7.065,6.982,7.552,6.994,7.44,6.572,7.165,7.298,6.802,7.013,6.652,6.613,6.664,7.019,6.398,6.19,6.975,6.084,7.111,7.156,6.212,5.947,7.161,6.46,6.513,6.978,6.0,6.311,6.533,6.502,5.844,6.196,6.592,5.861,6.625,5.707,6.03,4.178,3.93,4.493,4.549,4.11,4.234,4.647,5.014,4.171,4.419,4.895,4.568,3.733,4.424,4.191,5.074,9.411,9.662,10.165,10.407,9.338,9.959,10.139,9.896,9.343,10.635,10.385,8.995,8.735,9.469,8.899,9.115,10.123,9.183,10.754,8.681,5.304,5.64,5.718,5.56,6.02,6.352,5.93,5.732,6.195,6.117,6.084,6.52,5.811,5.552,6.333,6.598,6.816,6.832,6.956,7.54,6.503,6.699,7.192,7.788,6.925,7.194,7.268,7.366,7.155,7.07,7.417,3.085,2.421,2.897,2.875,1.986,2.236,2.658,3.035,2.09,2.762,2.92,2.332,2.363,3.042,2.052,2.974,7.166,7.031,7.203,7.382,7.478,7.247,7.98,8.183,7.687,6.837,7.979],[7.625,6.916,7.052,7.385,6.5,6.61,null,null,6.458,6.708,6.844,6.502,7.35,7.45,6.326,7.152,1.691,1.984,1.791,1.817,2.396,2.282,2.011,2.204,2.529,2.317,2.27,2.589,1.875,1.983,2.725,2.328,5.408,5.379,5.442,5.057,5.341,5.331,4.687,4.607,5.095,4.968,4.774,4.951,5.717,5.178,5.171,4.611,6.754,5.988,6.235,5.592,6.248,5.831,10.099,5.937,5.975,5.754,6.972,6.826,6.966,7.303,6.992,7.192,7.481,7.827,7.312,7.375,7.748,7.612,6.668,7.533,7.129,8.332,6.439,6.711,6.711,6.928,6.496,6.697,7.281,7.195,7.009,7.151,7.6,7.347,6.392,6.875,6.838,7.548,9.405,9.088,9.118,8.889,9.242,8.704,8.567,8.547,8.403,8.482,8.308,7.977,9.251,8.98,8.247,8.159,7.616,7.577,7.889,8.038,7.951,7.901,8.127,8.256,8.129,8.074,8.797,8.321,7.458,7.796,8.335,8.47,3.084,3.815,3.526,2.868,4.068,3.536,3.297,2.814,3.845,3.553,3.266,3.53,3.726,2.848,3.228,7.449,7.281,7.803,7.982,6.9,7.323,7.855,8.329,7.21,7.411,8.03,7.509,7.356,7.857,7.281,8.053,4.358,4.294,4.441,4.687,4.178,4.585,4.883,5.094,4.629,5.031,5.119,5.028,4.039,4.782,4.79,5.321,3.564,3.577,4.123,4.674,3.494,3.729,4.22,4.552,4.564,4.46,4.977,4.776,3.37,4.023,4.065,5.003,3.266,2.989,3.064,3.071,3.141,3.172,2.848,2.994,2.602,2.577,2.533,2.291,3.189,2.72,2.381,7.95,10.57,7.889,7.519,8.495,7.826,7.406,7.167,7.857,7.474,7.359,7.421,8.128,7.485,7.868,7.141,4.967,4.562,5.657,4.978,4.21,4.201,4.465,5.214,4.078,4.12,4.414,4.331,5.12,5.314,3.88,4.54,3.914,4.027,4.199,4.365,3.821,4.027,4.577,5.413,4.32,4.598,4.764,3.697,4.444,4.404,5.701,5.216,5.425,5.417,4.977,5.466,5.538,5.636,4.995,4.862,5.0,4.725,5.557,4.598,5.415,5.463,5.19,5.181,5.075,5.222,4.926,4.755,4.784,5.087,4.861,4.736,4.504,5.365,5.075,5.194,4.292,7.725,7.443,7.534,7.647,7.084,7.414,7.006,7.025,6.7,7.185,6.896,6.746,7.556,7.523,6.673,6.886,6.71,6.838,6.956,7.408,6.927,7.147,7.373,7.57,7.1,7.4,7.498,7.503,6.471,7.246,7.071,7.658,9.543,9.756,9.776,9.98,9.254,9.443,10.109,10.522,9.595,9.842,10.445,9.793,9.189,10.292,9.596,10.43,7.486,7.545,7.31,7.078,8.213,7.606,6.955,6.68,7.958,7.294,7.046,7.395,7.745,7.262,7.718,6.819,10.719,10.815,10.574,10.399,10.845,10.721,10.237,10.366,10.096,9.876,10.994,10.433,2.091,1.558,1.808,2.074,1.696,2.083,2.327,2.833,2.015,2.293,2.567,2.723,1.548,2.266,1.316,1.386,1.067,1.016,1.491,1.126,0.909,0.76,1.063,0.913,0.745,0.701,1.87,1.218,1.657,0.226,7.807,7.523,7.864,7.48,7.574,7.336,7.158,7.154,7.22,7.061,6.901,6.925,7.539,7.585,7.213,7.083,4.185,4.5,4.253,3.983,4.615,4.348,3.738,3.46,4.17,4.38,3.571,3.847,4.547,4.181,4.209,3.64,5.953,6.053,5.76,5.468,6.175,5.928,5.513,5.266,6.124,5.915,5.978,5.895,6.173,5.617,6.353,5.485,7.353,7.485,7.201,7.412,6.821,6.782,7.063,7.701,6.585,6.589,7.657,6.483,7.28,7.555,6.381,7.165,7.054,6.156,6.209,6.871,5.696,6.007,6.266,6.395,5.54,5.892,6.485,5.754,6.321,5.403,5.923,5.63,5.382,5.945,6.001,5.562,5.686,6.099,6.466,5.623,5.871,6.347,6.02,5.185,5.876,5.643,6.526,8.324,8.575,9.078,9.32,8.251,8.565,8.745,8.502,7.946,9.241,8.991,7.599,7.356,8.075,7.522,7.738,8.729,7.786,9.36,7.304,5.027,5.363,5.475,5.327,5.743,6.075,5.785,5.587,6.05,5.972,5.939,6.375,5.534,5.407,6.188,6.453,8.269,8.285,8.409,8.993,7.956,8.152,8.645,9.241,8.378,8.647,8.721,8.819,8.627,8.523,8.87,4.435,3.87,4.247,4.324,3.435,3.685,4.107,4.507,3.562,4.211,4.392,3.804,3.812,4.404,3.524,4.446,7.022,6.887,7.059,7.238,7.334,7.103,7.836,8.039,7.543,6.693,7.835],[8.126,7.426,7.614,7.947,7.062,7.172,null,null,7.02,7.27,7.406,7.064,7.644,8.012,6.888,7.714,2.803,2.924,3.274,3.3,2.995,3.222,3.668,3.839,3.641,3.784,3.904,4.056,2.815,3.643,3.442,3.958,3.409,3.399,3.443,3.077,3.446,3.436,2.707,2.627,3.2,3.073,2.794,3.056,3.737,3.179,3.276,2.631,7.316,6.663,6.91,6.267,6.923,6.756,11.35,6.868,6.537,6.679,8.43,8.284,8.424,8.761,8.45,8.65,8.939,9.264,8.799,8.833,9.206,9.07,8.126,8.991,8.616,9.79,8.085,8.255,8.357,8.574,7.921,8.166,8.927,8.841,8.463,8.576,9.074,8.772,7.936,8.521,8.263,9.022,7.969,7.652,7.682,7.453,7.806,7.268,7.131,7.111,6.967,7.046,6.872,6.541,7.815,7.544,6.811,6.723,8.968,8.929,9.241,9.438,9.303,9.253,9.479,9.681,9.481,9.426,10.149,9.673,8.81,9.221,9.687,9.822,2.959,3.842,3.553,2.945,4.145,3.588,3.349,2.866,3.897,3.605,3.318,3.582,3.597,2.925,3.28,8.7,8.532,9.054,9.233,8.151,8.574,9.106,9.58,8.461,8.662,9.281,8.928,8.607,9.108,8.626,9.304,5.902,5.838,5.985,6.231,5.72,6.129,6.427,6.74,6.171,6.575,6.663,6.572,5.581,6.428,6.332,6.865,5.221,5.234,5.78,6.327,5.152,5.387,5.878,6.205,6.222,6.118,6.635,6.434,5.027,5.676,5.723,6.661,2.069,1.559,1.766,1.947,1.224,1.255,1.84,2.13,1.2,1.175,1.397,0.889,1.778,0.803,1.373,6.055,8.675,5.994,5.624,6.6,5.931,5.511,5.272,5.962,5.579,5.464,5.526,6.233,5.59,5.973,5.246,3.995,3.557,4.702,4.114,3.205,3.229,3.601,4.35,3.214,3.256,3.55,3.467,4.115,4.394,3.016,3.676,5.351,5.464,5.636,5.802,5.37,5.569,6.047,6.883,5.862,6.14,6.411,5.134,5.881,6.057,6.263,5.778,5.987,5.979,5.406,6.028,6.1,6.198,5.557,5.424,5.562,5.287,6.119,5.16,5.977,4.027,3.713,3.704,3.639,3.633,3.432,3.278,3.348,3.298,3.001,3.259,2.644,3.888,3.639,3.405,2.815,6.289,6.007,6.098,6.211,5.648,5.978,5.57,5.589,5.264,5.749,5.46,5.31,6.12,6.087,5.237,5.45,8.206,8.334,8.452,8.904,8.435,8.655,8.869,9.066,8.608,8.896,8.994,9.011,7.967,8.742,8.579,9.154,10.794,11.007,11.027,11.231,10.505,10.694,11.36,11.773,10.846,11.093,11.696,11.044,10.44,11.543,10.847,11.681,5.591,5.65,5.415,5.183,6.318,5.711,5.06,4.785,6.063,5.399,5.151,5.5,5.85,5.367,5.823,4.924,8.824,8.92,8.679,8.504,8.95,8.826,8.342,8.471,8.201,7.981,9.099,8.538,3.288,3.212,3.462,3.728,3.353,3.737,3.981,4.446,3.669,3.947,4.221,4.377,3.205,3.92,1.158,0.729,1.1,1.286,1.186,1.031,1.409,1.75,1.29,1.386,1.656,1.988,1.216,1.342,1.82,1.904,6.371,6.032,6.428,6.044,5.794,5.845,5.667,5.663,5.44,5.57,5.41,5.145,6.048,6.149,5.353,5.592,2.29,2.605,2.358,2.088,2.72,2.453,1.843,1.565,2.47,2.742,1.94,2.352,2.652,2.286,2.578,2.009,5.055,5.517,5.291,4.999,5.938,5.691,5.276,5.029,5.966,5.757,5.897,5.972,5.528,5.096,6.27,5.562,6.468,6.6,6.316,6.527,5.936,5.897,6.178,6.816,5.7,5.704,6.772,5.598,6.395,6.67,5.496,6.28,6.023,4.72,4.773,5.84,4.26,4.571,4.997,5.364,4.316,4.623,5.316,4.587,4.885,4.395,4.915,7.288,7.04,7.591,7.659,7.208,7.332,7.757,8.124,7.269,7.517,8.005,7.666,6.843,7.534,7.289,8.172,8.401,8.652,9.155,9.397,8.328,7.045,7.144,7.271,6.5,7.64,7.39,6.793,6.62,7.179,7.073,7.5,7.128,6.4,7.759,7.067,5.886,6.251,6.363,6.215,6.631,6.963,6.714,6.516,6.978,6.9,6.868,7.304,6.371,6.336,7.115,7.382,9.757,9.781,9.897,10.481,9.452,9.648,10.133,10.729,9.874,10.143,10.217,10.315,10.085,10.019,10.366,4.997,4.545,4.809,4.999,4.11,4.36,4.782,5.429,4.309,4.886,5.111,4.701,4.487,4.966,4.228,5.368,8.374,8.239,8.411,8.59,8.686,8.455,9.188,9.391,8.895,8.045,9.187],[8.597,7.888,8.024,8.357,7.472,7.582,null,null,7.43,7.68,7.816,7.474,8.322,8.422,7.298,8.124,2.303,2.39,2.774,2.8,2.241,2.688,3.168,3.339,3.109,3.284,3.404,3.556,2.281,3.157,2.722,3.458,4.69,4.661,4.724,4.339,4.552,4.542,3.969,3.889,4.306,4.179,3.993,4.162,4.955,4.46,4.382,3.893,7.726,6.96,7.207,6.564,7.22,7.022,11.29,7.128,6.947,6.945,8.163,8.017,8.157,8.494,8.183,8.383,8.672,9.018,8.491,8.566,8.939,8.803,7.859,8.724,8.308,9.523,7.414,7.558,7.668,7.853,7.174,7.419,8.209,8.17,7.716,7.829,8.327,8.025,7.265,7.85,7.516,8.275,9.653,9.336,9.366,9.137,9.49,8.952,8.815,8.795,8.651,8.73,8.556,8.225,9.499,9.228,8.495,8.407,8.214,8.175,8.487,8.684,8.549,8.499,8.725,8.934,8.727,8.672,9.395,8.919,8.056,8.474,8.933,9.068,2.043,2.598,2.309,1.651,2.851,2.316,2.077,1.594,2.625,2.333,2.046,2.31,2.669,1.631,2.008,8.64,8.472,8.994,9.173,8.091,8.514,9.046,9.52,8.401,8.602,9.221,8.7,8.547,9.048,8.472,9.244,5.231,5.167,5.314,5.56,5.049,5.458,5.756,6.069,5.5,5.904,5.992,5.901,4.91,5.757,5.661,6.194,4.75,4.763,5.309,5.86,4.673,4.908,5.399,5.738,5.743,5.639,6.156,5.955,4.556,5.209,5.244,6.182,3.671,3.322,3.436,3.549,3.261,3.292,3.442,3.732,2.802,2.777,2.999,2.491,3.522,2.84,2.975,7.161,9.781,7.1,6.73,7.706,7.037,6.617,6.378,7.068,6.685,6.57,6.632,7.339,6.696,7.079,6.352,5.597,5.159,6.304,5.716,4.807,4.831,5.203,5.952,4.816,4.858,5.152,5.069,5.717,5.996,4.618,5.278,5.105,5.218,5.39,5.556,5.012,5.218,5.768,6.604,5.511,5.789,5.95,4.888,5.635,5.59,6.673,6.188,6.397,6.389,5.949,6.438,6.51,6.608,5.967,5.834,5.972,5.697,6.529,5.57,6.387,5.566,5.215,5.24,5.216,5.132,4.961,4.822,4.928,4.827,4.537,4.803,4.182,5.39,5.323,4.934,4.359,7.973,7.691,7.782,7.895,7.332,7.662,7.254,7.273,6.948,7.433,7.144,6.994,7.804,7.771,6.921,7.134,7.889,8.017,8.135,8.587,8.104,8.324,8.552,8.749,8.277,8.579,8.677,8.68,7.65,8.425,8.248,8.837,10.734,10.947,10.967,11.171,10.445,10.634,11.3,11.713,10.786,11.033,11.636,10.984,10.38,11.483,10.787,11.621,6.697,6.756,6.521,6.289,7.424,6.817,6.166,5.891,7.141,6.477,6.229,6.564,6.956,6.473,6.887,5.988,9.93,10.026,9.785,9.61,10.056,9.932,9.448,9.577,9.307,9.087,10.205,9.644,3.282,2.749,2.999,3.265,2.887,3.274,3.518,4.024,3.201,3.479,3.758,3.914,2.739,3.457,2.024,1.38,1.539,1.725,0.851,1.328,1.706,1.877,1.231,1.255,1.515,1.492,1.48,1.929,0.787,1.417,7.871,7.517,7.928,7.544,7.279,7.33,7.152,7.148,6.925,7.055,6.895,6.63,7.533,7.649,6.835,7.077,3.396,3.711,3.464,3.194,3.826,3.559,2.949,2.671,3.381,3.591,2.782,2.913,3.758,3.392,3.42,2.722,4.929,5.012,4.719,4.427,5.134,4.887,4.472,4.225,4.907,4.698,4.761,4.678,5.132,4.576,5.136,4.268,8.07,8.202,7.918,8.129,7.538,7.499,7.78,8.418,7.302,7.306,8.374,7.2,7.997,8.272,7.098,7.882,7.648,6.53,6.583,7.465,6.027,6.381,6.807,6.989,6.126,6.433,7.079,6.348,6.652,5.997,6.517,6.809,6.559,6.92,7.178,6.537,6.661,7.276,7.643,6.598,6.846,7.524,6.995,6.362,7.053,6.618,7.501,7.107,7.358,7.861,8.103,7.034,7.539,7.719,7.476,6.923,8.215,7.965,6.575,6.315,7.049,6.481,6.697,7.703,6.763,8.334,6.263,4.928,5.322,5.434,5.29,5.698,6.031,5.789,5.594,6.053,5.975,5.944,6.38,5.413,5.414,6.188,6.458,9.448,9.464,9.588,10.172,9.135,9.331,9.824,10.42,9.557,9.826,9.9,9.998,9.818,9.702,10.049,5.407,4.842,5.219,5.296,4.407,4.657,5.079,5.698,4.606,5.183,5.408,4.995,4.784,5.376,4.525,5.637,7.5,7.365,7.563,7.716,7.812,7.607,8.34,8.543,8.047,7.171,8.339],[6.271,5.571,5.875,6.361,5.519,5.79,null,null,5.64,5.89,6.161,5.843,5.789,6.426,5.656,6.469,4.242,4.563,4.297,4.323,4.634,4.861,4.308,4.229,5.064,4.816,4.567,5.085,4.454,3.943,5.081,4.625,4.658,4.95,4.692,4.376,5.001,4.991,4.258,4.178,4.755,4.628,4.345,4.611,5.288,4.384,4.831,4.182,6.119,5.746,5.993,5.631,6.281,6.156,10.942,6.46,5.498,6.237,8.111,7.965,8.105,8.442,8.131,8.331,8.62,8.945,8.48,8.514,8.887,8.751,7.807,8.672,8.297,9.471,7.784,8.066,8.056,8.273,8.443,8.181,8.626,8.54,8.364,8.526,8.945,8.758,7.747,8.22,8.379,8.893,7.681,7.364,7.394,7.165,7.518,6.98,6.843,6.823,6.679,6.758,6.584,6.253,7.527,7.256,6.523,6.435,9.724,9.685,9.997,9.872,10.059,10.009,10.22,10.09,10.237,10.182,10.845,10.429,9.566,9.669,10.443,10.509,4.756,5.531,5.242,4.584,5.784,5.227,4.988,4.505,5.536,5.244,4.957,5.221,5.398,4.564,4.919,8.292,8.124,8.646,8.825,7.743,8.166,8.698,9.172,8.053,8.254,8.873,8.52,8.199,8.7,8.218,8.896,5.766,5.863,5.925,6.042,6.018,6.154,6.238,6.439,6.318,6.515,6.474,6.597,5.671,6.127,6.479,6.676,4.902,4.915,5.461,6.008,4.839,5.074,5.565,5.886,5.909,5.805,6.322,6.121,4.708,5.357,5.41,6.348,1.851,1.921,1.649,1.385,2.073,2.152,1.235,0.974,1.631,1.381,1.56,1.367,2.121,1.822,1.018,7.61,10.24,7.559,7.179,8.165,7.496,7.076,6.827,7.527,7.144,7.029,7.091,7.798,7.145,7.538,6.811,2.161,1.697,2.868,2.229,1.345,1.395,1.926,2.623,1.251,1.604,1.898,1.815,2.255,2.539,1.071,2.024,5.032,5.145,5.317,5.483,5.051,5.25,5.728,6.564,5.543,5.821,6.092,4.815,5.562,5.738,4.529,4.068,4.411,4.717,3.681,4.486,4.838,4.963,4.133,4.203,4.341,4.066,4.857,3.789,4.756,3.739,3.982,3.67,3.351,4.014,3.718,3.244,3.06,3.901,3.793,3.225,3.436,4.157,3.351,4.008,3.106,6.001,5.719,5.81,5.923,5.36,5.69,5.282,5.301,4.976,5.461,5.172,5.022,5.832,5.799,4.949,5.162,7.887,8.015,8.133,8.585,8.116,8.336,8.55,8.747,8.289,8.577,8.675,8.692,7.648,8.423,8.26,8.835,10.386,10.599,10.619,10.823,10.097,10.286,10.952,11.365,10.438,10.685,11.288,10.636,10.032,11.135,10.439,11.273,7.156,7.215,6.98,6.748,7.883,7.276,6.625,6.35,7.628,6.964,6.716,7.065,7.415,6.932,7.388,6.489,10.389,10.485,10.244,10.069,10.515,10.391,9.907,10.036,9.766,9.546,10.664,10.103,2.969,2.893,3.143,3.409,3.034,3.418,3.662,4.127,3.35,3.628,3.902,4.058,2.886,3.601,2.174,2.36,2.491,2.416,2.825,2.67,2.587,2.338,2.929,2.96,2.837,3.292,2.919,2.08,3.459,3.085,6.083,6.315,6.14,5.756,6.366,6.128,5.836,5.831,6.012,5.853,5.693,5.717,6.331,5.861,6.145,5.76,3.855,4.243,3.996,3.808,4.44,4.173,3.563,3.285,4.19,4.462,3.809,4.221,4.217,3.858,4.447,3.878,6.924,7.386,7.16,6.868,7.807,7.56,7.145,6.898,7.835,7.626,7.694,7.611,7.397,6.965,8.069,7.201,4.613,4.745,4.461,4.672,4.081,4.042,4.323,4.961,3.845,3.849,4.917,3.743,4.54,4.815,3.641,4.425,4.288,4.069,4.029,4.105,3.786,3.92,3.66,3.629,3.356,3.341,3.719,2.988,4.405,3.219,3.157,6.975,6.727,7.29,7.346,6.907,7.031,7.444,7.811,6.968,7.216,7.692,7.365,6.53,7.221,6.988,7.871,10.04,10.291,10.794,11.036,9.967,8.765,8.864,8.991,8.22,9.36,9.11,8.574,8.446,8.899,8.793,9.22,8.848,8.12,9.479,8.936,7.268,7.604,7.682,7.524,7.984,8.316,7.894,7.696,8.159,8.081,8.048,8.484,7.775,7.516,8.297,8.562,9.438,9.462,9.578,10.162,9.133,9.329,9.814,10.41,9.555,9.824,9.898,9.996,9.766,9.7,10.047,3.776,4.137,4.016,4.591,3.702,3.952,4.374,5.021,3.901,4.478,4.703,4.293,4.079,4.169,3.82,4.96,9.13,8.995,9.167,9.346,9.442,9.211,9.944,10.147,9.651,8.801,9.943],[7.87,7.17,7.474,7.96,7.118,7.389,null,null,7.239,7.489,7.76,7.524,7.388,8.025,7.255,8.068,5.094,5.28,5.316,5.342,5.351,5.578,5.327,5.498,5.932,5.835,5.586,6.104,5.171,5.212,5.798,5.644,2.489,2.892,2.583,2.327,3.251,3.332,2.531,2.461,3.159,3.172,2.628,3.155,3.23,2.181,3.369,2.465,7.8,7.427,7.674,7.312,7.962,7.837,12.613,8.131,7.179,7.918,9.782,9.636,9.776,10.113,9.802,10.002,10.291,10.616,10.151,10.185,10.558,10.422,9.478,10.343,9.968,11.142,9.455,9.737,9.727,9.944,9.812,9.852,10.297,10.211,10.035,10.197,10.616,10.429,9.418,9.891,10.05,10.564,5.415,5.098,5.128,4.899,5.252,4.714,4.577,4.557,4.413,4.492,4.318,3.987,5.261,4.99,4.257,4.169,10.932,10.893,11.205,11.354,11.267,11.217,11.443,11.572,11.445,11.39,12.113,11.637,10.774,11.112,11.651,11.786,4.693,5.576,5.287,5.087,5.997,5.496,5.294,5.094,5.993,5.705,5.498,5.762,5.331,5.032,5.508,9.963,9.795,10.317,10.496,9.414,9.837,10.369,10.843,9.724,9.925,10.544,10.191,9.87,10.371,9.889,10.567,7.437,7.534,7.596,7.713,7.454,7.825,7.909,8.11,7.905,8.186,8.145,8.268,7.315,7.798,8.066,8.347,6.573,6.586,7.132,7.679,6.51,6.745,7.236,7.557,7.58,7.476,7.993,7.792,6.379,7.028,7.081,8.019,1.336,1.164,1.468,1.373,1.487,1.674,1.666,2.173,1.843,1.763,1.922,2.266,1.052,1.82,1.911,5.185,8.592,5.347,4.894,6.302,5.57,4.905,4.542,5.626,5.26,4.985,5.343,5.749,4.72,5.789,4.846,3.453,2.968,4.315,3.849,2.616,2.931,3.584,4.243,2.932,3.285,3.579,3.496,3.526,4.151,2.752,3.705,6.703,6.816,6.988,7.154,6.722,6.921,7.399,8.235,7.214,7.492,7.763,6.486,7.233,7.409,6.128,5.68,6.01,6.316,5.301,6.098,6.437,6.562,5.753,5.862,6.022,5.747,6.456,5.447,6.437,1.417,1.374,1.062,1.031,1.406,1.11,0.636,0.742,1.293,1.386,0.617,1.422,1.549,1.085,1.4,0.645,3.735,3.453,3.544,3.657,3.094,3.424,3.016,3.035,2.71,3.195,2.906,2.756,3.566,3.533,2.683,2.896,9.558,9.686,9.804,10.256,9.787,10.007,10.221,10.418,9.96,10.248,10.346,10.363,9.319,10.094,9.931,10.506,12.057,12.27,12.29,12.494,11.768,11.957,12.623,13.036,12.109,12.356,12.959,12.307,11.703,12.806,12.11,12.944,5.814,5.873,5.638,5.406,6.541,5.934,5.283,5.008,6.286,5.622,5.374,6.121,6.073,5.59,6.189,5.423,8.741,8.837,8.596,8.421,8.867,8.743,8.259,8.388,8.118,7.898,9.016,8.455,4.64,4.564,4.814,5.08,4.705,5.089,5.333,5.798,5.021,5.299,5.573,5.729,4.557,5.272,3.151,3.077,3.208,3.392,3.542,3.387,3.539,3.357,3.646,3.677,3.789,4.244,3.567,3.099,4.048,4.037,3.76,3.707,3.817,3.433,3.758,3.52,3.342,3.338,3.404,3.245,3.085,3.109,3.723,3.538,3.596,3.267,2.984,3.52,3.273,3.093,3.973,3.863,3.34,3.15,4.054,4.326,3.674,4.086,3.467,3.133,4.312,3.743,6.783,7.241,7.025,6.733,7.672,7.425,7.01,6.763,7.7,7.491,7.631,7.726,7.252,6.825,8.004,7.308,6.212,6.344,6.06,6.271,5.68,5.641,5.922,6.56,5.444,5.448,6.516,5.342,6.139,6.414,5.24,6.024,4.127,2.734,2.823,3.944,2.108,2.487,3.101,3.468,2.42,2.727,3.42,2.691,2.664,2.612,3.335,8.646,8.398,8.961,9.017,8.578,8.702,9.115,9.482,8.639,8.887,9.363,9.036,8.201,8.892,8.659,9.542,10.155,10.406,10.909,11.151,10.082,8.445,8.544,8.594,7.939,9.04,8.79,8.293,8.165,8.618,8.512,8.939,8.528,7.839,9.159,8.716,8.242,8.607,8.719,8.571,8.987,9.319,9.07,8.872,9.334,9.256,9.224,9.66,8.727,8.692,9.471,9.738,11.109,11.133,11.249,11.833,10.804,11.0,11.485,12.081,11.226,11.495,11.569,11.667,11.437,11.371,11.718,5.457,5.808,5.697,6.262,5.373,5.623,6.045,6.692,5.572,6.149,6.374,5.964,5.75,5.85,5.491,6.631,10.338,10.203,10.375,10.554,10.65,10.419,11.152,11.355,10.859,10.009,11.151],[9.158,8.458,8.762,9.248,8.406,8.677,null,null,8.527,8.777,9.048,8.943,8.676,9.313,8.543,9.356,6.228,6.414,6.45,6.476,6.485,6.712,6.461,6.654,7.066,6.969,6.72,7.238,6.305,6.423,6.932,6.778,1.079,1.482,1.173,0.917,1.841,1.922,1.285,1.648,1.749,1.786,1.68,2.223,1.82,0.771,1.959,1.812,9.373,9.0,9.247,8.885,9.535,9.41,14.018,9.536,8.752,9.347,11.174,11.028,11.168,11.505,11.194,11.394,11.683,12.008,11.543,11.577,11.95,11.814,10.87,11.735,11.36,12.534,10.841,11.123,11.113,11.33,10.946,11.147,11.683,11.597,11.421,11.583,12.002,11.797,10.804,11.277,11.288,11.95,4.584,4.267,4.297,4.068,4.421,3.883,3.746,3.726,3.582,3.661,3.487,3.156,4.43,4.159,3.426,3.338,12.066,12.027,12.339,12.488,12.401,12.351,12.577,12.706,12.579,12.524,13.247,12.771,11.908,12.246,12.785,12.92,4.715,5.279,5.309,5.109,5.242,5.518,5.316,5.599,5.696,5.727,5.52,5.784,5.034,5.125,5.88,11.368,11.2,11.722,11.901,10.819,11.242,11.774,12.248,11.129,11.33,11.949,11.596,11.275,11.776,11.294,11.972,8.808,8.744,8.891,9.099,8.628,9.035,9.295,9.496,9.079,9.481,9.531,9.478,8.489,9.184,9.24,9.733,7.965,7.978,8.524,9.071,7.896,8.131,8.622,8.949,8.966,8.862,9.379,9.178,7.771,8.42,8.467,9.405,2.765,2.298,2.602,2.796,2.592,2.779,3.184,3.741,2.977,2.897,3.056,3.4,2.157,2.925,3.318,3.775,7.182,3.937,3.484,4.892,4.16,3.495,3.132,4.216,3.85,3.575,3.933,4.339,3.31,4.379,3.436,4.741,4.541,5.603,5.137,4.189,4.504,5.157,5.531,4.505,4.858,5.152,5.069,4.922,5.439,4.325,5.278,8.095,8.208,8.38,8.546,8.114,8.313,8.791,9.627,8.606,8.884,9.155,7.878,8.625,8.801,7.416,6.968,7.298,7.604,6.589,7.386,7.725,7.85,7.041,7.15,7.441,7.32,7.744,7.02,8.01,0.848,0.619,0.81,1.184,0.358,0.514,1.03,1.451,0.658,0.997,1.355,1.429,0.537,1.437,0.765,1.453,2.904,2.622,2.713,2.949,2.263,2.593,2.626,2.938,1.879,2.621,2.625,2.501,2.735,2.789,1.852,2.799,10.95,11.078,11.196,11.648,11.179,11.399,11.613,11.81,11.352,11.64,11.738,11.755,10.711,11.486,11.323,11.898,13.462,13.675,13.695,13.899,13.173,13.362,14.028,14.441,13.514,13.761,14.364,13.712,13.108,14.211,13.515,14.349,4.405,4.689,4.372,4.153,5.429,4.926,4.275,4.0,5.278,4.614,4.366,5.113,4.681,4.225,5.181,4.415,7.331,7.427,7.186,7.011,7.457,7.333,6.849,6.978,6.708,6.488,7.606,7.045,6.032,5.956,6.206,6.472,6.097,6.481,6.725,7.19,6.413,6.691,6.965,7.121,5.949,6.664,4.285,4.211,4.342,4.526,4.646,4.521,4.673,4.491,4.78,4.811,4.923,5.378,4.537,4.233,5.018,5.171,2.702,2.348,2.759,2.375,2.348,2.161,1.983,1.979,1.994,1.886,1.726,1.699,2.364,2.48,2.186,1.908,3.005,3.541,3.294,3.114,3.749,3.639,3.361,3.431,3.83,4.062,3.696,4.108,3.243,3.154,4.049,3.765,5.775,6.233,6.023,5.734,6.673,6.426,6.011,5.764,6.701,6.492,6.632,6.727,6.244,5.817,7.005,6.309,7.5,7.632,7.348,7.559,6.968,6.929,7.21,7.848,6.732,6.736,7.804,6.63,7.427,7.702,6.528,7.312,5.415,4.022,4.111,5.232,3.396,3.775,4.389,4.756,3.708,4.015,4.708,3.979,3.952,3.9,4.623,10.032,9.784,10.347,10.403,9.964,10.088,10.501,10.868,10.025,10.273,10.749,10.422,9.587,10.278,10.045,10.928,9.156,9.407,9.91,10.152,9.083,7.437,7.536,7.586,6.931,8.032,7.782,7.285,7.157,7.61,7.504,7.931,7.52,6.831,8.151,7.708,8.89,9.285,9.398,9.255,9.523,9.856,9.754,9.746,9.962,9.884,9.909,10.292,9.298,9.405,10.013,10.37,12.501,12.525,12.641,13.225,12.196,12.392,12.877,13.473,12.618,12.887,12.961,13.059,12.829,12.763,13.11,7.03,7.213,7.27,7.667,6.778,7.028,7.45,8.097,6.977,7.554,7.779,7.369,7.155,7.423,6.896,8.036,11.472,11.337,11.509,11.688,11.784,11.553,12.286,12.489,11.993,11.143,12.285],[7.829,7.232,7.536,8.022,7.18,7.451,null,null,7.301,7.551,7.822,7.844,7.347,8.087,7.317,8.13,8.931,9.117,9.145,9.171,9.188,9.415,9.156,9.111,9.769,9.664,9.415,9.933,9.008,8.825,9.635,9.473,3.698,3.992,4.023,4.004,4.351,4.643,4.372,4.735,4.836,4.873,4.767,5.31,4.33,3.858,4.906,4.899,8.801,8.429,8.676,8.314,8.964,8.839,13.768,9.292,8.181,8.92,11.019,10.873,11.013,11.35,11.065,11.243,11.528,11.782,11.414,11.426,11.795,11.663,10.741,11.58,11.231,12.379,13.068,13.35,13.34,13.557,13.482,13.465,13.91,13.824,13.648,13.81,14.229,14.042,13.031,13.504,13.663,14.177,2.645,2.513,2.416,2.127,2.667,2.57,1.799,1.673,2.478,2.179,1.956,2.295,2.676,2.218,2.754,2.247,14.606,14.567,14.879,15.024,14.941,14.891,15.117,15.242,15.119,15.064,15.787,15.311,14.448,14.782,15.325,15.46,7.72,8.366,8.314,8.114,8.329,8.523,8.321,8.604,8.783,8.732,8.525,8.789,8.121,8.13,8.885,11.118,10.95,11.472,11.651,10.569,10.992,11.524,11.998,10.879,11.08,11.699,11.346,11.025,11.526,11.044,11.722,11.05,11.147,11.209,11.326,11.067,11.438,11.522,11.723,11.518,11.799,11.758,11.881,10.928,11.411,11.679,11.96,9.56,9.721,10.078,10.4,10.123,9.968,10.267,10.278,10.803,10.507,10.8,11.067,9.722,9.852,10.35,10.826,5.173,5.001,5.25,4.986,5.324,5.511,5.279,5.786,5.68,5.6,5.759,6.103,4.889,5.657,5.524,6.032,9.439,6.194,5.741,7.149,6.417,5.752,5.389,6.473,6.107,5.832,6.19,6.596,5.567,6.636,5.693,4.349,4.746,4.743,4.722,5.251,5.566,5.31,5.116,5.336,5.438,5.54,5.647,4.53,4.579,5.387,5.856,9.22,9.459,9.505,9.496,9.443,9.573,9.745,10.119,9.866,10.037,10.348,9.138,9.443,10.13,6.19,6.108,6.072,6.378,6.156,6.526,6.499,6.624,6.608,6.717,7.008,7.026,6.518,6.682,7.243,2.739,3.474,3.152,3.034,3.317,3.351,3.4,3.363,3.665,3.979,3.714,4.389,3.357,2.911,3.772,4.037,0.755,1.649,1.201,1.059,1.29,0.897,1.223,1.445,1.212,1.217,1.222,1.571,0.57,0.899,1.344,1.398,10.825,11.344,11.071,11.52,11.842,11.589,11.488,11.682,11.688,11.733,11.777,12.207,11.143,11.358,11.784,11.937,13.212,13.425,13.445,13.649,12.923,13.112,13.778,14.191,13.264,13.511,14.114,13.462,12.858,13.961,13.265,14.099,6.915,7.199,6.882,6.663,7.86,7.496,6.902,6.627,7.917,7.253,7.005,7.752,7.191,6.735,7.82,7.054,9.588,9.684,9.443,9.268,9.714,9.59,9.106,9.235,8.965,8.745,9.863,9.302,8.253,8.177,8.427,8.693,8.318,8.702,8.946,9.008,8.634,8.912,9.186,9.342,8.17,8.885,6.988,6.914,7.045,7.229,7.379,7.224,7.376,7.186,7.483,7.514,7.626,8.081,7.444,6.928,7.925,7.874,3.06,3.357,3.285,3.018,3.878,3.636,3.098,3.093,3.807,3.476,3.316,3.515,3.341,3.123,4.184,3.022,6.01,6.546,6.299,6.119,6.836,6.726,6.366,6.42,6.917,7.149,6.701,7.113,6.33,6.159,7.136,6.77,8.414,8.872,8.662,8.465,9.397,9.15,8.744,8.505,9.426,9.224,9.364,9.48,8.883,8.456,9.729,9.042,6.274,6.406,6.122,6.333,5.742,5.703,5.984,6.519,5.506,5.51,6.475,5.404,6.201,6.476,5.302,5.983,4.121,2.683,2.772,3.938,2.49,2.501,3.095,3.462,2.844,3.152,3.414,3.587,2.52,3.036,3.397,11.517,11.829,12.195,12.078,12.11,12.233,12.176,12.454,12.252,12.334,12.424,12.587,11.632,11.948,12.272,12.87,11.824,12.075,12.578,12.82,11.836,10.076,10.175,10.225,9.57,10.671,10.421,9.924,9.796,10.249,10.143,10.57,10.159,9.47,10.79,10.347,11.895,12.29,12.403,12.26,12.528,12.861,12.759,12.578,12.967,12.889,12.914,13.297,12.303,12.398,13.018,13.375,12.372,12.397,12.512,13.096,12.071,12.264,12.748,13.344,12.493,12.761,12.833,12.934,12.678,12.638,12.982,7.277,7.789,7.518,8.032,8.099,7.966,7.965,8.424,8.15,8.119,8.172,8.387,8.002,7.586,8.069,8.379,14.012,13.877,14.049,14.228,14.324,14.093,14.826,15.029,14.533,13.683,14.825],[10.266,9.566,9.87,10.356,9.514,9.785,null,null,9.635,9.885,10.156,10.178,9.784,10.421,9.651,10.464,9.588,9.774,9.802,9.828,9.845,10.072,9.813,9.768,10.426,10.321,10.072,10.59,9.665,9.482,10.292,10.13,4.351,4.633,4.68,4.661,4.992,5.3,5.029,5.392,5.493,5.53,5.424,5.967,4.839,4.515,5.563,5.556,11.057,10.684,10.931,10.569,11.219,11.094,16.023,11.547,10.436,11.175,13.274,13.128,13.268,13.605,13.32,13.498,13.783,14.037,13.669,13.681,14.05,13.918,12.996,13.835,13.486,14.634,13.725,14.007,13.997,14.214,14.139,14.122,14.567,14.481,14.305,14.467,14.886,14.699,13.688,14.161,14.32,14.834,1.016,0.536,0.783,1.069,0.69,0.261,0.795,1.131,0.156,0.526,0.801,0.654,0.699,1.16,0.313,0.926,15.263,15.224,15.536,15.681,15.598,15.548,15.774,15.899,15.776,15.721,16.444,15.968,15.105,15.439,15.982,16.117,8.377,9.023,8.971,8.771,8.986,9.18,8.978,9.261,9.44,9.389,9.182,9.446,8.778,8.787,9.542,13.373,13.205,13.727,13.906,12.824,13.247,13.779,14.253,13.134,13.335,13.954,13.601,13.28,13.781,13.299,13.977,11.707,11.804,11.866,11.983,11.724,12.095,12.179,12.38,12.175,12.456,12.415,12.538,11.585,12.068,12.336,12.617,10.843,10.856,11.402,11.949,10.78,11.015,11.506,11.827,11.85,11.746,12.263,12.062,10.649,11.298,11.351,12.289,5.83,5.658,5.907,5.643,5.981,6.168,5.936,6.443,6.337,6.257,6.416,6.76,5.546,6.314,6.181,3.82,7.307,3.982,4.003,4.937,4.323,4.149,3.786,4.608,4.504,4.229,4.587,4.384,3.964,5.033,4.203,6.054,6.451,6.826,6.45,6.846,7.161,7.038,6.844,7.041,7.166,7.268,7.375,6.235,6.662,6.982,7.584,10.973,11.086,11.258,11.392,10.992,11.191,11.641,12.044,11.484,11.762,12.033,10.756,11.368,11.679,8.524,8.191,8.406,8.712,7.902,8.609,8.833,8.958,8.354,8.463,8.754,8.772,8.852,8.428,9.462,3.396,4.131,3.809,3.691,3.974,4.008,4.057,4.02,4.322,4.636,4.371,5.046,4.014,3.568,4.429,4.694,3.267,3.056,2.973,3.457,2.697,2.884,3.254,3.568,2.868,3.248,3.253,3.229,2.392,3.297,2.912,3.429,13.079,13.372,13.325,13.775,13.767,13.617,13.742,13.937,13.716,13.77,13.868,14.179,13.127,13.613,13.756,14.028,15.467,15.68,15.7,15.904,15.178,15.367,16.033,16.446,15.519,15.766,16.369,15.717,15.113,16.216,15.52,16.354,5.541,6.025,5.778,5.741,6.412,6.322,6.046,6.146,6.591,6.371,6.308,6.838,5.985,5.714,6.652,6.357,7.456,7.552,7.311,7.136,7.582,7.458,6.974,7.103,6.833,6.613,7.731,7.17,8.91,8.834,9.084,9.35,8.975,9.359,9.603,10.068,9.291,9.569,9.843,9.999,8.827,9.542,7.645,7.571,7.702,7.886,8.036,7.881,8.033,7.843,8.14,8.171,8.283,8.738,8.101,7.585,8.582,8.531,1.369,1.666,1.598,1.638,2.187,1.947,1.957,2.324,2.272,2.231,2.545,2.586,1.65,1.791,2.539,2.411,6.667,7.203,6.956,6.776,7.493,7.383,7.023,7.077,7.574,7.806,7.358,7.77,6.987,6.816,7.793,7.427,7.723,7.992,7.971,8.067,8.517,8.27,8.346,8.23,8.546,8.533,8.673,8.819,8.003,8.058,8.849,8.644,8.608,8.74,8.456,8.667,8.076,8.037,8.318,8.956,7.84,7.844,8.912,7.738,8.535,8.81,7.636,8.42,6.519,5.081,5.17,6.336,4.663,4.835,5.493,5.86,4.927,5.235,5.812,5.292,4.918,5.119,5.731,12.916,12.668,13.231,13.287,12.848,12.972,13.385,13.752,12.909,13.157,13.633,13.306,12.471,13.162,12.929,13.812,10.652,10.903,11.406,11.648,11.376,8.845,8.944,8.916,8.398,9.44,9.19,8.752,8.624,9.077,8.971,9.398,8.928,8.298,9.559,9.175,12.552,12.947,13.06,12.917,13.185,13.518,13.416,13.235,13.624,13.546,13.571,13.954,12.96,13.055,13.675,14.032,14.627,14.652,14.767,15.351,14.325,14.519,15.003,15.599,14.747,15.016,15.088,15.188,14.933,14.892,15.237,9.023,9.535,9.264,9.778,9.643,9.712,9.711,10.349,9.842,9.865,9.918,10.133,9.748,9.332,9.761,10.172,14.669,14.534,14.706,14.885,14.981,14.75,15.483,15.686,15.19,14.34,15.482],[10.551,9.906,10.21,10.696,9.854,10.125,null,null,9.975,10.225,10.496,10.518,10.069,10.761,9.991,10.804,8.181,8.367,8.403,8.429,8.438,8.665,8.414,8.607,9.019,8.922,8.673,9.191,8.258,8.376,8.885,8.731,2.485,2.763,2.855,2.861,3.115,3.442,3.229,3.592,3.685,3.722,3.619,4.158,2.927,2.722,3.73,3.756,11.048,10.675,10.922,10.56,11.21,11.085,15.901,11.419,10.427,11.166,13.07,12.924,13.064,13.401,13.09,13.29,13.579,13.904,13.439,13.473,13.846,13.71,12.766,13.631,13.256,14.43,12.743,13.025,13.015,13.232,12.899,13.1,13.585,13.499,13.323,13.485,13.904,13.717,12.706,13.179,13.241,13.852,3.084,2.71,2.789,2.569,2.864,2.329,2.247,2.48,2.074,2.162,1.988,1.657,2.873,2.66,1.918,2.113,14.019,13.98,14.292,14.441,14.354,14.304,14.53,14.659,14.532,14.477,15.2,14.724,13.861,14.199,14.738,14.873,6.654,7.21,7.248,7.048,7.141,7.457,7.255,7.538,7.627,7.666,7.459,7.723,6.965,7.064,7.819,13.251,13.083,13.605,13.784,12.702,13.125,13.657,14.131,13.012,13.213,13.832,13.479,13.158,13.659,13.177,13.855,10.725,10.697,10.844,11.001,10.581,10.988,11.197,11.398,11.032,11.434,11.433,11.431,10.442,11.086,11.193,11.635,9.861,9.874,10.42,10.967,9.798,10.033,10.524,10.845,10.868,10.764,11.281,11.08,9.667,10.316,10.369,11.307,4.718,4.251,4.555,4.661,4.545,4.732,4.954,5.461,4.93,4.85,5.009,5.353,4.11,4.878,5.199,3.239,6.715,3.401,2.948,4.356,3.624,2.959,2.596,3.68,3.314,3.039,3.397,3.803,2.774,3.843,2.9,6.322,6.216,7.184,6.718,5.864,6.179,6.832,7.112,6.18,6.533,6.827,6.744,6.503,7.02,6.0,6.953,9.991,10.104,10.276,10.442,10.01,10.209,10.687,11.523,10.502,10.78,11.051,9.774,10.521,10.697,8.864,8.549,8.746,9.052,8.17,8.967,9.173,9.298,8.622,8.731,9.022,8.995,9.192,8.695,9.685,2.114,2.572,2.547,2.456,2.311,2.467,2.845,2.867,2.611,2.95,3.212,3.382,2.49,2.709,2.718,3.406,3.69,2.898,3.243,3.816,2.539,3.243,3.613,3.927,2.875,3.607,3.612,3.497,2.85,3.656,2.754,3.788,12.846,12.974,13.092,13.544,13.075,13.295,13.509,13.706,13.248,13.536,13.634,13.651,12.607,13.382,13.219,13.794,15.345,15.558,15.578,15.782,15.056,15.245,15.911,16.324,15.397,15.644,16.247,15.595,14.991,16.094,15.398,16.232,4.265,4.749,4.502,4.292,5.136,5.046,4.597,4.368,5.315,4.994,4.746,5.493,4.709,4.31,5.376,4.795,6.864,6.96,6.719,6.544,6.99,6.866,6.382,6.511,6.241,6.021,7.139,6.578,7.928,7.852,8.102,8.368,7.993,8.377,8.621,9.086,8.309,8.587,8.861,9.017,7.845,8.56,6.238,6.164,6.295,6.479,6.594,6.474,6.626,6.444,6.733,6.764,6.876,7.331,6.485,6.186,6.966,7.124,0.903,0.728,0.96,0.576,1.245,0.836,0.183,0.338,0.892,0.561,0.559,0.6,0.712,0.681,1.269,0.425,4.944,5.48,5.233,5.053,5.68,5.57,5.3,5.37,5.761,5.993,5.635,6.047,5.174,5.093,5.98,5.704,6.155,6.613,6.403,6.206,7.138,6.891,6.485,6.246,7.167,6.965,7.105,7.251,6.624,6.197,7.47,6.783,8.948,9.08,8.796,9.007,8.416,8.377,8.658,9.241,8.18,8.184,9.197,8.078,8.875,9.15,7.976,8.705,6.86,5.422,5.511,6.677,4.931,5.175,5.834,6.201,5.286,5.594,6.153,5.56,5.262,5.478,6.071,11.934,11.686,12.249,12.305,11.866,11.99,12.403,12.77,11.927,12.175,12.651,12.324,11.489,12.18,11.947,12.83,9.376,9.627,10.13,10.372,9.663,7.569,7.668,7.64,7.122,8.164,7.914,7.476,7.348,7.801,7.695,8.122,7.652,7.022,8.283,7.899,10.829,11.224,11.337,11.194,11.462,11.795,11.693,11.685,11.901,11.823,11.848,12.231,11.237,11.344,11.952,12.309,14.397,14.421,14.537,15.121,14.092,14.288,14.773,15.369,14.514,14.783,14.857,14.955,14.725,14.659,15.006,8.705,9.096,8.945,9.55,8.661,8.911,9.333,9.98,8.86,9.437,9.662,9.252,9.038,9.098,8.779,9.919,13.425,13.29,13.462,13.641,13.737,13.506,14.239,14.442,13.946,13.096,14.238],[11.55,10.85,11.154,11.64,10.798,11.069,null,null,10.919,11.169,11.44,11.335,11.068,11.705,10.935,11.748,7.853,7.94,8.324,8.35,7.791,8.238,8.718,8.889,8.659,8.834,8.954,9.106,7.831,8.707,8.272,9.008,1.99,1.954,2.36,2.366,2.289,2.619,2.424,2.79,2.863,2.9,2.797,3.332,2.105,2.418,2.904,2.954,11.738,11.365,11.612,11.25,11.9,11.775,16.383,11.901,11.117,11.712,13.539,13.393,13.533,13.87,13.559,13.759,14.048,14.373,13.908,13.942,14.315,14.179,13.235,14.1,13.725,14.899,13.004,13.156,13.266,13.451,12.772,13.017,13.807,13.76,13.314,13.427,13.925,13.623,12.855,13.44,13.114,13.873,4.055,3.681,3.763,3.543,3.835,3.3,3.221,3.454,3.048,3.136,2.962,2.631,3.844,3.634,2.892,3.087,13.812,13.773,14.085,14.282,14.147,14.097,14.323,14.532,14.325,14.27,14.993,14.517,13.654,14.072,14.531,14.666,5.832,6.379,6.426,6.226,6.301,6.635,6.433,6.716,6.788,6.844,6.637,6.901,6.134,6.242,6.997,13.733,13.565,14.087,14.266,13.184,13.607,14.139,14.613,13.494,13.695,14.314,13.961,13.64,14.141,13.659,14.337,10.821,10.757,10.904,11.15,10.639,11.048,11.346,11.659,11.09,11.494,11.582,11.491,10.5,11.347,11.251,11.784,10.295,10.308,10.854,11.405,10.223,10.458,10.949,11.283,11.293,11.189,11.706,11.505,10.101,10.754,10.794,11.732,5.13,4.644,4.967,5.161,4.687,4.874,5.549,6.106,5.341,5.262,5.421,5.646,4.522,4.966,5.683,1.271,4.758,1.433,1.112,2.388,1.661,1.266,0.903,1.717,1.621,1.346,1.704,1.835,1.081,2.15,1.32,7.133,6.906,7.995,7.529,6.554,6.869,7.522,7.923,6.87,7.223,7.517,7.434,7.314,7.831,6.69,7.643,10.46,10.573,10.745,10.911,10.479,10.678,11.156,11.992,10.971,11.249,11.495,10.243,10.99,11.135,9.808,9.36,9.69,9.996,8.981,9.778,10.117,10.242,9.433,9.542,9.833,9.685,10.136,9.385,10.375,3.362,3.011,3.313,3.576,2.75,2.906,3.422,3.843,3.023,3.362,3.747,3.505,2.929,3.829,3.13,3.818,5.442,4.635,4.98,5.553,4.276,4.98,5.143,5.443,4.485,5.198,5.142,5.018,4.644,5.393,4.394,5.304,13.315,13.443,13.561,14.013,13.544,13.764,13.978,14.175,13.717,14.005,14.103,14.12,13.076,13.851,13.688,14.263,15.827,16.04,16.06,16.264,15.538,15.727,16.393,16.806,15.879,16.126,16.729,16.077,15.473,16.576,15.88,16.714,2.658,3.142,2.895,2.858,3.529,3.439,3.163,3.28,3.708,3.488,3.425,3.955,3.102,2.831,3.769,3.474,4.907,5.003,4.762,4.587,5.033,4.909,4.425,4.554,4.284,4.064,5.182,4.621,8.397,8.289,8.539,8.805,8.427,8.814,9.058,9.555,8.746,9.024,9.298,9.454,8.279,8.997,6.283,5.847,6.218,6.404,5.804,6.149,6.527,6.856,6.337,6.439,6.709,7.041,5.695,6.46,6.176,6.957,1.874,1.519,2.103,2.142,1.175,1.502,1.968,2.226,1.349,1.831,1.972,1.592,1.535,2.267,1.143,2.313,4.122,4.658,4.411,4.231,4.849,4.739,4.478,4.548,4.93,5.162,4.813,5.225,4.343,4.271,5.149,4.882,4.84,5.109,5.088,5.21,5.634,5.387,5.489,5.404,5.663,5.65,5.79,5.936,5.12,5.201,5.966,5.787,9.892,10.024,9.74,9.951,9.36,9.321,9.602,10.24,9.124,9.128,10.196,9.022,9.819,10.094,8.92,9.704,7.807,6.414,6.503,7.624,5.788,6.167,6.781,7.148,6.1,6.407,7.1,6.371,6.344,6.292,7.015,12.359,12.111,12.51,12.73,12.127,12.251,12.828,13.195,12.188,12.436,13.076,12.585,11.914,12.605,12.208,13.091,7.769,8.02,8.523,8.765,8.493,5.962,6.061,6.033,5.515,6.557,6.307,5.869,5.741,6.194,6.088,6.515,6.045,5.415,6.676,6.292,10.0,10.395,10.508,10.365,10.633,10.966,10.864,10.856,11.072,10.994,11.019,11.402,10.408,10.515,11.123,11.48,14.866,14.89,15.006,15.59,14.561,14.757,15.242,15.838,14.983,15.252,15.326,15.424,15.194,15.128,15.475,9.395,9.578,9.635,10.032,9.143,9.393,9.815,10.462,9.342,9.919,10.144,9.734,9.52,9.788,9.261,10.401,12.61,12.475,12.673,12.826,12.922,12.717,13.45,13.653,13.157,12.281,13.449],[8.918,8.218,8.522,9.008,8.166,8.437,null,null,8.287,8.537,8.808,8.521,8.436,9.073,8.303,9.116,4.502,4.589,4.973,4.999,4.44,4.887,5.367,5.538,5.308,5.483,5.603,5.755,4.48,5.356,4.921,5.657,2.016,1.896,2.05,1.665,1.597,1.587,1.295,1.215,1.351,1.224,1.038,1.207,2.0,1.786,1.427,1.219,8.797,8.424,8.671,8.097,8.753,8.586,13.18,8.698,8.176,8.509,10.207,10.061,10.201,10.538,10.227,10.427,10.716,11.041,10.576,10.61,10.983,10.847,9.903,10.768,10.393,11.567,9.621,9.765,9.875,10.06,9.381,9.626,10.416,10.377,9.923,10.036,10.534,10.232,9.472,10.057,9.723,10.482,6.979,6.662,6.692,6.463,6.816,6.278,6.141,6.121,5.977,6.056,5.882,5.551,6.825,6.554,5.821,5.733,10.421,10.382,10.694,10.891,10.756,10.706,10.932,11.141,10.934,10.879,11.602,11.126,10.263,10.681,11.104,11.275,2.654,3.537,3.248,3.048,3.958,3.457,3.255,3.389,3.954,3.666,3.459,3.723,3.292,3.064,3.803,10.53,10.362,10.884,11.063,9.981,10.404,10.936,11.41,10.291,10.492,11.111,10.723,10.437,10.938,10.456,11.134,7.438,7.374,7.521,7.767,7.256,7.665,7.963,8.276,7.707,8.111,8.199,8.108,7.117,7.964,7.868,8.401,6.946,6.959,7.505,8.056,6.872,7.107,7.598,7.934,7.942,7.838,8.355,8.154,6.752,7.405,7.443,8.381,2.338,1.813,2.026,2.22,1.676,1.806,2.601,3.158,2.065,2.04,2.302,2.368,1.914,1.688,2.564,4.206,6.95,4.242,3.775,4.875,4.206,3.786,3.423,4.237,3.854,3.739,3.801,4.508,3.741,4.248,3.521,4.518,3.965,5.362,4.884,3.613,3.928,4.581,5.278,3.929,4.282,4.576,4.493,4.523,5.194,3.749,4.702,7.128,7.241,7.413,7.579,7.147,7.346,7.824,8.66,7.639,7.917,8.146,6.911,7.658,7.786,7.176,6.723,7.058,7.364,6.336,7.141,7.485,7.61,6.788,6.881,7.019,6.744,7.504,6.444,7.434,2.892,2.541,2.566,2.542,2.458,2.287,2.148,2.254,2.153,1.863,2.129,1.508,2.716,2.649,2.26,1.685,5.299,5.017,5.108,5.221,4.658,4.988,4.58,4.599,4.274,4.759,4.47,4.32,5.13,5.097,4.247,4.46,9.983,10.111,10.229,10.681,10.212,10.432,10.646,10.843,10.385,10.673,10.771,10.788,9.744,10.519,10.356,10.931,12.624,12.837,12.857,13.061,12.335,12.524,13.19,13.603,12.676,12.923,13.526,12.874,12.27,13.373,12.677,13.511,3.866,3.925,3.69,3.458,4.593,3.986,3.335,3.06,4.338,3.674,3.426,4.173,4.125,3.642,4.241,3.475,7.099,7.195,6.954,6.779,7.225,7.101,6.617,6.746,6.476,6.256,7.374,6.813,5.065,4.94,5.19,5.456,5.078,5.465,5.709,6.215,5.397,5.675,5.949,6.105,4.93,5.648,2.997,2.559,2.93,3.116,2.503,2.861,3.239,3.58,2.988,3.09,3.36,3.69,2.408,3.172,2.825,3.608,5.197,4.843,5.254,4.87,4.605,4.656,4.478,4.474,4.251,4.381,4.221,3.956,4.859,4.975,4.161,4.403,0.945,1.481,1.234,1.054,1.934,1.824,1.301,1.111,2.015,2.287,1.635,2.047,1.428,1.094,2.273,1.704,4.748,5.208,4.986,4.694,5.633,5.386,4.971,4.724,5.661,5.452,5.592,5.687,5.219,4.79,5.965,5.269,7.26,7.392,7.108,7.319,6.728,6.689,6.97,7.608,6.492,6.496,7.564,6.39,7.187,7.462,6.288,7.072,6.109,4.716,4.805,5.926,4.09,4.469,5.083,5.45,4.402,4.709,5.402,4.673,4.657,4.594,5.217,9.008,8.76,9.127,9.379,8.744,8.868,9.477,9.844,8.805,9.053,9.725,9.202,8.563,9.254,8.825,9.708,8.116,8.367,8.87,9.112,8.043,6.452,6.551,6.636,5.917,7.047,6.797,6.271,6.143,6.596,6.49,6.917,6.535,5.817,7.166,6.694,6.81,7.204,7.316,7.172,7.462,7.795,7.671,7.476,7.901,7.823,7.826,8.231,7.237,7.296,7.952,8.309,11.534,11.558,11.674,12.258,11.229,11.425,11.91,12.506,11.651,11.92,11.994,12.092,11.862,11.796,12.143,6.454,6.375,6.639,6.829,5.94,6.19,6.612,7.259,6.139,6.716,6.941,6.531,6.317,6.796,6.058,7.198,9.382,9.247,9.445,9.598,9.694,9.489,10.222,10.425,9.929,9.053,10.221],[10.8,10.1,10.404,10.89,10.048,10.319,null,null,10.169,10.419,10.69,10.403,10.318,10.955,10.185,10.998,6.53,6.617,7.001,7.027,6.468,6.915,7.395,7.566,7.336,7.511,7.631,7.783,6.508,7.384,6.949,7.685,1.548,1.114,1.918,1.824,0.763,1.084,1.806,2.172,1.082,1.363,1.735,1.326,1.217,1.97,0.898,2.336,10.679,10.306,10.553,10.125,10.781,10.614,15.208,10.726,10.058,10.537,12.235,12.089,12.229,12.566,12.255,12.455,12.744,13.069,12.604,12.638,13.011,12.875,11.931,12.796,12.421,13.595,11.297,11.411,11.521,11.706,11.027,11.272,12.062,12.119,11.569,11.682,12.18,11.878,11.139,11.911,11.369,12.128,6.587,6.27,6.3,6.071,6.424,5.886,5.749,5.729,5.585,5.664,5.49,5.159,6.433,6.162,5.429,5.341,11.92,11.814,12.126,12.323,11.664,11.985,12.364,12.636,12.102,12.311,13.034,12.558,11.695,12.327,12.011,12.707,3.56,3.464,3.711,3.954,3.427,3.92,4.161,4.444,3.881,4.128,4.365,4.387,3.219,3.97,4.725,12.558,12.39,12.912,13.091,12.009,12.432,12.964,13.438,12.319,12.52,13.139,12.751,12.465,12.966,12.484,13.162,9.292,9.228,9.375,9.621,9.11,9.519,9.817,10.13,9.561,9.965,10.053,9.962,8.971,9.818,9.722,10.255,8.974,8.987,9.533,10.084,8.9,9.135,9.626,9.962,9.97,9.866,10.383,10.182,8.78,9.433,9.471,10.409,4.22,3.695,3.908,4.102,3.558,3.688,4.483,5.04,4.093,4.068,4.33,4.396,3.796,3.716,4.592,3.211,5.453,2.772,2.602,3.378,2.709,2.289,2.25,2.74,2.357,2.242,2.304,3.011,2.742,2.751,2.024,6.4,5.847,7.244,6.766,5.495,5.81,6.463,7.16,5.811,6.164,6.458,6.375,6.405,7.076,5.631,6.584,9.156,9.269,9.441,9.607,9.175,9.374,9.852,10.688,9.667,9.945,10.174,8.939,9.686,9.814,9.058,8.605,8.94,9.246,8.218,9.023,9.367,9.492,8.67,8.763,8.901,8.626,9.386,8.326,9.316,3.352,3.001,3.303,3.566,2.74,2.896,3.412,3.833,3.013,3.352,3.737,3.39,2.919,3.819,3.12,3.567,5.432,4.625,4.97,5.543,4.266,4.97,5.133,5.433,4.475,5.188,5.132,5.008,4.738,5.383,4.384,5.294,12.011,12.139,12.257,12.709,12.165,12.385,12.674,12.871,12.338,12.697,12.799,12.741,11.772,12.547,12.309,12.959,14.652,14.865,14.885,15.089,14.363,14.552,15.218,15.631,14.704,14.951,15.554,14.902,14.298,15.401,14.705,15.539,2.159,2.217,1.9,1.681,3.065,2.514,1.92,1.645,2.935,2.271,2.023,2.77,2.417,1.812,2.838,2.072,5.602,5.698,5.457,5.282,5.728,5.604,5.12,5.249,4.979,4.759,5.877,5.316,7.093,6.968,7.218,7.484,7.106,7.493,7.737,8.243,7.425,7.703,7.977,8.133,6.958,7.676,5.025,4.587,4.958,5.144,4.531,4.889,5.267,5.608,5.016,5.118,5.388,5.718,4.436,5.2,4.853,5.636,4.584,4.229,4.644,4.293,3.975,4.042,3.901,3.897,3.674,3.804,3.644,3.379,4.245,4.398,3.527,3.826,1.85,2.014,2.139,1.959,1.934,1.824,2.206,2.484,2.015,2.247,2.541,2.51,1.428,1.999,2.234,2.61,3.432,3.89,3.68,3.483,4.415,4.168,3.762,3.523,4.444,4.242,4.382,4.498,3.901,3.474,4.747,4.06,9.142,9.274,8.99,9.201,8.61,8.571,8.852,9.49,8.374,8.378,9.446,8.272,9.069,9.344,8.17,8.954,7.797,6.404,6.493,7.614,5.778,6.157,6.771,7.138,6.09,6.397,7.09,6.361,6.334,6.282,7.005,11.036,10.62,10.981,11.239,10.598,10.722,11.337,11.704,10.659,10.907,11.585,11.056,10.423,11.114,10.679,11.562,6.842,7.093,7.596,7.838,6.854,5.094,5.193,5.243,4.588,5.689,5.439,4.942,4.814,5.267,5.161,5.588,5.177,4.488,5.808,5.365,7.402,7.797,7.91,7.767,8.035,8.368,8.266,8.258,8.474,8.396,8.421,8.804,7.81,7.917,8.525,8.882,13.562,13.586,13.702,14.286,13.257,13.453,13.938,14.534,13.679,13.948,14.022,14.12,13.89,13.824,14.171,8.336,8.403,8.576,8.857,7.968,8.218,8.64,9.287,8.167,8.744,8.969,8.559,8.345,8.729,8.086,9.226,10.012,9.877,10.075,10.228,10.324,10.119,10.852,11.055,10.559,9.683,10.851],[11.584,10.884,11.158,11.491,10.606,10.716,null,null,10.564,10.814,10.95,10.608,11.102,11.556,10.432,11.258,5.726,5.811,6.197,6.223,5.417,6.109,6.501,6.528,5.991,6.173,6.289,6.268,5.702,6.58,5.421,6.177,3.441,3.007,3.717,3.47,2.656,2.881,3.402,3.768,2.788,2.952,3.324,2.914,3.11,3.616,2.604,3.932,10.86,10.207,10.454,9.811,10.467,10.3,14.894,10.412,10.081,10.223,11.771,11.625,11.765,12.102,11.791,11.991,12.28,12.75,11.914,12.144,12.454,12.181,11.467,12.332,11.731,12.8,9.72,9.834,9.944,10.129,9.45,9.695,10.485,10.542,9.992,10.105,10.603,10.301,9.562,10.334,9.792,10.551,8.109,7.735,7.817,7.597,7.889,7.354,7.275,7.345,7.102,7.19,7.016,6.685,7.898,7.688,6.946,6.957,10.343,10.237,10.549,10.746,10.087,10.408,10.787,11.059,10.525,10.734,11.457,10.981,10.118,10.75,10.434,11.13,2.717,2.38,2.627,2.919,2.18,2.748,2.984,3.268,2.613,2.86,3.153,2.959,2.209,3.025,3.513,12.244,12.076,12.598,12.777,11.695,12.118,12.65,13.124,12.005,12.206,12.825,12.433,12.151,12.652,12.17,12.848,7.715,7.651,7.798,8.044,7.533,7.942,8.24,8.553,7.984,8.388,8.476,8.385,7.394,8.241,8.145,8.678,8.356,8.369,8.898,9.466,8.096,8.331,8.822,9.344,8.635,8.568,9.115,8.676,8.162,8.815,8.667,9.141,5.527,5.016,5.223,5.405,4.681,4.712,5.298,5.588,4.658,4.633,4.855,4.347,5.124,4.26,4.831,4.553,6.343,4.113,3.944,4.479,3.994,3.63,3.592,3.963,3.642,3.527,3.527,4.352,4.084,3.953,3.365,7.453,7.015,8.16,7.572,6.663,6.687,7.059,7.808,6.672,6.714,7.008,6.925,7.573,7.852,6.474,7.134,8.838,8.951,9.123,9.289,8.62,8.826,9.376,10.212,9.119,9.397,9.556,8.621,9.368,9.196,9.764,9.266,9.531,9.523,8.864,9.572,9.644,9.742,9.101,8.968,9.106,8.831,9.663,8.704,9.521,5.124,4.773,5.075,5.338,4.512,4.668,5.184,5.54,4.73,5.057,5.415,4.794,4.691,5.591,4.837,4.971,7.291,6.518,6.863,7.336,6.159,6.863,6.905,7.205,6.266,6.96,6.904,6.78,6.631,7.176,6.239,7.066,11.312,10.775,10.895,11.347,10.588,10.808,11.299,11.509,10.761,11.12,11.288,11.164,10.41,11.186,10.732,11.448,14.338,14.551,14.571,14.775,14.049,14.238,14.904,15.317,14.39,14.637,15.24,14.588,13.984,15.087,14.391,15.225,2.859,2.821,2.586,2.355,3.468,2.861,2.21,1.935,3.015,2.351,2.103,2.438,3.021,2.568,2.761,1.862,6.189,6.271,6.044,5.869,6.273,6.149,5.707,5.794,5.566,5.333,6.45,5.861,6.775,6.632,6.882,7.148,6.77,7.157,7.401,7.841,6.807,7.085,7.366,7.522,6.622,7.34,4.837,4.273,4.644,4.83,4.217,4.575,4.953,5.294,4.698,4.8,5.07,5.289,4.248,4.886,4.535,5.3,5.928,5.573,5.988,5.653,5.319,5.386,5.261,5.264,5.022,5.164,5.01,4.733,5.589,5.758,4.871,5.312,2.753,2.283,2.572,2.302,1.857,1.589,2.057,2.249,1.343,1.553,1.98,1.816,1.972,2.62,1.54,2.012,1.612,2.074,1.86,1.663,2.599,2.352,1.94,1.693,2.628,2.421,2.561,2.656,2.085,1.654,2.931,2.238,9.926,10.058,9.774,9.985,9.394,9.355,9.636,10.274,9.158,9.162,10.23,9.056,9.853,10.128,8.954,9.738,9.319,7.926,8.015,9.136,7.3,7.679,8.293,8.66,7.612,7.919,8.612,7.883,7.925,7.804,8.373,9.53,9.043,9.404,9.662,9.021,9.145,9.76,10.127,9.082,9.33,10.008,9.479,8.846,9.537,9.102,9.985,5.085,5.336,5.839,6.081,5.012,3.767,3.866,3.985,3.218,4.362,4.112,3.35,3.177,3.826,3.642,4.147,3.85,3.116,4.481,3.727,5.825,6.22,6.333,6.19,6.458,6.791,6.689,6.681,6.897,6.819,6.844,7.227,6.233,6.34,6.948,7.305,12.299,12.225,12.439,13.023,11.879,12.091,12.675,13.271,12.301,12.57,12.661,12.742,12.75,12.311,12.81,8.541,8.089,8.353,8.543,7.654,7.904,8.326,8.973,7.853,8.43,8.655,8.245,8.031,8.51,7.772,8.912,8.435,8.3,8.498,8.651,8.747,8.542,9.275,9.478,8.982,8.106,9.274],[12.475,11.775,12.079,12.565,11.723,11.994,null,null,11.844,12.094,12.365,12.078,11.993,12.63,11.86,12.673,8.205,8.292,8.676,8.702,8.143,8.59,9.07,9.241,8.966,9.148,9.264,9.243,8.183,9.059,8.396,9.152,3.191,2.763,3.561,3.467,2.405,2.735,3.449,3.815,2.733,3.014,3.386,2.977,2.695,3.613,2.549,3.979,12.354,11.981,12.228,11.8,12.456,12.289,16.883,12.401,11.733,12.212,13.91,13.764,13.904,14.241,13.93,14.13,14.419,14.744,14.279,14.313,14.686,14.55,13.606,14.471,14.096,15.27,12.695,12.809,12.919,13.104,12.425,12.67,13.46,13.517,12.967,13.08,13.578,13.276,12.537,13.309,12.767,13.526,5.416,5.042,5.124,4.904,5.196,4.661,4.582,4.815,4.409,4.497,4.323,3.992,5.205,4.995,4.253,4.448,13.318,13.212,13.524,13.721,13.062,13.383,13.762,14.034,13.5,13.709,14.432,13.956,13.093,13.725,13.409,14.105,5.235,5.355,5.602,5.629,5.155,5.723,5.836,6.119,5.588,5.835,6.04,5.934,5.184,5.645,6.4,14.233,14.065,14.587,14.766,13.684,14.107,14.639,15.113,13.994,14.195,14.814,14.426,14.14,14.641,14.159,14.837,10.69,10.626,10.773,11.019,10.508,10.917,11.215,11.528,10.959,11.363,11.451,11.36,10.369,11.216,11.12,11.653,10.649,10.662,11.208,11.759,10.575,10.81,11.301,11.637,11.61,11.541,12.058,11.651,10.455,11.108,11.146,12.084,5.895,5.37,5.583,5.777,5.233,5.363,6.158,6.715,5.768,5.743,6.005,6.071,5.471,5.391,6.267,1.803,3.489,1.362,1.194,1.19,0.948,0.879,0.842,0.639,0.523,0.481,0.203,1.601,1.334,0.532,0.783,8.075,7.522,8.919,8.441,7.17,7.485,8.138,8.835,7.486,7.839,8.133,8.05,8.08,8.751,7.306,8.259,10.831,10.944,11.116,11.282,10.85,11.049,11.527,12.363,11.342,11.62,11.849,10.614,11.361,11.489,10.733,10.28,10.615,10.921,9.893,10.698,11.042,11.167,10.345,10.438,10.576,10.301,11.061,10.001,10.991,4.649,4.298,4.6,4.863,4.037,4.193,4.709,5.13,4.31,4.649,5.034,4.792,4.216,5.116,4.417,5.105,6.729,5.922,6.267,6.84,5.563,6.267,6.43,6.73,5.772,6.485,6.429,6.305,5.931,6.68,5.681,6.591,13.686,13.75,13.87,14.322,13.563,13.783,14.274,14.484,13.736,14.095,14.263,14.139,13.385,14.161,13.707,14.423,16.327,16.54,16.56,16.764,16.038,16.227,16.893,17.306,16.379,16.626,17.229,16.577,15.973,17.076,16.38,17.214,1.039,1.523,1.276,1.245,1.91,1.82,1.55,1.667,2.089,1.869,1.812,2.336,1.483,1.218,2.15,1.861,3.638,3.734,3.493,3.318,3.764,3.64,3.156,3.285,3.015,2.795,3.913,3.352,8.768,8.643,8.893,9.159,8.781,9.168,9.412,9.918,9.1,9.378,9.652,9.808,8.633,9.351,6.7,6.262,6.633,6.819,6.206,6.564,6.942,7.283,6.691,6.793,7.063,7.393,6.111,6.875,6.528,7.311,3.235,2.88,3.464,3.429,2.625,2.863,3.255,3.513,2.636,3.118,3.259,2.879,2.896,3.554,2.43,3.6,3.525,4.061,3.814,3.634,4.018,3.908,3.881,4.159,4.099,4.331,4.216,4.594,3.512,3.674,4.318,4.285,3.227,3.49,3.475,3.653,4.015,3.768,3.932,4.058,4.044,4.037,4.177,4.323,3.501,3.644,4.347,4.23,10.817,10.949,10.665,10.876,10.285,10.246,10.527,11.165,10.049,10.053,11.121,9.947,10.744,11.019,9.845,10.629,9.094,7.701,7.79,8.911,7.075,7.454,8.068,8.435,7.387,7.694,8.387,7.658,7.631,7.579,8.302,12.505,12.018,12.379,12.637,11.996,12.12,12.735,13.102,12.057,12.305,12.983,12.454,11.821,12.512,12.077,12.96,6.15,6.401,6.904,7.146,6.874,4.343,4.442,4.414,3.896,4.938,4.688,4.25,4.122,4.575,4.469,4.896,4.426,3.796,5.057,4.673,8.8,9.195,9.308,9.165,9.433,9.766,9.664,9.656,9.872,9.794,9.819,10.202,9.208,9.315,9.923,10.28,15.237,15.2,15.377,15.961,14.854,15.066,15.613,16.209,15.276,15.545,15.636,15.717,15.565,15.286,15.785,10.011,10.078,10.251,10.532,9.643,9.893,10.315,10.962,9.842,10.419,10.644,10.234,10.02,10.404,9.761,10.901,11.41,11.275,11.473,11.626,11.722,11.517,12.25,12.453,11.957,11.081,12.249],[15.451,14.751,15.055,15.541,14.699,14.97,null,null,14.82,15.07,15.341,15.054,14.969,15.606,14.836,15.649,11.097,11.182,11.568,11.594,10.788,11.48,11.872,11.899,11.362,11.544,11.66,11.639,11.073,11.951,10.792,11.548,6.167,5.739,6.537,6.443,5.381,5.711,6.425,6.791,5.709,5.99,6.362,5.953,5.671,6.589,5.525,6.955,15.33,14.957,15.204,14.776,15.432,15.265,19.859,15.377,14.709,15.188,16.886,16.74,16.88,17.217,16.906,17.106,17.395,17.72,17.255,17.289,17.662,17.526,16.582,17.447,17.072,18.171,15.091,15.205,15.315,15.5,14.821,15.066,15.856,15.913,15.363,15.476,15.974,15.672,14.933,15.705,15.163,15.922,7.981,7.607,7.748,8.027,7.702,7.226,7.705,7.938,7.121,7.491,7.446,7.115,7.687,8.118,6.908,7.571,15.714,15.608,15.92,16.117,15.458,15.779,16.158,16.43,15.896,16.105,16.828,16.352,15.489,16.121,15.805,16.501,8.088,7.751,7.998,8.29,7.551,8.119,8.355,8.639,7.984,8.231,8.524,8.33,7.58,8.396,8.884,17.209,17.041,17.563,17.742,16.66,17.083,17.615,18.089,16.97,17.171,17.79,17.402,17.116,17.617,17.135,17.813,13.086,13.022,13.169,13.415,12.904,13.313,13.611,13.924,13.355,13.759,13.847,13.756,12.765,13.612,13.516,14.049,13.625,13.638,14.184,14.735,13.467,13.702,14.193,14.613,14.006,13.939,14.486,14.047,13.431,14.084,14.038,14.512,8.871,8.346,8.559,8.753,8.209,8.339,9.134,9.691,8.744,8.719,8.981,9.047,8.447,8.367,9.243,3.675,3.091,3.2,3.544,2.709,2.88,3.553,3.896,2.845,3.255,3.669,3.185,3.392,3.986,3.288,3.942,11.051,10.498,11.895,11.417,10.146,10.461,11.114,11.811,10.462,10.815,11.109,11.026,11.056,11.727,10.282,11.235,13.807,13.92,14.092,14.258,13.826,14.025,14.503,15.339,14.318,14.596,14.825,13.59,14.337,14.465,13.709,13.256,13.591,13.897,12.869,13.674,14.018,14.143,13.321,13.414,13.552,13.277,14.037,12.977,13.967,7.681,7.33,7.632,7.895,7.069,7.225,7.741,8.162,7.342,7.681,8.066,7.824,7.248,8.148,7.449,8.137,9.761,8.954,9.299,9.872,8.595,9.299,9.462,9.762,8.804,9.517,9.461,9.337,8.963,9.712,8.713,9.623,16.662,16.146,16.266,16.718,15.959,16.179,16.67,16.88,16.132,16.491,16.659,16.535,15.781,16.557,16.103,16.819,19.303,19.516,19.536,19.74,19.014,19.203,19.869,20.282,19.355,19.602,20.205,19.553,18.949,20.052,19.356,20.19,3.283,3.39,3.52,3.806,2.942,3.2,3.667,3.872,3.405,3.814,4.017,3.904,3.119,3.779,3.466,4.066,0.66,0.57,0.515,0.341,0.704,0.766,0.358,0.85,0.583,0.901,0.749,1.028,11.744,11.619,11.869,12.135,11.757,12.144,12.388,12.894,12.076,12.354,12.628,12.784,11.609,12.327,9.676,9.238,9.609,9.795,9.182,9.54,9.918,10.259,9.667,9.769,10.039,10.369,9.087,9.851,9.504,10.287,6.358,6.003,6.587,6.552,5.508,5.986,6.376,6.634,5.759,6.241,6.38,6.0,6.019,6.677,5.553,6.721,6.501,6.836,6.79,6.61,6.709,6.598,6.857,7.135,6.737,6.963,7.192,7.226,6.303,6.65,6.95,7.261,5.359,5.25,5.529,5.87,5.275,5.44,6.138,6.412,5.707,6.08,6.22,6.205,5.297,5.861,5.607,6.429,13.793,13.925,13.641,13.852,13.261,13.222,13.503,14.141,13.025,13.029,14.097,12.923,13.72,13.995,12.821,13.605,12.126,10.733,10.822,11.943,10.107,10.486,11.1,11.467,10.419,10.726,11.419,10.69,10.663,10.611,11.334,14.901,14.414,14.775,15.033,14.392,14.516,15.131,15.498,14.453,14.701,15.379,14.85,14.217,14.908,14.473,15.356,5.563,5.814,6.317,6.559,8.001,3.497,3.596,3.742,4.016,4.092,3.842,4.178,4.31,4.144,4.397,4.465,3.58,3.984,4.211,4.685,11.196,11.591,11.704,11.561,11.829,12.162,12.06,12.052,12.268,12.19,12.215,12.598,11.604,11.711,12.319,12.676,17.67,17.596,17.81,18.394,17.25,17.462,18.046,18.642,17.672,17.941,18.032,18.113,18.121,17.682,18.181,12.987,13.054,13.227,13.508,12.619,12.869,13.291,13.938,12.818,13.395,13.62,13.21,12.996,13.38,12.737,13.877,13.806,13.671,13.869,14.022,14.118,13.913,14.646,14.849,14.353,13.477,14.645],[12.505,11.805,12.109,12.595,11.753,12.024,null,null,11.874,12.124,12.267,11.925,12.023,12.66,11.749,12.575,7.036,7.121,7.507,7.533,6.727,7.419,7.811,7.838,7.301,7.483,7.599,7.578,7.012,7.89,6.731,7.487,4.254,3.82,4.53,4.283,3.469,3.694,4.215,4.581,3.601,3.765,4.137,3.727,3.923,4.429,3.417,4.745,12.177,11.524,11.771,11.128,11.784,11.617,16.211,11.729,11.398,11.54,13.081,12.935,13.075,13.412,13.101,13.301,13.59,14.06,13.224,13.454,13.764,13.491,12.777,13.642,13.041,14.11,11.03,11.144,11.254,11.439,10.76,11.005,11.795,11.852,11.302,11.415,11.913,11.611,10.872,11.644,11.102,11.861,7.824,7.45,7.532,7.312,7.604,7.069,6.99,7.223,6.817,6.905,6.731,6.4,7.613,7.403,6.661,6.856,11.653,11.547,11.859,12.056,11.397,11.718,12.097,12.369,11.835,12.044,12.767,12.291,11.428,12.06,11.744,12.44,4.027,3.69,3.937,4.229,3.49,4.058,4.294,4.578,3.923,4.17,4.463,4.269,3.519,4.335,4.823,13.561,13.393,13.915,14.094,13.012,13.435,13.967,14.441,13.322,13.523,14.142,13.754,13.468,13.969,13.487,14.165,9.025,8.961,9.108,9.354,8.843,9.252,9.55,9.863,9.294,9.698,9.786,9.695,8.704,9.551,9.455,9.988,9.666,9.679,10.208,10.776,9.406,9.641,10.132,10.654,9.945,9.878,10.425,9.986,9.472,10.125,9.977,10.451,6.438,5.913,6.126,6.32,5.584,5.615,6.242,6.656,5.579,5.554,5.781,5.415,6.014,5.167,5.899,4.198,4.933,3.755,3.589,3.409,3.341,3.272,3.25,3.03,2.915,2.889,2.611,3.994,3.729,2.883,3.191,8.374,7.959,9.081,8.564,7.607,7.608,8.127,8.876,7.74,7.782,8.076,7.993,8.517,8.773,7.542,8.202,10.159,10.272,10.444,10.61,9.93,10.136,10.686,11.522,10.429,10.707,10.866,9.942,10.689,10.506,10.763,10.302,10.645,10.84,9.932,10.72,10.961,11.059,10.384,10.285,10.423,10.148,10.98,10.01,10.838,5.937,5.586,5.888,6.151,5.325,5.481,5.997,6.354,5.543,5.87,6.229,5.608,5.504,6.404,5.65,5.785,8.104,7.331,7.676,8.149,6.972,7.676,7.718,8.018,7.079,7.773,7.717,7.593,7.444,7.989,7.052,7.879,12.622,12.085,12.205,12.657,11.898,12.118,12.609,12.819,12.071,12.43,12.598,12.474,11.72,12.496,12.042,12.758,15.655,15.868,15.888,16.092,15.366,15.555,16.221,16.634,15.707,15.954,16.557,15.905,15.301,16.404,15.708,16.542,1.449,1.411,1.176,1.396,1.901,1.451,1.234,1.366,1.425,0.763,1.022,0.786,1.611,1.608,1.17,0.781,4.726,4.808,4.581,4.406,4.81,4.686,4.244,4.331,4.103,3.87,4.987,4.398,8.096,7.942,8.192,8.458,8.08,8.467,8.711,9.151,8.117,8.395,8.676,8.832,7.932,8.65,6.028,5.59,5.961,6.147,5.534,5.892,6.27,6.611,6.019,6.121,6.391,6.599,5.439,6.203,5.856,6.61,5.643,5.288,5.872,5.837,5.033,5.271,5.596,5.607,5.044,5.499,5.353,5.076,5.304,5.962,4.838,5.694,3.57,3.101,3.39,3.12,2.833,2.652,3.123,3.315,2.676,2.902,3.329,3.165,2.789,3.437,2.889,3.361,1.132,1.1,1.38,1.696,1.625,1.378,1.975,2.229,1.654,1.942,2.082,2.098,1.111,1.687,1.957,2.273,10.847,10.979,10.695,10.906,10.315,10.276,10.557,11.195,10.079,10.083,11.151,9.977,10.774,11.049,9.875,10.659,10.209,8.816,8.905,10.026,8.19,8.569,9.183,9.55,8.502,8.809,9.502,8.773,8.757,8.694,9.317,10.84,10.353,10.714,10.972,10.331,10.455,11.07,11.437,10.392,10.64,11.318,10.789,10.156,10.847,10.412,11.295,4.098,4.349,4.852,5.094,4.484,2.389,2.488,2.591,1.844,2.984,2.734,2.198,2.07,2.523,2.417,2.844,2.472,1.744,3.103,2.621,7.135,7.53,7.643,7.5,7.768,8.101,7.999,7.991,8.207,8.129,8.154,8.537,7.543,7.65,8.258,8.615,13.609,13.535,13.749,14.333,13.189,13.401,13.985,14.581,13.611,13.88,13.971,14.052,14.06,13.621,14.12,9.858,9.406,9.67,9.86,8.971,9.221,9.643,10.29,9.17,9.747,9.972,9.562,9.348,9.827,9.089,10.229,9.745,9.61,9.808,9.961,10.057,9.852,10.585,10.788,10.292,9.416,10.584],[14.582,13.882,14.186,14.672,13.83,14.101,null,null,13.951,14.201,14.472,14.185,14.1,14.737,13.967,14.78,9.989,10.074,10.46,10.486,9.68,10.372,10.764,10.791,10.254,10.436,10.552,10.531,9.965,10.843,9.684,10.44,5.612,5.178,5.982,5.847,4.827,5.148,5.779,6.145,5.146,5.329,5.701,5.291,5.25,5.993,4.962,6.309,14.461,14.088,14.335,13.907,14.563,14.396,18.99,14.508,13.84,14.319,16.017,15.871,16.011,16.348,16.037,16.237,16.526,16.851,16.177,16.407,16.717,16.444,15.713,16.578,15.994,17.063,13.983,14.097,14.207,14.392,13.713,13.958,14.748,14.805,14.255,14.368,14.866,14.564,13.825,14.597,14.055,14.814,8.384,8.01,8.151,8.001,8.162,7.629,7.679,7.912,7.506,7.594,7.42,7.089,8.147,8.092,7.311,7.545,14.606,14.5,14.812,15.009,14.35,14.671,15.05,15.322,14.788,14.997,15.72,15.244,14.381,15.013,14.697,15.393,6.98,6.643,6.89,7.182,6.443,7.011,7.247,7.531,6.876,7.123,7.416,7.222,6.472,7.288,7.776,16.34,16.172,16.694,16.873,15.791,16.214,16.746,17.22,16.101,16.302,16.921,16.533,16.247,16.748,16.266,16.944,11.978,11.914,12.061,12.307,11.796,12.205,12.503,12.816,12.247,12.651,12.739,12.648,11.657,12.504,12.408,12.941,12.619,12.632,13.161,13.729,12.359,12.594,13.085,13.607,12.898,12.831,13.378,12.939,12.425,13.078,12.93,13.404,8.002,7.477,7.69,7.884,7.34,7.47,8.265,8.822,7.875,7.85,8.112,8.178,7.578,7.498,8.374,4.078,3.551,3.603,3.856,3.151,3.283,3.728,3.939,3.111,3.371,3.578,3.289,3.842,4.185,3.339,3.88,10.182,9.629,11.026,10.548,9.277,9.592,10.245,10.942,9.593,9.946,10.24,10.157,10.187,10.858,9.413,10.366,12.938,13.051,13.223,13.389,12.883,13.089,13.634,14.47,13.382,13.66,13.819,12.721,13.468,13.459,12.84,12.387,12.722,13.028,12.0,12.805,13.149,13.274,12.452,12.545,12.683,12.408,13.168,12.108,13.098,7.416,7.065,7.367,7.63,6.804,6.96,7.476,7.897,7.077,7.416,7.793,7.172,6.983,7.883,7.184,7.349,9.496,8.689,9.034,9.607,8.33,9.034,9.197,9.497,8.539,9.252,9.196,9.072,8.79,9.447,8.448,9.358,15.575,15.038,15.158,15.61,14.851,15.071,15.562,15.772,15.024,15.383,15.551,15.427,14.673,15.449,14.995,15.711,18.434,18.647,18.667,18.871,18.145,18.334,19.0,19.413,18.486,18.733,19.336,18.684,18.08,19.183,18.487,19.321,2.415,2.3,2.501,2.794,1.834,2.092,2.559,2.764,2.297,2.706,2.909,2.796,2.011,2.789,2.358,2.958,2.204,2.286,2.059,1.884,2.196,2.072,1.722,1.717,1.581,1.348,2.465,1.784,10.875,10.75,11.0,11.266,10.888,11.275,11.519,12.025,11.07,11.348,11.629,11.785,10.74,11.458,8.807,8.369,8.74,8.926,8.313,8.671,9.049,9.39,8.798,8.9,9.17,9.5,8.218,8.982,8.635,9.418,6.332,5.977,6.561,6.526,5.722,5.96,6.285,6.543,5.733,6.188,6.289,5.909,5.993,6.651,5.527,6.63,5.632,5.728,5.921,5.741,5.601,5.49,5.988,6.266,5.629,5.855,6.282,6.118,5.195,5.781,5.842,6.314,4.251,4.142,4.421,4.762,4.167,4.332,5.03,5.304,4.599,4.972,5.112,5.097,4.189,4.753,4.499,5.321,12.924,13.056,12.772,12.983,12.392,12.353,12.634,13.272,12.156,12.16,13.228,12.054,12.851,13.126,11.952,12.736,11.773,10.38,10.469,11.59,9.754,10.133,10.747,11.114,10.066,10.373,11.066,10.337,10.321,10.258,10.881,13.793,13.306,13.667,13.925,13.284,13.408,14.023,14.39,13.345,13.593,14.271,13.742,13.109,13.8,13.365,14.248,4.455,4.706,5.209,5.451,6.893,2.389,2.488,2.634,2.908,2.984,2.734,3.07,3.202,3.036,3.289,3.357,2.472,2.876,3.103,3.577,10.088,10.483,10.596,10.453,10.721,11.054,10.952,10.944,11.16,11.082,11.107,11.49,10.496,10.603,11.211,11.568,16.562,16.488,16.702,17.286,16.142,16.354,16.938,17.534,16.564,16.833,16.924,17.005,17.013,16.574,17.073,12.118,12.185,12.358,12.639,11.75,12.0,12.422,13.069,11.949,12.526,12.751,12.341,12.127,12.511,11.868,13.008,12.698,12.563,12.761,12.914,13.01,12.805,13.538,13.741,13.245,12.369,13.537],[11.214,10.505,10.641,10.974,10.089,10.199,null,null,10.047,10.297,10.433,10.091,10.939,11.039,9.915,10.741,3.725,3.683,3.524,3.55,3.051,3.385,3.291,3.286,3.083,3.08,3.064,2.85,3.792,3.808,2.808,2.832,8.996,8.761,9.03,8.645,8.433,8.423,8.275,8.195,8.137,8.06,7.924,7.813,8.865,8.766,8.033,8.199,10.078,9.306,9.553,8.91,9.566,9.062,11.619,8.997,9.299,8.985,8.059,7.766,7.753,7.973,7.451,7.398,7.949,8.22,7.309,7.482,7.792,7.519,8.261,8.203,7.126,8.033,3.362,3.476,3.586,3.771,3.092,3.337,4.127,4.184,3.634,3.747,4.245,3.943,3.204,4.038,3.434,4.193,13.786,13.469,13.499,13.27,13.623,13.085,12.948,12.928,12.784,12.863,12.689,12.358,13.632,13.361,12.628,12.54,3.985,3.879,4.191,4.388,3.792,4.113,4.429,4.701,4.23,4.376,5.099,4.623,3.76,4.392,4.139,4.772,5.004,5.129,5.056,4.521,5.066,4.765,4.471,4.217,4.57,4.494,4.207,4.074,5.2,4.501,3.905,9.731,9.272,9.94,9.949,9.005,9.216,9.822,10.086,8.972,9.255,9.55,9.011,9.658,10.084,8.783,9.573,2.862,2.655,2.716,2.953,2.193,2.619,2.82,2.998,2.122,2.456,2.518,2.108,2.422,3.317,2.125,2.461,5.091,4.639,4.981,5.556,4.319,4.378,4.905,5.784,4.153,4.476,5.022,4.194,5.035,5.55,4.33,5.048,7.647,7.368,7.445,7.452,7.308,7.339,7.229,7.363,6.848,6.823,6.914,6.537,7.568,6.887,6.762,10.99,12.749,10.613,10.381,10.885,10.494,10.13,10.029,10.369,10.142,10.027,9.933,10.852,10.521,10.359,9.865,9.336,8.931,10.026,9.347,8.579,8.57,8.834,9.583,8.447,8.489,8.783,8.7,9.489,9.683,8.249,8.909,7.145,6.878,7.015,7.344,6.178,6.384,6.866,7.143,6.377,6.656,6.144,6.928,7.675,5.928,9.29,8.785,9.014,9.006,8.546,9.055,9.127,9.225,8.543,8.451,8.589,8.314,9.146,8.146,9.004,9.844,9.521,9.546,9.456,9.438,9.267,9.128,9.165,9.133,8.843,9.109,8.488,9.696,9.456,9.24,8.665,12.106,11.824,11.915,12.028,11.465,11.795,11.387,11.406,11.081,11.566,11.277,11.127,11.937,11.904,11.054,11.267,6.909,5.957,6.47,6.401,5.654,5.709,6.2,6.55,5.472,5.826,6.183,5.831,6.171,6.478,5.294,6.343,10.796,10.858,10.878,11.054,10.204,10.495,11.173,11.486,10.535,10.782,11.297,10.514,10.442,11.504,10.39,11.151,9.265,9.227,8.992,8.761,9.874,9.267,8.616,8.341,9.421,8.757,8.509,8.844,9.427,8.974,9.167,8.268,12.595,12.677,12.45,12.275,12.679,12.555,12.113,12.2,11.972,11.739,12.856,12.267,5.694,5.566,5.563,5.677,5.057,5.172,5.434,5.537,4.957,5.069,5.082,5.006,5.556,5.643,5.833,5.427,5.385,5.533,5.157,5.268,5.426,5.119,5.088,5.055,5.085,4.745,5.603,5.599,4.973,4.743,12.177,11.823,12.234,11.85,11.585,11.636,11.458,11.454,11.231,11.361,11.201,10.936,11.839,11.955,11.141,11.383,6.916,6.909,6.733,6.463,6.805,6.537,6.216,6.408,6.288,6.498,5.979,6.007,6.921,6.783,6.327,5.89,7.204,7.285,6.992,6.7,7.407,7.16,6.692,6.445,7.066,6.787,6.847,6.653,7.407,6.849,7.222,6.287,11.722,11.854,11.57,11.781,11.19,11.151,11.432,11.958,10.954,10.958,11.914,10.852,11.649,11.924,10.75,10.886,11.435,10.537,10.59,11.252,10.077,10.388,10.647,10.776,9.921,10.273,10.866,10.135,10.702,9.784,10.304,4.705,4.19,4.181,4.455,3.445,3.939,4.233,4.739,3.446,3.694,4.481,3.481,3.993,4.684,3.466,3.987,8.723,8.974,9.477,9.719,8.648,9.814,9.994,9.751,9.198,10.49,10.24,8.85,8.59,9.324,8.754,8.97,9.978,9.038,10.609,8.536,1.073,1.409,1.052,0.735,1.73,1.775,0.882,0.511,1.341,0.983,0.863,1.299,1.58,0.699,1.479,1.377,7.532,7.325,7.573,8.129,6.774,7.127,7.781,8.364,7.19,7.45,7.742,7.467,7.983,7.013,7.856,8.024,7.188,7.795,7.642,6.753,7.003,7.425,7.738,6.794,7.529,7.623,7.035,7.13,7.809,6.759,7.677,2.417,2.282,2.48,2.633,2.729,2.524,3.257,3.46,2.964,2.088,3.256],[12.451,11.742,11.878,12.211,11.326,11.436,null,null,11.284,11.534,11.67,11.328,12.176,12.276,11.152,11.978,5.036,4.994,5.021,5.047,4.067,4.696,4.922,4.949,4.397,4.579,4.695,4.364,5.103,5.555,3.827,4.495,7.194,6.911,7.23,6.821,6.56,6.552,6.641,6.759,6.157,6.191,6.44,6.153,7.014,6.967,6.052,6.769,11.58,10.814,11.061,10.418,11.074,10.907,14.184,11.019,10.801,10.83,10.624,10.331,10.318,10.538,10.016,9.963,10.514,10.785,9.874,10.047,10.357,10.084,10.815,10.768,9.691,10.627,6.912,7.026,7.136,7.321,6.642,6.887,7.677,7.734,7.184,7.297,7.795,7.493,6.754,7.588,6.984,7.743,12.298,11.924,12.006,11.786,12.078,11.543,11.464,11.452,11.291,11.379,11.205,10.874,12.087,11.877,11.135,11.064,7.535,7.421,7.733,7.93,7.092,7.413,7.941,8.213,7.53,7.75,8.605,7.995,7.302,7.942,7.439,8.278,3.52,3.091,3.228,3.268,2.835,2.937,2.987,3.105,2.532,2.666,2.755,2.561,3.162,3.389,3.115,12.296,11.837,12.505,12.514,11.57,11.781,12.387,12.651,11.537,11.82,12.115,11.576,12.223,12.649,11.348,12.138,5.204,5.14,5.287,5.533,4.922,5.413,5.729,6.042,5.326,5.817,5.938,5.658,4.883,5.73,5.487,6.011,7.087,6.635,6.977,7.552,6.175,6.41,6.901,7.78,6.424,6.471,7.018,6.465,6.894,7.546,6.601,7.044,7.525,7.176,7.29,7.403,7.115,7.146,7.296,7.586,6.656,6.631,6.853,6.345,7.376,6.694,6.829,8.742,10.119,8.302,8.133,8.463,8.183,7.819,7.781,7.947,7.831,7.716,7.511,8.541,8.273,7.937,7.554,9.451,9.013,10.158,9.57,8.661,8.685,9.057,9.806,8.67,8.712,9.006,8.923,9.571,9.85,8.472,9.132,9.099,8.874,9.011,9.34,8.174,8.38,8.862,9.223,8.373,8.736,8.224,8.882,9.629,7.924,10.527,10.042,10.251,10.243,9.803,10.292,10.364,10.462,9.821,9.688,9.826,9.551,10.383,9.424,10.241,8.363,8.012,8.13,8.193,7.751,7.851,8.039,8.219,7.717,7.736,8.094,7.473,7.93,8.446,7.824,7.65,10.63,10.348,10.439,10.675,9.989,10.319,10.144,10.236,9.605,10.199,10.107,9.957,10.461,10.515,9.578,10.097,9.375,8.486,8.798,8.989,8.077,8.297,8.788,9.144,8.25,8.609,8.777,8.653,8.313,9.066,8.221,8.937,13.361,13.423,13.443,13.619,12.769,13.06,13.738,14.051,13.1,13.347,13.862,13.079,13.007,14.069,12.955,13.716,6.715,6.647,6.442,6.339,6.976,6.651,6.194,5.919,6.5,6.029,6.01,5.88,6.847,6.552,6.245,5.769,9.801,9.883,9.656,9.481,9.885,9.761,9.319,9.406,9.178,8.945,10.062,9.473,7.162,6.743,6.993,7.259,6.655,6.934,7.29,7.396,6.644,6.831,6.858,6.768,6.733,7.451,5.878,5.234,5.393,5.579,4.705,5.182,5.56,5.731,5.085,5.187,5.457,5.262,5.334,5.783,4.715,5.411,10.117,9.762,10.177,9.842,9.508,9.575,9.45,9.453,9.211,9.353,9.199,8.922,9.778,9.947,9.06,9.501,5.432,5.081,5.249,4.979,4.767,4.499,4.732,4.924,4.25,4.46,4.495,4.179,4.883,5.299,4.289,4.406,4.442,4.228,4.194,3.905,4.143,4.014,3.696,3.453,3.808,3.78,3.84,3.596,4.383,4.054,3.955,3.28,11.924,12.056,11.772,11.983,11.392,11.353,11.634,12.272,11.156,11.16,12.228,11.054,11.851,12.126,10.952,11.736,11.502,10.384,10.437,11.319,9.881,10.235,10.661,10.843,9.98,10.287,10.933,10.202,10.506,9.851,10.371,7.047,6.532,6.893,7.151,6.51,6.634,7.249,7.616,6.571,6.819,7.497,6.968,6.335,7.026,6.591,7.474,5.21,5.461,5.964,6.206,5.135,6.614,6.794,6.551,6.001,7.29,7.04,5.65,5.39,6.124,5.49,5.706,6.778,5.841,7.409,5.272,2.83,3.225,3.338,3.195,3.463,3.796,3.694,3.686,3.902,3.824,3.849,4.232,3.238,3.345,3.953,4.31,10.126,9.919,10.167,10.738,9.368,9.722,10.39,10.986,9.79,10.059,10.351,10.231,10.577,9.8,10.49,9.261,8.696,9.073,9.15,8.261,8.511,8.933,9.58,8.46,9.037,9.262,8.852,8.638,9.23,8.379,9.519,5.44,5.305,5.503,5.656,5.752,5.547,6.28,6.483,5.987,5.111,6.279],[11.633,10.924,11.06,11.393,10.508,10.618,null,null,10.466,10.716,10.852,10.51,11.358,11.458,10.334,11.16,4.51,4.595,4.676,4.702,3.566,4.351,4.577,4.604,4.052,4.234,4.35,4.329,4.486,5.21,3.482,4.238,6.374,6.091,6.41,6.001,5.74,5.732,5.821,5.941,5.337,5.371,5.622,5.333,6.194,6.147,5.232,5.951,10.762,9.996,10.243,9.6,10.256,10.089,14.133,10.201,9.983,10.012,10.782,10.489,10.476,10.696,10.174,10.121,10.672,10.943,10.032,10.205,10.515,10.242,10.486,10.926,9.849,10.861,7.763,7.877,7.987,8.172,7.493,7.738,8.528,8.585,8.035,8.148,8.646,8.344,7.605,8.378,7.835,8.594,11.326,10.952,11.034,10.814,11.106,10.571,10.492,10.609,10.319,10.407,10.233,9.902,11.115,10.905,10.163,10.221,8.386,8.28,8.592,8.789,7.967,8.288,8.816,9.088,8.405,8.625,9.48,8.87,8.161,8.793,8.314,9.153,2.702,2.273,2.41,2.45,2.015,2.119,2.169,2.287,1.714,1.848,1.937,1.743,2.344,2.571,2.297,11.816,11.648,12.17,12.349,11.267,11.69,12.222,12.6,11.486,11.769,12.064,11.525,11.723,12.224,11.297,12.087,5.759,5.695,5.842,6.088,5.577,5.986,6.284,6.597,6.028,6.432,6.52,6.429,5.438,6.285,6.189,6.722,7.245,6.793,7.135,7.71,6.333,6.568,7.059,7.938,6.696,6.629,7.176,6.737,7.052,7.704,6.873,7.202,6.707,6.358,6.472,6.585,6.297,6.328,6.478,6.768,5.838,5.813,6.035,5.527,6.558,5.876,6.011,7.77,9.035,7.33,7.161,7.171,7.084,6.847,6.809,6.792,6.676,6.617,6.356,7.569,7.301,6.645,6.582,8.633,8.195,9.34,8.752,7.843,7.867,8.239,8.988,7.852,7.894,8.188,8.105,8.753,9.032,7.654,8.314,8.281,8.339,8.544,8.732,7.639,7.845,8.395,9.231,8.138,8.416,8.382,8.064,8.811,8.082,9.709,9.224,9.433,9.425,8.985,9.474,9.546,9.644,9.003,8.87,9.008,8.733,9.565,8.606,9.423,7.543,7.192,7.312,7.375,6.931,7.033,7.221,7.401,6.899,6.918,7.276,6.655,7.11,7.628,7.006,6.832,9.81,9.528,9.619,9.855,9.169,9.499,9.324,9.418,8.785,9.379,9.289,9.139,9.641,9.695,8.758,9.279,9.533,8.836,8.956,9.408,8.632,8.852,9.343,9.57,8.805,9.164,9.332,9.208,8.471,9.247,8.776,9.492,13.519,13.581,13.601,13.777,12.927,13.218,13.896,14.209,13.258,13.505,14.02,13.237,13.165,14.227,13.113,13.874,5.551,5.513,5.278,5.047,5.921,5.553,4.902,4.627,5.445,4.94,4.715,4.825,5.713,5.26,5.19,4.474,8.746,8.828,8.601,8.426,8.83,8.706,8.264,8.351,8.123,7.89,9.007,8.418,6.344,5.925,6.175,6.441,5.837,6.207,6.694,6.86,5.826,6.104,6.385,6.541,5.915,6.633,5.06,4.416,4.575,4.761,3.887,4.364,4.742,4.913,4.267,4.369,4.639,4.444,4.516,4.965,3.897,4.593,9.145,8.79,9.205,8.87,8.536,8.603,8.478,8.481,8.239,8.381,8.227,7.95,8.806,8.975,8.088,8.568,4.614,4.263,4.431,4.161,3.949,3.681,3.914,4.106,3.432,3.642,3.677,3.361,4.065,4.481,3.471,3.588,3.129,3.173,2.881,2.592,3.088,2.959,2.383,2.232,2.753,2.467,2.527,2.333,3.328,2.741,2.902,1.967,11.106,11.238,10.954,11.165,10.574,10.535,10.816,11.454,10.338,10.342,11.41,10.236,11.033,11.308,10.134,10.918,10.684,9.566,9.619,10.501,9.063,9.417,9.843,10.025,9.162,9.469,10.115,9.384,9.688,9.033,9.553,7.591,7.087,7.448,7.706,7.065,7.189,7.804,8.171,7.126,7.374,8.052,7.523,6.89,7.581,7.146,8.029,4.463,4.714,5.217,5.459,4.388,5.559,5.739,5.496,4.946,6.235,5.985,4.595,4.335,5.069,4.435,4.651,5.723,4.786,6.354,4.217,3.705,4.1,4.213,4.07,4.338,4.671,4.569,4.561,4.777,4.699,4.724,5.107,4.113,4.22,4.828,5.185,10.36,10.286,10.5,11.084,9.923,10.152,10.736,11.332,10.345,10.614,10.722,10.786,10.811,10.355,10.871,8.443,7.878,8.255,8.332,7.443,7.693,8.115,8.762,7.642,8.219,8.444,8.034,7.82,8.412,7.561,8.701,6.315,6.18,6.378,6.531,6.627,6.422,7.155,7.358,6.862,5.986,7.154],[14.625,13.925,14.229,14.715,13.873,14.076,null,null,13.924,14.174,14.31,13.968,14.143,14.78,13.792,14.618,9.032,9.117,9.503,9.529,8.723,9.415,9.807,9.834,9.297,9.479,9.595,9.574,9.008,9.886,8.727,9.483,6.371,5.937,6.647,6.4,5.586,5.811,6.332,6.698,5.718,5.882,6.254,5.844,6.04,6.546,5.534,6.862,14.22,13.567,13.814,13.171,13.827,13.66,18.254,13.772,13.441,13.583,15.077,14.931,15.071,15.408,15.097,15.297,15.586,16.056,15.22,15.45,15.76,15.487,14.773,15.638,15.037,16.106,13.026,13.14,13.25,13.435,12.756,13.001,13.791,13.848,13.298,13.411,13.909,13.607,12.868,13.64,13.098,13.857,9.395,9.021,9.103,8.883,9.175,8.64,8.561,8.794,8.388,8.476,8.302,7.971,9.184,8.974,8.232,8.427,13.649,13.543,13.855,14.052,13.393,13.714,14.093,14.365,13.831,14.04,14.763,14.287,13.424,14.056,13.74,14.436,6.023,5.686,5.933,6.225,5.486,6.054,6.29,6.574,5.919,6.166,6.459,6.265,5.515,6.331,6.819,15.604,15.436,15.958,16.137,15.055,15.478,16.01,16.484,15.365,15.566,16.185,15.78,15.511,16.012,15.53,16.208,11.021,10.957,11.104,11.35,10.839,11.248,11.546,11.859,11.29,11.694,11.782,11.691,10.7,11.547,11.451,11.984,11.662,11.675,12.204,12.772,11.402,11.637,12.128,12.65,11.941,11.874,12.421,11.982,11.468,12.121,11.973,12.447,8.555,8.03,8.243,8.437,7.704,7.735,8.362,8.747,7.699,7.674,7.901,7.506,8.131,7.287,7.99,5.254,4.629,4.779,5.046,4.302,4.459,4.729,4.821,4.424,4.372,4.46,4.182,4.93,5.186,4.34,4.762,10.494,10.079,11.201,10.684,9.727,9.728,10.218,10.967,9.831,9.873,10.167,10.084,10.637,10.893,9.633,10.293,12.185,12.298,12.47,12.636,11.926,12.132,12.682,13.518,12.425,12.703,12.862,11.968,12.715,12.502,12.883,12.422,12.765,12.883,12.023,12.84,13.004,13.102,12.461,12.328,12.466,12.191,13.023,12.064,12.881,8.054,7.703,8.005,8.268,7.442,7.598,8.114,8.471,7.66,7.987,8.346,7.725,7.621,8.521,7.767,7.902,10.221,9.448,9.793,10.266,9.089,9.793,9.835,10.135,9.196,9.89,9.834,9.71,9.561,10.106,9.169,9.996,14.618,14.081,14.201,14.653,13.894,14.114,14.605,14.815,14.067,14.426,14.594,14.47,13.716,14.492,14.038,14.754,17.698,17.911,17.931,18.135,17.409,17.598,18.264,18.677,17.75,17.997,18.6,17.948,17.344,18.447,17.751,18.585,3.297,3.182,3.293,3.513,2.525,2.974,3.351,3.483,2.55,2.88,3.139,2.428,2.893,3.671,2.295,2.898,3.845,3.927,3.7,3.525,3.929,3.805,3.363,3.45,3.222,2.989,4.106,3.517,10.122,9.938,10.188,10.454,10.076,10.463,10.707,11.147,10.113,10.391,10.672,10.828,9.928,10.646,8.119,7.633,8.004,8.19,7.577,7.935,8.313,8.654,8.045,8.147,8.417,8.595,7.53,8.246,7.882,8.606,7.214,6.859,7.443,7.408,6.604,6.842,7.167,7.425,6.615,7.07,7.171,6.791,6.875,7.533,6.409,7.512,5.69,5.309,5.598,5.328,5.011,4.743,5.214,5.406,4.767,4.993,5.42,5.256,4.909,5.557,4.98,5.452,2.895,2.779,3.058,3.399,2.735,2.969,3.667,3.941,3.236,3.609,3.749,3.734,2.826,3.39,3.136,3.958,12.967,13.099,12.815,13.026,12.435,12.396,12.677,13.315,12.199,12.203,13.271,12.097,12.894,13.169,11.995,12.779,12.326,10.933,11.022,12.143,10.307,10.686,11.3,11.667,10.619,10.926,11.619,10.89,10.874,10.811,11.434,12.836,12.349,12.71,12.968,12.327,12.451,13.066,13.433,12.388,12.636,13.314,12.785,12.152,12.843,12.408,13.291,2.756,3.007,3.477,3.719,5.188,0.752,0.851,0.682,1.024,1.347,1.097,1.215,1.476,1.181,1.434,1.502,0.835,1.401,1.466,1.722,9.131,9.526,9.639,9.496,9.764,10.097,9.995,9.987,10.203,10.125,10.15,10.533,9.539,9.646,10.254,10.611,15.605,15.531,15.745,16.329,15.185,15.397,15.981,16.577,15.607,15.876,15.967,16.048,16.056,15.617,16.116,11.901,11.449,11.713,11.903,11.014,11.264,11.686,12.333,11.213,11.79,12.015,11.605,11.391,11.87,11.132,12.272,11.741,11.606,11.804,11.957,12.053,11.848,12.581,12.239,12.288,11.412,11.853],[12.727,12.018,12.154,12.487,11.602,11.712,null,null,11.56,11.81,11.946,11.604,12.452,12.552,11.428,12.254,5.96,6.045,6.431,6.457,5.59,6.343,6.641,6.668,6.116,6.298,6.414,6.288,5.936,6.814,5.546,6.302,6.012,5.578,6.288,6.041,5.227,5.452,5.973,6.339,5.359,5.523,5.895,5.485,5.681,6.187,5.175,6.503,11.856,11.09,11.337,10.694,11.35,11.183,15.354,11.295,11.077,11.106,12.011,11.865,12.005,12.342,12.031,12.185,12.52,12.99,12.096,12.269,12.579,12.306,11.707,12.572,11.913,12.925,9.652,9.766,9.876,10.061,9.382,9.627,10.417,10.474,9.924,10.037,10.535,10.233,9.494,10.318,9.724,10.483,10.177,9.803,9.885,9.665,9.957,9.422,9.343,9.576,9.17,9.258,9.084,8.753,9.966,9.756,9.014,9.209,10.275,10.164,10.476,10.673,9.835,10.156,10.684,10.956,10.273,10.493,11.348,10.738,10.045,10.682,10.182,11.021,3.41,2.748,2.995,3.159,2.42,2.988,3.224,3.508,2.853,3.1,3.387,3.193,2.819,3.393,3.747,12.91,12.742,13.264,13.443,12.361,12.784,13.316,13.79,12.671,12.872,13.285,12.746,12.817,13.318,12.518,13.308,7.699,7.635,7.782,8.028,7.517,7.926,8.224,8.537,7.968,8.372,8.46,8.369,7.378,8.225,8.129,8.662,8.596,8.609,9.132,9.706,8.33,8.565,9.056,9.584,8.76,8.693,9.24,8.801,8.402,9.055,8.901,9.266,7.801,7.452,7.566,7.679,7.14,7.171,7.572,7.862,6.932,6.907,7.129,6.621,7.583,6.719,7.105,6.551,7.206,6.108,5.942,5.762,5.694,5.625,5.603,5.383,5.268,5.242,4.964,6.347,6.082,5.236,5.457,9.727,9.289,10.434,9.846,8.937,8.961,9.333,10.082,8.946,8.988,9.282,9.199,9.847,10.126,8.748,9.408,9.375,9.488,9.66,9.826,8.86,9.066,9.616,10.452,9.359,9.637,9.796,9.158,9.905,9.436,10.803,10.318,10.527,10.519,10.079,10.568,10.64,10.738,10.097,9.964,10.102,9.827,10.659,9.7,10.517,7.695,7.344,7.646,7.909,7.083,7.239,7.755,8.003,7.301,7.52,7.878,7.257,7.262,8.162,7.408,7.434,9.862,9.089,9.434,9.907,8.73,9.434,9.476,9.776,8.837,9.531,9.475,9.351,9.202,9.747,8.81,9.637,11.546,10.9,11.02,11.472,10.572,10.792,11.283,11.634,10.745,11.104,11.272,11.148,10.535,11.311,10.716,11.432,14.798,15.011,15.031,15.235,14.509,14.698,15.364,15.777,14.85,15.097,15.7,15.048,14.444,15.547,14.851,15.685,3.802,3.734,3.529,3.497,4.063,3.738,3.35,3.426,3.587,3.116,3.122,2.967,3.934,3.71,3.332,2.881,6.888,6.97,6.743,6.568,6.972,6.848,6.406,6.493,6.265,6.032,7.149,6.56,7.438,7.019,7.269,7.535,7.058,7.428,7.788,8.081,7.047,7.325,7.606,7.762,7.009,7.727,6.154,5.51,5.669,5.855,4.981,5.458,5.836,6.007,5.361,5.463,5.733,5.529,5.61,6.059,4.991,5.687,7.996,7.641,8.186,7.971,7.386,7.584,7.579,7.582,7.275,7.482,7.328,7.051,7.657,8.076,7.069,7.669,5.216,4.746,5.033,4.763,4.321,4.053,4.516,4.708,3.804,4.014,4.279,3.844,4.437,5.083,3.843,4.19,1.515,1.315,1.267,1.391,1.23,1.101,1.182,1.339,0.895,0.853,0.913,0.683,1.47,1.54,1.042,0.766,12.2,12.332,12.048,12.259,11.668,11.629,11.91,12.548,11.432,11.436,12.504,11.33,12.127,12.402,11.228,12.012,11.778,10.385,10.474,11.595,9.759,10.138,10.752,11.119,10.071,10.378,11.071,10.342,10.384,10.127,10.647,9.542,9.027,9.388,9.646,9.005,9.129,9.744,10.111,9.066,9.314,9.992,9.463,8.83,9.521,9.086,9.969,2.297,2.548,3.051,3.293,2.224,3.701,3.881,3.638,3.088,4.377,4.127,2.737,2.477,3.211,2.577,2.793,3.865,2.928,4.496,2.359,5.573,5.968,6.081,5.938,6.206,6.539,6.437,6.429,6.645,6.567,6.592,6.975,5.981,6.088,6.696,7.053,12.424,12.35,12.564,13.148,11.863,12.216,12.8,13.396,12.285,12.554,12.786,12.726,12.875,12.295,12.935,9.537,8.972,9.349,9.426,8.537,8.787,9.209,9.856,8.736,9.313,9.538,9.128,8.914,9.506,8.655,9.795,8.183,8.048,8.246,8.399,8.495,8.29,9.023,9.226,8.73,7.854,9.022],[4.004,3.829,3.477,3.27,3.483,3.337,null,null,3.184,3.113,2.723,2.843,4.042,3.335,3.633,2.832,8.319,8.674,8.374,8.4,9.185,8.972,8.385,8.221,9.141,8.893,8.644,8.962,8.565,7.937,9.225,8.589,10.68,10.972,10.714,10.398,11.057,11.047,10.28,10.2,10.811,10.684,10.367,10.667,11.31,10.406,10.887,10.204,1.483,1.454,1.207,1.71,1.53,1.921,4.798,1.894,1.737,2.002,2.999,2.885,2.993,3.362,3.519,3.255,3.54,3.546,3.474,3.438,3.807,3.675,3.061,3.51,3.613,4.391,8.579,8.785,8.536,8.209,9.444,9.197,8.485,8.334,9.178,9.06,8.866,9.292,8.961,8.367,9.375,9.124,12.736,12.604,12.507,12.218,12.758,12.429,11.89,11.764,12.126,12.056,11.857,11.862,12.767,12.309,12.33,11.792,10.923,11.193,11.328,10.829,11.59,11.436,11.177,11.047,11.664,11.578,11.685,11.825,11.074,10.663,11.87,11.32,9.419,10.194,9.905,9.247,10.447,9.89,9.651,9.168,10.199,9.907,9.62,9.884,10.061,9.227,9.582,1.802,1.883,2.156,2.357,1.953,2.356,2.554,2.942,2.361,2.471,2.729,2.77,1.709,2.21,2.723,3.113,8.021,8.282,8.18,8.164,8.636,8.492,8.154,7.871,8.731,8.568,8.32,8.605,8.188,7.796,8.934,8.118,5.551,5.994,5.802,5.63,6.503,6.241,5.991,5.508,6.895,6.231,6.03,6.887,5.995,5.326,6.623,6.056,7.853,7.943,7.651,7.387,8.265,8.452,7.579,7.319,8.125,7.875,7.961,7.762,8.143,8.288,7.43,13.666,16.41,13.702,13.235,14.335,13.666,13.246,12.883,13.697,13.314,13.199,13.261,13.968,13.201,13.708,12.981,6.35,6.405,6.741,6.054,6.364,6.363,5.938,5.816,6.054,5.803,5.691,6.014,6.774,6.254,5.856,5.606,3.874,4.178,4.159,4.013,4.673,4.552,4.262,4.427,4.616,4.691,5.002,4.117,3.792,5.36,4.066,4.484,4.157,3.983,4.38,4.45,3.946,3.597,4.158,3.819,3.986,3.504,3.756,3.881,3.282,9.737,9.984,9.672,9.349,10.016,9.72,9.246,9.058,9.903,9.815,9.227,9.458,10.159,9.349,10.01,9.108,9.967,10.667,10.083,9.839,10.308,9.963,9.503,9.194,9.676,9.682,9.368,9.397,10.16,9.715,9.863,9.537,4.634,4.975,4.733,4.534,5.396,5.09,4.845,4.89,5.189,5.107,5.151,5.588,4.91,4.367,5.285,5.311,3.944,4.252,4.179,4.385,4.314,4.009,4.514,4.927,4.196,4.443,4.989,4.651,3.755,4.693,4.197,5.003,13.326,13.385,13.15,12.918,14.053,13.446,12.795,12.52,13.798,13.134,12.886,13.487,13.585,13.102,13.701,12.911,16.559,16.655,16.414,16.239,16.685,16.561,16.077,16.206,15.936,15.716,16.834,16.273,5.283,5.895,5.632,5.254,6.245,5.971,5.507,5.281,6.245,6.009,5.806,6.183,6.205,5.446,7.118,7.273,7.046,6.99,7.488,7.333,6.883,6.634,7.529,7.505,7.133,7.588,7.832,6.66,8.122,7.381,12.081,12.317,12.138,11.754,12.368,12.13,11.834,11.829,12.014,11.855,11.695,11.719,12.333,11.859,12.167,11.758,10.277,10.592,10.345,10.075,10.707,10.44,9.83,9.552,10.262,10.472,9.663,9.939,10.639,10.273,10.301,9.732,12.045,12.388,12.095,11.803,12.51,12.263,11.848,11.601,12.503,12.294,12.357,12.274,12.422,11.952,12.732,11.864,6.667,6.879,6.383,6.287,6.275,6.236,6.032,5.97,6.021,5.813,6.075,5.707,6.734,6.43,5.835,5.351,8.157,7.938,7.898,7.974,7.958,7.789,7.529,7.498,7.858,7.513,7.588,7.338,8.274,7.949,7.025,6.496,7.01,7.118,6.674,7.32,7.071,6.626,6.314,7.591,7.13,6.8,7.098,7.005,6.326,7.611,6.84,14.703,14.954,15.457,15.699,14.63,14.657,14.837,14.594,14.038,15.333,15.083,13.691,13.518,14.167,13.857,14.073,14.821,13.878,15.452,13.639,10.654,10.99,11.068,10.91,11.37,11.702,11.28,11.038,11.545,11.467,11.39,11.826,11.161,10.902,11.683,11.904,4.566,5.26,4.875,5.135,5.053,5.19,5.169,5.362,5.434,5.551,5.509,5.874,4.69,5.579,5.66,4.184,3.777,3.708,3.391,4.097,3.733,3.195,3.078,4.141,3.593,3.402,3.861,3.99,3.169,4.06,3.37,11.726,11.613,11.24,11.942,11.804,11.284,12.017,12.22,11.724,11.144,12.016],[6.826,6.126,6.43,6.916,6.074,6.345,null,null,6.195,6.445,6.716,6.738,6.344,6.981,6.211,7.024,7.549,7.735,7.763,7.789,7.806,8.033,7.774,7.729,8.387,8.282,8.033,8.551,7.626,7.443,8.253,8.091,3.861,4.264,3.955,3.699,4.623,4.704,3.903,3.879,4.531,4.568,4.046,4.73,4.602,3.553,4.741,3.883,7.695,7.323,7.57,7.208,7.858,7.733,12.662,8.186,7.075,7.814,9.913,9.767,9.907,10.244,9.959,10.137,10.422,10.676,10.308,10.32,10.689,10.557,9.635,10.474,10.125,11.273,11.686,11.968,11.958,12.175,12.1,12.083,12.479,12.393,12.266,12.428,12.847,12.66,11.649,12.073,12.281,12.795,4.129,3.981,3.899,3.61,4.135,3.62,3.267,3.141,3.528,3.217,3.018,3.312,4.144,3.701,3.78,3.242,13.224,13.185,13.497,13.642,13.559,13.509,13.735,13.86,13.737,13.682,14.405,13.929,13.066,13.4,13.943,14.078,6.426,7.309,7.02,6.82,7.73,7.229,7.027,7.31,7.726,7.438,7.231,7.495,7.064,6.836,7.591,10.012,9.844,10.366,10.545,9.463,9.886,10.418,10.892,9.773,9.974,10.593,10.24,9.919,10.42,9.938,10.616,9.668,9.765,9.827,9.944,9.685,10.056,10.14,10.341,10.136,10.417,10.376,10.499,9.546,10.029,10.297,10.578,8.096,8.257,8.614,8.936,8.741,8.504,8.803,8.814,9.339,9.043,9.336,9.603,8.258,8.388,8.886,9.362,3.791,3.619,3.868,3.604,3.942,4.129,3.897,4.404,4.298,4.218,4.377,4.721,3.507,4.275,4.142,6.557,9.964,6.719,6.266,7.674,6.942,6.277,5.914,6.998,6.632,6.357,6.715,7.121,6.092,7.161,6.218,2.885,3.282,3.279,3.258,3.787,4.102,3.846,3.652,3.872,3.974,4.076,4.183,3.066,3.115,3.923,4.392,7.817,7.995,8.102,8.143,7.979,8.109,8.392,8.8,8.402,8.634,8.945,7.674,8.124,8.666,5.084,4.644,4.966,5.272,4.692,5.062,5.393,5.518,5.144,5.253,5.544,5.562,5.412,5.218,6.137,2.233,2.746,2.354,1.849,2.778,2.482,2.062,1.981,2.665,2.758,2.332,3.007,2.921,1.69,2.772,2.655,1.422,2.147,1.244,1.303,1.788,1.124,0.664,0.658,1.156,0.843,0.529,0.877,1.321,1.179,1.343,0.926,9.719,10.162,9.965,10.414,10.523,10.407,10.382,10.576,10.506,10.56,10.658,10.935,9.883,10.252,10.512,10.818,12.106,12.319,12.339,12.543,11.817,12.006,12.672,13.085,12.158,12.405,13.008,12.356,11.752,12.855,12.159,12.993,7.187,7.471,7.154,6.935,8.211,7.664,7.013,6.738,8.016,7.352,7.104,7.851,7.463,7.007,7.919,7.153,10.113,10.209,9.968,9.793,10.239,10.115,9.631,9.76,9.49,9.27,10.388,9.827,6.871,6.795,7.045,7.311,6.936,7.32,7.564,7.544,7.252,7.53,7.804,7.96,6.788,7.503,5.606,5.532,5.663,5.847,5.997,5.842,5.994,5.804,6.101,6.132,6.244,6.699,6.062,5.546,6.543,6.492,4.077,4.374,4.302,3.948,4.895,4.653,4.028,4.023,4.737,4.406,4.246,4.445,4.358,4.053,4.968,3.952,4.716,5.252,5.005,4.825,5.705,5.595,5.072,5.038,5.786,6.058,5.407,5.819,5.199,4.865,6.045,5.476,8.513,8.971,8.758,8.466,9.405,9.158,8.743,8.496,9.433,9.224,9.364,9.459,8.982,8.555,9.737,9.041,5.168,5.3,5.016,5.227,4.636,4.597,4.878,5.516,4.4,4.404,5.472,4.298,5.095,5.37,4.196,4.98,3.08,1.642,1.731,2.897,1.143,1.395,2.054,2.421,1.38,1.688,2.373,2.123,1.482,1.572,2.291,10.053,10.365,10.731,10.616,10.646,10.769,10.714,11.051,10.796,10.87,10.962,11.123,10.168,10.486,10.816,11.408,11.888,12.139,12.642,12.884,11.815,10.175,10.274,10.324,9.669,10.77,10.52,10.023,9.895,10.348,10.242,10.669,10.258,9.569,10.889,10.446,10.601,10.996,11.109,10.966,11.234,11.567,11.394,11.196,11.659,11.581,11.548,11.984,11.009,11.016,11.724,12.062,11.266,11.291,11.406,11.99,10.965,11.158,11.642,12.238,11.387,11.655,11.727,11.828,11.572,11.532,11.876,5.813,6.325,6.054,6.568,6.635,6.502,6.501,7.105,6.686,6.655,6.708,6.923,6.538,6.122,6.605,6.915,12.63,12.495,12.667,12.846,12.942,12.711,13.444,13.647,13.151,12.301,13.443],[8.618,7.909,8.045,8.378,7.493,7.603,null,null,7.451,7.701,7.837,7.495,8.343,8.443,7.319,8.145,4.3,4.416,4.073,4.099,4.241,4.118,3.612,3.286,3.816,3.629,3.385,3.399,4.525,3.807,3.801,3.153,10.282,10.253,10.316,9.931,10.215,10.205,9.561,9.481,9.969,9.842,9.648,9.825,10.591,10.042,10.045,9.485,7.439,6.667,6.914,6.271,6.927,6.387,9.085,6.463,6.66,6.31,5.525,5.232,5.219,5.439,4.917,4.864,5.415,5.686,4.775,4.948,5.258,4.985,5.727,5.669,4.592,5.503,2.065,2.516,2.337,2.229,3.357,3.095,2.521,2.435,3.094,2.976,2.902,3.208,2.477,2.115,3.291,3.16,13.4,13.083,13.113,12.884,13.237,12.699,12.562,12.542,12.398,12.477,12.303,11.972,13.246,12.975,12.242,12.154,4.839,5.109,5.244,4.745,5.506,5.352,5.093,4.963,5.58,5.494,5.718,5.741,4.99,4.579,5.786,5.356,7.155,7.28,7.207,6.664,7.217,6.916,6.622,6.36,6.721,6.645,6.358,6.225,7.351,6.644,6.048,7.197,6.738,7.406,7.415,6.471,6.682,7.288,7.552,6.438,6.721,7.016,6.477,7.124,7.55,6.249,7.039,1.374,1.426,1.183,1.071,1.647,1.367,1.029,0.746,1.606,1.443,1.216,1.754,1.614,0.816,1.809,1.267,2.547,2.349,2.308,2.881,2.553,2.088,2.232,3.109,1.829,1.802,2.345,1.52,2.916,2.879,2.04,2.371,7.57,7.579,7.368,7.104,7.731,7.81,6.954,6.693,7.331,7.1,7.123,6.924,7.779,7.45,6.592,12.824,14.9,12.763,12.393,13.036,12.645,12.28,12.041,12.52,12.293,12.178,12.084,13.002,12.359,12.51,12.015,7.947,7.649,8.473,7.651,7.608,7.607,7.182,7.413,7.298,7.047,6.935,7.258,8.207,7.986,7.1,6.85,4.47,4.202,4.339,4.668,3.503,3.708,4.19,4.465,3.701,3.978,3.466,4.253,5.0,3.252,6.667,6.217,6.391,6.41,5.978,6.432,6.531,6.609,5.904,5.855,5.993,5.718,6.55,5.507,6.388,9.458,9.701,9.389,9.07,9.733,9.437,8.963,8.779,9.62,9.451,8.944,9.094,9.876,9.07,9.727,8.825,11.72,11.438,11.529,11.642,11.079,11.409,11.001,11.02,10.695,11.18,10.891,10.741,11.551,11.518,10.668,10.881,4.375,3.372,3.936,3.867,2.963,3.175,3.666,4.02,2.938,3.296,3.653,3.301,3.493,3.944,2.76,3.813,8.262,8.324,8.344,8.52,7.67,7.961,8.639,8.952,8.001,8.248,8.763,7.98,7.908,8.97,7.856,8.617,11.416,11.378,11.143,10.912,12.025,11.418,10.767,10.492,11.572,10.908,10.66,10.995,11.578,11.125,11.318,10.419,14.746,14.828,14.601,14.426,14.83,14.706,14.264,14.351,14.123,13.89,15.007,14.418,4.026,4.107,3.992,4.009,3.915,3.641,3.668,3.51,3.426,3.498,3.316,3.338,4.097,3.877,6.28,6.39,6.071,6.02,6.414,6.148,5.913,5.355,6.085,5.935,5.749,5.461,6.793,5.822,6.259,5.251,11.802,12.034,11.859,11.475,12.085,11.847,11.555,11.55,11.731,11.572,11.412,11.436,12.05,11.58,11.803,11.479,9.059,9.06,8.884,8.614,8.956,8.688,8.367,8.334,8.439,8.649,8.13,8.158,9.072,8.934,8.478,8.041,9.355,9.436,9.143,8.851,9.558,9.311,8.843,8.596,9.217,9.008,9.072,8.966,9.558,9.0,9.447,8.579,9.599,9.732,9.236,9.447,9.086,9.047,9.098,9.387,8.832,8.624,9.409,8.518,9.545,9.59,8.646,8.381,10.14,9.921,9.881,9.957,9.69,9.772,9.512,9.481,9.535,9.245,9.571,8.935,10.257,9.398,9.008,1.933,1.419,1.57,1.844,0.834,1.328,1.622,2.128,0.835,1.083,1.87,1.232,1.222,1.912,0.855,1.738,11.185,11.436,11.939,12.181,11.11,11.965,12.145,11.902,11.349,12.641,12.391,11.001,10.741,11.475,10.905,11.121,12.129,11.189,12.76,10.687,4.539,4.875,4.565,4.407,5.243,5.288,4.558,4.187,5.015,4.659,4.539,4.975,5.046,4.372,5.153,5.053,5.002,4.795,5.043,5.601,4.244,4.598,5.253,5.836,4.662,4.922,5.214,4.939,5.453,4.485,5.328,5.428,4.719,5.156,5.003,4.432,4.364,4.786,5.063,4.123,4.89,4.948,4.36,4.809,5.17,4.414,5.002,5.237,5.102,5.131,5.453,5.549,5.175,5.908,6.111,5.615,4.908,5.907],[11.503,10.794,10.93,11.263,10.378,10.488,null,null,10.336,10.586,10.722,10.38,11.228,11.328,10.204,11.03,3.499,3.457,3.483,3.509,2.533,3.159,3.383,3.41,2.858,3.04,3.156,2.81,3.566,3.992,2.29,2.941,7.997,7.716,8.031,7.622,7.388,7.378,7.278,7.198,7.092,7.015,6.879,6.768,7.82,7.767,6.988,7.208,10.366,9.594,9.841,9.198,9.854,9.408,12.534,9.514,9.587,9.331,8.974,8.681,8.668,8.888,8.366,8.313,8.864,9.135,8.224,8.397,8.707,8.434,9.165,9.118,8.041,8.977,4.636,4.75,4.86,5.045,4.366,4.611,5.401,5.458,4.908,5.021,5.519,5.217,4.478,5.312,4.708,5.467,13.055,12.738,12.768,12.539,12.892,12.354,12.217,12.197,12.053,12.132,11.958,11.627,12.901,12.63,11.897,11.809,5.19,5.074,5.386,5.583,4.746,5.067,5.595,5.867,5.184,5.404,6.259,5.649,4.955,5.663,5.093,5.932,3.959,4.001,3.947,3.599,3.889,3.656,3.426,3.295,3.442,3.385,3.162,2.965,4.072,3.579,2.983,10.646,10.187,10.855,10.864,9.92,10.131,10.737,11.001,9.887,10.17,10.465,9.926,10.573,10.999,9.698,10.488,3.554,3.49,3.637,3.883,3.173,3.664,3.935,4.113,3.237,3.728,3.849,3.442,3.233,4.08,3.398,3.795,5.437,4.985,5.327,5.902,4.525,4.76,5.251,6.13,4.774,4.821,5.368,4.815,5.244,5.896,4.951,5.394,7.199,6.85,6.964,7.077,6.79,6.821,6.97,7.26,6.33,6.305,6.527,6.019,7.05,6.369,6.503,9.876,11.572,9.436,9.267,9.708,9.317,8.953,8.915,9.192,8.965,8.85,8.756,9.675,9.407,9.182,8.688,9.125,8.687,9.832,9.244,8.335,8.359,8.731,9.48,8.344,8.386,8.68,8.597,9.245,9.524,8.146,8.806,7.491,7.224,7.361,7.69,6.524,6.73,7.212,7.573,6.723,7.086,6.574,7.274,8.021,6.274,9.579,9.073,9.303,9.295,8.834,9.344,9.416,9.514,8.831,8.74,8.878,8.603,9.435,8.434,9.293,8.895,8.544,8.569,8.632,8.461,8.29,8.428,8.534,8.156,8.143,8.409,7.788,8.719,8.885,8.263,7.965,11.375,11.093,11.184,11.297,10.734,11.064,10.656,10.675,10.35,10.835,10.546,10.396,11.206,11.173,10.323,10.536,7.725,6.836,7.148,7.339,6.427,6.647,7.138,7.494,6.6,6.959,7.127,7.003,6.663,7.416,6.551,7.287,11.711,11.773,11.793,11.969,11.119,11.41,12.088,12.401,11.45,11.697,12.212,11.429,11.357,12.419,11.305,12.066,8.088,8.05,7.815,7.584,8.697,8.09,7.439,7.164,8.244,7.58,7.332,7.667,8.25,7.797,7.99,7.091,11.418,11.5,11.273,11.098,11.502,11.378,10.936,11.023,10.795,10.562,11.679,11.09,5.806,5.678,5.675,5.789,5.169,5.284,5.64,5.746,5.069,5.181,5.208,5.118,5.668,5.849,5.521,4.909,5.055,5.221,4.571,4.75,5.174,5.207,4.57,4.537,4.774,4.434,5.085,5.425,4.482,4.676,11.192,10.837,11.252,10.873,10.583,10.65,10.481,10.477,10.254,10.384,10.224,9.959,10.853,10.978,10.135,10.406,5.871,5.8,5.688,5.418,5.677,5.409,5.171,5.363,5.16,5.37,4.934,4.898,5.793,5.738,5.199,4.845,6.027,6.108,5.815,5.523,6.151,5.983,5.444,5.198,5.816,5.53,5.59,5.396,6.23,5.672,5.965,5.03,11.598,11.73,11.446,11.657,11.066,11.027,11.308,11.946,10.83,10.834,11.902,10.728,11.525,11.8,10.626,11.175,11.176,10.058,10.111,10.993,9.598,9.909,10.335,10.517,9.654,9.961,10.607,9.876,10.223,9.525,10.045,5.397,4.882,5.243,5.501,4.625,4.984,5.413,5.919,4.626,4.874,5.661,4.815,4.685,5.376,4.646,5.321,7.466,7.717,8.22,8.462,7.391,8.622,8.802,8.559,8.009,9.298,9.048,7.658,7.398,8.132,7.498,7.714,8.786,7.849,9.417,7.28,0.714,1.109,1.222,1.079,1.466,1.799,1.578,1.465,1.842,1.764,1.733,2.169,1.191,1.229,1.956,2.247,8.476,8.269,8.517,9.088,7.718,8.072,8.74,9.336,8.14,8.409,8.701,8.581,8.927,8.15,8.84,8.313,7.476,8.083,7.93,7.041,7.291,7.713,8.084,7.14,7.817,7.969,7.381,7.418,8.097,7.105,8.023,3.324,3.189,3.387,3.54,3.636,3.431,4.164,4.367,3.871,2.995,4.163],[11.735,11.001,11.162,11.411,10.609,10.72,null,null,10.545,10.795,10.864,10.522,11.393,11.476,10.436,11.172,5.148,5.155,4.883,4.909,4.818,4.857,4.467,4.298,4.555,4.439,4.24,4.209,5.264,4.82,4.498,4.008,10.76,10.731,10.794,10.409,10.693,10.683,10.039,9.959,10.447,10.32,10.126,10.171,11.069,10.53,10.391,9.963,9.939,9.364,9.397,9.154,9.283,8.802,11.238,8.616,9.379,8.725,7.678,7.385,7.372,7.592,7.07,7.017,7.568,7.636,6.928,6.898,7.208,6.935,7.88,7.822,6.745,7.433,2.094,2.046,1.932,2.049,1.102,1.347,1.98,1.928,1.23,1.242,1.74,1.438,1.885,2.607,0.929,1.688,14.798,14.481,14.511,14.282,14.635,14.097,13.96,13.94,13.796,13.875,13.701,13.37,14.644,14.373,13.64,13.552,1.311,1.322,1.634,1.831,1.707,1.657,1.872,2.144,1.885,1.819,2.542,2.066,1.203,1.796,2.091,2.215,7.362,7.487,7.414,6.879,7.424,7.123,6.829,6.575,6.928,6.852,6.565,6.432,7.558,6.859,6.263,9.35,8.891,9.559,9.568,8.624,8.835,9.441,9.705,8.591,8.874,9.169,8.63,9.277,9.703,8.402,9.192,3.6,3.512,3.317,3.547,3.009,3.405,3.257,3.432,2.923,2.891,2.947,2.752,3.279,3.908,2.922,2.889,5.964,5.593,5.847,6.161,5.33,5.332,5.512,6.174,5.107,5.341,5.625,5.059,6.046,6.418,5.284,5.651,8.823,8.546,8.621,8.464,8.698,8.777,8.241,8.387,8.298,8.273,8.09,7.891,8.746,8.417,7.774,13.302,15.107,12.971,12.739,13.243,12.852,12.488,12.387,12.727,12.5,12.385,12.291,13.21,12.837,12.717,12.223,10.36,9.955,11.05,10.371,9.603,9.594,9.858,10.607,9.471,9.513,9.807,9.724,10.513,10.707,9.273,9.933,7.32,6.988,7.125,7.441,6.783,6.582,6.976,7.075,6.487,6.762,6.25,7.334,7.86,6.126,10.206,9.739,9.93,9.799,9.5,9.971,9.92,9.57,9.443,9.244,9.411,9.039,9.939,9.046,9.349,10.856,10.747,10.738,10.468,10.779,10.483,10.312,10.177,10.644,10.418,10.293,10.061,10.922,10.468,10.751,9.849,13.118,12.836,12.927,13.04,12.477,12.807,12.399,12.418,12.093,12.578,12.289,12.139,12.949,12.916,12.066,12.279,6.528,5.576,6.089,5.803,5.273,5.328,5.602,5.95,5.091,5.226,5.583,5.231,5.798,5.88,4.913,5.743,10.415,10.411,10.431,10.607,9.757,10.048,10.589,10.902,9.951,10.198,10.713,9.93,10.061,11.057,9.806,10.567,11.623,11.585,11.35,11.119,12.232,11.625,10.974,10.699,11.779,11.115,10.867,11.202,11.785,11.332,11.525,10.626,14.953,15.035,14.808,14.633,15.037,14.913,14.471,14.558,14.33,14.097,15.214,14.625,6.718,6.59,6.587,6.701,6.081,6.196,6.445,6.548,5.981,6.093,6.093,6.03,6.58,6.654,7.192,7.169,6.857,6.892,6.921,6.895,6.785,6.295,6.832,6.682,6.557,6.217,7.37,6.775,6.737,6.102,13.2,13.08,13.257,12.873,13.131,12.893,12.715,12.711,12.777,12.618,12.458,12.482,13.096,12.978,12.77,12.64,9.274,9.267,9.091,8.821,9.163,8.895,8.574,8.766,8.646,8.856,8.337,8.365,9.279,9.141,8.685,8.248,9.562,9.643,9.35,9.058,9.765,9.518,9.05,8.803,9.424,9.215,9.279,9.166,9.765,9.207,9.654,8.786,12.746,12.878,12.594,12.805,12.214,12.175,12.456,12.809,11.978,11.982,12.868,11.876,12.673,12.948,11.774,11.84,12.447,11.549,11.602,12.264,11.089,11.4,11.659,11.788,10.933,11.285,11.878,11.147,11.714,10.796,11.316,4.802,4.697,4.204,4.394,3.872,4.157,4.166,4.493,3.753,3.717,4.414,3.5,4.5,4.694,3.893,3.855,11.236,11.487,11.99,12.232,11.161,12.172,12.352,12.109,11.556,12.848,12.598,11.208,10.948,11.682,11.112,11.328,12.336,11.396,12.967,10.894,3.412,3.744,3.391,3.073,4.062,4.107,3.216,2.839,3.674,3.314,3.003,3.272,3.915,3.221,3.81,3.192,6.88,6.622,6.87,7.426,6.071,6.424,7.078,7.314,6.487,6.747,6.88,6.597,7.329,6.31,6.806,8.967,8.142,8.695,8.331,7.707,7.903,8.114,7.972,7.662,8.182,8.013,7.64,8.084,8.498,7.713,7.911,2.093,1.98,1.607,2.309,2.048,1.201,2.366,2.569,1.545,1.511,2.191],[8.079,7.37,7.506,7.839,6.954,7.064,null,null,6.912,7.162,7.298,6.956,7.804,7.904,6.78,7.606,0.817,1.172,0.837,0.863,1.787,1.47,0.847,0.685,1.604,1.355,1.106,1.624,1.063,0.399,1.8,1.164,6.861,6.919,6.895,6.579,6.881,6.871,6.227,6.147,6.635,6.508,6.314,6.491,7.257,6.587,6.711,6.151,7.052,6.28,6.527,5.884,6.54,6.095,9.524,6.201,6.273,6.018,5.885,5.505,5.769,6.01,5.565,5.512,6.063,6.504,5.423,5.766,6.076,5.803,5.919,6.24,5.24,6.551,4.53,4.811,4.802,5.019,4.983,4.915,5.372,5.286,5.109,5.271,5.691,5.503,4.492,4.966,5.113,5.639,10.182,9.865,9.895,9.666,10.019,9.481,9.344,9.324,9.18,9.259,9.085,8.754,10.028,9.757,9.024,8.936,6.107,6.068,6.38,6.525,6.442,6.392,6.618,6.743,6.62,6.565,7.288,6.812,5.949,6.283,6.826,6.961,4.347,4.758,4.469,3.813,4.842,4.179,3.882,3.509,4.409,4.119,3.691,3.918,4.829,3.793,3.363,7.636,7.177,7.845,7.854,6.804,7.121,7.727,7.991,6.877,7.16,7.455,6.916,7.54,7.989,6.688,7.478,2.5,2.532,2.659,2.787,2.443,2.823,2.983,3.185,2.894,3.249,3.219,3.266,2.304,2.873,3.055,3.421,2.128,2.065,2.407,2.982,1.605,1.84,2.331,3.116,2.675,2.571,3.088,2.867,1.934,2.587,2.176,3.114,4.314,4.124,4.112,3.848,4.276,4.355,3.625,3.644,3.876,3.851,3.668,3.469,4.324,3.995,3.158,9.49,12.11,9.429,9.059,10.035,9.366,8.946,8.707,9.397,9.014,8.899,8.961,9.668,9.025,9.408,8.681,5.617,5.212,6.307,5.628,4.86,4.851,5.115,5.864,4.728,4.77,5.064,4.981,5.77,5.964,4.53,5.19,4.178,3.912,4.055,4.384,3.212,3.418,3.906,4.327,3.417,3.84,3.328,3.961,4.708,2.968,6.155,5.67,5.879,5.871,5.431,5.92,5.992,6.09,5.449,5.316,5.454,5.179,6.011,5.052,5.869,6.24,6.325,6.133,5.852,6.357,6.061,5.707,5.561,6.222,5.996,5.688,5.639,6.5,5.852,6.329,5.427,8.502,8.22,8.311,8.424,7.861,8.191,7.783,7.802,7.477,7.962,7.673,7.523,8.333,8.3,7.45,7.663,4.821,4.949,5.067,5.519,5.018,5.238,5.484,5.681,5.191,5.511,5.609,5.594,4.582,5.357,5.162,5.769,8.823,8.896,8.916,9.12,8.318,8.562,9.249,9.61,8.659,8.906,9.509,8.798,8.469,9.569,8.66,9.435,9.026,9.085,8.85,8.618,9.753,9.146,8.495,8.22,9.3,8.636,8.388,8.723,9.285,8.802,9.046,8.147,12.259,12.355,12.114,11.939,12.385,12.261,11.777,11.906,11.636,11.416,12.534,11.973,2.465,1.937,2.187,2.448,1.427,1.797,2.326,2.433,1.416,1.694,1.878,1.788,1.927,2.535,2.825,2.908,2.589,2.538,3.031,2.666,2.431,1.873,2.603,2.453,2.267,1.979,3.41,2.353,2.93,1.769,8.584,8.658,8.641,8.257,8.709,8.471,8.293,8.289,8.355,8.196,8.036,8.06,8.674,8.362,8.348,8.218,5.725,6.04,5.793,5.523,6.155,5.888,5.278,5.0,5.71,5.92,5.111,5.387,6.087,5.721,5.749,5.18,7.083,7.164,6.871,6.579,7.286,7.039,6.571,6.324,6.945,6.736,6.8,6.717,7.286,6.728,7.175,6.307,8.003,8.135,7.851,8.062,7.471,7.432,7.713,8.351,7.235,7.239,8.307,7.133,7.93,8.205,7.031,7.751,7.803,6.933,6.986,7.62,6.473,6.784,7.043,7.144,6.317,6.669,7.234,6.503,7.098,6.18,6.672,3.721,3.473,4.036,4.092,3.653,3.777,4.19,4.557,3.714,3.962,4.438,4.111,3.276,3.967,3.734,4.617,9.146,9.397,9.9,10.142,9.073,9.693,9.873,9.63,9.077,10.369,10.119,8.729,8.469,9.203,8.633,8.849,9.857,8.917,10.488,8.415,3.651,3.987,4.065,3.907,4.367,4.699,4.277,4.079,4.542,4.464,4.431,4.867,4.158,3.899,4.68,4.945,6.38,6.396,6.52,7.104,6.067,6.263,6.756,7.352,6.489,6.758,6.832,6.93,6.818,6.634,6.981,4.889,4.162,4.701,4.616,3.727,3.977,4.399,4.771,3.827,4.503,4.656,4.068,4.104,4.783,3.792,4.71,5.513,5.378,5.55,5.729,5.825,5.594,6.327,6.53,6.034,5.184,6.326],[10.189,9.48,9.616,9.949,9.064,9.174,null,null,9.022,9.272,9.408,9.066,9.914,10.014,8.89,9.716,3.956,4.041,4.427,4.453,3.867,4.339,4.821,4.978,4.516,4.698,4.814,4.793,3.932,4.81,3.946,4.702,4.416,4.133,4.452,4.043,3.782,3.774,3.863,3.872,3.379,3.413,3.553,3.375,4.236,4.189,3.274,3.882,9.318,8.552,8.799,8.156,8.812,8.645,13.148,8.757,8.539,8.568,9.848,9.702,9.842,10.179,9.868,10.068,10.357,10.827,10.144,10.251,10.624,10.488,9.544,10.409,9.961,11.208,8.245,8.359,8.469,8.654,7.975,8.22,9.01,9.067,8.517,8.63,9.128,8.826,8.087,8.859,8.317,9.076,9.532,9.215,9.245,9.016,9.369,8.831,8.694,8.674,8.53,8.609,8.435,8.104,9.378,9.107,8.374,8.286,8.868,8.762,9.074,9.271,8.729,9.05,9.312,9.584,9.167,9.259,9.982,9.506,8.643,9.275,9.076,9.655,0.739,0.392,0.639,0.972,0.815,0.848,1.179,1.463,0.809,1.056,1.383,1.315,0.147,0.992,1.743,10.498,10.33,10.852,11.031,9.949,10.372,10.904,11.378,10.259,10.46,11.079,10.558,10.405,10.906,10.33,11.102,6.24,6.176,6.323,6.569,6.058,6.467,6.765,7.078,6.509,6.913,7.001,6.91,5.919,6.766,6.67,7.203,6.433,6.446,6.992,7.543,6.326,6.561,7.052,7.421,7.16,7.093,7.64,7.201,6.239,6.892,6.897,7.666,5.263,4.804,5.011,5.141,4.469,4.5,5.034,5.324,4.394,4.369,4.591,4.083,4.912,4.048,4.567,6.231,8.156,5.793,5.622,6.292,5.73,5.31,5.27,5.761,5.378,5.263,5.325,6.032,5.762,5.766,5.045,7.189,6.751,7.896,7.308,6.399,6.423,6.795,7.544,6.408,6.45,6.744,6.661,7.309,7.588,6.21,6.87,6.963,7.076,7.248,7.414,6.697,6.903,7.453,8.289,7.196,7.474,7.633,6.746,7.493,7.273,8.265,7.78,7.989,7.981,7.541,8.03,8.102,8.2,7.559,7.426,7.564,7.289,8.121,7.162,7.979,5.569,5.218,5.243,5.306,4.973,4.964,5.152,5.332,4.83,4.849,5.207,4.586,5.152,5.559,4.937,4.763,7.852,7.57,7.661,7.897,7.211,7.541,7.33,7.349,6.827,7.421,7.22,7.07,7.683,7.737,6.8,7.21,9.542,9.3,9.42,9.872,9.113,9.333,9.824,10.034,9.286,9.645,9.813,9.689,8.935,9.711,9.257,9.973,12.592,12.805,12.825,13.029,12.303,12.492,13.158,13.571,12.644,12.891,13.494,12.842,12.238,13.341,12.645,13.479,4.672,4.634,4.399,4.168,5.281,4.674,4.023,3.748,4.828,4.164,3.916,4.251,4.834,4.381,4.574,3.675,8.002,8.084,7.857,7.682,8.086,7.962,7.52,7.607,7.379,7.146,8.263,7.674,4.9,4.716,4.966,5.232,4.854,5.241,5.485,5.918,4.884,5.162,5.443,5.599,4.706,5.424,3.616,2.972,3.131,3.317,2.443,2.92,3.298,3.469,2.823,2.925,3.195,3.341,3.072,3.521,2.66,3.384,7.604,7.249,7.664,7.302,6.995,7.062,6.91,6.906,6.683,6.813,6.653,6.388,7.265,7.407,6.547,6.835,2.545,2.075,2.362,2.092,1.65,1.382,1.845,2.037,1.133,1.343,1.608,1.173,1.766,2.412,1.172,1.519,2.611,2.694,2.401,2.109,2.816,2.569,2.154,1.907,2.625,2.416,2.556,2.642,2.814,2.258,2.929,2.232,9.662,9.794,9.51,9.721,9.13,9.091,9.372,10.01,8.894,8.898,9.966,8.792,9.589,9.864,8.69,9.474,9.107,7.714,7.803,8.924,7.088,7.467,8.081,8.448,7.4,7.707,8.4,7.671,7.713,7.589,8.109,8.055,7.568,7.929,8.187,7.546,7.67,8.285,8.652,7.607,7.855,8.533,8.004,7.371,8.062,7.627,8.51,5.071,5.322,5.825,6.067,4.998,5.221,5.401,5.158,4.605,5.897,5.647,4.257,3.997,4.731,4.163,4.379,5.385,4.445,6.016,3.945,4.467,4.862,4.975,4.832,5.1,5.433,5.331,5.323,5.539,5.461,5.486,5.869,4.875,4.982,5.59,5.947,10.824,10.75,10.964,11.548,10.404,10.616,11.2,11.796,10.826,11.095,11.186,11.267,11.275,10.836,11.335,6.999,6.434,6.811,6.888,5.999,6.249,6.671,7.318,6.198,6.775,7.0,6.59,6.376,6.968,6.117,7.257,7.077,6.942,7.14,7.293,7.389,7.184,7.917,8.12,7.624,6.748,7.916],[13.871,13.162,13.298,13.631,12.746,12.856,null,null,12.704,12.954,13.09,12.748,13.596,13.696,12.572,13.398,7.104,7.189,7.575,7.601,6.795,7.487,7.879,7.906,7.369,7.551,7.667,7.646,7.08,7.958,6.799,7.555,6.994,6.56,7.27,7.023,6.209,6.434,6.955,7.321,6.341,6.505,6.877,6.467,6.663,7.169,6.157,7.485,13.0,12.234,12.481,11.838,12.494,12.327,16.498,12.439,12.221,12.25,13.155,13.009,13.149,13.486,13.175,13.375,13.664,14.134,13.292,13.522,13.832,13.559,12.851,13.716,13.109,14.178,11.098,11.212,11.322,11.507,10.828,11.073,11.863,11.92,11.37,11.483,11.981,11.679,10.94,11.712,11.17,11.929,10.156,9.782,9.864,9.644,9.936,9.401,9.322,9.555,9.149,9.237,9.063,8.732,9.945,9.735,8.993,9.188,11.721,11.615,11.927,12.124,11.406,11.727,12.165,12.437,11.844,12.064,12.835,12.309,11.496,12.128,11.753,12.508,4.425,3.764,4.011,4.303,3.564,4.132,4.368,4.652,3.997,4.244,4.531,4.337,3.833,4.409,4.891,14.054,13.886,14.408,14.587,13.505,13.928,14.46,14.934,13.815,14.016,14.429,13.89,13.961,14.462,13.662,14.452,9.093,9.029,9.176,9.422,8.911,9.32,9.618,9.931,9.362,9.766,9.854,9.763,8.772,9.619,9.523,10.056,9.74,9.753,10.276,10.85,9.474,9.709,10.2,10.728,10.013,9.946,10.493,10.054,9.546,10.199,10.045,10.519,8.945,8.485,8.692,8.823,8.15,8.181,8.716,9.006,8.076,8.051,8.273,7.765,8.593,7.729,8.249,6.416,6.207,5.973,5.807,5.627,5.559,5.49,5.582,5.248,5.133,5.221,4.943,6.212,5.947,5.101,5.523,10.871,10.433,11.578,10.99,10.081,10.105,10.477,11.226,10.09,10.132,10.426,10.343,10.991,11.27,9.892,10.552,10.519,10.632,10.804,10.97,10.004,10.21,10.76,11.596,10.503,10.781,10.94,10.302,11.049,10.58,11.947,11.462,11.671,11.663,11.223,11.712,11.784,11.882,11.241,11.108,11.246,10.971,11.803,10.844,11.661,8.677,8.326,8.628,8.891,8.065,8.221,8.737,9.013,8.283,8.53,8.888,8.267,8.244,9.144,8.39,8.444,10.844,10.071,10.416,10.889,9.712,10.416,10.458,10.758,9.819,10.513,10.457,10.333,10.184,10.729,9.792,10.619,12.69,12.153,12.273,12.725,11.966,12.186,12.677,12.887,12.139,12.498,12.666,12.542,11.788,12.564,12.11,12.826,15.942,16.155,16.175,16.379,15.653,15.842,16.508,16.921,15.994,16.241,16.844,16.192,15.588,16.691,15.995,16.829,4.058,3.807,3.916,4.136,3.239,3.67,3.974,4.106,3.173,3.503,3.762,3.051,3.654,4.348,2.918,3.521,5.652,5.734,5.507,5.332,5.736,5.612,5.17,5.257,5.029,4.796,5.913,5.324,8.582,8.163,8.413,8.679,8.202,8.572,8.932,9.225,8.191,8.469,8.75,8.906,8.153,8.871,7.298,6.654,6.813,6.999,6.125,6.602,6.98,7.151,6.505,6.607,6.877,6.673,6.754,7.203,6.135,6.831,7.975,7.62,8.204,8.169,7.365,7.603,7.928,8.186,7.376,7.831,7.932,7.552,7.636,8.294,7.17,8.273,6.226,5.756,6.043,5.773,5.331,5.063,5.526,5.718,4.814,5.024,5.289,4.854,5.447,6.093,4.853,5.2,2.178,1.716,1.933,2.179,1.191,1.438,1.972,2.248,1.222,1.522,1.582,1.567,1.871,2.327,0.969,1.791,13.344,13.476,13.192,13.403,12.812,12.773,13.054,13.692,12.576,12.58,13.648,12.474,13.271,13.546,12.372,13.156,12.788,11.395,11.484,12.605,10.769,11.148,11.762,12.129,11.081,11.388,12.081,11.352,11.394,11.271,11.791,10.908,10.421,10.782,11.04,10.399,10.523,11.138,11.505,10.46,10.708,11.386,10.857,10.224,10.915,10.48,11.363,2.01,2.261,2.764,3.006,3.008,2.313,2.493,2.25,1.702,2.989,2.739,1.35,1.089,1.823,1.13,1.341,2.477,1.542,3.108,0.912,7.144,7.539,7.652,7.509,7.777,8.11,8.008,8.0,8.216,8.138,8.163,8.546,7.552,7.659,8.267,8.624,13.677,13.603,13.817,14.401,13.257,13.469,14.053,14.649,13.679,13.948,14.039,14.12,14.128,13.689,14.188,10.681,10.116,10.493,10.57,9.681,9.931,10.353,11.0,9.88,10.457,10.682,10.272,10.058,10.65,9.799,10.939,9.754,9.619,9.817,9.97,10.066,9.861,10.594,10.797,10.301,9.425,10.593],[4.209,3.5,3.636,3.785,3.084,3.194,null,null,2.919,3.169,3.238,2.896,3.934,3.85,2.91,3.546,6.01,6.365,6.065,6.091,6.876,6.663,6.076,5.912,6.832,6.584,6.335,6.642,6.256,5.628,6.905,6.269,8.739,9.031,8.773,8.457,9.116,9.106,8.339,8.259,8.87,8.743,8.426,8.726,9.369,8.465,8.946,8.263,2.544,1.759,1.974,1.308,1.964,1.681,6.275,1.793,1.774,1.604,3.526,3.38,3.52,3.857,3.548,3.748,4.035,4.289,3.897,3.931,4.302,4.168,3.224,4.087,3.714,4.886,6.423,6.825,6.643,6.407,7.642,7.395,6.683,6.532,7.376,7.258,7.064,7.49,6.805,6.212,7.573,7.322,11.334,11.053,11.076,10.815,11.207,10.664,10.472,10.346,10.361,10.418,10.223,10.097,11.216,10.906,10.565,10.027,9.121,9.391,9.526,9.027,9.788,9.634,9.375,9.245,9.862,9.776,9.883,10.023,9.272,8.861,10.068,9.518,7.211,7.986,7.697,7.039,8.239,7.682,7.443,6.96,7.991,7.699,7.412,7.676,7.853,7.019,7.374,3.625,3.457,3.979,4.158,3.076,3.499,4.031,4.505,3.386,3.587,4.206,3.853,3.532,4.033,3.551,4.229,5.638,5.899,5.797,5.774,6.253,6.166,5.828,5.548,6.408,6.245,6.018,6.441,5.805,5.406,6.611,5.954,3.146,3.613,3.368,3.151,4.129,3.86,3.557,3.029,4.484,3.795,3.545,4.449,3.621,2.871,4.243,3.571,5.912,6.002,5.71,5.446,6.324,6.44,5.56,5.299,5.956,5.706,5.753,5.554,6.202,6.08,5.222,11.725,14.454,11.761,11.294,12.379,11.71,11.29,10.942,11.741,11.358,11.243,11.305,12.012,11.26,11.752,11.025,4.585,4.464,5.107,4.289,4.423,4.422,3.997,4.051,4.113,3.862,3.75,4.073,5.009,4.62,3.915,3.665,1.378,1.682,1.663,1.56,2.194,2.056,1.809,1.986,2.12,2.195,2.506,1.621,1.31,2.881,2.951,2.85,2.775,2.609,2.615,2.816,2.73,2.27,2.393,2.054,2.221,1.739,2.641,2.116,1.781,7.796,8.043,7.731,7.408,8.075,7.779,7.305,7.117,7.962,7.874,7.286,7.517,8.218,7.408,8.069,7.167,8.584,8.902,8.449,8.456,8.543,8.329,7.869,7.811,7.911,8.048,7.734,7.632,8.526,8.332,8.098,7.772,3.305,3.436,3.551,4.003,3.709,3.758,3.968,4.165,3.711,3.996,4.094,4.121,3.069,3.841,3.698,4.254,5.719,5.932,5.952,6.156,5.43,5.619,6.285,6.698,5.771,6.018,6.621,5.969,5.365,6.468,5.772,6.606,11.37,11.429,11.194,10.962,12.097,11.49,10.839,10.564,11.842,11.178,10.93,11.279,11.629,11.146,11.602,10.703,14.603,14.699,14.458,14.283,14.729,14.605,14.121,14.25,13.98,13.76,14.878,14.317,3.075,3.687,3.424,3.046,3.936,3.662,3.198,2.907,3.936,3.7,3.497,3.874,3.997,3.238,4.91,5.065,4.838,4.782,5.28,5.125,4.675,4.426,5.321,5.297,4.925,5.38,5.624,4.452,5.914,5.173,10.14,10.376,10.197,9.813,10.427,10.189,9.893,9.888,10.073,9.914,9.754,9.778,10.392,9.918,10.226,9.817,8.069,8.384,8.137,7.867,8.499,8.232,7.622,7.344,8.054,8.264,7.455,7.731,8.431,8.065,8.093,7.524,9.837,10.18,9.887,9.595,10.302,10.055,9.64,9.393,10.295,10.086,10.149,10.066,10.214,9.744,10.524,9.656,6.187,6.32,5.824,5.826,5.674,5.635,5.569,5.509,5.42,5.212,5.612,5.106,6.133,5.969,5.234,4.883,6.774,6.555,6.515,6.591,6.324,6.406,6.146,6.115,6.256,5.879,6.205,5.573,6.891,6.184,5.642,4.0,4.521,4.678,4.236,4.998,4.753,4.334,4.612,5.269,4.817,4.582,4.985,4.52,4.106,5.289,5.028,12.495,12.746,13.249,13.491,12.422,12.449,12.629,12.386,11.83,13.125,12.875,11.483,11.31,11.959,11.649,11.865,12.613,11.67,13.244,11.431,8.334,8.67,8.748,8.59,9.05,9.382,8.96,8.689,9.225,9.147,9.041,9.477,8.841,8.582,9.363,9.555,4.855,4.88,4.995,5.579,4.551,4.747,5.231,5.827,4.973,5.242,5.316,5.414,5.183,5.118,5.465,2.243,1.697,1.767,1.311,1.889,1.525,1.115,0.852,1.933,1.232,0.976,1.5,1.91,1.228,1.852,0.874,9.602,9.467,9.438,9.818,9.914,9.482,10.215,10.418,9.922,9.273,10.214],[5.773,5.598,5.246,5.039,5.252,5.106,null,null,4.889,4.975,4.492,4.548,5.811,5.104,5.338,4.748,6.909,7.264,6.964,6.99,7.908,7.562,6.975,6.811,7.462,7.213,6.964,7.229,7.155,6.527,7.492,6.856,12.156,12.39,12.19,11.874,12.433,12.423,11.698,11.618,12.187,12.06,11.785,12.043,12.728,11.882,12.263,11.622,3.216,3.088,2.939,3.155,2.825,2.837,3.775,2.297,3.317,2.76,0.855,0.896,0.758,0.65,1.246,0.946,0.378,0.281,1.165,1.016,0.642,1.076,1.474,0.402,1.304,1.226,6.257,6.345,6.096,5.769,7.004,6.757,6.045,5.894,6.738,6.62,6.426,6.852,6.62,6.058,6.935,6.684,14.441,14.309,14.212,13.923,14.463,14.008,13.595,13.469,13.705,13.761,13.562,13.441,14.472,14.014,13.909,13.371,8.483,8.753,8.888,8.389,9.15,8.996,8.737,8.607,9.224,9.138,9.245,9.385,8.634,8.223,9.43,8.88,10.015,10.428,10.139,9.481,10.681,10.046,9.807,9.324,10.355,10.063,9.776,10.009,10.499,9.461,9.454,2.165,1.706,2.211,2.22,1.439,1.39,2.027,2.242,1.146,1.429,1.706,1.185,2.092,2.355,0.957,1.729,5.926,6.187,5.999,5.887,6.463,6.183,5.845,5.562,6.422,6.259,6.011,6.296,6.093,5.591,6.625,5.809,4.56,4.498,4.25,4.032,5.016,4.745,4.439,3.91,5.071,4.503,4.253,4.92,5.065,4.317,4.951,4.279,9.386,9.419,9.184,8.92,9.571,9.65,8.77,8.509,9.166,8.916,8.963,8.764,9.619,9.29,8.432,15.042,17.662,14.981,14.611,15.587,14.918,14.498,14.259,14.949,14.566,14.451,14.513,15.22,14.577,14.96,14.233,7.929,7.984,8.446,7.633,7.943,7.942,7.517,7.395,7.633,7.382,7.27,7.593,8.353,7.959,7.435,7.185,3.712,3.769,3.517,3.353,4.282,3.937,3.333,2.858,3.522,3.195,3.265,4.45,3.888,3.855,5.771,6.189,5.862,5.688,5.959,6.155,5.651,5.302,5.737,5.398,5.565,5.083,5.461,5.46,4.861,11.274,11.517,11.205,10.886,11.549,11.253,10.779,10.595,11.436,11.291,10.76,10.934,11.692,10.886,11.543,10.641,11.672,12.246,11.788,11.544,11.887,11.668,11.208,10.899,11.255,11.387,11.073,10.976,11.865,11.42,11.442,11.116,2.325,2.666,2.424,2.094,3.087,2.781,2.405,2.45,2.88,2.667,2.711,3.148,2.601,1.927,2.976,2.871,2.704,2.683,2.703,2.879,2.029,2.32,3.008,3.321,2.37,2.617,3.22,2.557,2.35,3.329,2.371,3.194,14.578,14.637,14.402,14.17,15.305,14.698,14.047,13.772,15.05,14.386,14.138,14.487,14.837,14.354,14.81,13.911,17.811,17.907,17.666,17.491,17.937,17.813,17.329,17.458,17.188,16.968,18.086,17.525,5.898,6.261,6.01,5.743,6.069,5.795,5.331,5.04,5.728,5.667,5.228,5.332,6.251,5.54,8.12,8.275,8.048,7.992,8.49,8.218,7.885,7.365,8.155,8.107,7.759,7.636,8.834,7.662,8.587,7.426,13.618,13.85,13.675,13.291,13.901,13.663,13.371,13.366,13.547,13.388,13.228,13.252,13.866,13.396,13.643,13.295,11.277,11.592,11.345,11.075,11.707,11.44,10.83,10.552,11.262,11.472,10.663,10.939,11.639,11.273,11.301,10.732,12.875,12.956,12.663,12.371,13.078,12.831,12.363,12.116,12.737,12.528,12.591,12.508,13.078,12.52,12.966,12.098,8.436,8.648,8.152,8.056,8.044,8.005,7.801,7.739,7.79,7.582,7.844,7.476,8.503,8.199,7.604,7.12,9.862,9.643,9.603,9.679,9.663,9.494,9.234,9.203,9.563,9.218,9.293,8.917,9.979,9.528,8.73,4.307,4.701,4.809,4.365,5.011,4.762,4.317,4.005,5.282,4.821,4.491,4.789,4.702,4.017,5.302,4.531,14.937,15.188,15.691,15.933,14.864,15.485,15.665,15.422,14.869,16.161,15.911,14.521,14.261,14.995,14.425,14.641,15.649,14.709,16.28,14.207,8.921,9.257,9.221,8.953,9.637,9.944,9.1,8.729,9.559,9.201,9.081,9.517,9.428,8.917,9.697,9.595,1.926,2.594,2.235,2.187,2.526,2.581,2.221,2.152,2.652,2.616,2.418,2.783,1.861,2.797,2.45,5.763,5.218,5.287,4.832,5.514,5.15,4.636,4.385,5.114,4.857,4.601,4.632,5.431,4.748,5.438,4.499,9.286,9.173,8.8,9.502,9.364,8.844,9.577,9.78,9.284,8.704,9.576],[7.99,7.29,7.594,8.08,7.238,7.509,null,null,7.359,7.609,7.88,7.777,7.508,8.145,7.375,8.188,7.237,7.423,7.451,7.477,7.494,7.721,7.462,7.417,8.075,7.97,7.721,8.239,7.314,7.131,7.941,7.779,3.073,3.476,3.167,2.911,3.835,3.916,3.209,3.287,3.743,3.78,3.573,4.217,3.814,2.765,3.953,3.41,8.578,8.205,8.452,8.09,8.74,8.615,13.544,9.068,7.957,8.696,10.795,10.649,10.789,11.126,10.841,11.019,11.304,11.558,11.19,11.202,11.571,11.439,10.517,11.356,11.007,12.155,11.374,11.656,11.646,11.863,11.788,11.771,12.216,12.13,11.954,12.116,12.535,12.348,11.337,11.81,11.969,12.483,4.32,4.003,4.033,3.804,4.157,3.619,3.482,3.462,3.318,3.397,3.223,2.892,4.166,3.895,3.162,3.074,12.912,12.873,13.185,13.33,13.247,13.197,13.423,13.548,13.425,13.37,14.093,13.617,12.754,13.088,13.631,13.766,6.026,6.909,6.62,6.42,7.236,6.829,6.627,6.91,7.326,7.038,6.831,7.095,6.664,6.436,7.191,10.894,10.726,11.248,11.427,10.345,10.768,11.3,11.774,10.655,10.856,11.475,11.122,10.801,11.302,10.82,11.498,9.356,9.453,9.515,9.632,9.373,9.744,9.828,10.029,9.824,10.105,10.064,10.187,9.234,9.717,9.985,10.266,8.492,8.505,9.051,9.598,8.429,8.664,9.155,9.476,9.499,9.395,9.912,9.711,8.298,8.947,9.0,9.938,3.479,3.307,3.556,3.292,3.63,3.817,3.585,4.092,3.986,3.906,4.065,4.409,3.195,3.963,3.83,5.769,9.176,5.931,5.478,6.886,6.154,5.489,5.126,6.21,5.844,5.569,5.927,6.333,5.304,6.373,5.43,3.575,3.972,4.437,3.971,4.477,4.792,4.559,4.365,4.562,4.687,4.789,4.896,3.756,4.273,4.613,5.105,8.587,8.735,8.872,8.913,8.641,8.84,9.162,9.565,9.133,9.404,9.682,8.405,8.889,9.328,6.248,5.802,6.13,6.436,5.423,6.22,6.557,6.682,5.875,5.984,6.275,6.293,6.576,5.949,6.983,1.314,1.915,1.593,1.34,1.99,1.788,1.706,1.669,1.971,2.285,2.02,2.695,2.088,1.217,2.078,2.343,1.949,1.704,1.771,1.832,1.345,1.651,1.191,1.187,0.927,1.37,1.058,0.908,1.816,1.708,0.9,1.048,10.6,10.893,10.846,11.296,11.288,11.138,11.263,11.458,11.237,11.291,11.389,11.7,10.648,11.134,11.277,11.549,12.988,13.201,13.221,13.425,12.699,12.888,13.554,13.967,13.04,13.287,13.89,13.238,12.634,13.737,13.041,13.875,6.399,6.683,6.366,6.147,7.423,6.92,6.269,5.994,7.272,6.608,6.36,7.107,6.675,6.219,7.175,6.409,9.325,9.421,9.18,9.005,9.451,9.327,8.843,8.972,8.702,8.482,9.6,9.039,6.559,6.483,6.733,6.999,6.624,7.008,7.252,7.717,6.94,7.218,7.492,7.648,6.476,7.191,5.294,5.22,5.351,5.535,5.685,5.53,5.682,5.492,5.789,5.82,5.932,6.387,5.75,5.234,6.231,6.18,3.372,3.669,3.566,3.182,4.19,3.915,3.262,3.257,3.971,3.64,3.48,3.679,3.653,3.287,4.18,3.186,4.316,4.852,4.605,4.425,5.305,5.195,4.672,4.726,5.386,5.658,5.007,5.419,4.799,4.465,5.645,5.076,7.769,8.227,8.017,7.728,8.667,8.42,8.005,7.758,8.695,8.486,8.626,8.721,8.238,7.811,8.999,8.303,6.332,6.464,6.18,6.391,5.8,5.761,6.042,6.68,5.564,5.568,6.636,5.462,6.259,6.534,5.36,6.144,4.244,2.806,2.895,4.061,2.184,2.559,3.218,3.585,2.542,2.849,3.537,2.813,2.646,2.734,3.455,10.565,10.317,10.88,10.936,10.497,10.621,11.034,11.401,10.558,10.806,11.282,10.955,10.12,10.811,10.578,11.461,11.15,11.401,11.904,12.146,11.077,9.431,9.53,9.58,8.925,10.026,9.776,9.279,9.151,9.604,9.498,9.925,9.514,8.825,10.145,9.702,10.201,10.596,10.709,10.566,10.834,11.167,11.065,10.884,11.273,11.195,11.22,11.603,10.609,10.704,11.324,11.681,12.148,12.173,12.288,12.872,11.846,12.04,12.524,13.12,12.268,12.537,12.609,12.709,12.454,12.413,12.758,6.544,7.056,6.785,7.299,7.292,7.233,7.232,7.87,7.417,7.386,7.439,7.654,7.269,6.853,7.336,7.693,12.318,12.183,12.355,12.534,12.63,12.399,13.132,13.335,12.839,11.989,13.131],[11.457,10.757,11.061,11.547,10.705,10.976,null,null,10.826,11.076,11.347,11.06,10.975,11.612,10.842,11.655,7.187,7.274,7.658,7.684,6.998,7.572,8.052,8.109,7.572,7.754,7.87,7.849,7.165,8.041,7.002,7.758,2.323,1.889,2.693,2.599,1.538,1.859,2.553,2.919,1.857,2.103,2.475,2.065,1.992,2.745,1.673,3.083,11.336,10.963,11.21,10.782,11.438,11.271,15.865,11.383,10.715,11.194,12.892,12.746,12.886,13.223,12.912,13.112,13.401,13.726,13.261,13.295,13.668,13.532,12.588,13.453,13.078,14.252,11.301,11.415,11.525,11.71,11.031,11.276,12.066,12.123,11.573,11.686,12.184,11.882,11.143,11.915,11.373,12.132,6.572,6.198,6.28,6.06,6.352,5.817,5.738,5.935,5.565,5.653,5.479,5.148,6.361,6.151,5.409,5.547,11.924,11.818,12.13,12.327,11.668,11.989,12.368,12.64,12.106,12.315,13.038,12.562,11.699,12.331,12.015,12.711,4.217,3.961,4.208,4.5,3.761,4.329,4.565,4.849,4.194,4.441,4.734,4.54,3.79,4.606,5.094,13.215,13.047,13.569,13.748,12.666,13.089,13.621,14.095,12.976,13.177,13.796,13.408,13.122,13.623,13.141,13.819,9.296,9.232,9.379,9.625,9.114,9.523,9.821,10.134,9.565,9.969,10.057,9.966,8.975,9.822,9.726,10.259,9.631,9.644,10.19,10.741,9.557,9.792,10.283,10.619,10.216,10.149,10.696,10.257,9.437,10.09,10.128,10.722,4.877,4.352,4.565,4.759,4.215,4.345,5.14,5.697,4.75,4.725,4.987,5.053,4.453,4.373,5.249,3.016,4.842,2.576,2.407,2.768,2.33,2.093,2.055,2.13,1.978,1.863,1.694,2.815,2.547,2.141,1.828,7.057,6.504,7.901,7.423,6.152,6.467,7.12,7.817,6.468,6.821,7.115,7.032,7.062,7.733,6.288,7.241,9.813,9.926,10.098,10.264,9.832,10.031,10.509,11.345,10.324,10.602,10.831,9.596,10.343,10.471,9.715,9.262,9.597,9.903,8.875,9.68,10.024,10.149,9.327,9.42,9.558,9.283,10.043,8.983,9.973,4.127,3.776,4.078,4.341,3.515,3.671,4.187,4.608,3.788,4.127,4.512,4.047,3.694,4.594,3.895,4.224,6.207,5.4,5.745,6.318,5.041,5.745,5.908,6.208,5.25,5.963,5.907,5.783,5.513,6.158,5.159,6.069,12.668,12.356,12.476,12.928,12.169,12.389,12.88,13.09,12.342,12.701,12.869,12.745,11.991,12.767,12.313,13.029,15.309,15.522,15.542,15.746,15.02,15.209,15.875,16.288,15.361,15.608,16.211,15.559,14.955,16.058,15.362,16.196,1.342,1.4,1.083,0.864,2.248,1.697,1.169,0.94,2.141,1.566,1.318,2.065,1.6,0.995,2.133,1.367,4.969,5.051,4.824,4.649,5.053,4.929,4.487,4.574,4.346,4.113,5.23,4.641,7.75,7.625,7.875,8.141,7.763,8.15,8.394,8.9,8.082,8.36,8.634,8.79,7.615,8.333,5.682,5.244,5.615,5.801,5.188,5.546,5.924,6.265,5.673,5.775,6.045,6.375,5.093,5.857,5.51,6.293,4.391,4.036,4.451,4.116,3.782,3.849,3.724,3.727,3.485,3.627,3.473,3.196,4.052,4.221,3.334,3.814,2.507,2.671,2.796,2.616,2.591,2.481,2.863,3.141,2.672,2.904,3.198,3.167,2.085,2.656,2.891,3.267,2.727,3.185,2.975,2.778,3.71,3.463,3.057,2.818,3.739,3.537,3.677,3.823,3.196,2.769,4.042,3.355,9.799,9.931,9.647,9.858,9.267,9.228,9.509,10.147,9.031,9.035,10.103,8.929,9.726,10.001,8.827,9.611,8.572,7.179,7.268,8.389,6.553,6.932,7.546,7.913,6.865,7.172,7.865,7.136,7.109,7.057,7.756,11.111,10.624,10.985,11.243,10.602,10.726,11.341,11.708,10.663,10.911,11.589,11.06,10.427,11.118,10.683,11.566,6.137,6.388,6.891,7.133,6.235,4.389,4.488,4.538,3.883,4.984,4.734,4.237,4.109,4.562,4.456,4.883,4.472,3.783,5.103,4.66,7.406,7.801,7.914,7.771,8.039,8.372,8.27,8.262,8.478,8.4,8.425,8.808,7.814,7.921,8.529,8.886,13.88,13.806,14.02,14.604,13.46,13.672,14.256,14.852,13.882,14.151,14.242,14.323,14.331,13.892,14.391,8.993,9.06,9.233,9.514,8.625,8.875,9.297,9.944,8.824,9.401,9.626,9.216,9.002,9.386,8.743,9.883,10.016,9.881,10.079,10.232,10.328,10.123,10.856,11.059,10.563,9.687,10.855],[6.4,5.691,5.827,6.16,5.275,5.385,null,null,5.233,5.483,5.619,5.277,6.125,6.225,5.101,5.927,2.658,3.013,2.713,2.739,3.657,3.311,2.724,2.56,3.48,3.232,2.983,3.285,2.904,2.276,3.548,2.912,7.818,7.907,7.852,7.536,7.869,7.859,7.215,7.135,7.623,7.496,7.302,7.479,8.245,7.544,7.699,7.139,5.267,4.495,4.742,4.099,4.755,4.251,7.862,4.357,4.488,4.174,4.505,4.125,4.389,4.63,4.211,4.413,4.8,5.307,4.555,4.596,5.067,4.833,4.215,4.86,4.372,5.581,4.215,4.666,4.487,4.686,5.308,5.046,4.978,4.892,5.114,5.126,5.359,5.358,4.497,4.572,5.244,5.324,10.895,10.578,10.608,10.379,10.732,10.194,10.057,10.037,9.893,9.972,9.798,9.467,10.741,10.47,9.737,9.649,6.792,7.062,7.199,6.737,7.459,7.307,7.085,6.955,7.535,7.449,7.71,7.696,6.943,6.534,7.741,7.374,5.342,5.755,5.466,4.808,5.899,5.236,4.997,4.514,5.545,5.253,4.966,5.23,5.826,4.788,4.83,5.869,5.427,6.095,6.104,5.022,5.445,5.977,6.329,5.215,5.498,5.793,5.254,5.758,6.239,5.026,5.816,2.631,2.892,2.79,2.792,3.246,3.183,2.988,2.87,3.445,3.38,3.224,3.626,2.798,2.558,3.606,3.417,0.748,0.761,1.307,1.858,0.769,1.004,1.495,1.736,1.839,1.735,2.252,2.189,0.554,1.207,1.34,2.278,5.065,5.081,4.863,4.599,5.233,5.312,4.449,4.188,4.833,4.595,4.625,4.426,5.281,4.952,4.094,10.478,13.098,10.417,10.047,11.023,10.354,9.934,9.695,10.385,10.002,9.887,9.949,10.656,10.013,10.396,9.669,5.637,5.337,6.164,5.341,5.159,5.158,4.87,5.103,4.849,4.598,4.59,4.809,5.895,5.677,4.651,4.401,2.334,2.068,2.273,2.602,1.368,1.574,2.124,2.947,1.867,2.145,1.948,2.117,2.864,1.588,4.476,3.974,4.2,4.192,3.735,4.241,4.313,4.411,3.732,3.637,3.775,3.5,4.332,3.335,4.19,6.953,7.196,6.884,6.565,7.228,6.932,6.458,6.274,7.115,6.953,6.439,6.596,7.371,6.565,7.222,6.32,9.215,8.933,9.024,9.137,8.574,8.904,8.496,8.515,8.19,8.675,8.386,8.236,9.046,9.013,8.163,8.376,3.953,4.081,4.199,4.651,4.181,4.401,4.616,4.813,4.354,4.643,4.741,4.757,3.714,4.489,4.325,4.901,7.306,7.516,7.536,7.74,6.986,7.182,7.869,8.278,7.327,7.574,8.177,7.525,6.952,8.055,7.328,8.162,10.014,10.073,9.838,9.606,10.741,10.134,9.483,9.208,10.383,9.719,9.471,9.806,10.273,9.79,10.129,9.23,13.247,13.343,13.102,12.927,13.373,13.249,12.765,12.894,12.624,12.404,13.522,12.961,1.314,1.599,1.379,1.182,1.407,1.133,0.77,0.589,0.918,0.857,0.418,0.522,1.589,0.979,3.782,3.893,3.574,3.523,4.016,3.654,3.416,2.858,3.591,3.441,3.252,2.969,4.395,3.324,3.92,2.759,9.297,9.529,9.354,8.97,9.58,9.342,9.05,9.045,9.226,9.067,8.907,8.931,9.545,9.075,9.305,8.974,6.713,7.028,6.781,6.511,7.143,6.876,6.266,5.988,6.698,6.908,6.099,6.375,7.075,6.709,6.737,6.168,8.166,8.247,7.954,7.662,8.369,8.122,7.654,7.407,8.028,7.819,7.915,7.832,8.369,7.811,8.29,7.422,7.29,7.423,6.927,7.138,6.777,6.738,6.789,7.144,6.523,6.315,7.1,6.209,7.236,7.281,6.337,6.072,7.831,7.612,7.572,7.648,7.186,7.463,7.203,7.172,7.03,6.936,7.262,6.625,7.811,6.893,6.699,3.037,3.052,3.617,3.603,3.234,3.358,3.701,4.054,3.295,3.543,3.949,3.692,2.855,3.473,3.315,4.198,10.261,10.512,11.015,11.257,10.188,10.776,10.956,10.713,10.16,11.452,11.202,9.812,9.552,10.286,9.716,9.932,10.94,10.0,11.571,9.498,4.977,5.313,5.391,5.233,5.693,6.025,5.603,5.405,5.868,5.79,5.757,6.193,5.484,5.225,6.006,6.271,5.512,5.528,5.652,6.236,5.199,5.395,5.888,6.484,5.621,5.89,5.964,6.062,5.848,5.766,6.113,3.21,2.377,2.984,2.831,1.942,2.192,2.614,2.927,1.983,2.718,2.812,2.224,2.319,2.998,1.948,2.866,6.776,6.641,6.813,6.992,7.088,6.857,7.59,7.793,7.297,6.447,7.589],[1.697,0.823,1.127,1.477,0.431,0.702,null,null,0.552,0.802,1.073,1.33,1.215,1.542,0.568,1.381,7.44,7.779,7.495,7.521,8.142,8.077,7.506,7.358,8.262,8.014,7.765,8.253,7.67,7.072,8.458,7.823,9.104,9.396,9.138,8.822,9.481,9.471,8.704,8.624,9.235,9.108,8.791,9.091,9.734,8.83,9.311,8.628,2.585,2.388,2.546,2.571,2.871,2.798,7.598,3.235,2.105,2.879,4.849,4.703,4.843,5.18,5.011,5.073,5.358,5.612,5.292,5.256,5.625,5.493,4.684,5.41,5.203,6.209,8.336,8.738,8.556,8.32,9.555,9.308,8.596,8.445,9.289,9.171,8.977,9.403,8.718,8.125,9.486,9.235,9.588,9.456,9.359,9.07,9.61,9.34,8.742,8.616,9.248,8.937,8.738,9.032,9.619,9.161,9.5,8.962,11.034,11.304,11.439,10.94,11.701,11.547,11.288,11.158,11.775,11.689,11.796,11.936,11.185,10.774,11.981,11.431,8.327,9.102,8.813,8.155,9.355,8.798,8.559,8.076,9.107,8.815,8.528,8.792,8.969,8.135,8.49,4.948,4.78,5.302,5.481,4.399,4.822,5.354,5.828,4.709,4.91,5.529,5.176,4.855,5.356,4.874,5.494,7.312,7.573,7.471,7.455,7.927,7.864,7.558,7.302,8.126,7.999,7.772,8.307,7.479,7.087,8.287,7.823,4.865,5.285,5.116,5.043,5.794,5.532,5.305,4.921,6.209,5.545,5.443,6.201,5.286,4.689,5.914,5.469,6.277,6.367,6.075,5.811,6.689,6.876,5.982,5.721,6.47,6.22,6.584,6.469,6.567,6.661,6.091,11.884,14.834,12.046,11.593,12.759,12.09,11.604,11.241,12.121,11.738,11.623,11.685,12.392,11.419,12.132,11.405,3.68,4.238,3.812,3.71,4.834,4.971,4.026,3.487,4.407,4.312,3.779,4.205,4.493,3.568,4.464,4.095,3.58,3.855,3.865,3.856,4.086,3.969,4.105,4.479,4.262,4.397,4.708,3.534,3.803,4.773,1.302,1.908,1.396,1.222,2.093,1.874,1.185,0.996,1.871,1.532,1.571,1.669,0.995,1.924,1.603,7.807,8.073,7.761,7.419,8.105,7.809,7.335,7.128,7.992,8.085,7.382,7.882,8.248,7.416,8.099,7.487,6.819,7.921,7.083,6.691,7.562,6.963,6.503,6.046,6.993,6.682,6.368,6.714,7.041,6.567,7.18,6.765,5.185,5.704,5.431,5.82,6.202,5.949,5.848,6.042,6.048,6.093,6.137,6.567,5.503,5.653,6.144,6.297,6.984,7.197,7.217,7.421,6.695,6.884,7.55,7.963,7.036,7.283,7.886,7.234,6.63,7.733,7.037,7.871,11.75,11.809,11.574,11.342,12.477,11.87,11.219,10.944,12.222,11.558,11.31,11.904,12.009,11.526,12.125,11.328,14.983,15.079,14.838,14.663,15.109,14.985,14.501,14.63,14.36,14.14,15.258,14.697,4.403,4.902,4.752,4.545,5.359,5.132,4.798,4.572,5.359,5.238,5.097,5.474,5.212,4.737,5.953,6.108,5.954,5.883,6.396,6.241,5.791,5.542,6.437,6.413,6.041,6.496,6.667,5.495,7.03,6.289,9.797,10.094,10.022,9.644,10.457,10.219,9.724,9.719,10.103,9.944,9.784,9.808,10.078,9.749,10.295,9.648,8.694,9.082,8.835,8.655,9.536,9.275,8.665,8.387,9.097,9.307,8.498,8.774,9.056,8.697,9.136,8.567,10.88,11.246,11.003,10.711,11.418,11.171,10.756,10.509,11.411,11.202,11.265,11.182,11.257,10.823,11.64,10.772,3.244,3.44,2.944,2.864,2.794,2.755,2.607,2.547,2.54,2.332,2.65,2.226,3.253,3.007,2.354,1.921,5.009,4.79,4.75,4.826,4.958,4.641,4.381,4.35,4.71,4.513,4.44,4.658,5.126,4.902,3.877,5.913,6.434,6.591,6.438,6.896,6.666,6.536,6.814,7.167,6.73,6.784,6.983,6.418,6.308,7.187,7.23,13.611,13.862,14.365,14.607,13.538,13.492,13.672,13.429,12.873,14.168,13.918,12.526,12.353,13.002,12.765,12.981,13.656,12.713,14.287,12.547,9.945,10.281,10.359,10.201,10.661,10.993,10.571,10.363,10.836,10.758,10.715,11.151,10.452,10.193,10.974,11.229,6.384,6.56,6.589,6.953,6.339,6.476,6.825,7.18,6.761,6.973,6.996,7.202,6.508,6.906,7.145,2.315,2.827,2.556,2.675,3.411,3.047,2.608,2.784,3.455,2.956,2.815,3.224,3.04,2.255,3.374,2.775,11.515,11.38,11.351,11.731,11.827,11.395,12.128,12.331,11.835,11.186,12.127],[5.073,4.364,4.5,4.833,3.948,4.058,null,null,3.906,4.156,4.292,3.95,4.774,4.898,3.774,4.6,6.052,6.373,6.107,6.133,6.444,6.671,6.118,6.039,6.874,6.626,6.377,6.895,6.264,5.753,6.891,6.435,5.996,6.288,6.03,5.714,6.373,6.363,5.596,5.516,6.127,6.0,5.683,5.983,6.626,5.722,6.203,5.52,4.226,3.853,4.1,3.738,4.388,4.263,9.192,4.716,3.605,4.344,6.443,6.297,6.437,6.774,6.489,6.667,6.952,7.206,6.838,6.85,7.219,7.087,6.165,7.004,6.655,7.803,7.537,7.988,7.809,8.008,8.782,8.52,8.3,8.214,8.566,8.448,8.681,8.68,7.949,7.894,8.718,8.646,8.995,8.678,8.708,8.479,8.832,8.294,8.157,8.047,7.993,8.072,7.898,7.567,8.841,8.57,7.837,7.715,10.266,10.536,10.673,10.211,10.933,10.781,10.559,10.429,11.009,10.923,11.184,11.17,10.417,10.008,11.215,10.848,6.566,7.341,7.052,6.394,7.594,7.037,6.798,6.315,7.346,7.054,6.767,7.031,7.208,6.374,6.729,6.542,6.374,6.896,7.075,5.993,6.416,6.948,7.422,6.303,6.504,7.123,6.77,6.449,6.95,6.468,7.146,6.105,6.366,6.264,6.248,6.72,6.657,6.444,6.192,6.919,6.854,6.678,7.1,6.272,5.88,7.08,6.739,3.917,4.078,4.502,4.983,4.587,4.325,4.691,4.861,5.16,4.931,5.383,5.424,4.079,4.363,4.707,5.409,3.169,3.259,2.967,2.703,3.581,3.768,2.895,2.635,3.441,3.191,3.37,3.177,3.459,3.632,2.828,8.982,11.726,9.018,8.551,9.651,8.982,8.562,8.199,9.013,8.63,8.515,8.577,9.284,8.517,9.024,8.297,2.273,1.931,2.82,1.977,1.68,1.679,1.464,1.756,1.37,1.119,1.111,1.33,2.489,2.333,1.172,0.922,3.903,4.081,4.188,4.229,4.026,4.195,4.478,5.058,4.488,4.72,5.031,3.76,4.252,4.713,2.763,2.263,2.645,2.865,1.861,2.681,2.986,3.084,2.313,2.31,2.448,2.173,3.005,1.939,2.863,5.053,5.3,4.988,4.665,5.332,5.036,4.562,4.374,5.219,5.131,4.543,4.774,5.475,4.665,5.326,4.424,6.324,6.59,6.162,6.196,6.231,6.042,5.582,5.551,5.599,5.761,5.447,5.32,6.228,6.072,5.786,5.46,6.248,6.508,6.494,6.944,6.781,6.786,6.911,7.106,6.783,6.939,7.037,7.193,6.141,6.782,6.77,7.197,8.636,8.849,8.869,9.073,8.347,8.536,9.202,9.615,8.688,8.935,9.538,8.886,8.282,9.385,8.689,9.523,8.642,8.701,8.466,8.234,9.369,8.762,8.111,7.836,9.114,8.45,8.202,8.875,8.901,8.418,9.017,8.251,11.875,11.971,11.73,11.555,12.001,11.877,11.393,11.522,11.252,11.032,12.15,11.589,3.196,3.695,3.545,3.338,4.152,3.925,3.591,3.365,4.152,4.031,3.89,4.267,4.005,3.53,3.984,4.17,4.301,4.226,4.635,4.48,4.397,4.148,4.739,4.77,4.647,5.102,4.729,3.89,5.269,4.895,7.397,7.633,7.454,7.07,7.684,7.446,7.15,7.145,7.33,7.171,7.011,7.035,7.649,7.175,7.483,7.074,5.665,6.053,5.806,5.618,6.25,5.983,5.373,5.095,6.0,6.272,5.619,6.031,6.027,5.668,6.257,5.688,8.734,9.196,8.97,8.678,9.617,9.37,8.955,8.708,9.645,9.436,9.504,9.421,9.207,8.775,9.879,9.011,3.995,4.128,3.632,3.843,3.482,3.443,3.494,4.132,3.228,3.02,4.088,2.914,3.941,3.986,3.042,3.251,4.514,4.295,4.255,4.331,4.037,4.146,3.886,3.855,3.691,3.592,3.945,3.261,4.631,3.554,3.382,6.139,6.253,6.817,6.702,6.534,6.657,6.8,7.137,6.617,6.865,7.048,7.014,6.056,6.572,6.637,7.494,11.85,12.101,12.604,12.846,11.777,10.575,10.674,10.801,10.03,11.17,10.92,10.384,10.256,10.709,10.603,11.03,10.658,9.93,11.289,10.746,8.738,9.074,9.152,8.994,9.454,9.786,9.364,9.156,9.629,9.551,9.508,9.944,9.245,8.986,9.767,10.022,7.796,7.821,7.936,8.52,7.494,7.688,8.172,8.768,7.916,8.185,8.257,8.357,8.102,8.061,8.406,1.883,2.394,2.123,2.792,2.456,2.323,2.636,3.363,2.507,2.74,2.932,2.899,2.607,2.276,2.426,3.186,10.405,10.27,10.433,10.621,10.717,10.477,11.21,11.413,10.917,10.076,11.209],[6.483,5.774,5.91,6.243,5.358,5.468,null,null,5.316,5.566,5.702,5.36,6.208,6.308,5.184,6.01,2.468,2.823,2.523,2.549,3.467,3.121,2.534,2.37,3.29,3.042,2.793,3.224,2.714,2.086,3.482,2.846,6.988,7.079,7.022,6.706,7.041,7.031,6.387,6.307,6.795,6.668,6.474,6.651,7.417,6.714,6.871,6.311,5.491,4.719,4.966,4.323,4.979,4.539,8.239,4.645,4.712,4.462,4.896,4.75,4.89,5.227,4.916,5.116,5.405,5.875,5.258,5.299,5.672,5.536,4.592,5.457,5.075,6.256,4.402,4.853,4.674,4.891,5.406,5.144,5.244,5.158,5.212,5.313,5.563,5.545,4.595,4.838,5.342,5.511,10.065,9.748,9.778,9.549,9.902,9.364,9.227,9.207,9.063,9.142,8.968,8.637,9.911,9.64,8.907,8.819,6.89,7.16,7.297,6.835,7.557,7.405,7.183,7.053,7.633,7.547,7.808,7.794,7.041,6.632,7.839,7.472,4.521,4.934,4.645,3.987,5.187,4.682,4.462,3.979,5.01,4.718,4.431,4.695,5.005,3.967,4.393,6.157,5.804,6.472,6.481,5.399,5.822,6.354,6.706,5.592,5.875,6.17,5.631,6.064,6.565,5.403,6.193,2.729,2.99,2.888,2.89,3.344,3.281,3.086,3.057,3.543,3.478,3.322,3.724,2.896,2.745,3.704,3.524,1.487,1.5,2.046,2.597,1.44,1.675,2.166,2.475,2.51,2.406,2.923,2.739,1.293,1.946,2.011,2.949,4.235,4.251,4.033,3.769,4.403,4.482,3.619,3.358,4.003,3.765,3.795,3.596,4.451,4.122,3.264,9.65,12.27,9.589,9.219,10.195,9.526,9.106,8.867,9.557,9.174,9.059,9.121,9.828,9.185,9.568,8.841,5.331,4.926,6.021,5.342,4.574,4.565,4.829,5.186,4.442,4.484,4.673,4.695,5.484,5.678,4.244,4.484,2.622,2.445,2.65,2.979,1.745,1.951,2.501,3.337,2.244,2.522,2.687,2.405,3.152,2.327,4.559,4.074,4.283,4.275,3.835,4.324,4.396,4.494,3.853,3.72,3.858,3.583,4.415,3.456,4.273,6.123,6.366,6.054,5.735,6.398,6.102,5.628,5.444,6.285,6.123,5.609,5.766,6.541,5.735,6.392,5.49,8.385,8.103,8.194,8.307,7.744,8.074,7.666,7.685,7.36,7.845,7.556,7.406,8.216,8.183,7.333,7.546,4.656,4.784,4.902,5.354,4.883,5.103,5.319,5.516,5.056,5.346,5.444,5.459,4.417,5.192,5.027,5.604,7.683,7.896,7.916,8.12,7.394,7.583,8.249,8.662,7.735,7.982,8.585,7.933,7.329,8.432,7.736,8.57,9.186,9.245,9.01,8.778,9.913,9.306,8.655,8.38,9.598,8.934,8.686,9.021,9.445,8.962,9.344,8.445,12.419,12.515,12.274,12.099,12.545,12.421,11.937,12.066,11.796,11.576,12.694,12.133,0.799,0.767,0.516,0.782,0.575,0.301,0.859,0.966,0.575,0.21,0.491,0.647,0.757,0.974,2.952,3.063,2.744,2.693,3.186,2.826,2.586,2.028,2.763,2.613,2.422,2.142,3.565,2.494,3.093,1.932,8.467,8.699,8.524,8.14,8.75,8.512,8.22,8.215,8.396,8.237,8.077,8.101,8.715,8.245,8.475,8.144,5.885,6.2,5.953,5.683,6.315,6.048,5.438,5.16,5.87,6.08,5.271,5.547,6.247,5.881,5.909,5.34,7.381,7.462,7.169,6.877,7.584,7.337,6.869,6.622,7.243,7.034,7.097,7.014,7.584,7.026,7.472,6.604,7.373,7.506,7.01,7.221,6.86,6.821,6.872,7.227,6.606,6.398,7.183,6.292,7.319,7.364,6.42,6.155,7.517,6.816,6.869,7.334,6.356,6.667,6.889,6.858,6.2,6.552,6.948,6.217,6.981,6.063,6.386,3.593,3.345,3.908,3.964,3.525,3.649,4.062,4.429,3.586,3.834,4.31,3.983,3.148,3.839,3.606,4.489,9.443,9.694,10.197,10.439,9.37,9.991,10.171,9.928,9.375,10.667,10.417,9.027,8.767,9.501,8.931,9.147,10.155,9.215,10.786,8.713,4.916,5.252,5.33,5.172,5.632,5.964,5.542,5.344,5.807,5.729,5.696,6.132,5.423,5.164,5.945,6.21,6.215,6.231,6.355,6.939,5.902,6.098,6.591,7.187,6.324,6.593,6.667,6.765,6.551,6.469,6.816,3.293,2.601,3.105,3.055,2.166,2.416,2.838,3.215,2.27,2.942,3.1,2.512,2.543,3.222,2.232,3.154,6.767,6.632,6.804,6.983,7.079,6.848,7.581,7.784,7.288,6.438,7.58],[9.287,8.587,8.891,9.377,8.535,8.806,null,null,8.656,8.906,9.177,8.89,8.805,9.442,8.672,9.485,5.105,5.192,5.576,5.602,5.043,5.49,5.97,6.141,5.911,6.086,6.206,6.358,5.083,5.959,5.524,6.26,1.319,1.29,1.353,0.968,1.649,1.642,0.598,0.518,1.412,1.295,0.685,1.369,1.628,1.089,1.589,0.522,9.166,8.793,9.04,8.637,9.293,9.126,13.72,9.238,8.545,9.049,10.8,10.654,10.794,11.131,10.82,11.02,11.309,11.634,11.169,11.203,11.576,11.44,10.496,11.361,10.986,12.16,10.256,10.408,10.518,10.703,10.024,10.269,11.059,11.012,10.566,10.679,11.177,10.875,10.107,10.692,10.366,11.125,6.355,6.038,6.068,5.839,6.192,5.654,5.517,5.497,5.353,5.432,5.258,4.927,6.201,5.93,5.197,5.109,11.064,11.025,11.337,11.534,11.399,11.349,11.575,11.784,11.577,11.522,12.245,11.769,10.906,11.324,11.783,11.918,3.138,4.021,3.732,3.532,4.442,3.941,3.739,4.022,4.438,4.15,3.943,4.207,3.776,3.548,4.303,11.07,10.902,11.424,11.603,10.521,10.944,11.476,11.95,10.831,11.032,11.651,11.298,10.977,11.478,10.996,11.674,8.073,8.009,8.156,8.402,7.891,8.3,8.598,8.911,8.342,8.746,8.834,8.743,7.752,8.599,8.503,9.036,7.547,7.56,8.106,8.657,7.475,7.71,8.201,8.535,8.545,8.441,8.958,8.757,7.353,8.006,8.046,8.984,2.558,2.033,2.395,2.589,2.045,2.175,2.97,3.527,2.595,2.57,2.832,2.898,2.134,2.218,3.094,4.191,7.263,4.353,3.9,5.188,4.519,3.911,3.548,4.55,4.167,3.991,4.114,4.755,3.726,4.561,3.834,4.887,4.334,5.731,5.253,3.982,4.297,4.95,5.647,4.298,4.651,4.945,4.862,4.892,5.563,4.118,5.071,7.721,7.834,8.006,8.172,7.74,7.939,8.417,9.253,8.232,8.51,8.747,7.504,8.251,8.387,7.545,7.092,7.427,7.733,6.705,7.51,7.854,7.979,7.157,7.25,7.388,7.113,7.873,6.813,7.803,2.195,1.844,1.869,1.932,1.761,1.59,1.705,2.126,1.456,1.331,2.03,1.288,2.019,2.169,1.563,1.78,4.675,4.393,4.484,4.597,4.034,4.364,3.956,3.975,3.65,4.135,3.846,3.696,4.506,4.473,3.623,3.836,10.576,10.704,10.822,11.274,10.805,11.025,11.239,11.436,10.978,11.266,11.364,11.381,10.337,11.112,10.949,11.524,13.164,13.377,13.397,13.601,12.875,13.064,13.73,14.143,13.216,13.463,14.066,13.414,12.81,13.913,13.217,14.051,4.185,4.315,4.08,3.848,4.983,4.376,3.725,3.45,4.728,4.064,3.816,4.563,4.461,4.005,4.631,3.865,7.412,7.508,7.267,7.092,7.538,7.414,6.93,7.059,6.789,6.569,7.687,7.126,5.658,5.541,5.791,6.057,5.679,6.066,6.31,6.816,5.998,6.276,6.55,6.706,5.531,6.249,3.535,3.099,3.47,3.656,3.056,3.401,3.779,4.12,3.589,3.691,3.961,4.293,2.947,3.712,3.428,4.209,4.5,4.146,4.557,4.173,3.908,3.959,3.781,3.777,3.554,3.684,3.524,3.259,4.162,4.278,3.464,3.706,1.428,1.964,1.717,1.537,2.417,2.307,1.784,1.854,2.498,2.77,2.119,2.531,1.911,1.577,2.757,2.188,5.225,5.683,5.47,5.178,6.117,5.87,5.455,5.208,6.145,5.936,6.076,6.171,5.694,5.267,6.449,5.753,7.629,7.761,7.477,7.688,7.097,7.058,7.339,7.977,6.861,6.865,7.933,6.759,7.556,7.831,6.657,7.441,6.09,4.697,4.786,5.907,4.071,4.45,5.064,5.431,4.383,4.69,5.383,4.654,4.627,4.575,5.298,9.611,9.363,9.762,9.982,9.379,9.503,10.08,10.447,9.44,9.688,10.328,9.837,9.166,9.857,9.46,10.343,8.6,8.851,9.354,9.596,8.527,6.887,6.986,7.036,6.381,7.482,7.232,6.735,6.607,7.06,6.954,7.381,6.97,6.281,7.601,7.158,7.313,7.708,7.821,7.678,7.946,8.279,8.177,8.169,8.385,8.307,8.332,8.715,7.721,7.828,8.436,8.793,12.127,12.151,12.267,12.851,11.822,12.018,12.503,13.099,12.244,12.513,12.587,12.685,12.455,12.389,12.736,6.823,6.915,7.063,7.369,6.48,6.73,7.152,7.799,6.679,7.256,7.481,7.071,6.857,7.216,6.598,7.738,9.923,9.788,9.986,10.139,10.235,10.03,10.763,10.966,10.47,9.594,10.762],[13.955,13.246,13.382,13.715,12.83,12.94,null,null,12.788,13.038,13.174,12.832,13.68,13.78,12.656,13.482,6.488,6.446,6.473,6.499,5.519,6.148,6.374,6.401,5.849,6.031,6.147,5.813,6.555,7.007,5.279,5.944,8.698,8.415,8.734,8.325,8.064,8.056,8.145,8.263,7.661,7.695,7.944,7.657,8.518,8.471,7.556,8.273,13.084,12.318,12.565,11.922,12.578,12.411,15.633,12.523,12.305,12.334,12.073,11.78,11.767,11.987,11.465,11.412,11.963,12.234,11.323,11.496,11.806,11.533,12.264,12.217,11.14,12.076,8.361,8.475,8.585,8.77,8.091,8.336,9.126,9.183,8.633,8.746,9.244,8.942,8.203,9.037,8.433,9.192,13.593,13.219,13.301,13.081,13.373,12.838,12.759,12.956,12.586,12.674,12.5,12.169,13.382,13.172,12.43,12.568,8.984,8.87,9.182,9.379,8.541,8.862,9.39,9.662,8.979,9.199,10.054,9.444,8.751,9.391,8.888,9.727,5.024,4.595,4.732,4.772,4.339,4.441,4.491,4.609,4.036,4.17,4.259,4.03,4.666,4.893,4.619,13.745,13.286,13.954,13.963,13.019,13.23,13.836,14.1,12.986,13.269,13.564,13.025,13.672,14.098,12.797,13.587,6.653,6.589,6.736,6.982,6.371,6.862,7.178,7.491,6.775,7.266,7.387,7.107,6.332,7.179,6.936,7.46,8.536,8.084,8.426,9.001,7.624,7.859,8.35,9.229,7.873,7.92,8.467,7.914,8.343,8.995,8.05,8.493,9.029,8.68,8.794,8.907,8.619,8.65,8.8,9.09,8.16,8.135,8.357,7.849,8.88,8.198,8.333,9.967,10.622,9.524,9.358,9.178,9.11,9.041,9.019,8.799,8.684,8.658,8.38,9.763,9.498,8.652,8.907,10.955,10.517,11.662,11.074,10.165,10.189,10.561,11.31,10.174,10.216,10.51,10.427,11.075,11.354,9.976,10.636,10.59,10.323,10.46,10.789,9.623,9.829,10.311,10.672,9.822,10.185,9.673,10.373,11.12,9.373,12.031,11.546,11.755,11.747,11.307,11.796,11.868,11.966,11.325,11.192,11.33,11.055,11.887,10.928,11.745,9.867,9.516,9.634,9.697,9.255,9.355,9.543,9.723,9.221,9.24,9.598,8.977,9.434,9.95,9.328,9.154,12.134,11.852,11.943,12.179,11.493,11.823,11.648,11.74,11.109,11.703,11.611,11.461,11.965,12.019,11.082,11.601,10.824,9.935,10.247,10.438,9.526,9.746,10.237,10.593,9.699,10.058,10.226,10.102,9.762,10.515,9.67,10.386,14.81,14.872,14.892,15.068,14.218,14.509,15.187,15.5,14.549,14.796,15.311,14.528,14.456,15.518,14.404,15.165,7.218,7.15,6.945,7.021,7.479,7.154,6.874,6.908,7.003,6.532,6.646,6.383,7.35,7.234,6.748,6.405,10.304,10.386,10.159,9.984,10.388,10.264,9.822,9.909,9.681,9.448,10.565,9.976,8.666,8.247,8.497,8.763,8.159,8.383,8.739,8.845,8.148,8.28,8.307,8.217,8.237,8.948,7.382,6.738,6.897,7.083,6.209,6.686,7.064,7.235,6.589,6.691,6.961,6.766,6.838,7.287,6.219,6.915,11.412,11.057,11.53,11.195,10.802,10.928,10.803,10.806,10.564,10.706,10.552,10.275,11.073,11.3,10.413,10.893,6.936,6.585,6.753,6.483,6.271,6.003,6.236,6.428,5.754,5.964,5.999,5.683,6.387,6.803,5.793,5.91,5.039,4.731,4.791,4.873,4.646,4.517,4.664,4.557,4.311,4.377,4.437,4.099,4.886,5.022,4.458,4.248,13.428,13.56,13.276,13.487,12.896,12.857,13.138,13.776,12.66,12.664,13.732,12.558,13.355,13.63,12.456,13.24,13.006,11.888,11.941,12.823,11.385,11.739,12.165,12.347,11.484,11.791,12.437,11.706,12.01,11.355,11.875,8.496,7.981,8.342,8.6,7.959,8.083,8.698,9.065,8.02,8.268,8.946,8.417,7.784,8.475,8.04,8.923,5.713,5.964,6.467,6.709,5.638,7.117,7.297,7.054,6.504,7.793,7.543,6.153,5.893,6.627,5.993,6.209,7.281,6.344,7.912,5.775,4.279,4.674,4.787,4.644,4.912,5.245,5.143,5.135,5.351,5.273,5.298,5.681,4.687,4.794,5.402,5.759,11.575,11.368,11.616,12.187,10.817,11.171,11.839,12.435,11.239,11.508,11.8,11.68,12.026,11.249,11.939,10.765,10.2,10.577,10.654,9.765,10.015,10.437,11.084,9.964,10.541,10.766,10.356,10.142,10.734,9.883,11.023,6.889,6.754,6.952,7.105,7.201,6.996,7.729,7.932,7.436,6.56,7.728],[8.724,8.549,8.197,7.99,8.203,8.057,null,null,7.84,7.926,7.443,7.499,8.762,8.055,8.289,7.699,10.546,10.901,10.601,10.627,11.545,11.199,10.612,10.448,11.092,10.843,10.594,10.859,10.792,10.164,11.122,10.486,15.21,15.502,15.244,14.928,15.587,15.577,14.81,14.73,15.341,15.214,14.897,15.197,15.84,14.936,15.417,14.734,6.167,6.039,5.89,6.106,5.776,6.042,5.331,5.502,6.268,5.965,4.357,4.586,4.319,4.34,4.883,4.583,4.068,3.835,4.802,4.607,4.158,4.592,5.068,4.092,4.941,4.742,9.773,9.861,9.612,9.285,10.52,10.273,9.561,9.41,10.254,10.136,9.942,10.368,10.136,9.686,10.451,10.2,17.392,17.26,17.163,16.874,17.414,16.959,16.546,16.42,16.656,16.712,16.513,16.392,17.423,16.965,16.86,16.322,11.999,12.269,12.404,11.905,12.666,12.512,12.253,12.123,12.74,12.654,12.761,12.901,12.15,11.739,12.946,12.396,13.43,13.843,13.554,12.896,14.096,13.591,13.371,12.888,13.919,13.627,13.34,13.604,13.914,12.876,13.091,3.973,4.016,3.66,3.382,4.155,3.82,3.583,3.035,4.163,3.905,3.262,3.434,4.079,3.503,4.082,3.081,9.554,9.815,9.627,9.515,10.091,9.811,9.473,9.19,10.05,9.887,9.639,9.924,9.721,9.219,10.253,9.437,8.186,8.135,7.887,7.669,8.653,8.382,8.076,7.547,8.701,8.14,7.89,8.548,8.662,7.911,8.588,7.916,12.383,12.473,12.181,11.917,12.795,12.908,12.028,11.767,12.424,12.174,12.221,12.022,12.673,12.548,11.69,18.196,20.922,18.232,17.765,18.847,18.178,17.758,17.413,18.209,17.826,17.711,17.773,18.48,17.731,18.22,17.493,10.88,10.935,11.397,10.584,10.894,10.893,10.468,10.346,10.584,10.333,10.221,10.544,11.304,10.91,10.386,10.136,7.127,7.184,6.932,6.768,7.742,7.531,6.927,6.452,7.116,6.789,6.902,7.723,7.244,7.492,8.722,9.14,8.813,8.639,8.91,9.106,8.602,8.253,8.688,8.349,8.516,8.034,8.412,8.411,7.812,14.267,14.514,14.202,13.879,14.546,14.25,13.776,13.588,14.433,14.345,13.757,13.988,14.689,13.879,14.54,13.638,14.623,15.197,14.739,14.495,14.838,14.619,14.159,13.85,14.206,14.338,14.024,13.927,14.816,14.371,14.393,14.067,5.962,6.294,6.052,5.61,6.715,6.409,5.921,5.966,6.508,6.183,6.227,6.664,6.229,5.443,6.604,6.387,1.342,1.582,1.196,1.068,2.38,2.053,1.531,1.614,2.086,1.979,1.679,2.19,1.731,0.828,2.087,1.693,17.838,17.897,17.662,17.43,18.565,17.958,17.307,17.032,18.31,17.646,17.398,17.747,18.097,17.614,18.07,17.171,21.071,21.167,20.926,20.751,21.197,21.073,20.589,20.718,20.448,20.228,21.346,20.785,9.313,9.676,9.425,9.158,9.484,9.21,8.746,8.455,9.322,9.248,8.822,8.926,9.666,8.955,11.378,11.533,11.306,11.25,11.748,11.593,11.143,10.78,11.57,11.522,11.174,11.051,12.092,10.92,12.002,10.841,16.611,16.847,16.668,16.284,16.898,16.66,16.364,16.359,16.544,16.385,16.225,16.249,16.863,16.389,16.697,16.288,14.537,14.852,14.605,14.335,14.967,14.7,14.09,13.812,14.522,14.732,13.923,14.199,14.899,14.533,14.561,13.992,16.29,16.371,16.078,15.786,16.493,16.246,15.778,15.531,16.152,15.943,16.006,15.923,16.493,15.935,16.381,15.513,11.387,11.599,11.103,11.007,10.995,10.956,10.752,10.69,10.741,10.533,10.795,10.427,11.454,11.15,10.555,10.071,12.813,12.594,12.554,12.63,12.614,12.445,12.185,12.154,12.514,12.169,12.244,11.868,12.93,12.479,11.681,7.935,8.329,8.437,7.993,8.639,8.39,7.945,7.633,8.91,8.449,8.119,8.417,8.33,7.645,8.93,8.159,18.352,18.603,19.106,19.348,18.279,18.9,19.08,18.837,18.284,19.576,19.326,17.936,17.676,18.41,17.84,18.056,19.064,18.124,19.695,17.622,12.551,12.887,12.849,12.581,13.267,13.572,12.728,12.357,13.187,12.829,12.709,13.145,13.058,12.545,13.325,13.223,5.234,5.889,5.543,5.477,5.834,5.889,5.511,5.442,5.947,5.906,5.708,6.073,5.151,6.092,5.74,8.714,8.169,8.238,7.783,8.489,8.125,7.587,7.336,8.372,7.985,7.73,7.89,8.382,7.699,8.452,7.628,12.802,12.689,12.316,13.018,12.88,12.36,13.093,13.296,12.8,12.22,13.092],[11.559,10.859,11.163,11.649,10.807,10.94,null,null,10.788,11.038,11.174,10.832,11.077,11.714,10.656,11.482,6.012,6.097,6.483,6.509,5.703,6.395,6.787,6.814,6.277,6.459,6.575,6.554,5.988,6.866,5.707,6.463,3.411,2.977,3.687,3.44,2.626,2.851,3.372,3.738,2.758,2.922,3.294,2.884,3.08,3.586,2.574,3.902,11.084,10.431,10.678,10.035,10.691,10.524,15.118,10.636,10.305,10.447,12.057,11.911,12.051,12.388,12.077,12.277,12.566,12.979,12.2,12.43,12.74,12.467,11.753,12.618,12.017,13.086,10.006,10.12,10.23,10.415,9.736,9.981,10.771,10.828,10.278,10.391,10.889,10.587,9.848,10.62,10.078,10.837,8.079,7.705,7.787,7.567,7.859,7.324,7.245,7.315,7.072,7.16,6.986,6.655,7.868,7.658,6.916,6.927,10.629,10.523,10.835,11.032,10.373,10.694,11.073,11.345,10.811,11.02,11.743,11.267,10.404,11.036,10.72,11.416,3.003,2.666,2.913,3.205,2.466,3.034,3.27,3.554,2.899,3.146,3.439,3.245,2.495,3.311,3.799,12.468,12.3,12.822,13.001,11.919,12.342,12.874,13.348,12.229,12.43,13.049,12.661,12.375,12.876,12.394,13.072,8.001,7.937,8.084,8.33,7.819,8.228,8.526,8.839,8.27,8.674,8.762,8.671,7.68,8.527,8.431,8.964,8.642,8.655,9.184,9.752,8.382,8.617,9.108,9.63,8.921,8.854,9.401,8.962,8.448,9.101,8.953,9.427,5.502,4.991,5.198,5.38,4.656,4.687,5.273,5.563,4.633,4.608,4.83,4.322,5.099,4.235,4.806,4.523,6.313,4.083,3.914,4.449,3.964,3.6,3.562,3.933,3.612,3.497,3.497,4.322,4.054,3.923,3.335,7.428,6.99,8.135,7.547,6.638,6.662,7.034,7.783,6.647,6.689,6.983,6.9,7.548,7.827,6.449,7.109,9.066,9.179,9.351,9.517,8.906,9.112,9.662,10.498,9.405,9.683,9.842,8.849,9.596,9.482,9.739,9.241,9.621,9.747,8.839,9.659,9.868,9.966,9.291,9.192,9.33,9.055,9.887,8.917,9.745,5.094,4.743,5.045,5.308,4.482,4.638,5.154,5.51,4.7,5.027,5.385,4.764,4.661,5.561,4.807,4.941,7.261,6.488,6.833,7.306,6.129,6.833,6.875,7.175,6.236,6.93,6.874,6.75,6.601,7.146,6.209,7.036,11.598,11.061,11.181,11.633,10.874,11.094,11.585,11.795,11.047,11.406,11.574,11.45,10.696,11.472,11.018,11.734,14.562,14.775,14.795,14.999,14.273,14.462,15.128,15.541,14.614,14.861,15.464,14.812,14.208,15.311,14.615,15.449,2.829,2.791,2.556,2.325,3.438,2.831,2.18,1.905,2.985,2.321,2.073,2.408,2.991,2.538,2.731,1.832,6.159,6.241,6.014,5.839,6.243,6.119,5.677,5.764,5.536,5.303,6.42,5.831,7.003,6.878,7.128,7.394,7.016,7.403,7.647,8.127,7.093,7.371,7.652,7.808,6.868,7.586,4.935,4.497,4.868,5.054,4.441,4.799,5.177,5.518,4.926,5.028,5.298,5.575,4.346,5.11,4.763,5.546,5.898,5.543,5.958,5.623,5.289,5.356,5.231,5.234,4.992,5.134,4.98,4.703,5.559,5.728,4.841,5.282,2.723,2.253,2.542,2.272,1.827,1.559,2.03,2.222,1.471,1.681,2.108,1.944,1.942,2.59,1.668,2.14,1.898,2.36,2.146,1.949,2.885,2.638,2.226,1.979,2.914,2.707,2.847,2.942,2.371,1.94,3.217,2.524,9.901,10.033,9.749,9.96,9.369,9.33,9.611,10.249,9.133,9.137,10.205,9.031,9.828,10.103,8.929,9.713,9.294,7.901,7.99,9.111,7.275,7.654,8.268,8.635,7.587,7.894,8.587,7.858,7.9,7.779,8.348,9.816,9.329,9.69,9.948,9.307,9.431,10.046,10.413,9.368,9.616,10.294,9.765,9.132,9.823,9.388,10.271,5.371,5.622,6.125,6.367,5.298,3.737,3.836,3.963,3.192,4.332,4.082,3.546,3.418,3.871,3.765,4.192,3.82,3.092,4.451,3.969,6.111,6.506,6.619,6.476,6.744,7.077,6.975,6.967,7.183,7.105,7.13,7.513,6.519,6.626,7.234,7.591,12.585,12.511,12.725,13.309,12.165,12.377,12.961,13.557,12.587,12.856,12.947,13.028,13.036,12.597,13.096,8.765,8.313,8.577,8.767,7.878,8.128,8.55,9.197,8.077,8.654,8.879,8.469,8.255,8.734,7.996,9.136,8.721,8.586,8.784,8.937,9.033,8.828,9.561,9.764,9.268,8.392,9.56],[16.54,15.84,16.144,16.63,15.788,16.059,null,null,15.909,16.159,16.43,16.143,16.058,16.695,15.925,16.738,12.179,12.264,12.65,12.676,11.87,12.562,12.954,12.981,12.444,12.626,12.742,12.721,12.155,13.033,11.874,12.63,7.256,6.828,7.626,7.532,6.47,6.8,7.514,7.88,6.798,7.079,7.451,7.042,6.76,7.678,6.614,8.044,16.419,16.046,16.293,15.865,16.521,16.354,20.948,16.466,15.798,16.277,17.975,17.829,17.969,18.306,17.995,18.195,18.484,18.809,18.344,18.378,18.751,18.615,17.671,18.536,18.161,19.253,16.173,16.287,16.397,16.582,15.903,16.148,16.938,16.995,16.445,16.558,17.056,16.754,16.015,16.787,16.245,17.004,9.07,8.696,8.837,9.116,8.791,8.315,8.794,9.027,8.21,8.58,8.535,8.204,8.776,9.207,7.997,8.66,16.796,16.69,17.002,17.199,16.54,16.861,17.24,17.512,16.978,17.187,17.91,17.434,16.571,17.203,16.887,17.583,9.17,8.833,9.08,9.372,8.633,9.201,9.437,9.721,9.066,9.313,9.606,9.412,8.662,9.478,9.966,18.298,18.13,18.652,18.831,17.749,18.172,18.704,19.178,18.059,18.26,18.879,18.491,18.205,18.706,18.224,18.902,14.168,14.104,14.251,14.497,13.986,14.395,14.693,15.006,14.437,14.841,14.929,14.838,13.847,14.694,14.598,15.131,14.714,14.727,15.273,15.824,14.549,14.784,15.275,15.702,15.088,15.021,15.568,15.129,14.52,15.173,15.12,15.594,9.96,9.435,9.648,9.842,9.298,9.428,10.223,10.78,9.833,9.808,10.07,10.136,9.536,9.456,10.332,4.764,4.18,4.289,4.633,3.798,3.969,4.642,4.985,3.934,4.344,4.758,4.274,4.481,5.075,4.377,5.031,12.14,11.587,12.984,12.506,11.235,11.55,12.203,12.9,11.551,11.904,12.198,12.115,12.145,12.816,11.371,12.324,14.896,15.009,15.181,15.347,14.915,15.114,15.592,16.428,15.407,15.685,15.914,14.679,15.426,15.554,14.798,14.345,14.68,14.986,13.958,14.763,15.107,15.232,14.41,14.503,14.641,14.366,15.126,14.066,15.056,8.77,8.419,8.721,8.984,8.158,8.314,8.83,9.251,8.431,8.77,9.155,8.913,8.337,9.237,8.538,9.226,10.85,10.043,10.388,10.961,9.684,10.388,10.551,10.851,9.893,10.606,10.55,10.426,10.052,10.801,9.802,10.712,17.751,17.228,17.348,17.8,17.041,17.261,17.752,17.962,17.214,17.573,17.741,17.617,16.863,17.639,17.185,17.901,20.392,20.605,20.625,20.829,20.103,20.292,20.958,21.371,20.444,20.691,21.294,20.642,20.038,21.141,20.445,21.279,4.372,4.472,4.609,4.895,4.024,4.282,4.749,4.954,4.487,4.896,5.099,4.986,4.201,4.868,4.548,5.148,0.834,0.792,1.05,1.429,1.219,1.281,1.447,1.408,1.672,1.99,0.789,1.61,12.833,12.708,12.958,13.224,12.846,13.233,13.477,13.983,13.165,13.443,13.717,13.873,12.698,13.416,10.765,10.327,10.698,10.884,10.271,10.629,11.007,11.348,10.756,10.858,11.128,11.458,10.176,10.94,10.593,11.376,7.447,7.092,7.676,7.641,6.597,7.075,7.465,7.723,6.848,7.33,7.469,7.089,7.108,7.766,6.642,7.81,7.59,7.918,7.879,7.699,7.791,7.68,7.946,8.224,7.819,8.045,8.281,8.308,7.385,7.739,8.032,8.35,6.441,6.332,6.611,6.952,6.357,6.522,7.22,7.494,6.789,7.162,7.302,7.287,6.379,6.943,6.689,7.511,14.882,15.014,14.73,14.941,14.35,14.311,14.592,15.23,14.114,14.118,15.186,14.012,14.809,15.084,13.91,14.694,13.215,11.822,11.911,13.032,11.196,11.575,12.189,12.556,11.508,11.815,12.508,11.779,11.752,11.7,12.423,15.983,15.496,15.857,16.115,15.474,15.598,16.213,16.58,15.535,15.783,16.461,15.932,15.299,15.99,15.555,16.438,6.645,6.896,7.399,7.641,9.083,4.579,4.678,4.824,5.098,5.174,4.924,5.26,5.392,5.226,5.479,5.547,4.662,5.066,5.293,5.767,12.278,12.673,12.786,12.643,12.911,13.244,13.142,13.134,13.35,13.272,13.297,13.68,12.686,12.793,13.401,13.758,18.752,18.678,18.892,19.476,18.332,18.544,19.128,19.724,18.754,19.023,19.114,19.195,19.203,18.764,19.263,14.076,14.143,14.316,14.597,13.708,13.958,14.38,15.027,13.907,14.484,14.709,14.299,14.085,14.469,13.826,14.966,14.888,14.753,14.951,15.104,15.2,14.995,15.728,15.931,15.435,14.559,15.727],[9.185,8.588,8.892,9.378,8.536,8.807,null,null,8.657,8.907,9.178,9.2,8.703,9.443,8.673,9.486,9.701,9.887,9.915,9.941,9.958,10.185,9.926,9.881,10.539,10.434,10.185,10.703,9.778,9.595,10.405,10.243,4.464,4.746,4.793,4.774,5.105,5.413,5.142,5.505,5.606,5.643,5.537,6.08,4.952,4.628,5.676,5.669,10.157,9.785,10.032,9.67,10.32,10.195,15.124,10.648,9.537,10.276,12.375,12.229,12.369,12.706,12.421,12.599,12.884,13.138,12.77,12.782,13.151,13.019,12.097,12.936,12.587,13.735,13.838,14.12,14.11,14.327,14.252,14.235,14.68,14.594,14.418,14.58,14.999,14.812,13.801,14.274,14.433,14.947,1.329,2.093,1.568,1.282,2.143,2.182,1.411,1.288,2.09,1.833,1.568,1.949,1.757,1.373,2.408,1.905,15.376,15.337,15.649,15.794,15.711,15.661,15.887,16.012,15.889,15.834,16.557,16.081,15.218,15.552,16.095,16.23,8.49,9.136,9.084,8.884,9.099,9.293,9.091,9.374,9.553,9.502,9.295,9.559,8.891,8.9,9.655,12.474,12.306,12.828,13.007,11.925,12.348,12.88,13.354,12.235,12.436,13.055,12.702,12.381,12.882,12.4,13.078,11.82,11.917,11.979,12.096,11.837,12.208,12.292,12.493,12.288,12.569,12.528,12.651,11.698,12.181,12.449,12.73,10.876,10.969,11.394,11.716,10.893,11.128,11.583,11.594,11.963,11.823,12.116,12.175,10.762,11.168,11.464,12.142,5.943,5.771,6.02,5.756,6.094,6.281,6.049,6.556,6.45,6.37,6.529,6.873,5.659,6.427,6.294,5.774,9.261,5.936,5.483,6.891,6.159,5.494,5.131,6.215,5.849,5.574,5.932,6.338,5.309,6.378,5.525,5.626,6.023,6.059,6.022,6.528,6.843,6.61,6.416,6.613,6.738,6.84,6.947,5.807,5.895,6.664,7.156,10.576,10.775,10.861,10.852,10.759,10.889,11.101,11.475,11.182,11.393,11.704,10.454,10.799,11.446,7.546,7.424,7.428,7.734,7.472,7.842,7.855,7.98,7.924,8.033,8.324,8.342,7.874,7.998,8.599,3.509,4.244,3.922,3.804,4.087,4.121,4.17,4.133,4.435,4.749,4.484,5.159,4.127,3.681,4.542,4.807,1.723,2.647,2.421,2.027,2.288,2.117,2.487,2.65,2.44,2.481,2.486,2.801,1.625,1.867,2.503,2.662,12.181,12.7,12.427,12.876,13.198,12.945,12.844,13.038,13.044,13.089,13.133,13.563,12.499,12.714,13.14,13.293,14.568,14.781,14.801,15.005,14.279,14.468,15.134,15.547,14.62,14.867,15.47,14.818,14.214,15.317,14.621,15.455,6.757,7.201,6.908,6.689,7.692,7.498,6.994,6.765,7.871,7.391,7.143,7.89,7.163,6.707,7.932,7.192,9.41,9.506,9.265,9.09,9.536,9.412,8.928,9.057,8.787,8.567,9.685,9.124,9.023,8.947,9.197,9.463,9.088,9.472,9.716,10.181,9.404,9.682,9.956,10.112,8.94,9.655,7.758,7.684,7.815,7.999,8.149,7.994,8.146,7.956,8.253,8.284,8.396,8.851,8.214,7.698,8.695,8.644,2.714,3.011,2.939,2.787,3.532,3.29,2.93,2.925,3.53,3.247,3.148,3.347,2.995,2.892,3.882,2.854,6.78,7.316,7.069,6.889,7.606,7.496,7.136,7.19,7.687,7.919,7.471,7.883,7.1,6.929,7.906,7.54,8.552,9.01,8.8,8.603,9.535,9.288,8.882,8.643,9.564,9.362,9.502,9.648,9.021,8.594,9.867,9.18,7.63,7.762,7.478,7.689,7.098,7.059,7.34,7.875,6.862,6.866,7.831,6.76,7.557,7.832,6.658,7.339,5.089,3.651,3.74,4.906,3.846,3.857,4.063,4.43,4.16,4.468,4.382,4.864,3.488,4.352,4.753,12.833,12.781,13.344,13.396,12.961,13.085,13.494,13.81,13.022,13.27,13.742,13.419,12.584,13.266,13.042,13.925,11.932,12.183,12.686,12.928,12.06,10.125,10.224,10.196,9.678,10.72,10.47,10.032,9.904,10.357,10.251,10.678,10.208,9.578,10.839,10.455,12.665,13.06,13.173,13.03,13.298,13.631,13.529,13.348,13.737,13.659,13.684,14.067,13.073,13.168,13.788,14.145,13.728,13.753,13.868,14.452,13.427,13.62,14.104,14.7,13.849,14.117,14.189,14.29,14.034,13.994,14.338,8.593,9.105,8.834,9.348,9.415,9.282,9.281,9.78,9.466,9.435,9.488,9.703,9.318,8.902,9.385,9.695,14.782,14.647,14.819,14.998,15.094,14.863,15.596,15.799,15.303,14.453,15.595],[9.892,9.192,9.496,9.982,9.14,9.411,null,null,9.261,9.511,9.782,9.677,9.41,10.047,9.277,10.09,6.918,7.039,7.157,7.183,7.043,7.337,7.168,7.361,7.756,7.676,7.427,7.945,6.93,7.13,7.524,7.485,0.947,1.225,1.317,1.323,1.577,1.904,1.691,2.054,2.147,2.184,2.081,2.62,1.389,1.184,2.192,2.218,10.08,9.707,9.954,9.592,10.242,10.117,14.725,10.243,9.459,10.054,11.881,11.735,11.875,12.212,11.901,12.101,12.39,12.715,12.25,12.284,12.657,12.521,11.577,12.442,12.067,13.241,11.548,11.83,11.82,12.037,11.653,11.854,12.39,12.304,12.128,12.29,12.709,12.504,11.511,11.984,11.995,12.657,4.407,4.09,4.12,3.891,4.244,3.706,3.569,3.549,3.405,3.484,3.31,2.979,4.253,3.982,3.249,3.161,12.773,12.734,13.046,13.195,13.108,13.058,13.284,13.413,13.286,13.231,13.954,13.478,12.615,12.953,13.492,13.627,5.116,5.672,5.71,5.51,5.635,5.919,5.717,6.0,6.089,6.128,5.921,6.185,5.427,5.526,6.281,12.075,11.907,12.429,12.608,11.526,11.949,12.481,12.955,11.836,12.037,12.656,12.303,11.982,12.483,12.001,12.679,9.515,9.451,9.598,9.806,9.335,9.742,10.002,10.203,9.786,10.188,10.238,10.185,9.196,9.891,9.947,10.44,8.672,8.685,9.231,9.778,8.603,8.838,9.329,9.656,9.673,9.569,10.086,9.885,8.478,9.127,9.174,10.112,3.472,2.986,3.309,3.503,3.029,3.216,3.891,4.448,3.683,3.604,3.763,4.105,2.864,3.362,4.025,3.008,6.415,3.17,2.717,4.125,3.393,2.728,2.365,3.449,3.083,2.808,3.166,3.572,2.543,3.612,2.669,5.475,5.248,6.337,5.871,4.896,5.211,5.864,6.265,5.212,5.565,5.859,5.776,5.656,6.173,5.032,5.985,8.802,8.915,9.087,9.253,8.821,9.02,9.498,10.334,9.313,9.591,9.862,8.585,9.332,9.508,8.15,7.702,8.032,8.338,7.323,8.12,8.459,8.584,7.775,7.884,8.175,8.027,8.478,7.727,8.717,1.704,1.353,1.655,1.918,1.092,1.248,1.764,2.185,1.365,1.704,2.089,1.847,1.271,2.171,1.472,2.16,3.775,2.968,3.313,3.886,2.609,3.313,3.485,3.785,2.827,3.54,3.484,3.36,3.01,3.726,2.736,3.646,11.657,11.785,11.903,12.355,11.886,12.106,12.32,12.517,12.059,12.347,12.445,12.462,11.418,12.193,12.03,12.605,14.169,14.382,14.402,14.606,13.88,14.069,14.735,15.148,14.221,14.468,15.071,14.419,13.815,14.918,14.222,15.056,3.716,4.16,3.867,3.648,4.766,4.457,3.953,3.724,4.925,4.35,4.102,4.849,4.122,3.666,4.917,4.151,6.564,6.66,6.419,6.244,6.69,6.566,6.082,6.211,5.941,5.721,6.839,6.278,6.739,6.663,6.913,7.179,6.804,7.188,7.432,7.897,7.12,7.398,7.672,7.828,6.656,7.371,4.99,4.844,5.047,5.231,5.056,5.146,5.378,5.198,5.405,5.501,5.628,6.083,4.947,4.94,5.428,5.876,2.392,2.038,2.449,2.065,1.792,1.851,1.673,1.669,1.438,1.576,1.416,1.143,2.054,2.17,1.344,1.598,3.406,3.942,3.695,3.515,4.142,4.032,3.762,3.832,4.223,4.455,4.097,4.509,3.636,3.555,4.442,4.166,5.511,5.969,5.759,5.562,6.494,6.247,5.841,5.602,6.523,6.321,6.461,6.607,5.98,5.553,6.826,6.139,8.234,8.366,8.082,8.293,7.702,7.663,7.944,8.582,7.466,7.47,8.538,7.364,8.161,8.436,7.262,8.046,6.149,4.756,4.845,5.966,4.13,4.509,5.123,5.49,4.442,4.749,5.442,4.713,4.686,4.634,5.357,10.739,10.491,11.054,11.11,10.671,10.795,11.208,11.575,10.732,10.98,11.456,11.129,10.294,10.985,10.752,11.635,8.921,9.172,9.675,9.917,9.019,7.173,7.272,7.27,6.667,7.768,7.518,7.021,6.893,7.346,7.24,7.667,7.256,6.567,7.887,7.444,9.291,9.686,9.799,9.656,9.924,10.257,10.155,10.147,10.363,10.285,10.31,10.693,9.699,9.806,10.414,10.771,13.208,13.232,13.348,13.932,12.903,13.099,13.584,14.18,13.325,13.594,13.668,13.766,13.536,13.47,13.817,7.737,7.92,7.977,8.374,7.485,7.735,8.157,8.804,7.684,8.261,8.486,8.076,7.862,8.13,7.603,8.743,11.901,11.766,11.964,12.117,12.213,12.008,12.741,12.944,12.448,11.572,12.74]]}""")
    nrow(sites) == length(bundled.sites) && length(offsets) == length(bundled.targets) || return nothing
    coordinates_match(point, ref) =
        abs(point.lat - ref.lat) <= 1e-9 && abs(point.lon - ref.lon) <= 1e-9
    for (site, ref) in zip(eachrow(sites), bundled.sites)
        site.site_id == ref.id && coordinates_match(site, ref) || return nothing
    end
    for (point, ref) in zip(offsets, bundled.targets)
        point.point_id == ref.id && coordinates_match(point, ref) || return nothing
    end
    println("Using bundled grid routes for the matching sites and centroid positions.")
    return [isnothing(bundled.routes[j][k]) ? NaN : 4 * bundled.routes[j][k] / 1.609344
            for j in 1:nrow(sites), k in eachindex(offsets)]
end


bundled_offset_times (generic function with 1 method)

In [18]:
function in_ring(lon, lat, ring)
    inside = false
    for k in 2:length(ring)
        a, b = ring[k-1][1:2]
        c, d = ring[k][1:2]
        if (b > lat) != (d > lat) && lon < a + (lat - b) * (c - a) / (d - b)
            inside = !inside
        end
    end
    return inside
end

function in_zip(lon, lat, geom)
    polygons = geom.type == "Polygon" ? [geom.coordinates] : geom.coordinates
    return any(poly -> in_ring(lon, lat, first(poly)) &&
                       !any(hole -> in_ring(lon, lat, hole), Iterators.drop(poly, 1)), polygons)
end

function centroid_offsets(demand, radius, spacing, zip_file)
    offsets = NamedTuple{(:area_id, :point_id, :lat, :lon), Tuple{String, String, Float64, Float64}}[]
    radius == 0 && return offsets
    geo = JSON3.read(read(zip_file, String))
    boundaries = Dict(String(f.properties.modzcta) => f.geometry for f in geo.features)
    count_steps = floor(Int, radius / spacing)
    shifts = [(a * spacing, b * spacing)
              for a in -count_steps:count_steps for b in -count_steps:count_steps
              if 0 < (a * spacing)^2 + (b * spacing)^2 <= radius^2]
    diagonal = radius / sqrt(2)
    append!(shifts, [(diagonal * a, diagonal * b)
                     for (a, b) in [(-1, -1), (-1, 1), (1, -1), (1, 1)]])
    for row in eachrow(demand), (k, (dx, dy)) in enumerate(shifts)
        lat = row.lat + dy / 111195
        lon = row.lon + dx / (111195 * cosd(row.lat))
        if in_zip(lon, lat, boundaries[row.area_id])
            push!(offsets, (area_id=row.area_id, point_id="$(row.area_id)_G$k", lat=lat, lon=lon))
        end
    end
    return offsets
end

matrix_location(point) = (lat=point.lat, lon=point.lon, search_cutoff=MAX_SNAP_M)
function snap_meters(lat, lon, snapped_lat, snapped_lon)
    return 111195 * hypot(lat - snapped_lat, (lon - snapped_lon) * cosd(lat))
end

function route_offsets(sites, offsets)
    result = fill(NaN, nrow(sites), length(offsets))
    isempty(offsets) && return result
    # Cache key includes the ordered IDs and actual coordinates of all endpoints.
    site_points = [(id=s.site_id, lat=s.lat, lon=s.lon) for s in eachrow(sites)]
    cache_key = bytes2hex(sha256(JSON3.write((site_points, offsets, MATRIX_URL, MAX_SNAP_M))))
    mkpath(OUTPUT_DIR)
    path = joinpath(OUTPUT_DIR, "pr_offsets_$(cache_key[1:16]).csv")
    if isfile(path)
        cached = Matrix{Float64}(CSV.read(path, DataFrame))
        size(cached) == size(result) || error("Offset cache dimensions do not match")
        return cached
    end
    println("Routing ", length(offsets), " offsets from ", nrow(sites), " sites.")
    for site_batch in Iterators.partition(1:nrow(sites), 2)
        for target_batch in Iterators.partition(eachindex(offsets), 48)
            payload = JSON3.write((
                sources=[matrix_location(sites[j, :]) for j in site_batch],
                targets=[matrix_location(offsets[k]) for k in target_batch],
                costing="auto",
                costing_options=(auto=(shortest=true, disable_hierarchy_pruning=true),),
                units="kilometers", verbose=true,
            ))
            response_body = IOBuffer()
            response = Downloads.request(MATRIX_URL; method="POST",
                input=IOBuffer(payload), output=response_body, timeout=60,
                headers=["Content-Type" => "application/json",
                         "User-Agent" => "15C57-Brooklyn-EMS-Study/1.0",
                         "X-Client-Id" => "15C57-Brooklyn-EMS-Study"])
            body = String(take!(response_body))
            response.status == 200 || error("Valhalla HTTP $(response.status): $body")
            data = JSON3.read(body)
            data.units == "kilometers" || error("Expected Valhalla distances in kilometers")
            rows = data.sources_to_targets
            length(rows) == length(site_batch) || error("Incomplete Valhalla source batch")
            for (j, routes) in zip(site_batch, rows)
                length(routes) == length(target_batch) || error("Incomplete Valhalla target batch")
                for (k, route) in zip(target_batch, routes)
                    distance = get(route, :distance, nothing)
                    cost = get(route, :cost, nothing)
                    if distance === nothing || !isfinite(distance) || distance < 0
                        continue
                    end
                    cost !== nothing && isfinite(cost) && abs(cost - 1000 * distance) <= 1.1 || continue
                    site, point = sites[j, :], offsets[k]
                    any(key -> get(route, key, nothing) === nothing,
                        (:begin_lat, :begin_lon, :end_lat, :end_lon)) && continue
                    snap_meters(site.lat, site.lon, route.begin_lat, route.begin_lon) <= MAX_SNAP_M + 1 || continue
                    snap_meters(point.lat, point.lon, route.end_lat, route.end_lon) <= MAX_SNAP_M + 1 || continue
                    result[j, k] = 4 * distance / 1.609344
                end
            end
        end
        (last(site_batch) % 10 == 0 || last(site_batch) == nrow(sites)) &&
            println("Routed ", last(site_batch), " / ", nrow(sites), " sites.")
    end
    CSV.write(path, DataFrame(result, :auto))
    return result
end

offsets = centroid_offsets(demand, RADIUS_M, GRID_SPACING_M, ZIP_FILE)
offset_times = bundled_offset_times(sites, offsets)
if offset_times === nothing
    offset_times = route_offsets(sites, offsets)
end
valid_offsets = [k for k in eachindex(offsets) if all(isfinite, view(offset_times, :, k))]
dropped = setdiff(eachindex(offsets), valid_offsets)
isempty(dropped) || println("Offsets excluded by the routing checks: ",
                           [offsets[k].point_id for k in dropped])
all_times = hcat(nominal_times, offset_times[:, valid_offsets])
point_ids = vcat(I_areas, [offsets[k].point_id for k in valid_offsets])
zip_index = Dict(id => i for (i, id) in enumerate(I_areas))
point_zip = vcat(collect(eachindex(I_areas)), [zip_index[offsets[k].area_id] for k in valid_offsets])
points_by_zip = [findall(==(i), point_zip) for i in eachindex(I_areas)]
println("ZIPs: ", length(w), " | routable offsets: ", length(valid_offsets),
        " | ZIPs with offsets: ", count(p -> length(p) > 1, points_by_zip))

Using bundled grid routes for the matching sites and centroid positions.
Offsets excluded by the routing checks: ["11201_G7", "11201_G8"]
ZIPs: 37 | routable offsets: 554 | ZIPs with offsets: 37


## 4. Choose five sites

For each ZIP, `worst[i]` bounds the expected drive time at each of its possible centroid positions. Primary and backup assignments may change with the position. Open-post decisions are shared. For each position, only the fastest two existing posts and candidates at least as fast as the second existing post need to be considered.

In [19]:
function solve_standby_model(times, point_zip, weights, n_existing, P, p)
    n_sites, n_points = size(times)
    model = Model(Gurobi.Optimizer)
    set_silent(model)
    set_time_limit_sec(model, TIME_LIMIT_SEC)
    @variable(model, y[1:n_sites], Bin)
    @variable(model, worst[1:length(weights)] >= 0)
    @constraint(model, [j in 1:n_existing], y[j] == 1)
    @constraint(model, sum(y[j] for j in n_existing+1:n_sites) == P)

    for k in 1:n_points
        existing_two = partialsortperm(view(times, 1:n_existing, k), 1:2)
        cutoff = times[existing_two[2], k]
        eligible = vcat(existing_two,
            [j for j in n_existing+1:n_sites if times[j, k] <= cutoff])
        primary = @variable(model, [eligible], Bin)
        backup = @variable(model, [eligible], Bin)
        @constraint(model, sum(primary[j] for j in eligible) == 1)
        @constraint(model, sum(backup[j] for j in eligible) == 1)
        @constraint(model, [j in eligible], primary[j] + backup[j] <= y[j])
        first_time = sum(times[j, k] * primary[j] for j in eligible)
        second_time = sum(times[j, k] * backup[j] for j in eligible)
        @constraint(model, first_time <= second_time)
        @constraint(model, worst[point_zip[k]] >= (1-p) * first_time + p * second_time)
    end

    @objective(model, Min, sum(weights[i] * worst[i] for i in eachindex(weights)) / sum(weights))
    optimize!(model)
    is_solved_and_feasible(model) || error("Solve failed: $(termination_status(model))")
    selected = [j for j in n_existing+1:n_sites if value(y[j]) > 0.5]
    return (selected=selected, objective=objective_value(model))
end

sol = solve_standby_model(all_times, point_zip, w, n_existing, P, BUSY_PROB)
println("Selected candidate pins: ", site_ids[sol.selected])
@printf("Worst-case expected drive time: %.6f minutes per call\n", sol.objective)

Set parameter Username
Set parameter LicenseID to value 2857222
Academic license - for non-commercial use only - expires 2027-08-25
Selected candidate pins: ["1", "3", "4", "10", "11"]
Worst-case expected drive time: 3.850568 minutes per call


## 5. Report selected sites and ZIP-level worst-case scores

In [20]:
function score_layout(selected)
    open_sites = vcat(collect(1:n_existing), selected)
    ranked = [open_sites[partialsortperm(view(all_times, open_sites, k), 1:2)]
              for k in axes(all_times, 2)]
    expected = [(1-BUSY_PROB) * all_times[pair[1], k] + BUSY_PROB * all_times[pair[2], k]
                for (k, pair) in enumerate(ranked)]
    worst_points = [points[argmax(expected[points])] for points in points_by_zip]
    return (
        worst=sum(w .* expected[worst_points]) / sum(w),
        ranked=ranked, expected=expected, worst_points=worst_points,
    )
end

best = score_layout(sol.selected)
existing_only = score_layout(Int[])
@assert isapprox(best.worst, sol.objective; atol=1e-6)
@printf("Existing-only worst-case score: %.6f min\n", existing_only.worst)
@printf("Worst-case savings vs. existing posts: %.3f seconds per call\n",
        60 * (existing_only.worst - best.worst))

selected_sites = candidates[sol.selected .- n_existing, :]
display(selected_sites)
assignment = DataFrame(
    area_id=I_areas,
    calls=w,
    worst_position=point_ids[best.worst_points],
    worst_primary=[site_ids[best.ranked[k][1]] for k in best.worst_points],
    worst_backup=[site_ids[best.ranked[k][2]] for k in best.worst_points],
    worst_expected_minutes=best.expected[best.worst_points],
)
mkpath(OUTPUT_DIR)
CSV.write(joinpath(OUTPUT_DIR, "selected_sites_pr.csv"), selected_sites)
CSV.write(joinpath(OUTPUT_DIR, "area_assignment_pr.csv"), assignment)

Existing-only worst-case score: 4.027081 min
Worst-case savings vs. existing posts: 10.591 seconds per call


Row,site_id,lat,lon,source
,String,Float64,Float64,String15
1,1,40.6382,-73.9125,alternate
2,3,40.6547,-73.9392,original
3,4,40.6217,-73.9494,original
4,10,40.6676,-73.9451,original
5,11,40.688,-73.9894,original


"results\\area_assignment_pr.csv"

In [21]:
function greedy_layout(P)
    selected = Int[]
    remaining = collect((n_existing + 1):length(site_ids))

    for _ in 1:P
        scores = [score_layout(vcat(selected, j)).worst for j in remaining]
        best_index = argmin(scores)
        push!(selected, splice!(remaining, best_index))
    end

    return selected
end

layouts = [Int[], greedy_layout(P), sol.selected]
worst_minutes = [score_layout(layout).worst for layout in layouts]

comparison = DataFrame(
    method = ["Status quo", "Greedy", "Optimization"],
    selected_pins = [join(site_ids[layout], ", ") for layout in layouts],
    worst_case_minutes = round.(worst_minutes; digits=4),
    seconds_saved_vs_status_quo =
        round.(60 .* (worst_minutes[1] .- worst_minutes); digits=3),
)

display(comparison)

@printf("Optimization saves %.3f seconds per call vs. greedy\n",
        60 * (worst_minutes[2] - worst_minutes[3]))

CSV.write(joinpath(OUTPUT_DIR, "comparison_pr.csv"), comparison)

Row,method,selected_pins,worst_case_minutes,seconds_saved_vs_status_quo
,String,String,Float64,Float64
1,Status quo,,4.0271,0.0
2,Greedy,"10, 1, 11, 4, 3",3.8506,10.591
3,Optimization,"1, 3, 4, 10, 11",3.8506,10.591


Optimization saves 0.000 seconds per call vs. greedy


"results\\comparison_pr.csv"

## Verified result for the default inputs

Independent enumeration of all 15,504 five-site combinations at a 500 m radius, 250 m grid spacing, and 25% busy probability selected **Pins 1, 3, 4, 10, 11**. The weighted worst-case expected drive time was **3.851 min**, compared with **4.027 min** for existing posts only. The route checks retained 554 shifted positions across all 37 ZIPs.

These figures were verified separately from the Julia/Gurobi cells.